# The Dual Impact of Solar Power Penetration on the Turkish Electricity Market

## Full Data Pipeline

This notebook implements the complete empirical analysis pipeline for studying the dual effect of solar power penetration on the Turkish electricity market (Day-Ahead and Balancing Markets). 

### Pipeline Overview
| Phase | Description | Method |
|-------|-------------|--------|
| 1 | Data Ingestion & Pre-processing | Temporal indexing, integrity checks |
| 2 | Multi-Model Estimation | Baseline SARIMAX(1,1,1)(1,0,1)24 without intercept (Models 1–6, 7.2, 7.3) by profile maximum likelihood; Logistic Regression (Model 7.1) |
| 3 | Parameter Justification | ACF/PACF diagnostics; orders from a preliminary AIC grid search (note in Phase 3) |
| 4 | Robust Estimation | Optimized SARIMAX for the eight models of Supplementary Table 3: MCP (2,0,2)(1,0,1)24 with intercept, a-FRR (1,1,2)(1,0,1)24 without intercept, profile maximum likelihood; diagnostics and likelihood-ratio tests |
| 5 | Advanced Visualization | Market profiles, coefficient plots |
| 6 | Non-Linear Analysis | XGBoost PDP & Flexibility Trap; out-of-sample CV (6.4, Referee 2.3) |
| 7 | Robustness Checks | Quantile Regression (Q10/Q50/Q90); convergence diagnostics & clustered SE (7.2–7.4, Referee 2.5) |
| 8 | GAM Robustness | Cubic Spline non-linear regime analysis; Hansen threshold (8.2, Referee 2.4) |
| 9-10 | Appendices & Framework | Appendices A–J generation (incl. Botaş robustness, Appendix I; reviewer robustness, Appendix J) |
| 11 | Inflation Robustness | FE-OLS on a-FRR/MCP ratio & cap-normalised prices; timing/confounding (11.2, Referee 2.1) & sub-period (11.3, Referee 2.2) |

Code of the article *Solar PV Penetration, Merit-Order Effects and Balancing Reserve Costs in Türkiye's Electricity Market* (Renewable Energy, RENE-D-26-09232), release R2.

**Numbering note.** Phase numbers, result folders and file names are those of the R1 release (`RENE-D-26-09232-R1`). Phase 6.3 is not used (the out-of-sample check keeps its R1 number 6.4, matching its `Phase_6_4_*` files). Appendix J (Phase 10) reads the outputs of Phase 11, so its cell runs after Phase 11. `Appendix_J5_Clustered_QR.csv` keeps its R1 file name and holds Supplementary Table J6; Supplementary Table J5 (threshold support) is `Appendix_J5_Threshold_Support.csv`. 'Referee 2.x' marks first-revision (R1) analyses for Reviewer #2, comment x (code tag `Revision_Review_Referee_2.x`); 'R2' marks the second revision.

**What the code does.**
- *Data.* The data file is described in `data/README.md`. The setup cell verifies its SHA-256 and Phase 1 verifies its structure; the month terms are on a 12-month cycle in the file itself and are not recomputed.
- *Estimation.* All SARIMAX models (Phases 2 and 4, Appendix C) are estimated by profile maximum likelihood from several starting points; every start is recorded and every estimate carries a convergence status (certified maximum, plateau or not converged; setup cell). Coefficients are in the original units.
- *Specification.* Phase 2: SARIMAX(1,1,1)(1,0,1)24 without an intercept. Phase 4: MCP models SARIMAX(2,0,2)(1,0,1)24 with an intercept; a-FRR models SARIMAX(1,1,2)(1,0,1)24 without an intercept. The a-FRR model in levels, SARIMAX(1,0,2)(1,0,1)24 with an intercept, is a robustness row of Supplementary Table C1.
- *Inference.* OPG, GLS, Hessian, sandwich, day-clustered sandwich and HAC (24 and 48 lags) standard errors are stored for every SARIMAX coefficient; Supplementary Table D1 reports the day-clustered sandwich.
- *Outputs.* Phase 2 lists all nine models; Phase 4 estimates all eight SARIMAX models of Supplementary Table 3 and saves diagnostics, likelihood-ratio tests and the Phase 2 / Phase 4 comparison of Supplementary S1.5; Supplementary Table C1 is estimated in Phase 10 (Appendix C), Table D1 is built from the Phase 4 output and Table J5 is computed from the dataset; Phase 6.2 saves the crossover point of the flexibility-trap curves.
- *Run record.* Every phase up to the cell that closes the run appends a section to `results/RUN_REPORT.md`: its output files with checksums and a comparison with an independent recomputation of the same specification (20 November 2026, `validation/pretest_reference_2026-11-20.csv`). `results/RUN_MANIFEST.json` ties the results folder to the data file and the method. Every section also lists the checkpoints of its step and the warnings raised in it; `results/RUN_CHECKPOINTS.csv` lists all checkpoints of the run.
- *Checkpoints.* Self-checks K01-K91 print `[OK]`, `[WARNING]`, `[FAILED]` or `[NOT CHECKED]` with what was checked; they are listed in `results/RUN_CHECKPOINTS.csv` and at the end of `RUN_REPORT.md`. A failed check of the data or of the design (setup cell, Phase 1) stops the notebook at once; any other failed check is reported, the run continues, and the cell that closes the run (Supplementary Figures) raises an error after writing the summary. WARNING marks a documented property of the results (for example rows of the pre-test comparison explained by the mirror image of a seasonal MA term, KPSS p-values at the bounds of the look-up table, bootstrap replicates at the iteration limit) or a difference that can occur between machines. NOT CHECKED means that the quantity was not available: the result was loaded from the results folder, or the sample is not the full sample. The record covers one kernel session (a complete record needs one session that starts from an empty `results/` folder); a check that is recorded again by a later execution of its cell supersedes the earlier one. No blanket warning filter is installed (the only filter that suppresses a warning is scoped to one call: the PIL size warning when the notebook reads its own 1000-dpi figures in the submission export); every remaining warning is shown and also listed in the run report.

**How to run.** Run the cells from top to bottom in a fresh kernel, starting from an empty `results/` folder (`requirements.txt` pins the environment). Phases 2 and 4 and Appendix C (Phase 10) estimate the SARIMAX models and took about 1 h 45 min in total in the reference run (the whole run about 2 h 10 min on 8 logical CPUs with four models in parallel; longer with fewer cores): `SARIMAX_N_JOBS` models run in parallel, every finished start is saved under `results/Phase_2/starts`, `results/Phase_4/starts` and `results/Phase_10/starts` (Appendix C), and a cell that is run again resumes with the missing ones. The other cells take minutes, except Phase 7.4 (the day-clustered bootstrap of Supplementary Table J6, about 20-25 min, including the diagnostic re-solve of the replicates that reach the iteration limit; longer on a loaded machine).

In [ ]:
# =============================================================================
# GLOBAL CONFIGURATION: OUTPUT PATHS
# =============================================================================

import os
# One BLAS thread per process: the SARIMAX models are estimated in parallel worker processes (one model each),
# so multi-threaded linear algebra would only oversubscribe the cores. This must be set before numpy is imported; it has no
# effect in a kernel that has already imported numpy, so run the notebook from a fresh kernel. XGBoost is not affected.
for _v in ('VECLIB_MAXIMUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_v, '1')
import re
import sys
import json
import time
import hashlib
import platform
import warnings
# -----------------------------------------------------------------------------
# WARNING RECORDER AND CHECKPOINTS (installed before the other imports, so that every warning is recorded)
#   RUN_WARNINGS     one entry per distinct warning and step: step, where ('main' or the job of run_parallel), category,
#                    message (cut to 300 characters), location (file:line), count. Nothing is hidden: in the main process
#                    warnings.showwarning is wrapped and still shows every warning as Python shows it; every job executed
#                    through run_parallel counts its own warnings (_rw_job) and the counts are merged here.
#                    Count: in the main process the number of times Python showed the warning (under the default filter a
#                    repeated warning from the same line is shown, and counted, once); in a job every occurrence.
#                    A warning captured around a single fit and passed on with warnings.warn_explicit (Phases 7.1-7.4,
#                    Appendix B) is shown once per fit: the capture resets Python's once-per-location registry.
#   RUN_CHECKS       one entry per self-check: step, name, status, detail (see checkpoint()).
#   RUN_COMPARISONS  one entry per comparison with the independent pre-test (compare_with_reference; checkpoint K90).
# Step label: checkpoint() and the recorder add their entries without a step; the next report() call gives them its own
# title (the step that this report() call closes) and lists them at the end of its section of RUN_REPORT.md. An entry made
# after the last report() call of a cell therefore belongs to the next step that is reported.
# Executing this cell again starts a new record (the lists are emptied) and wraps the original showwarning only once.
# -----------------------------------------------------------------------------
RUN_WARNINGS, RUN_CHECKS, RUN_COMPARISONS = [], [], []

def _rw_add(where, category, message, location, count=1):
    """Adds `count` occurrences of one warning to the open (not yet reported) entries of RUN_WARNINGS."""
    for w in RUN_WARNINGS:
        if w['step'] is None and (w['where'], w['category'], w['message'], w['location']) == (where, category, message, location):
            w['count'] += count
            return
    RUN_WARNINGS.append({'step': None, 'where': where, 'category': category, 'message': message, 'location': location,
                         'count': count})

def _rw_install():
    """Wraps warnings.showwarning of the main process: the warning is recorded, then shown by the original handler."""
    original = getattr(warnings.showwarning, '_rw_original', warnings.showwarning)     # never wrap a wrapper
    def showwarning(message, category, filename, lineno, file=None, line=None):
        try:
            _rw_add('main', getattr(category, '__name__', str(category)), str(message)[:300],
                    f"{os.path.basename(str(filename))}:{lineno}")
        except Exception:                                   # the recorder never interrupts the analysis
            pass
        return original(message, category, filename, lineno, file, line)
    showwarning._rw_original = original
    warnings.showwarning = showwarning

_rw_install()

def _ck_line(entry):
    return f"[{entry['status']}] {entry['name']}" + (f": {entry['detail']}" if entry['detail'] else '')

def checkpoint(name, ok, detail='', level='fail'):
    """Records and prints one self-check as [OK] / [WARNING] / [FAILED] / [NOT CHECKED] name: detail.
    level 'stop': not ok -> FAILED and RuntimeError immediately (data and design errors; setup and Phase 1 only);
          'fail': not ok -> FAILED, the run continues and checkpoint_summary() raises at the end (estimation problems);
          'warn': not ok -> WARNING, the run continues (may differ between machines, or a documented property of the results).
    ok None: NOT CHECKED (the quantity is not available; the detail says why). A checkpoint only reads values that have been
    computed already: it changes no number and no output file other than RUN_REPORT.md and RUN_CHECKPOINTS.csv.
    An earlier entry with the same name (its cell executed again) is marked superseded: it stays in RUN_CHECKPOINTS.csv and
    no longer counts in checkpoint_summary(). Returns True if the status is OK."""
    if level not in ('stop', 'fail', 'warn'):
        raise ValueError(f"checkpoint level {level!r}")
    status = 'NOT CHECKED' if ok is None else ('OK' if bool(ok) else ('WARNING' if level == 'warn' else 'FAILED'))
    entry = {'step': None, 'name': str(name), 'status': status, 'detail': str(detail)}
    for c in RUN_CHECKS:                                   # a cell executed again replaces its earlier result of this check
        if c['name'] == entry['name']:
            c['superseded'] = True
    RUN_CHECKS.append(entry)
    print(_ck_line(entry))
    if status == 'FAILED' and level == 'stop':
        if callable(globals().get('report')):              # the run report records why the run stopped
            report(f"Run stopped at a failed checkpoint: {name}")
        raise RuntimeError(f"checkpoint failed: {name}: {detail}")
    return status == 'OK'

import matplotlib.pyplot as plt
import seaborn as sns

# no global warnings filter: a warning is removed at its source, or captured and reported where it carries
# information (convergence status, iteration limits, KPSS table bounds), so that every remaining warning is visible.
RUN_STARTED = time.strftime('%Y-%m-%d %H:%M:%S')          # start of this kernel session (run record)

# Global figure export resolution.
# Renewable Energy (Elsevier/EM) technical checklist asks for high-resolution
# figures (above 650 dpi); all savefig calls below use this single knob.
# [Revision_R2_Reviewer4_C9] raised from 650 to 1000 dpi, Elsevier's standard for
# line art / charts. The methodological flowchart (Phase 9, a 16 x 22 in canvas) is
# capped at FIG_DPI_LARGE (~9,500 px wide after tight cropping, above the R1 export).
FIG_DPI = 1000
FIG_DPI_LARGE = 750

def retag_png_dpi(path, dpi=FIG_DPI):
    """Sets the PNG resolution tag (pHYs chunk) to `dpi` without touching the image data:
    the pixels stay byte-identical; only the print-size metadata changes (all figures are tagged 1000 dpi)."""
    import struct, zlib
    with open(path, 'rb') as fh:                         # files are closed explicitly (no ResourceWarning)
        b = bytearray(fh.read())
    ppm = int(round(dpi / 0.0254))                       # pixels per metre
    data = struct.pack('>IIB', ppm, ppm, 1)
    chunk = struct.pack('>I', 9) + b'pHYs' + data + struct.pack('>I', zlib.crc32(b'pHYs' + data) & 0xffffffff)
    i = 8
    while i < len(b):
        n = struct.unpack('>I', b[i:i + 4])[0]
        t = bytes(b[i + 4:i + 8])
        if t == b'pHYs':
            b[i:i + 12 + n] = chunk; break
        if t == b'IDAT':
            b[i:i] = chunk; break                         # no pHYs yet: insert before the image data
        i += 12 + n
    with open(path, 'wb') as fh:
        fh.write(bytes(b))

# [Revision] CSV line endings: the revision cells write their CSVs with LF on every platform,
# so runs on Windows and Linux write the same line endings. pandas >= 1.5 calls the keyword 'lineterminator'.
import inspect
import pandas as pd
_LT_KEY = 'lineterminator' if 'lineterminator' in inspect.signature(pd.DataFrame.to_csv).parameters else 'line_terminator'
CSV_KW = {_LT_KEY: '\n'}


# Auto-detect repo root: look for the results/ directory
if os.path.isdir("results"):
    RESULTS_BASE = "results"
elif os.path.isdir(os.path.join("..", "results")):
    RESULTS_BASE = os.path.join("..", "results")
else:
    RESULTS_BASE = "results"

os.makedirs(RESULTS_BASE, exist_ok=True)
print(f">> Results base directory: {os.path.abspath(RESULTS_BASE)}")

# Data directory Configuration
if os.path.isdir("data"):
    DATA_DIR = "data"
elif os.path.isdir(os.path.join("..", "data")):
    DATA_DIR = os.path.join("..", "data")
else:
    DATA_DIR = "."

DATA_FILE = os.path.join(DATA_DIR, "Model_Data.xlsx")
# The data file is also accepted in the working folder of the kernel (normally the folder of the notebook).
if not os.path.exists(DATA_FILE) and os.path.exists("Model_Data.xlsx"):
    DATA_FILE = "Model_Data.xlsx"
print(f">> Data file path: {os.path.abspath(DATA_FILE)}")

# -----------------------------------------------------------------------------
# DATA FILE IDENTITY. Every result of this release is produced from one data file. Its SHA-256 is checked
# here, before anything is estimated or loaded from the results folder; any other file stops the notebook.
# The data file is documented in data/README.md.
# -----------------------------------------------------------------------------
DATA_SHA256_EXPECTED = "71e299d35f23f818e059e7e952700fc669ac778823074f10ec68b3f35369a78c"
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        "Data file not found: Model_Data.xlsx. Please put Model_Data.xlsx into the data/ folder of the package "
        "(the data/ folder sits next to the notebooks/ folder), or move it into the same folder as this notebook "
        "(more precisely, the working folder of the Jupyter kernel, which is normally the folder of the notebook), "
        "then restart the kernel and run the notebook again from the first cell. "
        f"The notebook looked for: {' and '.join(dict.fromkeys([DATA_FILE, os.path.join('.', 'Model_Data.xlsx')]))} "
        "(relative to the working folder of the kernel).")
with open(DATA_FILE, "rb") as _f:
    DATA_SHA256 = hashlib.sha256(_f.read()).hexdigest()
if DATA_SHA256 != DATA_SHA256_EXPECTED:
    raise RuntimeError(f"Unexpected data file: SHA-256 {DATA_SHA256}\n   expected {DATA_SHA256_EXPECTED} (the R2 dataset). "
                       "Results must not be produced from another file.")
print(f">> Data file SHA-256: {DATA_SHA256[:16]}... (verified: R2 dataset)")

# Global Academic Plot Styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# [Revision_Editor_E6] no background colour in figures: white figure, axes and saved-file background
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white'})

# -----------------------------------------------------------------------------
# FORCE_FIGURES = True  -> the 10 cached PNGs listed below are deleted in this cell; Phases 6.1,
#                          6.2, 8, Appendix F/H and Appendix I (XGBoost + GAM, random_state=42)
#                          recompute them in the same run and save the figures at FIG_DPI.
#                          Phase 5 and Phase 9 figures are redrawn on every run anyway.
# FORCE_FIGURES = False -> existing figures are kept.
# Set it back to False once the refresh is done.
# -----------------------------------------------------------------------------
FORCE_FIGURES = False

_figure_cache_targets = [
    ("Phase_6", "Phase_6_Figure_5_PDP_Merit_Order_and_Volatility.png"),
    ("Phase_6", "Phase_6_Figure_7_PDP_Merit_Order_and_Volatility.png"),
    ("Phase_6", "Phase_6_Figure_8_Surplus_Probability_and_Distribution.png"),
    ("Phase_6", "Phase_6_Figure_9_Flexibility_Trap.png"),
    ("Phase_8", "Phase_8_Figure_10_GAM_Flexibility_Stress.png"),
    # [Revision_Review_Referee_2.7] Appendix I figures are cached too; without these two
    # entries a FORCE_FIGURES run left Supplementary Figures I.1 / I.2 with their old axis labels.
    ("Phase_10", "Appendix_I_Figure_I1_Flexibility_Trap_Botas.png"),
    ("Phase_10", "Appendix_I_Figure_I2_GAM_Flexibility_Stress_Botas.png"),
    # [Revision_R2_Reviewer4_C9] Appendix F and H figures are cached as well; listed so that a
    # FORCE_FIGURES run re-exports every supplementary figure at the current FIG_DPI.
    ("Phase_10", "Appendix_F1_MCP_Correlation.png"),
    ("Phase_10", "Appendix_F2_aFRR_Correlation.png"),
    ("Phase_10", "Appendix_H_Down_Regulation_Volume.png"),
]
if FORCE_FIGURES:
    print(f">> FORCE_FIGURES: clearing the figure caches (they will be regenerated at {FIG_DPI} dpi).")
    for _sub, _name in _figure_cache_targets:
        _p = os.path.join(RESULTS_BASE, _sub, _name)
        if os.path.exists(_p):
            os.remove(_p); print("   Deleted:", _p)
        else:
            print("   Already absent:", _p)


# -----------------------------------------------------------------------------
# SARIMAX SPECIFICATION AND ESTIMATION (used by Phase 2, Phase 4 and Appendix C)
# -----------------------------------------------------------------------------
# SPECIFICATION OF RELEASE R2 (one specification for the whole notebook).
#   Month terms   : Month_Sin / Month_Cos with a 12-month cycle, as stored in the data file (verified in Phase 1).
#   Phase 2       : baseline models SARIMAX(1,1,1)(1,0,1)24 without an intercept.
#   Phase 4, MCP  : SARIMAX(2,0,2)(1,0,1)24 with an intercept (a regressor column of ones, 'const').
#   Phase 4, a-FRR: SARIMAX(1,1,2)(1,0,1)24 without an intercept and without drift (with d = 1 a constant in the level
#                   equation is not identified; no trend term is used anywhere in this notebook).
#   Robustness    : the a-FRR alternative in levels, SARIMAX(1,0,2)(1,0,1)24 with an intercept, is estimated only as a row
#                   of Supplementary Table C1 (Phase 10, Appendix C).
ORDER_MCP, SEASONAL_MCP = (2, 0, 2), (1, 0, 1, 24)
ORDER_AFRR, SEASONAL_AFRR = (1, 1, 2), (1, 0, 1, 24)
ORDER_BASELINE, SEASONAL_BASELINE = (1, 1, 1), (1, 0, 1, 24)
SARIMAX_INTERCEPT = {'MCP': True, 'aFRR': False}

# ESTIMATION: PROFILE MAXIMUM LIKELIHOOD. The likelihood is extremely flat in the regression directions, so the
# regression coefficients are not left to a numerical search. The models are regressions with
# seasonal ARIMA errors, y_t = x_t'beta + u_t. For given AR/MA parameters and innovation variance the Kalman gains K_t and
# forecast-error variances F_t do not depend on the data and the filter is linear in the data, so the innovations of y - X b
# are v_t(y) - v_t(X) b and the log-likelihood is exactly quadratic in b: beta_hat is weighted least squares (GLS) of v_t(y)
# on v_t(X) with weights 1/F_t. The regression coefficients are therefore solved in closed form at every step, and the
# numerical search is left with the 5-6 AR/MA parameters and sigma2:
#   search  - Nelder-Mead over the AR/MA parameters (beta by GLS, sigma2 in closed form);
#   polish  - safeguarded Newton steps over (AR/MA, log sigma2) on the exact profile log-likelihood with exact gradients;
#   check   - a point is a CERTIFIED maximum only if a central-difference Hessian computed at that point is negative
#             definite and the predicted remaining gain 0.5 g'(-H)^-1 g is below 1e-8 log-likelihood units;
#   plateau - the a-FRR likelihood with d = 1 has a flat ridge (an MA root close to the unit circle), on which the
#             certificate may not be attainable in floating point. A point that is not certified is accepted as a PLATEAU
#             if a full extra search-and-polish round gains less than 1e-4 log-likelihood units, or if the Hessian is
#             negative definite and the predicted remaining gain is below 1e-4 (the rule that applied is recorded with
#             every start: 'stagnation' or 'hessian'); anything else is reported as NOT CONVERGED;
#   starts  - several genuinely different AR/MA starting points per model; every start is recorded. (In profile ML a start
#             is defined by its AR/MA values alone.)
#   reported representation - the INVERTIBLE representation of the best point is reported (non-seasonal MA roots
#             outside the unit circle and |seasonal MA coefficient| < 1). Mirror-image MA solutions have the same
#             autocovariances and regression coefficients, but statsmodels' approximate-diffuse initialisation gives the
#             non-invertible one a slightly higher log-likelihood. Among the starts within SARIMAX_TIE_TOL of the highest
#             log-likelihood a certified maximum is preferred, then a plateau, then an invertible point. If that point is not
#             invertible, its mirror image is estimated as an additional start ('mirror') and the choice is repeated; if it is
#             still not invertible, the best certified or plateau invertible run within SARIMAX_MIRROR_TOL of the highest
#             log-likelihood is reported. The higher non-invertible log-likelihood stays in the start records only
#             (info['peak_llf']). Without such a run the point is reported as it is and info['representation'] says so.
#             The reported log-likelihood, AIC, BIC, sigma2 and MA coefficients refer to the reported representation.
# Every reported log-likelihood is mod.loglike(params) of the statsmodels SARIMAX model itself (enforce_stationarity=False,
# enforce_invertibility=False). Regressors are rescaled for numerical conditioning only - solar-share terms in percentage
# points (x100), MWh and TL variables in thousands (/1000) - and coefficients and standard errors are transformed back, so
# every reported number keeps its units. Independent models are estimated in parallel (SARIMAX_N_JOBS, one thread each);
# the results do not depend on the number of parallel jobs.
import numpy as np
import statsmodels.api as sm
from scipy import stats as _sx_stats
from scipy.optimize import minimize as _sx_minimize
from threadpoolctl import threadpool_limits
from statsmodels.tsa.statespace import kalman_filter as _sx_kfm
from statsmodels.tools.sm_exceptions import ConvergenceWarning

SARIMAX_METHOD_TAG = ('R2 final | profile ML (multi-start; certified maximum or plateau) | 12-month terms; Phase 2 (1,1,1) no intercept; '
                      'Phase 4 MCP (2,0,2) intercept, a-FRR (1,1,2) no intercept | data ' + DATA_SHA256[:12])
SARIMAX_SCALE_LEVEL_VARS = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation', 'Hydroelectric_Generation',
                            'Domestic_Coal_Generation', 'Price_Cap']
# optimiser budgets: Nelder-Mead evaluations (first search / restart), Newton steps, search-polish rounds, tolerances
SARIMAX_SEARCH = dict(nm1=1500, nm2=500, newton_it=4, rounds=3, step1=0.05, step2=0.002,
                      xatol1=1e-4, fatol1=1e-4, xatol2=1e-5, fatol2=1e-5, tol_gain=1e-8, plateau_gain=1e-4)
SARIMAX_TIE_TOL = 1e-3      # log-likelihood units: starts within this distance of the best one are numerically equivalent
# log-likelihood units: an invertible run this close to the best point is taken as its invertible representation (gap
# between a mirror point and its invertible representation in the reference run of this release: at most 0.011, 0.022
# against the pre-test value of Model 5; distinct local maxima lie 776-801 units lower). Deliberately kept out of
# SARIMAX_SEARCH, which is part of the checkpoint signature of every start.
SARIMAX_MIRROR_TOL = 1.0
SARIMAX_SUMMARY_COLS = ['coef', 'std err', 'z', 'P>|z|', '[0.025', '0.975]']       # layout of the Phase 2 summaries (OPG)
# additional standard errors (see _sx_inference); Cluster_Day is the default inference of the paper (Supplementary Table D1)
SARIMAX_EXTRA_SE = ['GLS', 'Hessian', 'Robust', 'Cluster_Day', 'Cluster_Day_adj', 'HAC24', 'HAC48']
SARIMAX_DEFAULT_STARTS = {'default': 'default', 'grid': 'grid'}                     # starts used when none are given (Phase 2)
try:
    import psutil as _psutil
    _ram_jobs = int(_psutil.virtual_memory().total / 2**30 // 2.5)       # up to ~1.7 GB per parallel estimation (standard-error stage)
except Exception:
    _ram_jobs = 4
SARIMAX_N_JOBS = max(1, min(int(os.environ.get('SARIMAX_N_JOBS', 4)), os.cpu_count() or 2, _ram_jobs))   # the main process only waits

def sarimax_scale_factors(columns):
    """x100 for solar-share terms, /1000 for MWh and TL level variables, 1 otherwise (the intercept included)."""
    return {c: 100.0 if (c.startswith('Solar_') or c.startswith('Interaction_Solar'))
            else (1e-3 if c in SARIMAX_SCALE_LEVEL_VARS else 1.0) for c in columns}

_SX_LEAN = (_sx_kfm.MEMORY_NO_PREDICTED | _sx_kfm.MEMORY_NO_FILTERED | _sx_kfm.MEMORY_NO_SMOOTHING | _sx_kfm.MEMORY_NO_STD_FORECAST
            | _sx_kfm.MEMORY_NO_LIKELIHOOD)          # keep only forecast errors, their variances and the Kalman gains
_SX_LOG2PI, _SX_EPS = float(np.log(2.0 * np.pi)), float(np.finfo(float).eps)

def _sx_model_index(index):
    """Time index handed to statsmodels. Without a stated frequency statsmodels infers it from a regular
    date index (and warns: 'No frequency information was provided, so inferred frequency h will be used') or, for an
    irregular one, ignores the dates and uses 0, 1, ..., n-1 internally (and warns: 'A date index has been provided, but
    it has no associated frequency information ...'). Both cases are stated here, so that statsmodels builds the same
    model without a warning: a regular date index gets its inferred frequency (same timestamps), an irregular one (e.g.
    a thinned test sample) is replaced by 0, 1, ..., n-1. Only the statsmodels model sees this index; the calendar-day
    clusters and the number of days are taken from the time index of the data passed in."""
    if isinstance(index, pd.DatetimeIndex) and index.freq is None:
        freq = index.inferred_freq
        return pd.DatetimeIndex(index, freq=freq) if freq is not None else pd.RangeIndex(len(index))
    return index

class SarimaxProfile:
    """Regression with SARIMA errors: exact profile likelihood over the regression coefficients.
    self.mod is the statsmodels SARIMAX model on the rescaled regressors; every reported log-likelihood is
    self.mod.loglike(params). `burn` overrides the number of initial observations left out of the log-likelihood
    (statsmodels' default with enforce_stationarity=False is the number of states: 27 or 28 for the orders used here)."""

    def __init__(self, endog, exog, order, seasonal_order, burn=None):
        self.cols = list(exog.columns); self.k = len(self.cols); self.f = sarimax_scale_factors(self.cols)
        Xs = exog.astype(float).mul(pd.Series(self.f))
        kw = {} if burn is None else {'loglikelihood_burn': int(burn)}
        # the model is built on the time index with its frequency stated (_sx_model_index): no ValueWarning
        # about an inferred or missing frequency; the alignment check of statsmodels is kept here
        if not Xs.index.equals(endog.index):
            raise ValueError("The indices for endog and exog are not aligned")
        mi = _sx_model_index(endog.index)
        self.mod = sm.tsa.SARIMAX(endog=endog.astype(float).set_axis(mi), exog=Xs.set_axis(mi), order=order,
                                  seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False, **kw)
        self.names = list(self.mod.param_names)
        self.arma_names = self.names[self.k:-1]
        assert self.names[:self.k] == self.cols and self.names[-1] == 'sigma2', "unexpected parameter layout"
        assert all(n.startswith(('ar.', 'ma.')) for n in self.arma_names), "unexpected parameter layout"
        # specification guards: no trend term; no constant regressor in a model with d > 0
        assert getattr(self.mod, 'k_trend', 0) == 0, "no trend term may be used (the intercept is a regressor column)"
        if order[1] > 0 and any(float(np.ptp(Xs[col].to_numpy())) == 0.0 for col in self.cols):
            raise ValueError("a constant regressor is not identified in a model with d > 0")
        self.fvec = np.array([self.f.get(n, 1.0) for n in self.names])            # back-transformation to original units
        self.burn = int(self.mod.loglikelihood_burn); self.nobs = int(self.mod.nobs); self.n_eff = self.nobs - self.burn
        self.Xv = np.ascontiguousarray(Xs.to_numpy(dtype=float))
        self.n_pass = 0                                                            # filter passes (real + complex)

    def gls(self, arma, sigma2, cov=False):
        """One filter pass: exact GLS for beta at (AR/MA, sigma2)."""
        self.n_pass += 1
        arma = np.asarray(arma, dtype=float); out = {'ok': False, 'll': -np.inf}
        if not (np.all(np.isfinite(arma)) and np.isfinite(sigma2) and sigma2 > 0):
            return out
        try:
            m = self.mod
            m.update(np.r_[np.zeros(self.k), arma, sigma2], transformed=True)      # beta = 0: the filter sees y itself
            kf = m.ssm.filter(conserve_memory=_SX_LEAN)
            F = np.array(kf.forecasts_error_cov[0, 0], dtype=float); vy = np.array(kf.forecasts_error[0], dtype=float)
            K = np.ascontiguousarray(np.asarray(kf.kalman_gain, dtype=float)[:, 0, :].T)                # nobs x k_states
            Zm, Tm = np.asarray(m.ssm['design'], dtype=float), np.asarray(m.ssm['transition'], dtype=float)
            Z = Zm[0] if Zm.ndim == 2 else Zm[0, :, 0]; T = np.ascontiguousarray(Tm if Tm.ndim == 2 else Tm[:, :, 0])
            s = slice(self.burn, None)
            if not (np.all(np.isfinite(F)) and np.all(F[s] > 0) and np.all(np.isfinite(vy)) and np.all(np.isfinite(K))):
                return out
            # regressors through the same filter: a_{t+1} = T a_t + K_t v_t, v_t = x_t - Z a_t, a_1 = 0
            Xv = self.Xv; a = np.zeros((T.shape[0], self.k)); b = np.empty_like(a); VX = np.empty_like(Xv)
            for t in range(self.nobs):
                v = Xv[t] - Z @ a
                VX[t] = v
                np.dot(T, a, out=b); b += K[t][:, None] * v
                a, b = b, a
            sw = 1.0 / np.sqrt(F[s]); Xw = VX[s] * sw[:, None]; yw = vy[s] * sw
            if not np.all(np.isfinite(Xw)):
                return out
            beta, _, rank, sv = np.linalg.lstsq(Xw, yw, rcond=None)
            e = yw - Xw @ beta; S = float(e @ e)
            ll = -0.5 * (self.n_eff * _SX_LOG2PI + float(np.log(F[s]).sum()) + S)
            if not np.isfinite(ll):
                return out
            out.update(ok=True, ll=ll, beta=beta, S=S, q=S / self.n_eff, sigma2=float(sigma2), arma=arma.copy(),
                       full=np.r_[beta, arma, sigma2], cond=float(sv[0] / sv[-1]) if sv[-1] > 0 else np.inf)
            if cov:                    # inverse of the WLS normal matrix = covariance of beta_hat given (AR/MA, sigma2)
                _, s_, Vt = np.linalg.svd(Xw, full_matrices=False)
                keep = s_ > s_[0] * max(Xw.shape) * _SX_EPS
                out['cov_beta'] = (Vt[keep].T / s_[keep] ** 2) @ Vt[keep]
                out['e_std'] = e                                                    # standardized one-step-ahead residuals
        except Exception as exc:                                                   # e.g. non-finite values for explosive parameters
            out['error'] = repr(exc)[:200]
        return out

    def conc(self, arma, s2c):
        """Search objective: profile log-likelihood over beta at sigma2 = s2c plus the closed-form gain from re-scaling
        sigma2 to SSR/n (exact up to the fixed variance, 1e6, of statsmodels' approximate-diffuse prior)."""
        r = self.gls(arma, s2c)
        if not r['ok'] or r['q'] <= 0:
            return -np.inf, r
        return r['ll'] + 0.5 * self.n_eff * (r['q'] - 1.0 - np.log(r['q'])), r

    def grad_z(self, full):
        """Exact gradient of the profile log-likelihood with respect to (AR/MA, log sigma2): by the envelope theorem the
        partial derivatives of the full log-likelihood at beta_hat (complex-step differentiation of mod.loglike)."""
        base = np.asarray(full, dtype=complex); g = np.empty(len(full) - self.k)
        for i, j in enumerate(range(self.k, len(full))):
            h = np.sqrt(_SX_EPS) * max(abs(float(full[j])), 0.1)
            p = base.copy(); p[j] += 1j * h
            g[i] = np.imag(self.mod.loglike(p, complex_step=True)) / h
        g[-1] *= float(full[-1])                                                   # d / d log(sigma2)
        self.n_pass += len(g)
        return g

    def inner(self, arma, maxit=6):
        """Exact maximum over (beta, sigma2) for given AR/MA parameters."""
        r = self.gls(arma, 1.0)
        if not r['ok']:
            return r
        u = float(np.log(r['q'])); r = self.gls(arma, np.exp(u))
        for _ in range(maxit):                                                     # Newton on log(sigma2); curvature ~ -n/2
            if not r['ok']:
                break
            gu = self.grad_z(r['full'])[-1]
            if abs(gu) < 1e-7:
                break
            u += gu / (0.5 * self.n_eff); r = self.gls(arma, np.exp(u))
        return r

def _sx_sides(P, z, hstep, sgn):
    """Exact profile gradient and beta_hat at z + sgn * h_j e_j for every coordinate j (as columns), or None."""
    G, B = [], []
    for j in range(len(z)):
        zj = z.copy(); zj[j] += sgn * hstep[j]; rj = P.gls(zj[:-1], np.exp(zj[-1]))
        if not rj['ok']:
            return None
        G.append(P.grad_z(rj['full'])); B.append(rj['beta'])
    return np.column_stack(G), np.column_stack(B)

def _sx_quad(H, g):
    """Symmetrised Hessian, its eigenvalues, the solver of its negative-definite version, whether H is negative definite,
    and the predicted remaining gain 0.5 g'(-H)^-1 g."""
    H = 0.5 * (H + H.T); lam, V = np.linalg.eigh(H)
    lam_m = -np.maximum(np.abs(lam), 1e-12 * np.max(np.abs(lam)))
    solve = lambda gg: V @ ((V.T @ gg) / -lam_m)
    return H, lam, solve, bool(lam.max() < 0), 0.5 * float(g @ solve(g))

def _sx_profile_hessian(P, z):
    """Central-difference Hessian of the profile log-likelihood at z = (AR/MA, log sigma2) from the exact gradient, and
    J = d beta_hat / dz from the same filter passes; (H, J) or None."""
    z = np.asarray(z, dtype=float); hstep = np.r_[np.full(len(z) - 1, 1e-6), 1e-5]
    up, dn = _sx_sides(P, z, hstep, 1.0), _sx_sides(P, z, hstep, -1.0)
    if up is None or dn is None:
        return None
    H = (up[0] - dn[0]) / (2.0 * hstep)
    return 0.5 * (H + H.T), (up[1] - dn[1]) / (2.0 * hstep)

def _sx_newton(P, arma, sigma2, maxit, tol):
    """Safeguarded Newton iterations on the exact profile log-likelihood in z = (AR/MA, log sigma2): exact gradient, Hessian
    by differences of that gradient, step halving, only improvements are accepted. A point is CERTIFIED only if a
    central-difference Hessian computed AT THAT POINT is negative definite and the predicted remaining gain is below tol;
    that Hessian and J = d beta_hat / dz are returned in info (they are reused for the standard errors)."""
    z = np.r_[np.asarray(arma, dtype=float), np.log(sigma2)]; nz = len(z)
    info = {'converged': False, 'neg_def': False, 'remaining_gain': np.nan, 'max_abs_grad': np.nan, 'H': None, 'J': None,
            'status': 'start point could not be evaluated'}
    r = P.gls(z[:-1], np.exp(z[-1]))
    if not r['ok']:
        return z, r, info
    g = P.grad_z(r['full']); hstep = np.r_[np.full(nz - 1, 1e-6), 1e-5]
    for it in range(maxit + 1):
        info.update(max_abs_grad=float(np.max(np.abs(g))), H=None, J=None)
        up, dn = _sx_sides(P, z, hstep, 1.0), None                                  # one-sided Hessian at the current point
        if up is None:
            dn = _sx_sides(P, z, hstep, -1.0)
            if dn is None:
                info['status'] = 'Hessian could not be evaluated'; break
        H = (up[0] - g[:, None]) / hstep if up is not None else (g[:, None] - dn[0]) / hstep
        if not np.all(np.isfinite(H)):
            info['status'] = 'Hessian not finite'; break
        H, lam, solve, nd, pred = _sx_quad(H, g)
        info.update(neg_def=nd, remaining_gain=pred)
        if nd and pred < tol:                                                       # candidate: certify with a central Hessian here
            up = _sx_sides(P, z, hstep, 1.0) if up is None else up
            dn = _sx_sides(P, z, hstep, -1.0) if dn is None else dn
            if up is not None and dn is not None and np.all(np.isfinite(up[0])) and np.all(np.isfinite(dn[0])):
                H, lam, solve, nd, pred = _sx_quad((up[0] - dn[0]) / (2.0 * hstep), g)
                info.update(neg_def=nd, remaining_gain=pred, H=H, J=(up[1] - dn[1]) / (2.0 * hstep))
                if nd and pred < tol:
                    info.update(converged=True, status='certified'); break
        if it == maxit:
            info['status'] = 'Newton iterations exhausted'; break
        step = solve(g)                                                             # Newton step (ascent direction)
        mx = float(np.max(np.abs(step)))
        if mx > 0.2:
            step *= 0.2 / mx
        acc = None; t = 1.0
        for _ in range(12):
            zt = z + t * step; rt = P.gls(zt[:-1], np.exp(zt[-1]))
            if rt['ok'] and rt['ll'] > r['ll']:
                acc = (zt, rt); break
            t *= 0.5
        if acc is None:
            info['status'] = 'no improving step found (numerical precision)'; break
        z, r = acc; g = P.grad_z(r['full'])
    # flat-ridge indicator at the returned point (see 'plateau' in the notes above)
    info['plateau'] = bool((not info['converged']) and info['neg_def'] and np.isfinite(info['remaining_gain'])
                           and info['remaining_gain'] < SARIMAX_SEARCH['plateau_gain'])
    info['plateau_rule'] = 'hessian' if info['plateau'] else ''
    return z, r, info

def _sx_simplex(x, step):
    """Initial Nelder-Mead simplex: one vertex per parameter, moved by `step` towards zero for |x| >= 0.5 (away from unit
    roots) and by +step otherwise."""
    x = np.asarray(x, dtype=float); S = np.tile(x, (len(x) + 1, 1))
    for j in range(len(x)):
        S[j + 1, j] += -step * np.sign(x[j]) if abs(x[j]) >= 0.5 else step
    return S

def _sx_optimise(P, x0, cfg):
    """Search-and-polish from one AR/MA starting point: Nelder-Mead (beta by GLS, sigma2 in closed form), then Newton
    polishing; if Newton does not certify a maximum, Nelder-Mead restarts from the current point with a small simplex and
    the polishing is repeated (at most cfg['rounds'] rounds). Returns the solution (with its Newton record) or None."""
    x = np.asarray(x0, dtype=float)
    r0 = P.gls(x, 1.0)
    if not r0['ok']:
        return None
    s2c = r0['q']; J0, r1 = P.conc(x, s2c)                                          # closed-form scale at the start
    if not np.isfinite(J0):
        return None
    s2c *= r1['q']
    step, nfev, xatol, fatol = cfg['step1'], cfg['nm1'], cfg['xatol1'], cfg['fatol1']
    best_r, best_info, s2 = None, {'converged': False}, s2c
    for rnd in range(cfg['rounds']):
        best = {'J': -np.inf, 'x': x.copy(), 's2': s2c}
        def obj(xx):
            J, rr = P.conc(xx, s2c)
            if J > best['J']:
                best.update(J=J, x=np.array(xx, dtype=float), s2=s2c * rr['q'])
            return -J if np.isfinite(J) else 1e30
        _sx_minimize(obj, x, method='Nelder-Mead', options={'initial_simplex': _sx_simplex(x, step), 'xatol': xatol, 'fatol': fatol,
                                                             'maxfev': nfev, 'adaptive': True})
        if not np.isfinite(best['J']):
            break
        x, s2 = best['x'], best['s2']
        try:
            z, r, info = _sx_newton(P, x, s2, cfg['newton_it'], cfg['tol_gain'])
        except Exception as exc:                                                    # keep the Nelder-Mead point
            z, r = np.r_[x, np.log(s2)], P.gls(x, s2)
            info = {'converged': False, 'neg_def': False, 'remaining_gain': np.nan, 'max_abs_grad': np.nan, 'H': None, 'J': None,
                    'status': f'Newton stage failed: {exc!r}'[:120]}
        prev_ll = None if best_r is None else best_r['ll']
        if r['ok'] and (best_r is None or r['ll'] >= best_r['ll']):
            best_r, best_info = r, info; x, s2 = z[:-1].copy(), float(np.exp(z[-1]))
        if info['converged']:
            break
        # flat ridge: stop when the Newton record already predicts a remaining gain below cfg['plateau_gain'] at a
        # negative-definite Hessian, or when this full extra round gained less than that
        if best_r is not None and (best_info.get('plateau') or (prev_ll is not None and best_r['ll'] - prev_ll < cfg['plateau_gain']
                                                                and np.isfinite(best_info.get('max_abs_grad', np.nan)))):
            best_info = dict(best_info, plateau=True, plateau_rule=best_info.get('plateau_rule') or 'stagnation')
            break
        s2c = s2; step, nfev, xatol, fatol = cfg['step2'], cfg['nm2'], cfg['xatol2'], cfg['fatol2']
    if best_r is None:
        return None
    best_r = dict(best_r); best_r['newton'] = best_info
    return best_r

def _sx_grid_points(arma_names):
    """Coarse fixed grid of AR/MA starting values: first AR and MA coefficients x three seasonal pairs (all other terms at
    zero); ma.L1 = -0.95 with ar.L1 = 0.9 covers the over-differenced region of d = 1 models."""
    pts = []
    s_ar = [n for n in arma_names if n.startswith('ar.S.')][:1]; s_ma = [n for n in arma_names if n.startswith('ma.S.')][:1]
    for a in ((-0.5, 0.0, 0.5, 0.9, 0.98) if 'ar.L1' in arma_names else (0.0,)):
        for m in ((-0.95, -0.5, 0.0, 0.5) if 'ma.L1' in arma_names else (0.0,)):
            for sa, sm_ in ((0.9, -0.6), (0.98, -0.85), (0.995, -0.93)):
                d = dict.fromkeys(arma_names, 0.0)
                if 'ar.L1' in d: d['ar.L1'] = a
                if 'ma.L1' in d: d['ma.L1'] = m
                for n in s_ar: d[n] = sa
                for n in s_ma: d[n] = sm_
                pts.append([d[n] for n in arma_names])
    return pts

def sarimax_root_moduli(arma, kind):
    """Moduli of the roots of the non-seasonal AR polynomial 1 - ar.L1 z - ar.L2 z^2 ... (kind='ar') or MA polynomial
    1 + ma.L1 z + ma.L2 z^2 ... (kind='ma'), statsmodels sign conventions. Stationary / invertible if all moduli exceed 1."""
    c = [arma[n] for n in sorted((n for n in arma if n.startswith(f'{kind}.L')), key=lambda n: int(n.split('L')[1]))]
    if not c or not np.any(np.asarray(c) != 0):
        return []
    return sorted(float(v) for v in np.abs(np.roots([(-v if kind == 'ar' else v) for v in c[::-1]] + [1.0])))

def _sx_invertible(arma):
    return all(v > 1.0 for v in sarimax_root_moduli(arma, 'ma')) and all(abs(v) < 1.0 for n, v in arma.items() if n.startswith('ma.S.'))

def _sx_mirror(arma):
    """Invertible mirror image of an AR/MA point: non-seasonal MA roots inside the unit circle are replaced by their
    reciprocals (same autocorrelations; sigma2 is rescaled by the estimation), a seasonal MA coefficient with modulus above
    1 by its reciprocal."""
    out = dict(arma)
    names = sorted((n for n in arma if n.startswith('ma.L')), key=lambda n: int(n.split('L')[1]))
    c = [float(arma[n]) for n in names]
    if c and np.any(np.asarray(c) != 0):
        r = np.roots(c[::-1] + [1.0])                                       # roots of 1 + ma.L1 z + ... + ma.Lq z^q
        r = np.where(np.abs(r) < 1.0, 1.0 / np.conj(r), r)
        p = np.real(np.poly(r)); p = p / p[-1]
        out.update({n: float(v) for n, v in zip(names, list(p[::-1][1:]) + [0.0] * len(names))})
    out.update({n: 1.0 / v for n, v in arma.items() if n.startswith('ma.S.') and abs(v) > 1.0})
    return out

def _sx_scaled_inverse(M, sign=1.0):
    """Inverse of the symmetric positive-definite matrix sign * M, computed after scaling it to unit diagonal: the
    parameters differ in scale by many orders of magnitude and a plain (pseudo-)inverse would drop the weak directions
    (statsmodels' cov_type 'opg' / 'approx' use a pseudo-inverse with a relative cut-off of 1e-15 on the unscaled matrix).
    NaN if the scaled matrix is not positive definite."""
    M = sign * 0.5 * (np.asarray(M, dtype=float) + np.asarray(M, dtype=float).T)
    d = np.sqrt(np.abs(np.diag(M))); d[~(d > 0)] = 1.0
    lam, V = np.linalg.eigh(M / np.outer(d, d))
    if not np.all(np.isfinite(lam)) or lam.min() <= 1e-13 * lam.max():
        return np.full(M.shape, np.nan)
    return ((V / lam) @ V.T) / np.outer(d, d)

def _sx_ljung_box(e_std, lags=(24, 48)):
    """Ljung-Box statistics of the standardized one-step-ahead residuals (p-value: chi2 with as many degrees of freedom as lags)."""
    from statsmodels.stats.diagnostic import acorr_ljungbox
    out = {}
    for L in lags:
        try:
            lb = acorr_ljungbox(e_std, lags=[L], return_df=True)
            out[f'LjungBox_Q{L}'], out[f'LjungBox_p{L}'] = float(lb['lb_stat'].iloc[0]), float(lb['lb_pvalue'].iloc[0])
        except Exception:
            out[f'LjungBox_Q{L}'], out[f'LjungBox_p{L}'] = np.nan, np.nan
    return out

def _sx_inference(P, run, dates):
    """Standard errors at the selected point, all built here and inverted after diagonal scaling (rescaled units):
      OPG          (S'S)^-1 with S = scores per observation (mod.score_obs) - the 'std err' column of the coefficient tables;
      Hessian      exact block form: cov(z) = (-H_p)^-1 for z = (AR/MA, log sigma2), H_p = central-difference profile Hessian
                   at this point; cov(beta) = GLS covariance + J cov(z) J', J = d beta_hat / dz;
      Robust       White-type sandwich C (S'S) C with C the Hessian covariance (quasi-maximum-likelihood);
      Cluster_Day  the same sandwich with the scores summed within calendar days (the default inference of the paper);
      Cluster_Day_adj  Cluster_Day times sqrt(G / (G - 1)), G = number of days (small-sample factor);
      HAC24, HAC48 Newey-West sandwiches C M C with M the Bartlett-kernel long-run covariance of the scores (24 / 48 lags);
      GLS          conditional standard errors of the regression coefficients given the AR/MA parameters (NaN for the others).
    Returns ({kind: standard errors}, the score summed over the observations, the failed consistency checks, the GLS step)."""
    k, npar = P.k, len(P.names); arma, s2 = run['arma'], float(run['sigma2']); flags = []
    r = P.gls(arma, s2, cov=True); full = r['full']
    S = np.asarray(P.mod.score_obs(full), dtype=float); S = S if S.shape == (P.nobs, npar) else S.T
    G = S.T @ S
    day = pd.factorize(pd.DatetimeIndex(dates).normalize())[0]; Sd = np.zeros((int(day.max()) + 1, npar)); np.add.at(Sd, day, S)
    Hp, J = run['newton'].get('H'), run['newton'].get('J')
    if Hp is None or J is None:                                                    # not certified in the search: computed here
        ph = _sx_profile_hessian(P, np.r_[arma, np.log(s2)]); Hp, J = ph if ph is not None else (None, None)
    cov_hes = np.full((npar, npar), np.nan); nd = False
    if Hp is not None:
        nd = bool(np.linalg.eigvalsh(0.5 * (Hp + Hp.T)).max() < 0)
        czz = _sx_scaled_inverse(Hp, -1.0)
        cov_hes[:k, :k] = r['cov_beta'] + J @ czz @ J.T; cov_hes[:k, k:] = J @ czz; cov_hes[k:, :k] = cov_hes[:k, k:].T; cov_hes[k:, k:] = czz
        tr = np.ones(npar); tr[-1] = s2; cov_hes = cov_hes * np.outer(tr, tr)       # log sigma2 -> sigma2
    covs = {'OPG': _sx_scaled_inverse(G), 'Hessian': cov_hes, 'Robust': cov_hes @ G @ cov_hes, 'Cluster_Day': cov_hes @ (Sd.T @ Sd) @ cov_hes}
    for L in (24, 48):                                                             # HAC (Newey-West, Bartlett kernel)
        M = G.copy()
        for lag in range(1, L + 1):
            Gl = S[lag:].T @ S[:-lag]; M += (1.0 - lag / (L + 1.0)) * (Gl + Gl.T)
        covs[f'HAC{L}'] = cov_hes @ M @ cov_hes
    se = {kind: np.sqrt(np.where(np.diag(c) > 0, np.diag(c), np.nan)) for kind, c in covs.items()}
    se_gls = np.sqrt(np.diag(r['cov_beta']))
    n_days = Sd.shape[0]
    se['Cluster_Day_adj'] = se['Cluster_Day'] * np.sqrt(n_days / (n_days - 1.0))
    se['GLS'] = np.r_[se_gls, np.full(npar - k, np.nan)]
    # consistency checks of the standard errors
    if not nd:
        flags.append('profile Hessian not negative definite')
    if not np.all(np.isfinite(se['OPG'])):
        flags.append('OPG matrix singular')
    _hr = se['Hessian'][:k] / se_gls          # all NaN when the Hessian standard errors are not available
    if not np.any(np.isfinite(_hr)):
        flags.append('Hessian standard errors of the regression coefficients not available')
    elif not np.nanmin(_hr) >= 1.0 - 1e-6:
        flags.append('a Hessian standard error is below the GLS (conditional) one')
    if not se['Hessian'][-1] >= 0.99 * s2 * np.sqrt(2.0 / P.n_eff):
        flags.append('sigma2 standard error below sigma2*sqrt(2/n)')
    return se, S.sum(axis=0), flags, r

def sarimax_status_text(info):
    """Convergence status of an estimate returned by fit_sarimax, as printed in the reports."""
    return {'certified': 'CERTIFIED MAXIMUM', 'plateau': 'PLATEAU (flat ridge, accepted)'}.get(info.get('status_class'), 'NOT CONVERGED')

def fit_sarimax(endog, exog, order, seasonal_order, starts=None, burn=None, inference=True, job_dir=None, job_key=None):
    """Profile maximum-likelihood fit of a SARIMAX model from several AR/MA starting points.
    starts: {name: 'default' (statsmodels' start_params) | 'arma0' (all AR and MA terms at zero) | 'grid' (best point of a
            fixed grid) | {AR/MA parameter: value} (terms not listed start at zero)}; default: SARIMAX_DEFAULT_STARTS.
    job_dir, job_key: if given, the result of every finished start is written to job_dir/<job_key>__<start>.json and read
            back on a re-run (same method, data, model and start), so an interrupted estimation resumes with the missing
            starts; job_dir/progress.log receives one line per finished start. Every record also holds the warnings
            raised while that start was estimated (key 'warnings': [category, message, location, count]; not part of the
            signature); for a record without this key the warnings are reported as not recorded.
    Selection: the INVERTIBLE representation of the best point is reported. Mirror-image MA solutions (a non-seasonal
    MA root z and 1/z, a seasonal MA coefficient c and 1/c) have the same autocovariances and regression coefficients and
    differ in log-likelihood only through the initial conditions (the non-invertible one is slightly higher).
      (1) Best point: the highest log-likelihood; among the starts within SARIMAX_TIE_TOL
          of it a certified maximum is preferred, then a plateau, then an invertible point. If it is not invertible, its
          mirror image is estimated as start 'mirror' and the rule is applied again. An invertible best point is reported.
      (2) If the best point is still not invertible, the invertible run that is certified or a plateau and lies within
          SARIMAX_MIRROR_TOL of the highest log-likelihood is reported (the highest such run; within SARIMAX_TIE_TOL of it a
          certified maximum first, then a plateau).
      (3) If there is no such run (the mirror start failed or did not converge, or the only invertible runs are other,
          lower local maxima), the best point is reported in its non-invertible representation and info['representation']
          says so. info['peak_llf'] / info['peak_start'] keep the highest log-likelihood of all starts; every start stays in
          info['starts'].
    Returns (table, info, summary_text). table: coef, std err, z, P>|z|, [0.025, 0.975] in ORIGINAL units with OPG standard
    errors (the 'std err' column), plus 'std err (X)' and 'P>|z| (X)' for X = Hessian, Robust (sandwich) and
    Cluster_Day (sandwich clustered by calendar day). info: log-likelihood, AIC, BIC, the record of every start, AR/MA
    roots, Ljung-Box statistics, the gradient at the solution and the certificate ('converged' = certified maximum).
    inference=False skips the standard errors (likelihood-ratio tests, Appendix C); burn: see SarimaxProfile.
    info['status_class'] is 'certified', 'plateau' or 'not converged' (see the notes above); info['converged'] is True only
    for a certified maximum."""
    import gc as _gc
    import warnings as _warnings
    # no blanket filter: fit_sarimax runs in a worker process, or in the main process when SARIMAX_N_JOBS == 1
    # or a single model is estimated (run_parallel); a process-wide 'ignore' would persist into every later job or cell
    _warnings.simplefilter('always', ConvergenceWarning)
    cfg = SARIMAX_SEARCH
    with threadpool_limits(1):
        P = SarimaxProfile(endog, exog, order, seasonal_order, burn=burn)
        names, an = P.names, P.arma_names
        key = next((p for p in names if p.startswith(('Solar_', 'Interaction_Solar', 'System_Direction'))), names[0])
        runs, failed = [], {}
        def run_start(name, start):
            # checkpoint of this start: identified by method, data, model, order and the start itself
            ck, t0 = None, time.time()
            if job_dir:
                os.makedirs(job_dir, exist_ok=True)
                spec = start if isinstance(start, str) else sorted((str(a), float(b)) for a, b in start.items())
                sig = hashlib.sha256(json.dumps([SARIMAX_METHOD_TAG, names, P.nobs, P.burn, list(order), list(seasonal_order), name, spec,
                                                 sorted(cfg.items())], default=str).encode()).hexdigest()[:16]
                ck = os.path.join(job_dir, re.sub(r'[^A-Za-z0-9_.-]+', '_', f"{job_key}__{name}") + '.json')
                if os.path.exists(ck):
                    with open(ck, encoding='utf-8') as fh:
                        saved = json.load(fh)
                    if saved.get('signature') == sig:
                        runs.append({'name': name, 'arma': np.array(saved['arma'], dtype=float), 'sigma2': float(saved['sigma2']),
                                     'full': np.array(saved['full'], dtype=float), 'newton': dict(saved['newton'], H=None, J=None),
                                     'llf': float(saved['llf']), 'certified': bool(saved['certified']), 'plateau': bool(saved['plateau']),
                                     'invertible': bool(saved['invertible']), 'record': saved['record'],
                                     'warnings': saved.get('warnings')})     # None: a start record without the 'warnings' key
                        return
            # the warnings raised while this start is estimated are counted and stored with its record (key 'warnings', not
            # part of the signature); they are passed on unchanged to the handler that was active before (shown or counted there)
            sw_prev, sw_counts = _warnings.showwarning, {}
            def sw_count(message, category, filename, lineno, file=None, line=None):
                k = (getattr(category, '__name__', str(category)), str(message)[:300], f"{os.path.basename(str(filename))}:{lineno}")
                sw_counts[k] = sw_counts.get(k, 0) + 1
                sw_prev(message, category, filename, lineno, file, line)
            with _warnings.catch_warnings():
                _warnings.showwarning = sw_count
                if isinstance(start, str) and start == 'default':
                    sp = dict(zip(names, np.asarray(P.mod.start_params, dtype=float))); x0 = [sp[n] for n in an]
                elif isinstance(start, str) and start == 'arma0':
                    x0 = [0.0] * len(an)
                elif isinstance(start, str) and start == 'grid':
                    pts = _sx_grid_points(an); x0 = pts[int(np.argmax([P.conc(p, 1.0)[0] for p in pts]))]
                else:
                    x0 = [float(start.get(n, 0.0)) for n in an]
                n0 = P.n_pass; fail = 'the starting point could not be evaluated'
                try:
                    r = _sx_optimise(P, x0, dict(cfg, nm1=2 * cfg['nm1']) if start == 'arma0' else cfg)   # 'arma0' starts far away
                except Exception as exc:
                    r, fail = None, f'failed: {exc!r}'[:200]
            sw_list = [list(k) + [n] for k, n in sorted(sw_counts.items())]
            if r is None:                                                           # recorded with the other starts
                failed[name] = {'llf': np.nan, 'certified': False, 'status': fail,
                                'start_ARMA': dict(zip(an, [round(float(v), 6) for v in x0]))}
                if job_dir:
                    with open(os.path.join(job_dir, 'progress.log'), 'a', encoding='utf-8') as fh:
                        fh.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {job_key} | start '{name}' | FAILED: {fail}\n")
                return
            arma = dict(zip(an, map(float, r['arma']))); nw = r['newton']
            plateau = bool(nw.get('plateau', False)) and not bool(nw['converged'])
            runs.append({'name': name, 'arma': r['arma'], 'sigma2': r['sigma2'], 'full': r['full'], 'newton': nw,
                         'llf': float(P.mod.loglike(r['full'])), 'certified': bool(nw['converged']), 'plateau': plateau,
                         'invertible': _sx_invertible(arma), 'warnings': sw_list,
                         'record': {'llf': None, key: float(r['full'][names.index(key)] * P.fvec[names.index(key)]),
                                    'certified': bool(nw['converged']), 'plateau': plateau,
                                    'plateau_rule': nw.get('plateau_rule', '') if plateau else '', 'status': nw.get('status', ''),
                                    'remaining_gain': float(nw['remaining_gain']), 'max_abs_gradient': float(nw['max_abs_grad']),
                                    'AR_roots': [round(v, 5) for v in sarimax_root_moduli(arma, 'ar')],
                                    'MA_roots': [round(v, 5) for v in sarimax_root_moduli(arma, 'ma')],
                                    'start_ARMA': dict(zip(an, [round(float(v), 6) for v in x0])), 'filter_passes': P.n_pass - n0}})
            runs[-1]['record']['llf'] = runs[-1]['llf']
            if job_dir:                                                             # atomic write, then one progress line
                t = runs[-1]
                out = {'signature': sig, 'job_key': job_key, 'start': name, 'arma': [float(v) for v in t['arma']],
                       'sigma2': float(t['sigma2']), 'full': [float(v) for v in t['full']], 'llf': t['llf'],
                       'certified': t['certified'], 'plateau': t['plateau'], 'invertible': bool(t['invertible']), 'record': t['record'],
                       'warnings': t['warnings'],                       # not part of the signature
                       'newton': {a: (bool(b) if isinstance(b, (bool, np.bool_)) else (float(b) if isinstance(b, (int, float, np.floating)) else str(b)))
                                  for a, b in nw.items() if a not in ('H', 'J')},
                       'seconds': round(time.time() - t0, 1), 'finished': time.strftime('%Y-%m-%d %H:%M:%S')}
                with open(ck + '.tmp', 'w', encoding='utf-8') as fh:
                    json.dump(out, fh)
                for _k in range(6):              # Windows: a virus scanner or indexer may hold the file briefly
                    try:
                        os.replace(ck + '.tmp', ck)
                        break
                    except PermissionError:
                        if _k == 5:
                            raise
                        time.sleep(0.5 * (_k + 1))
                with open(os.path.join(job_dir, 'progress.log'), 'a', encoding='utf-8') as fh:
                    fh.write(f"{out['finished']} | {job_key} | start '{name}' | llf {t['llf']:.4f} | "
                             f"{'certified' if t['certified'] else ('plateau' if t['plateau'] else 'not converged')} | "
                             f"{t['record']['filter_passes']} filter passes | {out['seconds'] / 60:.1f} min\n")
            _gc.collect()
        for name, start in (starts or SARIMAX_DEFAULT_STARTS).items():
            run_start(name, start)
        if not runs:
            raise RuntimeError(f"no starting point could be estimated: {failed}")
        pick = lambda pool: max((t for t in pool if t['llf'] >= max(u['llf'] for u in pool) - SARIMAX_TIE_TOL),
                                key=lambda t: (t['certified'], t['plateau'], t['invertible'], t['llf']))
        best = pick(runs)                                                          # selection rule (1) of the docstring
        if not best['invertible']:                                                 # estimate the invertible representation
            run_start('mirror', _sx_mirror(dict(zip(an, map(float, best['arma'])))))
            best = pick(runs)
        if not best['invertible']:
            # report the invertible representation: the best converged invertible run within SARIMAX_MIRROR_TOL of the
            # highest log-likelihood; without one the best point is reported as it is (info['representation'] says so)
            top = max(t['llf'] for t in runs)
            pool = [t for t in runs if t['invertible'] and (t['certified'] or t['plateau']) and t['llf'] >= top - SARIMAX_MIRROR_TOL]
            if pool:
                best = pick(pool)
        peak = max(runs, key=lambda t: t['llf'])                                   # highest log-likelihood of all starts
        full = best['full']; llf = best['llf']; arma = dict(zip(an, map(float, best['arma']))); npar = len(names)
        c = pd.Series(P.fvec, index=names); coef = pd.Series(full, index=names) * c
        info = {'converged': best['certified'], 'status': best['newton'].get('status', ''), 'iterations': int(P.n_pass),
                'best_start': best['name'], 'llf': llf, 'aic': -2.0 * llf + 2.0 * npar, 'bic': -2.0 * llf + npar * float(np.log(P.n_eff)),
                'k_params': npar, 'nobs': P.nobs, 'burn': P.burn, 'starts': {**{t['name']: t['record'] for t in runs}, **failed},
                'arma': arma, 'sigma2': float(best['sigma2']), 'key': key, 'key_coef': float(coef[key]),
                'AR_roots': sarimax_root_moduli(arma, 'ar'), 'MA_roots': sarimax_root_moduli(arma, 'ma'),
                'remaining_gain': float(best['newton']['remaining_gain']), 'max_abs_gradient': np.nan, 'se_flags': '',
                'convergence_warnings': int(not (best['certified'] or best['plateau'])),
                # three-valued status, agreement between the starts, and the number of calendar days (clusters)
                'status_class': 'certified' if best['certified'] else ('plateau' if best['plateau'] else 'not converged'),
                'starts_within_001': int(sum(t['llf'] >= llf - 0.01 for t in runs)),
                'n_days': int(pd.DatetimeIndex(endog.index).normalize().nunique()),
                # reported representation and the highest log-likelihood of all starts (recorded, not reported)
                'representation': 'invertible' if best['invertible'] else
                                  'NON-INVERTIBLE (no converged invertible run within SARIMAX_MIRROR_TOL of the best point)',
                'peak_llf': float(peak['llf']), 'peak_start': peak['name'],
                # warnings stored with every start (None: a start record without the 'warnings' key)
                'start_warnings': {t['name']: t.get('warnings') for t in runs}}
        note = (f"profile maximum likelihood, best of {len(runs)} starts ('{best['name']}'), "
                + (sarimax_status_text(info).lower() if info['status_class'] != 'not converged' else f"NOT CONVERGED ({info['status']})") + "\n"
                "log-likelihood by start: " + ", ".join(f"{t['name']} {t['llf']:.4f} ({key} {t['record'][key]:,.2f}"
                                                         f"{'' if t['certified'] else (', plateau' if t['plateau'] else ', not converged')})" for t in runs)
                + ("".join(f"\n!! start '{n}': {v['status']}" for n, v in failed.items())))
        if info['peak_llf'] > llf + SARIMAX_TIE_TOL or not best['invertible']:      # says so in the estimation report
            note += (f"\nreported representation: {info['representation']}; highest log-likelihood of all starts "
                     f"{info['peak_llf']:.4f} (start '{info['peak_start']}', not reported)")
        if not inference:
            rg = P.gls(best['arma'], best['sigma2'], cov=True)
            info.update(_sx_ljung_box(rg['e_std']))
            table = pd.DataFrame({'coef': coef})
            return table, info, note
        se, score, flags, rg = _sx_inference(P, best, endog.index)
        info.update(_sx_ljung_box(rg['e_std']))
        info.update(max_abs_gradient=float(np.max(np.abs(score))), se_flags='; '.join(flags))
        z = full / se['OPG']
        table = pd.DataFrame({'coef': coef, 'std err': se['OPG'] * c.values, 'z': z, 'P>|z|': 2.0 * _sx_stats.norm.sf(np.abs(z)),
                              '[0.025': coef - 1.959963984540054 * se['OPG'] * c.values, '0.975]': coef + 1.959963984540054 * se['OPG'] * c.values},
                             index=names)
        for kind in SARIMAX_EXTRA_SE:
            table[f'std err ({kind})'] = se[kind] * c.values
            table[f'P>|z| ({kind})'] = 2.0 * _sx_stats.norm.sf(np.abs(full / se[kind]))
        # statsmodels' own summary at the same parameters (its OPG standard errors, rescaled units), for the report
        try:
            res = P.mod.filter(full); summary = str(res.summary())
            info['opg_max_rel_diff_statsmodels'] = float(np.nanmax(np.abs(np.asarray(res.bse, dtype=float) / se['OPG'] - 1.0)))
            del res
        except Exception as exc:
            summary = f"(statsmodels summary not available: {exc!r})"; info['opg_max_rel_diff_statsmodels'] = np.nan
        _gc.collect()
    note += ("\nstatsmodels summary at the same parameters; exogenous coefficients in rescaled units (solar-share terms x100, "
             + ", ".join(SARIMAX_SCALE_LEVEL_VARS) + " /1000); the table above is in original units."
             + (f"\n!! standard-error checks: {info['se_flags']}" if info['se_flags'] else ""))
    return table, info, note + "\n" + summary

def sarimax_profile_loglike(endog, exog, order, seasonal_order, arma, burn=None):
    """Log-likelihood of a SARIMAX model at given AR/MA values with the regression coefficients and sigma2
    re-solved (likelihood-ratio consistency check)."""
    with threadpool_limits(1):
        P = SarimaxProfile(endog, exog, order, seasonal_order, burn=burn)
        r = P.inner([float(arma.get(n, 0.0)) for n in P.arma_names])
        return float(P.mod.loglike(r['full'])) if r['ok'] else np.nan

def _rw_job(func, args, key):
    """Runs one job of run_parallel and counts the warnings raised in it (filter 'always' inside the job; a counter only,
    nothing is stored per occurrence). Self-contained: it runs in a worker process (also under the 'spawn' start method, in
    which nothing of the notebook exists but what is shipped with the job) or in the main process (sequential branch).
    Returns (result, {(category, message, location): count}, key)."""
    import os as _os
    import warnings as _w
    counts = {}
    def _count(message, category, filename, lineno, file=None, line=None):
        k = (getattr(category, '__name__', str(category)), str(message)[:300], f"{_os.path.basename(str(filename))}:{lineno}")
        counts[k] = counts.get(k, 0) + 1
    with _w.catch_warnings():
        _w.simplefilter('always')
        _w.showwarning = _count
        result = func(*args)
    return result, counts, key

def _rw_job_key(func, args):
    """Name of a job in RUN_WARNINGS ('where'): the function and its leading identifying arguments."""
    parts = [getattr(func, '__name__', 'job')]
    for a in args[:3]:
        a = a.get('id', a) if isinstance(a, dict) else a
        if isinstance(a, bool) or not isinstance(a, (str, int)):
            break
        parts.append(str(a))
    return ' '.join(parts)

def run_parallel(func, jobs):
    """Runs independent estimations in parallel and yields each result as soon as it is ready.
    Every job runs inside _rw_job, which counts the warnings raised in it (in a worker process or, in the sequential
    branch, in the main process); the counts are merged into RUN_WARNINGS (where = the job) and one line is printed for every
    job that raised any."""
    jobs = [tuple(j) for j in jobs]
    keys = [_rw_job_key(func, j) for j in jobs]
    if SARIMAX_N_JOBS == 1 or len(jobs) <= 1:
        results = (_rw_job(func, j, k) for j, k in zip(jobs, keys))
    else:
        from joblib import Parallel, delayed
        results = Parallel(n_jobs=min(SARIMAX_N_JOBS, len(jobs)), return_as='generator_unordered')(
            delayed(_rw_job)(func, j, k) for j, k in zip(jobs, keys))
    for result, counts, key in results:
        for (cat, msg, loc), n in sorted(counts.items()):
            _rw_add(key, cat, msg, loc, n)
        if counts:
            print(f"   [warnings] {key}: " + "; ".join(f"{cat} x{n} at {loc}: {' '.join(msg.split())[:200]}"
                                                     for (cat, msg, loc), n in sorted(counts.items())))
        yield result

def _sx_cross_job(model_id, endog, exog, order, seasonal_order, candidates):
    """Profile log-likelihood of one model at the AR/MA estimates of other models (runs in a parallel worker)."""
    return model_id, {s: sarimax_profile_loglike(endog, exog, order, seasonal_order, a) for s, a in candidates.items()}

def sarimax_cross_check(models):
    """Cross-check between models with the same dependent variable and order.
    models: {model id: (endog, exog, order, seasonal_order, info)} with info as returned by fit_sarimax. The profile
    log-likelihood of every model is evaluated at the AR/MA estimates of the other models of its group; if one of them is
    higher than the model's own maximum, the model has to be estimated again. Returns {model id: starts} for those models:
    the better point and the model's own solution as starting points."""
    jobs = []
    for m, (y, X, order, seasonal, info) in models.items():
        cand = {s: i['arma'] for s, (ys, _, o, se, i) in models.items() if s != m and ys.name == y.name and o == order and se == seasonal}
        if cand:
            jobs.append((m, y, X, order, seasonal, cand))
    redo = {}
    for m, vals in run_parallel(_sx_cross_job, jobs):
        vals = {s: v for s, v in vals.items() if np.isfinite(v)}
        best = max(vals, key=vals.get) if vals else None
        if best is not None and vals[best] > models[m][4]['llf'] + 1e-6:
            print(f"   [cross-check] {m}: the AR/MA estimate of {best} gives log-likelihood {vals[best]:.4f}, above its own "
                  f"{models[m][4]['llf']:.4f} -> estimated again from there")
            redo[m] = {f'sibling ({best})': models[best][4]['arma'], 'previous best': models[m][4]['arma']}
    if jobs and not redo:
        print(f"   [cross-check] {len(jobs)} model(s) evaluated at the AR/MA estimates of the models with the same dependent variable: "
              "no higher log-likelihood found.")
    return redo

# -----------------------------------------------------------------------------
# RUN RECORD: environment, results manifest, run report and comparison with the independent pre-test
# -----------------------------------------------------------------------------
# Reference environment in which the results of release R2 were produced (requirements.txt pins the full environment).
REFERENCE_ENVIRONMENT = {'numpy': '2.3.5', 'pandas': '2.3.3', 'scipy': '1.16.3', 'statsmodels': '0.14.6', 'scikit-learn': '1.8.0',
                         'xgboost': '3.2.0', 'patsy': '1.0.3', 'joblib': '1.6.0', 'matplotlib': '3.10.9', 'seaborn': '0.13.2',
                         'openpyxl': '3.1.5'}

def _installed_versions():
    from importlib import metadata
    out = {}
    for _p in REFERENCE_ENVIRONMENT:
        try:
            out[_p] = metadata.version(_p)
        except Exception:
            out[_p] = 'not installed'
    return out

ENVIRONMENT = {'python': platform.python_version(), 'platform': platform.platform(), 'machine': platform.machine(),
               'cpu_count': os.cpu_count(), 'packages': _installed_versions()}
_env_diff = {p: (v, REFERENCE_ENVIRONMENT[p]) for p, v in ENVIRONMENT['packages'].items() if v != REFERENCE_ENVIRONMENT[p]}

# Results manifest: the cached outputs of the later phases are valid only for this data file and method. A results folder
# produced with another data file or method, or one that holds outputs without a manifest, stops the notebook here.
RUN_MANIFEST_FILE = os.path.join(RESULTS_BASE, 'RUN_MANIFEST.json')
RUN_REPORT_FILE = os.path.join(RESULTS_BASE, 'RUN_REPORT.md')
_stamp = {'data_sha256': DATA_SHA256, 'method': SARIMAX_METHOD_TAG}
_present = [n for n in os.listdir(RESULTS_BASE) if not n.startswith('.') and n not in ('RUN_MANIFEST.json', 'RUN_REPORT.md')]
if os.path.exists(RUN_MANIFEST_FILE):
    with open(RUN_MANIFEST_FILE, encoding='utf-8') as _f:
        RUN_MANIFEST = json.load(_f)
    if {k: RUN_MANIFEST.get(k) for k in _stamp} != _stamp:
        raise RuntimeError(f"The results folder '{os.path.abspath(RESULTS_BASE)}' was produced with another data file or method "
                           "(see RUN_MANIFEST.json). Start from an empty results folder.")
    _fresh = False
elif _present:
    raise RuntimeError(f"The results folder '{os.path.abspath(RESULTS_BASE)}' holds outputs but no RUN_MANIFEST.json ({', '.join(sorted(_present)[:6])} ...). "
                       "They may come from another data file or method; start from an empty results folder.")
else:
    RUN_MANIFEST = dict(_stamp, created=RUN_STARTED, environment=ENVIRONMENT, reference_environment=REFERENCE_ENVIRONMENT)
    with open(RUN_MANIFEST_FILE, 'w', encoding='utf-8', newline='\n') as _f:
        json.dump(RUN_MANIFEST, _f, indent=1)
    _fresh = True

def _report_file_line(path):
    if not os.path.exists(path):
        return f"- MISSING: {path}"
    with open(path, 'rb') as fh:
        digest = hashlib.sha256(fh.read()).hexdigest()[:12]
    return f"- {os.path.relpath(path, RESULTS_BASE)} ({os.path.getsize(path):,} bytes, sha256 {digest})"

def _rk_take(title):
    """Closes the current step: the checkpoints and warnings recorded since the previous report() call get `title` as
    their step. Returns them as (checks, warnings)."""
    checks = [c for c in RUN_CHECKS if c['step'] is None]
    warns = [w for w in RUN_WARNINGS if w['step'] is None]
    for e in checks + warns:
        e['step'] = title
    return checks, warns

def _rk_lines(checks, warns):
    """Lines appended to a section of the run report: the checkpoints and the warnings of the step."""
    out = (["Checkpoints of this step:"] + [_ck_line(c) for c in checks] + [""]) if checks else []
    if not warns:
        return out + ["Warnings raised in this step: none.", ""]
    return (out + [f"Warnings raised in this step ({sum(w['count'] for w in warns):,}; category | where | location | message | count; "
                   "count: main process = times shown, a job = every occurrence):"]
            + [f"- {w['category']} | {w['where']} | {w['location']} | {' '.join(w['message'].split())} | x{w['count']}" for w in warns]
            + [""])

def report(title, lines=None, table=None, files=None, _taken=None):
    """Appends a timestamped section to results/RUN_REPORT.md and prints it, so that every step of the run
    leaves a written record. lines: a string or a list of strings; table: a pandas DataFrame (written as fixed-width text);
    files: output files, each listed with its size and the first 12 hex digits of its SHA-256 (missing files are flagged).
    Every section closes a step: the checkpoints and the warnings recorded since the previous report() call get this
    title as their step and are listed at the end of the section (_taken: entries already taken by checkpoint_summary).
    A failure of the report itself is printed and never interrupts the analysis."""
    try:
        taken = _rk_take(title) if _taken is None else _taken
        text = [f"## {time.strftime('%Y-%m-%d %H:%M:%S')} | {title}", ""]
        if lines is not None:
            text += ([lines] if isinstance(lines, str) else [str(v) for v in lines]) + [""]
        if table is not None and len(table):
            text += ["```", table.to_string(index=False), "```", ""]
        if files:
            text += [_report_file_line(p) for p in files] + [""]
        text += _rk_lines(*taken)                  # checkpoints and warnings of the step
        block = "\n".join(text)
        with open(RUN_REPORT_FILE, 'a', encoding='utf-8', newline='\n') as fh:
            fh.write(block + "\n")
        print(block)
    except Exception as exc:
        print(f"!! run report could not be written ({title}): {exc!r}")

# Independent pre-test of 20 November 2026: the same specification recomputed by a second implementation (co-author).
# Its values are used only for comparison; nothing in this notebook is estimated from them.
VALIDATION_DIR = os.path.join(os.path.dirname(os.path.normpath(DATA_DIR)) or '.', 'validation')
REFERENCE_FILE = os.path.join(VALIDATION_DIR, 'pretest_reference_2026-11-20.csv')
# every reference row carries its own tolerance (tol_abs, tol_rel): a value matches if |difference| is within either bound;
# rows without a tolerance (p-values, condition numbers) are reported only
REFERENCE = {}
if os.path.exists(REFERENCE_FILE):
    _ref = pd.read_csv(REFERENCE_FILE, float_precision='round_trip')
    REFERENCE = {i: (float(v), float(a), float(r)) for i, v, a, r in zip(_ref['id'], _ref['value'], _ref['tol_abs'], _ref['tol_rel'])}

def compare_with_reference(title, rows, check=None, explained=None):
    """Compares results of this run with the independent pre-test and writes the table to the run report.
    rows: iterable of (label, value, reference id). Returns the table (Quantity, This run, Reference, Difference, Tolerance,
    Status: 'match' | 'DIFFERS' | 'reported' (no tolerance defined) | 'no reference'). Differences never stop the notebook.
    Every comparison is recorded in RUN_COMPARISONS (total of the run: checkpoint K90). check: name of a checkpoint
    (level 'warn') that states the match / DIFFERS rows of this table in its section; explained: {Quantity: reason} for
    DIFFERS rows with a known cause, which are named as such. The pre-test values refer to the full sample: on any other
    sample (sample_is_full) that checkpoint is NOT CHECKED."""
    try:
        rows = list(rows)
        if not REFERENCE:
            if check:
                checkpoint(check, None, f"reference file not found ({REFERENCE_FILE})", level='warn')
            report(title, f"Reference file not found ({REFERENCE_FILE}); comparison skipped.")
            return pd.DataFrame()
        fmt = lambda v: 'nan' if not np.isfinite(v) else f"{v:.10g}"
        out = []
        for label, value, ref_id in rows:
            try:
                v = float(value)
            except Exception:
                v = np.nan
            ref = REFERENCE.get(ref_id)
            if ref is None:
                out.append({'Quantity': label, 'This run': fmt(v), 'Reference': '-', 'Difference': '-', 'Tolerance': '-', 'Status': 'no reference'})
                continue
            ref_value, tol_abs, tol_rel = ref; d = v - ref_value
            if not (np.isfinite(tol_abs) or np.isfinite(tol_rel)):
                status, shown = 'reported', '-'
            else:
                ok = bool(np.isfinite(d) and ((np.isfinite(tol_abs) and abs(d) <= tol_abs) or (np.isfinite(tol_rel) and abs(d) <= tol_rel * abs(ref_value))))
                status = 'match' if ok else 'DIFFERS'
                shown = " or ".join(t for t in (f"abs {tol_abs:g}" if np.isfinite(tol_abs) else '', f"rel {tol_rel:g}" if np.isfinite(tol_rel) else '') if t)
            out.append({'Quantity': label, 'This run': fmt(v), 'Reference': fmt(ref_value), 'Difference': fmt(d), 'Tolerance': shown, 'Status': status})
        tab = pd.DataFrame(out)
        n = tab['Status'].value_counts().to_dict() if len(tab) else {}
        full, why = sample_is_full()                                   # record of the comparison (K17, K90)
        RUN_COMPARISONS[:] = [c for c in RUN_COMPARISONS if c['title'] != title]   # a cell executed again replaces its table
        RUN_COMPARISONS.append({'title': title, 'full_sample': full, 'sample': why,
                                **{s: int(n.get(s, 0)) for s in ('match', 'DIFFERS', 'reported', 'no reference')}})
        if check:
            _ck_comparison(check, tab, n, explained or {}, full, why)
        report(title, f"{n.get('match', 0)} match, {n.get('DIFFERS', 0)} differ, {n.get('reported', 0)} reported without tolerance, "
                      f"{n.get('no reference', 0)} without reference (independent pre-test of 20 November 2026).", table=tab)
        return tab
    except Exception as exc:
        print(f"!! comparison with the reference could not be made ({title}): {exc!r}")
        if check and not any(c['name'] == check and c['step'] is None for c in RUN_CHECKS):
            checkpoint(check, None, f"the comparison could not be made: {exc!r}"[:300], level='warn')
        return pd.DataFrame()

# -----------------------------------------------------------------------------
# CHECKPOINT HELPERS (used by Phase 1, Phase 2, Phase 4 and the last cell; they only read computed values and files)
# -----------------------------------------------------------------------------
def sample_is_full(data=None):
    """(True, '') if `data` (default: the analysis data frame df) holds every hour of the data file read in Phase 1
    (DATA_RANGE: first hour, last hour, number of hours), otherwise (False, reason). Checks against values that refer to the
    full sample (the independent pre-test) are NOT CHECKED on any other sample."""
    data = globals().get('df') if data is None else data
    ref = globals().get('DATA_RANGE')
    if data is None or ref is None:
        return False, "the data file has not been read yet (Phase 1)"
    idx = data.index
    if len(idx) == ref[2] and len(idx) and idx.is_unique and idx[0] == ref[0] and idx[-1] == ref[1]:
        return True, ''
    return False, f"the analysis sample holds {len(idx):,} of the {ref[2]:,} hours of the data file"

def _ck_comparison(name, tab, counts, explained, full, why):
    """Checkpoint (level 'warn') of one comparison with the independent pre-test: the match / DIFFERS rows; DIFFERS rows
    with a known cause (explained: {Quantity: reason}) are named as such."""
    if not len(tab):
        return checkpoint(name, None, "no row to compare", level='warn')
    differs = tab.loc[tab['Status'] == 'DIFFERS', 'Quantity'].tolist()
    known, other = {}, []
    for q in differs:
        if q in explained:
            known.setdefault(explained[q], []).append(q)
        else:
            other.append(q)
    detail = (f"{counts.get('match', 0)} match, {len(differs)} DIFFERS, {counts.get('reported', 0)} reported without tolerance, "
              f"{counts.get('no reference', 0)} without reference"
              + "".join(f"; DIFFERS explained by the {reason}: {', '.join(qs)}" for reason, qs in known.items())
              + (f"; DIFFERS without a known cause: {', '.join(other[:8])}{' ...' if len(other) > 8 else ''}" if other else ""))
    if not full:
        return checkpoint(name, None, f"{why}, while the pre-test values refer to the full sample ({detail})", level='warn')
    return checkpoint(name, not differs, detail, level='warn')

def sarimax_start_record(job_dir, job_key, start):
    """The start record written by fit_sarimax (job_dir/<job_key>__<start>.json) as a dict, or None."""
    p = os.path.join(job_dir, re.sub(r'[^A-Za-z0-9_.-]+', '_', f"{job_key}__{start}") + '.json')
    try:
        with open(p, encoding='utf-8') as fh:
            return json.load(fh)
    except Exception:
        return None

def sarimax_start_warnings(job_dir):
    """One line for the run report: the warnings stored with the start records in job_dir (key 'warnings'). A record
    without the key is reported as 'warnings not recorded', never as 'no warning'."""
    recs = []
    for name in (sorted(os.listdir(job_dir)) if os.path.isdir(job_dir) else []):
        if name.endswith('.json'):
            try:
                with open(os.path.join(job_dir, name), encoding='utf-8') as fh:
                    d = json.load(fh)
            except Exception:
                continue
            recs.append((d.get('job_key', name), d.get('start', ''), d.get('warnings')))
    where = os.path.relpath(job_dir, RESULTS_BASE)
    if not recs:
        return f"Warnings stored with the start records: no start record in {where}."
    nr = [r for r in recs if r[2] is None]; none = [r for r in recs if r[2] == []]; some = [r for r in recs if r[2]]
    parts = ([f"warnings not recorded for {len(nr)} (start estimated before warnings were stored with the start records)"] if nr else []) \
        + ([f"no warning for {len(none)}"] if none else []) \
        + ([f"warnings for {len(some)}: " + "; ".join(f"{k} '{s}': " + ", ".join(f"{w[0]} x{w[3]} ({w[2]})" for w in ws)
                                                       for k, s, ws in some)] if some else [])
    return f"Warnings stored with the {len(recs)} start records in {where}: " + "; ".join(parts) + "."

def sarimax_peak(starts):
    """(highest log-likelihood, start) over the start records of a model (the 'Starts' JSON of the result files)."""
    try:
        st = json.loads(starts) if isinstance(starts, str) else dict(starts or {})
        vals = [(float(v.get('llf')), k) for k, v in st.items() if v.get('llf') is not None and np.isfinite(float(v.get('llf')))]
    except Exception:
        return None
    return max(vals) if vals else None

def sarimax_implied_clusters(se_cd, se_adj):
    """Number of day clusters G used for the stored standard errors, recovered from their small-sample factor:
    Cluster_Day_adj = Cluster_Day * sqrt(G / (G - 1))  ->  G = r^2 / (r^2 - 1), r = Cluster_Day_adj / Cluster_Day (median
    over the parameters); NaN if not available."""
    if se_cd is None or se_adj is None:
        return np.nan
    # a parameter whose two standard errors coincide gives r^2 = 1 (division by zero) and is dropped below; the numpy
    # floating-point state is changed for these two lines of the check only (no warning of the analysis is affected)
    with np.errstate(all='ignore'):
        r2 = (np.asarray(se_adj, dtype=float) / np.asarray(se_cd, dtype=float)) ** 2
        g = r2 / (r2 - 1.0)
    g = g[np.isfinite(g) & (g > 0)]
    return float(np.median(g)) if g.size else np.nan

def sarimax_invertible(row):
    """Whether the reported point of a model is invertible: every non-seasonal MA root modulus above 1 and |ma.S.L24|
    below 1, and info['representation'] == 'invertible' when the fit record of this run is available."""
    sma = row.get('seasonal_ma')
    ok = all(float(v) > 1.0 for v in (row.get('ma_roots') or [])) and not (sma is not None and np.isfinite(sma) and abs(sma) >= 1.0)
    return bool(ok and row.get('representation') in (None, 'invertible'))

def sarimax_checkpoints(step, rows):
    """Checkpoints K10-K14 and K16 of the reported models of one step (Phase 2, Phase 4). rows: one dict per model,
    read from the files the step has written (and from the fit records of this run for the models estimated in it):
      model, kind ('SARIMAX' | 'LOGIT'), status (certified | plateau | not converged; logit: converged), within_001,
      ma_roots, seasonal_ma, representation (info key, or None), llf, peak ((log-likelihood, start) of the best start, or None),
      regressors, intercept (whether the specification gives the model a constant; None: unknown),
      n_days ({source: number of day clusters}), days_expected (calendar days of the estimation sample), se_checks."""
    sx = [r for r in rows if str(r.get('kind', 'SARIMAX')).upper() == 'SARIMAX']
    names = lambda ms: ', '.join(ms) if ms else 'none'
    fin = lambda v: v is not None and np.isfinite(float(v))
    # K10 convergence ('fail' if not converged, 'warn' for a plateau)
    bad = [r['model'] for r in rows if r['status'] not in ('certified', 'plateau', 'converged')]
    plat = [r['model'] for r in sx if r['status'] == 'plateau']
    det = (f"{sum(r['status'] == 'certified' for r in sx)} of {len(sx)} SARIMAX models certified maxima"
           + (f"; plateau (flat ridge, accepted): {names(plat)}" if plat else "") + (f"; NOT CONVERGED: {names(bad)}" if bad else "")
           + "".join(f"; {r['model']} ({str(r.get('kind')).lower()}): {r['status']}" for r in rows if r not in sx))
    checkpoint(f"K10 {step}: every reported model certified or plateau", (not bad and not plat) if rows else None,
               det if rows else "no model", level='fail' if bad else 'warn')
    # K11 reported representation invertible ('fail')
    nonv = [r['model'] for r in sx if not sarimax_invertible(r)]
    roots = [float(v) for r in sx for v in (r.get('ma_roots') or [])]
    smas = [abs(float(r['seasonal_ma'])) for r in sx if fin(r.get('seasonal_ma'))]
    peaks = [f"{r['model']} (start '{r['peak'][1]}', {r['peak'][0] - r['llf']:+.4f})" for r in sx
             if r.get('peak') and r['peak'][0] > r['llf'] + SARIMAX_TIE_TOL]
    det = ((f"NOT invertible: {names(nonv)}" if nonv else f"all {len(sx)} reported points invertible")
           + (f"; smallest non-seasonal MA root modulus {min(roots):.5f}" if roots else "")
           + (f", largest |ma.S.L24| {max(smas):.4f}" if smas else "")
           + (f"; higher log-likelihood of another start, recorded with the starts and not reported: {', '.join(peaks)}" if peaks else ""))
    checkpoint(f"K11 {step}: reported representation invertible", (not nonv) if sx else None, det, level='fail')
    # K12 agreement between the starts ('warn')
    w = [(r['model'], r.get('within_001')) for r in sx]
    few = [m for m, v in w if fin(v) and float(v) < 2]; na = [m for m, v in w if not fin(v)]
    det = ("starts within 0.01 of the reported log-likelihood: " + ", ".join(f"{m} {int(float(v))}" for m, v in w if fin(v))
           + (f"; fewer than two: {names(few)}" if few else "") + (f"; not available: {names(na)}" if na else ""))
    checkpoint(f"K12 {step}: at least two starts within 0.01 of the reported log-likelihood",
               None if len(na) == len(sx) else not few, det, level='warn')
    # K13 'const' exactly in the models whose specification has an intercept ('fail')
    has = [r['model'] for r in sx if 'const' in (r.get('regressors') or [])]
    exp = [r['model'] for r in sx if r.get('intercept')]
    unk = [r['model'] for r in sx if r.get('intercept') is None]
    wrong = [r['model'] for r in sx if r.get('intercept') is not None and (r['model'] in has) != bool(r['intercept'])]
    det = (f"'const' among the regressors of: {names(has)}; specification with an intercept: {names(exp)}"
           + (f"; DISAGREE: {names(wrong)}" if wrong else "") + (f"; specification not known: {names(unk)}" if unk else ""))
    checkpoint(f"K13 {step}: 'const' among the regressors exactly in the models with an intercept",
               None if len(unk) == len(sx) else not wrong, det, level='fail')
    # K14 day clusters = calendar days of the estimation sample ('fail')
    res, parts, src, days = [], [], set(), set()
    for r in sx:
        vals, e = {k: float(v) for k, v in (r.get('n_days') or {}).items() if fin(v)}, r.get('days_expected')
        if e is None or not vals:
            res.append(None); parts.append(f"{r['model']}: not available"); continue
        src |= set(vals); days.add(int(e))
        okm = all(abs(v - e) < 0.01 for v in vals.values()); res.append(okm)
        if not okm:
            parts.append(f"{r['model']}: " + ", ".join(f"{k} {v:,.2f}" for k, v in vals.items()) + f" vs {int(e):,} calendar days")
    det = (f"day clusters = calendar days of the estimation sample ({', '.join(f'{d:,}' for d in sorted(days)) or '-'}) for "
           f"{sum(v is True for v in res)} of {len(sx)} models (from: {'; '.join(sorted(src)) or '-'})" + "".join(f"; {p}" for p in parts))
    checkpoint(f"K14 {step}: day clusters = calendar days of the estimation sample",
               None if all(v is None for v in res) else (False not in res), det, level='fail')
    # K16 standard-error flags ('warn')
    fl = [f"{r['model']}: {r['se_checks']}" for r in sx if r.get('se_checks')]
    checkpoint(f"K16 {step}: no standard-error flag", (not fl) if sx else None,
               ("flagged: " + "; ".join(fl)) if fl else f"SE_checks empty for all {len(sx)} SARIMAX models", level='warn')

def checkpoint_summary():
    """End of the run (called at the end of the last code cell): prints all checkpoints and all recorded warnings,
    writes results/RUN_CHECKPOINTS.csv (columns step, name, status, detail) and a final section of RUN_REPORT.md, and raises
    RuntimeError after everything is written if a checkpoint FAILED. A check recorded again by a later execution of its cell
    counts once (its latest result; the earlier one stays in the CSV, step marked superseded). RUN_CHECKS is emptied by the
    setup cell, so the record covers one kernel session. Returns the table of the checkpoints."""
    title = "End of run: checkpoints and recorded warnings"
    taken = _rk_take(title)
    live = [c for c in RUN_CHECKS if not c.get('superseded')]       # a check recorded again counts once (its latest result)
    tab = pd.DataFrame([dict(c, step=f"{c['step']} (superseded by a later execution of the cell)") if c.get('superseded') else c
                        for c in RUN_CHECKS], columns=['step', 'name', 'status', 'detail'])
    path = os.path.join(RESULTS_BASE, 'RUN_CHECKPOINTS.csv')
    try:
        tab.to_csv(path, index=False, **CSV_KW)
        csv_note = []
    except Exception as exc:                     # e.g. the file is open in a spreadsheet program (Windows file lock)
        csv_note = [f"!! {path} could not be written ({exc!r}); the table is in this section.", ""]
        print(csv_note[0])
    n = {s: sum(c['status'] == s for c in live) for s in ('OK', 'WARNING', 'FAILED', 'NOT CHECKED')}
    lines = [f"Checkpoints: {n['OK']} ok, {n['WARNING']} warning, {n['FAILED']} failed, {n['NOT CHECKED']} not checked; "
             f"warnings recorded: {sum(w['count'] for w in RUN_WARNINGS):,} ({len(RUN_WARNINGS)} distinct entries).",
             f"The record covers this kernel session (setup cell executed {RUN_STARTED}). A check that needs a value of the "
             "estimation itself is NOT CHECKED when that result was loaded from the results folder; a complete record needs one "
             "kernel session that starts from an empty results folder."
             + (f" {len(RUN_CHECKS) - len(live)} check(s) superseded by a later execution of their cell (listed in the CSV, "
                "not counted)." if len(live) < len(RUN_CHECKS) else ""), ""] + csv_note
    if n['OK'] < len(live):
        lines += ["Checkpoints that are not OK (step | [status] name: detail):"]
        lines += [f"- {c['step']} | {_ck_line(c)}" for c in live if c['status'] != 'OK'] + [""]
    if RUN_WARNINGS:
        lines += ["All recorded warnings (step | category | where | location | message | count):"]
        lines += [f"- {w['step']} | {w['category']} | {w['where']} | {w['location']} | {' '.join(w['message'].split())} | x{w['count']}"
                  for w in RUN_WARNINGS]
    else:
        lines += ["No warning was recorded in this run."]
    report(title, lines, table=tab[['step', 'name', 'status']], files=[] if csv_note else [path], _taken=taken)
    if n['FAILED']:
        raise RuntimeError(f"{n['FAILED']} checkpoint(s) FAILED (results/RUN_CHECKPOINTS.csv, last section of RUN_REPORT.md): "
                           + "; ".join(c['name'] for c in live if c['status'] == 'FAILED'))
    return tab

print(f">> SARIMAX: R2 specification | profile maximum likelihood, several starts per model, up to {SARIMAX_N_JOBS} models in parallel.")
print(f">> Results folder: {'new run, manifest written' if _fresh else 'continuing the run stamped ' + str(RUN_MANIFEST.get('created'))} "
      f"| run report: {RUN_REPORT_FILE}")
print(f">> Reference values of the independent pre-test: {len(REFERENCE):,} loaded" if REFERENCE else f">> Reference file not found ({REFERENCE_FILE}): comparisons will be skipped.")
if _env_diff:
    print(">> NOTE: package versions differ from the reference environment (results can differ in the last digits; XGBoost results "
          "depend on the library version and platform):\n   " + "; ".join(f"{p} {v} (reference {r})" for p, (v, r) in _env_diff.items()))
# checkpoints of the setup: environment (K05) and specification (K06). The documented specification is the method
# tag SARIMAX_METHOD_TAG (part of the signature of every cached start), so orders and intercepts are checked against it.
checkpoint("K05 environment: package versions equal the reference environment", not _env_diff,
           f"Python {ENVIRONMENT['python']}; " + (f"all {len(REFERENCE_ENVIRONMENT)} packages equal the reference environment" if not _env_diff
                                                 else "differing: " + "; ".join(f"{p} {v} (reference {r})" for p, (v, r) in _env_diff.items())),
           level='warn')
_k06_fmt = lambda o: "(" + ",".join(str(v) for v in o) + ")"
_k06 = [(f"Phase 2 {_k06_fmt(ORDER_BASELINE)} no intercept", f"Phase 2 {_k06_fmt(ORDER_BASELINE)} no intercept" in SARIMAX_METHOD_TAG),
        (f"MCP {_k06_fmt(ORDER_MCP)} with intercept", f"MCP {_k06_fmt(ORDER_MCP)} intercept" in SARIMAX_METHOD_TAG
         and bool(SARIMAX_INTERCEPT.get('MCP'))),
        (f"a-FRR {_k06_fmt(ORDER_AFRR)} no intercept", f"a-FRR {_k06_fmt(ORDER_AFRR)} no intercept" in SARIMAX_METHOD_TAG
         and not SARIMAX_INTERCEPT.get('aFRR', True)),
        ("a constant for the MCP models only", sorted(k for k, v in SARIMAX_INTERCEPT.items() if v) == ['MCP']),
        ("no intercept in a model with d > 0", not any(SARIMAX_INTERCEPT.get(k) and o[1] > 0 for k, o in (('MCP', ORDER_MCP), ('aFRR', ORDER_AFRR)))),
        ("seasonal part (1,0,1)24 in every model", SEASONAL_BASELINE == SEASONAL_MCP == SEASONAL_AFRR == (1, 0, 1, 24))]
checkpoint("K06 design: a constant for the MCP models only; SARIMAX orders as documented (SARIMAX_METHOD_TAG)", all(ok for _, ok in _k06),
           "; ".join(t + ('' if ok else ': FAILED') for t, ok in _k06), level='stop')
report("Setup" + (" (new run)" if _fresh else " (kernel started again)"),
       [f"Data file: {os.path.basename(DATA_FILE)}, SHA-256 {DATA_SHA256}", f"Method: {SARIMAX_METHOD_TAG}",
        f"Python {ENVIRONMENT['python']} on {ENVIRONMENT['platform']} ({ENVIRONMENT['cpu_count']} logical CPUs); parallel SARIMAX jobs: {SARIMAX_N_JOBS}",
        "Packages: " + ", ".join(f"{p} {v}" for p, v in ENVIRONMENT['packages'].items()),
        ("Package versions equal the reference environment." if not _env_diff else
         "Package versions differing from the reference environment: " + "; ".join(f"{p} {v} (reference {r})" for p, (v, r) in _env_diff.items()))])

## Phase 1: Data Ingestion and Pre-Processing
Reads the raw market dataset (`Model_Data.xlsx`), establishes the temporal index, and performs initial integrity checks.

The dataset is verified before it is used: 35,064 hourly observations on the complete grid 2022-01-01 00:00 – 2025-12-31 23:00, no missing values, 1,461 calendar days, `Date` equal to the calendar day of the time index, `Month_Sin` / `Month_Cos` on a 12-month cycle (as stored in the data file; they are not recomputed), `Weekend` equal to Saturday or Sunday. A failed check stops the notebook. `day_key()` returns the calendar-day code used by the day-clustered statistics of the later phases.

In [ ]:
# =============================================================================
# PHASE 1: DATA INGESTION AND PRE-PROCESSING
# =============================================================================
import pandas as pd
import numpy as np
import sys

def ingest_and_structure_data(file_path: str, date_col: str = 'Date_Hour') -> pd.DataFrame:
    if not os.path.exists(file_path):
        print(f"!! ERROR: File not found at {file_path}")
        return None

    try:
        print(f">> Ingesting data from: {file_path}")
        df = pd.read_excel(file_path)

        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col])
            df.set_index(date_col, inplace=True)
            df.sort_index(inplace=True)
        else:
            print(f"!! WARNING: '{date_col}' not found. Time-series logic may fail.")
        
        return df

    except Exception as e:
        print(f"!! FATAL ERROR during data ingestion: {e}")
        return None

def generate_data_audit_report(df: pd.DataFrame):
    print("\n" + "="*60 + "\nDATASET AUDIT REPORT\n" + "="*60)
    print(f"Temporal Range : {df.index.min()} to {df.index.max()}")
    print(f"Total Observations : {len(df)}\n" + "-" * 60)
    print(f"{'Variable Name':<30} | {'Type':<10} | {'Missing Values'}\n" + "-" * 60)
    
    for col in df.columns:
        print(f"{col:<30} | {str(df[col].dtype):<10} | {df[col].isnull().sum()}")
    print("="*60 + "\n")

def day_key(index_like):
    """Integer code of the calendar day of every element of a time index: the cluster key of all day-clustered
    statistics. The day is always derived from the time index, never from the 'Date' column."""
    return pd.factorize(pd.DatetimeIndex(index_like).normalize())[0]

def verify_dataset(df: pd.DataFrame):
    """Structural checks of the analysis dataset; any failure stops the notebook. Month_Sin / Month_Cos are
    read from the data file (12-month cycle) and verified here; they are not recomputed."""
    grid = pd.date_range('2022-01-01 00:00', '2025-12-31 23:00', freq='h')
    on_grid = isinstance(df.index, pd.DatetimeIndex) and len(df) == len(grid) == 35064 and bool((df.index == grid).all())
    checks = [("35,064 hourly observations on the complete grid 2022-01-01 00:00 .. 2025-12-31 23:00", on_grid)]
    if on_grid:
        month = df.index.month.to_numpy()
        month_err = max(float(np.abs(df['Month_Sin'].to_numpy() - np.sin(2 * np.pi * month / 12)).max()),
                        float(np.abs(df['Month_Cos'].to_numpy() - np.cos(2 * np.pi * month / 12)).max()))
        checks += [
            ("no missing values", int(df.isnull().sum().sum()) == 0),
            ("1,461 calendar days", int(df.index.normalize().nunique()) == 1461),
            ("'Date' equals the calendar day of the time index",
             bool((pd.to_datetime(df['Date']).to_numpy() == df.index.normalize().to_numpy()).all())),
            (f"Month_Sin / Month_Cos on a 12-month cycle (largest deviation {month_err:.1e})", month_err < 1e-12),
            ("Weekend = Saturday or Sunday", bool((df['Weekend'].to_numpy() == (df.index.dayofweek >= 5).astype(int)).all())),
            ("solar generation share between 0 and 1", bool(df['Solar_Generation_Share'].between(0, 1).all())),
        ]
    for text, ok in checks:
        print(f"   [{'OK' if ok else 'FAILED'}] {text}")
    verify_dataset.last = list(checks)          # read by the checkpoints K01-K04 (_p1_checkpoints)
    failed = [text for text, ok in checks if not ok]
    if failed:
        raise RuntimeError("Dataset check failed: " + "; ".join(failed))
    return [f"[OK] {text}" for text, _ in checks]

def _p1_checkpoints(df):
    """Checkpoints K01-K04 (level 'stop': a data error stops the notebook before anything is estimated or loaded).
    Every expectation is derived from the data in memory (no typed number of hours or days); where verify_dataset() (fixed
    2022-2025 grid) checks the same property, its result is routed through the same checkpoint."""
    v = list(getattr(verify_dataset, 'last', []))
    def ver(prefix):
        r = [ok for text, ok in v if text.startswith(prefix)]
        return None if not r else bool(all(r))
    vtxt = lambda prefix: {None: 'not evaluated', True: 'OK', False: 'FAILED'}[ver(prefix)]
    idx = df.index
    # K01: hourly, strictly increasing, no duplicate, no gap between the first and the last hour
    name = "K01 data: hourly time index, strictly increasing, no duplicate, no gap"
    if not isinstance(idx, pd.DatetimeIndex) or len(idx) < 2:
        checkpoint(name, False, "the index is not a time index with at least two time stamps", level='stop')
    hour = pd.Timedelta(hours=1)
    step = idx[1:] - idx[:-1]
    dup, gaps = idx[idx.duplicated()], idx[:-1][step > hour]
    back, off = int((step < pd.Timedelta(0)).sum()), int((idx != idx.floor('h')).sum())
    span = int((idx.max() - idx.min()) // hour) + 1
    ok = len(dup) == 0 and len(gaps) == 0 and back == 0 and off == 0 and span == len(idx)
    checkpoint(name, ok, f"{len(idx):,} time stamps {idx.min()} .. {idx.max()} ({span:,} hours from the first to the last): "
               + ("step 1 h, strictly increasing, no duplicate, no gap" if ok else
                  f"{len(dup)} duplicated hour(s)" + (f" (first {dup[0]})" if len(dup) else "") + f", {len(gaps)} gap(s)"
                  + (f" (first after {gaps[0]})" if len(gaps) else "") + f", {back} step(s) backwards, {off} time stamp(s) not on the full hour")
               + f"; verify_dataset, complete 2022-2025 grid: {vtxt('35,064 hourly observations')}", level='stop')
    # K02: solar share in [0, 1] and equal to solar generation / total generation. Total generation is not a column of the
    # data file; every other generation column implies it as generation / share (they must agree with each other). The
    # check is therefore one of consistency: a denominator shared by every share column cannot be verified from this file.
    name = "K02 data: Solar_Generation_Share in [0, 1] and equal to solar generation / the total implied by the other shares"
    share_cols = [c for c in df.columns if c.endswith('_Generation_Share') and c[:-len('_Share')] in df.columns]
    others = [c for c in share_cols if c != 'Solar_Generation_Share']
    if 'Solar_Generation_Share' not in share_cols or not others:
        checkpoint(name, False, "the data file lacks Solar_Generation / Solar_Generation_Share or the other generation columns", level='stop')
    s = pd.to_numeric(df['Solar_Generation_Share'], errors='coerce')
    implied = pd.DataFrame({c: pd.to_numeric(df[c[:-len('_Share')]], errors='coerce')
                            / pd.to_numeric(df[c], errors='coerce').where(lambda x: x > 0) for c in others}, index=df.index)
    total = implied.median(axis=1)
    spread = float(implied.sub(total, axis=0).abs().div(total, axis=0).max().max())
    n_tot = int(total.notna().sum())
    dev_max = float((s - pd.to_numeric(df['Solar_Generation'], errors='coerce') / total).abs().max()) if n_tot else np.nan
    in01, ssum = bool(s.between(0, 1).all()), float(df[share_cols].sum(axis=1).max())
    ok = (in01 and np.isfinite(spread) and spread <= 1e-9 and np.isfinite(dev_max) and dev_max <= 1e-9 and ssum <= 1.0 + 1e-9
          and ver('solar generation share') is not False)
    checkpoint(name, ok, f"range {s.min():.6f} .. {s.max():.6f} ({'within' if in01 else 'NOT within'} [0, 1]); largest deviation from "
               f"Solar_Generation / total generation {dev_max:.1e} (tolerance 1e-9, {n_tot:,} hours); total generation is implied by "
               f"the {len(others)} other generation columns (generation / share), which agree to {spread:.1e} (relative); the "
               f"{len(share_cols)} generation shares sum to at most {ssum:.4f}; the denominator itself is not a column of the data file, so a "
               f"total shared by every share column is not verified here; verify_dataset: {vtxt('solar generation share')}", level='stop')
    # K03: calendar terms follow the time index; month terms on the 12-month cycle, hour terms on the 24-hour cycle
    name = "K03 data: calendar terms follow the time index (weekend flag, weekday dummies, hour, month; 12-month and 24-hour cycles)"
    dow, bad = idx.dayofweek, []
    flags = {'Weekend': (dow >= 5).astype(int), 'Hour': idx.hour, 'Month': idx.month}
    flags.update({d: (dow == k).astype(int) for k, d in enumerate(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])})
    for c, e in flags.items():
        if c in df.columns:
            nbad = int((pd.to_numeric(df[c], errors='coerce').to_numpy() != np.asarray(e)).sum())
            if nbad:
                bad.append(f"{c} disagrees with the calendar in {nbad:,} hours")
    cyc = {}
    for c, per, val in (('Month', 12, idx.month), ('Hour', 24, idx.hour)):
        for f, fn in (('Sin', np.sin), ('Cos', np.cos)):
            if f"{c}_{f}" in df.columns:
                cyc[f"{c}_{f}"] = float(np.nanmax(np.abs(pd.to_numeric(df[f"{c}_{f}"], errors='coerce').to_numpy(dtype=float)
                                                         - fn(2 * np.pi * np.asarray(val) / per))))
    bad += [f"{c} deviates from the {12 if c.startswith('Month') else 24}-{'month' if c.startswith('Month') else 'hour'} cycle by {d:.1e}"
            for c, d in cyc.items() if not d < 1e-12]
    bad += [f"{c} missing" for c in ('Weekend', 'Month_Sin', 'Month_Cos') if c not in df.columns]
    ok = not bad and ver('Weekend') is not False and ver('Month_Sin') is not False
    checkpoint(name, ok, ("; ".join(bad) + "; " if bad else
                          f"{', '.join(c for c in flags if c in df.columns)} equal the calendar of the time index; "
                          + ", ".join(f"{c} {d:.1e}" for c, d in cyc.items()) + " largest deviation from the cycle; ")
               + f"verify_dataset: weekend {vtxt('Weekend')}, month terms {vtxt('Month_Sin')}", level='stop')
    # K04: no missing value (every column of the data file is checked; the model columns are among them)
    name = "K04 data: no missing value in any column used by a model"
    na = df.isnull().sum(); na = na[na > 0]
    checkpoint(name, na.empty and ver('no missing values') is not False,
               (f"no missing value in any of the {df.shape[1]} columns of the data file (they include every model column)" if na.empty
                else "missing values: " + ", ".join(f"{c} {int(k):,}" for c, k in na.items())) + f"; verify_dataset: {vtxt('no missing values')}",
               level='stop')

# Execute Phase 1
df = ingest_and_structure_data(DATA_FILE)
if df is not None:
    print(">> Dataset checks:")
    verify_dataset.last, _verify_error = [], None
    try:
        _checks = verify_dataset(df)
    except RuntimeError as _exc:        # the checkpoints K01-K04 below name the failure first; then it is raised
        _verify_error = _exc
    _p1_checkpoints(df)
    if _verify_error is not None:
        raise _verify_error
    DATA_RANGE = (df.index[0], df.index[-1], len(df))     # complete hourly range of the data file (sample_is_full)
    generate_data_audit_report(df)
    print(">> PHASE 1 COMPLETED. Data is ready for modeling.")
else:
    sys.exit("!! SYSTEM EXIT: Data loading failed.")

# run report and comparison with the independent pre-test: sample support of the solar-share thresholds
_s, _yr = df['Solar_Generation_Share'], df.index.year
_rows = []
for _thr, _tag in ((0.20, 'gt20'), (0.34, 'gt34'), (0.45, 'gt45')):      # descriptive upper threshold 34%
    _rows.append((f"hours with solar share > {int(_thr * 100)}%", int((_s > _thr).sum()), f"ESIK.duzeltilmis.{_tag}.Saat"))
    _rows += [(f"   of which in {_y}", int(((_s > _thr) & (_yr == _y)).sum()), f"ESIK.duzeltilmis.{_tag}.{_y}") for _y in (2022, 2023, 2024, 2025)]
_rows.append(("highest solar share", float(_s.max()), "ESIK.duzeltilmis.en_yuksek_pay"))
report("Phase 1: data ingestion and checks",
       [f"{len(df):,} hourly observations, {df.index.min()} .. {df.index.max()}, {df.shape[1]} columns, {int(df.index.normalize().nunique()):,} days."] + _checks
       + [f"Solar generation share: mean {_s.mean():.6f}, standard deviation {_s.std():.6f}, maximum {_s.max():.6f}."])
compare_with_reference("Phase 1: solar-share thresholds", _rows)


## Phase 2: Comprehensive Multi-Model Estimation Engine

Estimates 9 models (6 SARIMAX + 1 Logit + 2 ancillary SARIMAX: the eight SARIMAX Models 1–6, 7.2 and 7.3 of Supplementary Table 3 and the logit Model 7.1) at the baseline specification SARIMAX(1,1,1)(1,0,1)24 to test the merit-order and volatility hypotheses. Results are cached to the `Phase 2` directory to avoid redundant computation on re-runs.

The configuration lists all nine models. The eight SARIMAX models are estimated by profile maximum likelihood (setup cell) from two AR/MA starting points – statsmodels' default and the best point of a fixed grid – plus, if the best point is not invertible, its mirror image as a further start ('mirror'); the invertible representation of the best point is reported (a non-invertible mirror image with a slightly higher log-likelihood stays in the start records; setup cell); models with the same dependent variable are cross-checked at each other's solutions. The baseline models have no intercept: with d = 1 an intercept drops out when the series is differenced. Every finished start is saved in `results/Phase_2/starts`, so an interrupted run resumes with the missing starts.

`Phase2_Convergence.csv` records for every model the log-likelihood, the convergence status (`Status`: certified, plateau or not converged; `Converged` = certified maximum), the number of starts within 0.01 log-likelihood units of the best one, the result of every start, the AR/MA root moduli, the gradient at the solution and the Ljung–Box statistics at 24 and 48 lags (`Iterations` = Kalman-filter passes for the SARIMAX models). The summary files of the SARIMAX models report `std err`, `z` and `P>|z|` from OPG standard errors, with further columns for the other standard errors (GLS, Hessian, sandwich, day-clustered sandwich, HAC); the logit summary (Model 7.1) has the statsmodels standard errors only.

In [ ]:
# =============================================================================
# PHASE 2: COMPREHENSIVE MULTI-MODEL ESTIMATION ENGINE (CACHED)
# =============================================================================
import pandas as pd
import statsmodels.api as sm
import time
import json
import os
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings('always', category=ConvergenceWarning)   # a convergence problem is never hidden in the SARIMAX cells

RESULTS_DIR_P2 = os.path.join(RESULTS_BASE, "Phase_2")
os.makedirs(RESULTS_DIR_P2, exist_ok=True)

def generate_interaction_terms(df: pd.DataFrame) -> tuple:
    data = df.copy()
    if 'Sunday' not in data.columns:
        data['Sunday'] = (data.index.dayofweek == 6).astype(int)

    data['Interaction_Solar_Sunday'] = data['Solar_Generation_Share'] * data['Sunday']
    data['Interaction_Solar_Holiday'] = data['Solar_Generation_Share'] * data.get('Public_Holiday', 0)

    hourly_cols = []
    for h in range(8, 18):
        col_name = f'Solar_Share_x_Hour_{h}'
        data[col_name] = data['Solar_Generation_Share'] * (data['Hour'] == h).astype(int)
        hourly_cols.append(col_name)

    data['Interaction_Solar_Peak'] = data['Solar_Generation_Share'] * ((data['Hour'] >= 10) & (data['Hour'] <= 15)).astype(int)
    data['Interaction_Solar_Dir_Down'] = data['Solar_Generation_Share'] * data['System_Direction_Down']
    data['Interaction_Solar_Dir_Up'] = data['Solar_Generation_Share'] * data.get('System_Direction_Up', 0)

    return data, hourly_cols

def _p2_job(config, endog, exog, starts=None):
    """One Phase 2 estimation (runs in a parallel worker)."""
    t0 = time.time()
    if config['type'] == 'LOGIT':
        res = sm.Logit(endog, sm.add_constant(exog, prepend=True)).fit(disp=False)
        ci = res.conf_int()
        table = pd.DataFrame({'coef': res.params, 'std err': res.bse, 'z': res.tvalues, 'P>|z|': res.pvalues,
                              '[0.025': ci[0], '0.975]': ci[1]})
        rv = res.mle_retvals or {}
        info = {'converged': bool(rv.get('converged', False)), 'iterations': int(rv.get('iterations', -1)),
                'llf': float(res.llf), 'aic': float(res.aic), 'bic': float(res.bic), 'convergence_warnings': 0}
    else:
        # baseline order for every model, no intercept (d = 1); profile ML from two different AR/MA starts
        # ('default', 'grid'), see fit_sarimax in the setup cell; every finished start is saved (an interrupted run resumes)
        table, info, _ = fit_sarimax(endog, exog, ORDER_BASELINE, SEASONAL_BASELINE, starts=starts,
                                     job_dir=os.path.join(RESULTS_DIR_P2, "starts"), job_key="P2_" + config['id'].replace(' ', '_'))
    return config, table, info, time.time() - t0

_P2_RUN = {'samples': {}, 'info': {}}   # estimation samples and fit records of this run (read by the checkpoints)

def run_full_analysis_suite(df: pd.DataFrame):
    df_processed, hourly_cols = generate_interaction_terms(df)
    
    base_controls = [
        'Consumption', 'Wind_Generation', 'Import_Coal_Generation',
        'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
        'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
        'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday'
    ]

    # all nine models: the eight SARIMAX models of Supplementary Table 3 and the logit Model 7.1.
    # X = the 16 core controls + the hypothesis variables of each model (Model 7.1: solar share).
    baseline_vars = ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday']
    regime_vars = ['Solar_Generation_Share', 'System_Direction_Down', 'System_Direction_Up',
                   'Interaction_Solar_Dir_Down', 'Interaction_Solar_Dir_Up']
    models_config = [
        {'id': 'Model 1', 'name': 'a-FRR Baseline', 'type': 'SARIMAX', 'y': 'a-FRR_Price', 'x_vars': base_controls + baseline_vars},
        {'id': 'Model 2', 'name': 'a-FRR Hourly', 'type': 'SARIMAX', 'y': 'a-FRR_Price', 'x_vars': base_controls + hourly_cols},
        {'id': 'Model 3', 'name': 'a-FRR Peak Stress', 'type': 'SARIMAX', 'y': 'a-FRR_Price', 'x_vars': base_controls + ['Interaction_Solar_Peak']},
        {'id': 'Model 4', 'name': 'MCP Baseline', 'type': 'SARIMAX', 'y': 'Mcp', 'x_vars': base_controls + baseline_vars},
        {'id': 'Model 5', 'name': 'MCP Hourly', 'type': 'SARIMAX', 'y': 'Mcp', 'x_vars': base_controls + hourly_cols},
        {'id': 'Model 6', 'name': 'MCP Peak Depression', 'type': 'SARIMAX', 'y': 'Mcp', 'x_vars': base_controls + ['Interaction_Solar_Peak']},
        {'id': 'Model 7.1', 'name': 'Surplus Prob (Logit)', 'type': 'LOGIT', 'y': 'System_Direction_Down', 'x_vars': ['Solar_Generation_Share'] + base_controls},
        {'id': 'Model 7.2', 'name': 'a-FRR Regime Check', 'type': 'SARIMAX', 'y': 'a-FRR_Price', 'x_vars': base_controls + ['System_Direction_Down']},
        {'id': 'Model 7.3', 'name': 'a-FRR Flexibility Trap', 'type': 'SARIMAX', 'y': 'a-FRR_Price', 'x_vars': base_controls + regime_vars},
    ]

    print(f"\n{'='*80}\nSTARTING PHASE 2 ANALYSIS: {len(models_config)} MODELS QUEUED (DIR: {RESULTS_DIR_P2})\n{'='*80}")

    # a cached summary is used only if it was produced with the current method, specification and data file
    # (SARIMAX_METHOD_TAG names all three)
    conv_path = os.path.join(RESULTS_DIR_P2, "Phase2_Convergence.csv")
    conv = pd.read_csv(conv_path) if os.path.exists(conv_path) else pd.DataFrame(
        columns=['Model_ID', 'Type', 'Converged', 'Iterations', 'LogLik', 'AIC', 'Seconds', 'Method'])
    done = set(conv.loc[conv['Method'] == SARIMAX_METHOD_TAG, 'Model_ID']) if len(conv) else set()
    jobs, data_by_model = [], {}
    for idx, config in enumerate(models_config, 1):
        model_id = config['id']
        csv_path = os.path.join(RESULTS_DIR_P2, f"{model_id.replace(' ', '_')}_summary.csv")
        print(f">> [{idx}/{len(models_config)}] {model_id}: {config['name']}")
        # Drop rows with missing values before estimation
        model_data = df_processed[[config['y']] + config['x_vars']].dropna()
        data_by_model[model_id] = (config, model_data[config['y']], model_data[config['x_vars']])
        _P2_RUN['samples'][model_id] = model_data.index               # estimation sample (checkpoint K14)
        if os.path.exists(csv_path) and model_id in done:
            print(f"   [CACHE] Results loaded from '{RESULTS_DIR_P2}'.")
            try:
                print(pd.read_csv(csv_path).to_string(index=False))
            except Exception as e:
                print(f"   !! Cache read error: {e}")
                raise
            continue
        if os.path.exists(csv_path):
            print("   cached summary was produced with another method or specification -> re-estimated.")
        jobs.append(data_by_model[model_id])

    if jobs:
        print(f"\n>> Estimating {len(jobs)} model(s), up to {min(SARIMAX_N_JOBS, len(jobs))} in parallel "
              f"(SARIMAX(1,1,1)(1,0,1)[24] by profile maximum likelihood, {len(SARIMAX_DEFAULT_STARTS)} starts each; Logit for Model 7.1)...")
        rows = [r for r in conv.to_dict('records') if r['Model_ID'] not in {j[0]['id'] for j in jobs}]
        order = {c['id']: k for k, c in enumerate(models_config)}
        fitted = {}

        def record(config, table, info, secs):
            """Writes the summary of one model and its row of Phase2_Convergence.csv."""
            model_id = config['id']
            csv_path = os.path.join(RESULTS_DIR_P2, f"{model_id.replace(' ', '_')}_summary.csv")
            table.to_csv(csv_path, index_label='', **CSV_KW)
            _P2_RUN['info'][model_id] = info                                 # fit record of this run (checkpoints)
            # three-valued status of the SARIMAX estimates: certified maximum, plateau (flat ridge) or not converged
            status = ("CONVERGED" if info['converged'] else "NOT CONVERGED") if config['type'] == 'LOGIT' else sarimax_status_text(info)
            status_class = info.get('status_class', 'converged' if info['converged'] else 'not converged')
            print("-" * 60 + f"\nRESULTS SUMMARY FOR {model_id}  [{status}, {info['iterations']} "
                  + ("iterations" if config['type'] == 'LOGIT' else "filter passes") + f", {secs / 60:.1f} min]\n" + "-" * 60)
            print(table.to_string(float_format=lambda v: f"{v:,.4f}"))
            if config['type'] != 'LOGIT':
                print("   NaN in the GLS columns: the GLS standard errors are conditional on the AR/MA parameters and are "
                      "defined for the regression coefficients only.")
                print(f"   log-likelihood {info['llf']:.4f} | AR root moduli {[round(v, 4) for v in info['AR_roots']]} | "
                      f"MA root moduli {[round(v, 4) for v in info['MA_roots']]} | max |gradient| {info['max_abs_gradient']:.3g}")
                if status_class == 'not converged':
                    warnings.warn(f"Phase 2, {model_id}: the selected estimate did NOT converge ({info['status']})", ConvergenceWarning)
                if info['se_flags']:
                    print(f"   !! standard-error checks: {info['se_flags']}")
            rows[:] = [r for r in rows if r['Model_ID'] != model_id]
            rows.append({'Model_ID': model_id, 'Type': config['type'], 'Converged': info['converged'], 'Status': status_class,
                         'Starts_within_0.01': info.get('starts_within_001', np.nan),
                         'Iterations': info['iterations'], 'LogLik': info['llf'], 'AIC': info['aic'],
                         'Best_start': info.get('best_start', ''), 'Starts': json.dumps(info.get('starts', {})),
                         # AR/MA roots, gradient at the solution and Ljung-Box statistics (SARIMAX models)
                         'AR_root_moduli': json.dumps([round(v, 5) for v in info.get('AR_roots', [])]),
                         'MA_root_moduli': json.dumps([round(v, 5) for v in info.get('MA_roots', [])]),
                         'Max_abs_gradient': info.get('max_abs_gradient', np.nan), 'Remaining_gain': info.get('remaining_gain', np.nan),
                         'LjungBox_Q24': info.get('LjungBox_Q24', np.nan), 'LjungBox_p24': info.get('LjungBox_p24', np.nan),
                         'LjungBox_Q48': info.get('LjungBox_Q48', np.nan), 'LjungBox_p48': info.get('LjungBox_p48', np.nan),
                         'SE_checks': info.get('se_flags', ''),
                         'Seconds': round(secs, 1), 'Method': SARIMAX_METHOD_TAG})
            pd.DataFrame(rows).sort_values('Model_ID', key=lambda s: s.map(order)).to_csv(conv_path, index=False, **CSV_KW)

        for config, table, info, secs in run_parallel(_p2_job, jobs):
            record(config, table, info, secs)
            if config['type'] != 'LOGIT':
                fitted[config['id']] = info

        # cross-check between models with the same dependent variable (they share the order): if the AR/MA
        # estimate of another model gives a model a higher log-likelihood than its own solution, it is re-estimated from there
        redo = sarimax_cross_check({m: (data_by_model[m][1], data_by_model[m][2], ORDER_BASELINE, SEASONAL_BASELINE, i) for m, i in fitted.items()})
        for config, table, info, secs in run_parallel(_p2_job, [data_by_model[m] + (s,) for m, s in redo.items()]):
            info['starts'] = {**fitted[config['id']]['starts'], **info['starts']}
            record(config, table, info, secs)

        conv = pd.read_csv(conv_path)
        print("\n>> Convergence: " + "; ".join(f"{s}: {', '.join(g)}" for s, g in conv.groupby('Status', sort=False)['Model_ID']))
    print("=" * 80)

# Run the analysis
run_full_analysis_suite(df)

# run report and comparison with the independent pre-test (built from the files of results/Phase_2)
_conv = pd.read_csv(os.path.join(RESULTS_DIR_P2, "Phase2_Convergence.csv"), float_precision='round_trip')
_files = [os.path.join(RESULTS_DIR_P2, f"{_m.replace(' ', '_')}_summary.csv") for _m in _conv['Model_ID']]
# checkpoints K10-K14 and K16 of the reported models, read from the files above (and from the fit records of this run
# for the models estimated in it); K17 is written with the comparison below
_ck_rows = []
for _m, _f in zip(_conv['Model_ID'], _files):
    _r, _i = _conv[_conv['Model_ID'] == _m].iloc[0], _P2_RUN['info'].get(_m, {})
    _t = pd.read_csv(_f, index_col=0, float_precision='round_trip')
    _nd = {'fit record (n_days)': _i.get('n_days'),
           'day-cluster factor of the standard errors': sarimax_implied_clusters(_t.get('std err (Cluster_Day)'), _t.get('std err (Cluster_Day_adj)'))}
    _ck_rows.append({'model': _m, 'kind': _r['Type'], 'status': str(_r['Status']), 'within_001': _r['Starts_within_0.01'],
                     'ma_roots': json.loads(_r['MA_root_moduli']) if isinstance(_r['MA_root_moduli'], str) else [],
                     'seasonal_ma': float(_t.loc['ma.S.L24', 'coef']) if 'ma.S.L24' in _t.index else np.nan,
                     'representation': _i.get('representation'), 'llf': float(_r['LogLik']),
                     'peak': (_i['peak_llf'], _i['peak_start']) if 'peak_llf' in _i else sarimax_peak(_r['Starts']),
                     'regressors': [str(_v) for _v in _t.index if not str(_v).startswith(('ar.', 'ma.')) and _v != 'sigma2'],
                     'intercept': False,                                # Phase 2: no model has an intercept (d = 1)
                     'n_days': {_k: _v for _k, _v in _nd.items() if _v is not None and np.isfinite(_v)},
                     'days_expected': int(pd.DatetimeIndex(_P2_RUN['samples'][_m]).normalize().nunique()) if _m in _P2_RUN['samples'] else None,
                     'se_checks': '' if pd.isna(_r.get('SE_checks', np.nan)) else str(_r['SE_checks'])})
sarimax_checkpoints("Phase 2", _ck_rows)
_k45_files = _files + [os.path.join(RESULTS_DIR_P2, "Phase2_Convergence.csv")]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 2: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 2: baseline models (SARIMAX(1,1,1)(1,0,1)24 without intercept; logit Model 7.1)",
       [f"{len(_conv)} models. Status: " + "; ".join(f"{s}: {', '.join(g)}" for s, g in _conv.groupby('Status', sort=False)['Model_ID']) + ".",
        sarimax_start_warnings(os.path.join(RESULTS_DIR_P2, "starts"))],
       table=_conv[['Model_ID', 'Type', 'Status', 'LogLik', 'AIC', 'Best_start', 'Starts_within_0.01', 'Iterations', 'Seconds']],
       files=_files + [os.path.join(RESULTS_DIR_P2, "Phase2_Convergence.csv")])
_rows = []
for _m, _f in zip(_conv['Model_ID'], _files):
    _t = pd.read_csv(_f, index_col=0, float_precision='round_trip')
    _k, _ll = _m.replace('Model ', 'M'), float(_conv.loc[_conv['Model_ID'] == _m, 'LogLik'].iloc[0])
    if _m == 'Model 7.1':
        _rows += [("Model 7.1 (logit): solar share coefficient", _t.loc['Solar_Generation_Share', 'coef'], "LOJ.duzeltilmis.Solar_Generation_Share"),
                  ("Model 7.1 (logit): standard error", _t.loc['Solar_Generation_Share', 'std err'], "LOJ.duzeltilmis.SE"),
                  ("Model 7.1 (logit): log-likelihood", _ll, "LOJ.duzeltilmis.LogLik")]
        continue
    _rows.append((f"{_m}: log-likelihood", _ll, f"P2.{_k}.LogLik"))
    _rows += [(f"{_m}: {_v}", _t.loc[_v, 'coef'], f"P2.{_k}.coef.{_v}") for _v in _t.index
              if str(_v).startswith(('Solar_', 'Interaction_Solar', 'System_Direction'))]
    _rows += [(f"{_m}: {_v}", _t.loc[_v, 'coef'], f"P2.{_k}.arma.{_v}") for _v in _t.index if str(_v).startswith(('ar.', 'ma.'))]
compare_with_reference("Phase 2: baseline models", _rows, check="K17 Phase 2: comparison with the independent pre-test")

## Phase 3: Parameter Justification and Optimization
Performs ACF/PACF diagnostics to confirm the `s=24` hourly seasonality (Phase 3.1). The lag-24 autocorrelations printed by the cell are also written to `results/Phase_3/Phase3_Lag24_Autocorrelation.csv` and entered, with the figure file, in the run report (`results/RUN_REPORT.md`).

**Phase 3.2 – order selection.** The SARIMAX order selection (AIC-minimising grid search, 216 combinations per model on a stratified four-week sample) was run as a preliminary script that is not part of this package. It selected SARIMAX(2,0,2)(1,0,1)24 for the MCP and SARIMAX(1,1,2)(1,0,1)24 (d = 1) for the a-FRR price. These orders are kept and fixed in Phase 4: the MCP models are estimated with an intercept, the a-FRR models (d = 1) without an intercept or drift. The a-FRR alternative in levels, SARIMAX(1,0,2)(1,0,1)24 with an intercept, is reported as a robustness row in Supplementary Table C1, which is estimated in Phase 10 (Appendix C) together with the MA root moduli.

In [ ]:
# =============================================================================
# PHASE 3: PARAMETER JUSTIFICATION AND OPTIMIZATION
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
import sys
import time
import os

# no blanket warning filter (see the setup cell)

OUTPUT_DIR = os.path.join(RESULTS_BASE, "Phase_3")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

DIAG_PLOT_PATH = os.path.join(OUTPUT_DIR, "Phase3_Seasonality_Diagnostics.png")
ACF24_PATH = os.path.join(OUTPUT_DIR, "Phase3_Lag24_Autocorrelation.csv")   # record of the printed lag-24 autocorrelations

# =============================================================================
# PART 1: SEASONALITY DIAGNOSTICS (ACF/PACF)
# =============================================================================

def perform_seasonality_diagnostics(df):
    """
    Generates ACF/PACF plots for seasonality validation.
    Titles are omitted from saved figures to meet academic standards.
    """
    print("\n" + "="*80)
    print("PHASE 3.1: SEASONALITY DIAGNOSTICS (ACF/PACF ANALYSIS)")
    print("="*80)

    try:
        # Representative weekly samples across seasons
        winter = df[(df.index >= '2022-01-10') & (df.index < '2022-01-17')]
        autumn = df[(df.index >= '2022-10-10') & (df.index < '2022-10-17')]
        spring = df[(df.index >= '2023-04-10') & (df.index < '2023-04-17')]
        summer = df[(df.index >= '2024-07-08') & (df.index < '2024-07-15')]

        subset = pd.concat([winter, autumn, spring, summer])
        print(f">> Representative subset created. Observations: {len(subset)}")
    except Exception as e:
        print(f"!! Sampling Error: {e}")
        raise   # errors stop the cell

    target_vars = ['Solar_Generation_Share', 'a-FRR_Price', 'Mcp', 'Consumption']
    fig, axes = plt.subplots(len(target_vars), 2, figsize=(16, 16))
    plt.subplots_adjust(hspace=0.4)

    print(f"{'VARIABLE':<30} | {'LAG-24 AUTOCORRELATION':<20}")
    print("-" * 60)
    acf24_rows = []   # the printed values are also written to ACF24_PATH

    for i, var in enumerate(target_vars):
        if var not in subset.columns: continue
        series = subset[var].dropna()

        plot_acf(series, lags=72, ax=axes[i, 0], title='')
        axes[i, 0].axvline(x=24, color='red', linestyle='--', alpha=0.8, label='$s=24$ Seasonality')
        axes[i, 0].legend(loc='upper right')

        plot_pacf(series, lags=72, ax=axes[i, 1], title='')
        axes[i, 1].axvline(x=24, color='red', linestyle='--', alpha=0.8, label='$s=24$ Seasonality')
        axes[i, 1].legend(loc='upper right')

        acf_vals = sm.tsa.acf(series, nlags=72)
        print(f"{var:<30} | {acf_vals[24]:.4f}")
        acf24_rows.append({'Variable': var, 'Observations': len(series), 'Lag24_Autocorrelation': float(acf_vals[24])})

    # numerical record of the table printed above (read back by the run report)
    pd.DataFrame(acf24_rows, columns=['Variable', 'Observations', 'Lag24_Autocorrelation']).to_csv(ACF24_PATH, index=False, **CSV_KW)

    plt.savefig(DIAG_PLOT_PATH, dpi=FIG_DPI_LARGE, bbox_inches='tight')  # large canvas (16 x 16 in): saved at FIG_DPI_LARGE, like the Phase 9 flowchart
    retag_png_dpi(DIAG_PLOT_PATH, FIG_DPI)  # same pixels, dpi tag FIG_DPI like every other figure
    print(f">> Diagnostic plots saved to '{DIAG_PLOT_PATH}'")
    plt.show()

# =============================================================================
# PART 2: ORDER SELECTION (PRELIMINARY SCRIPT, NOT PART OF THIS PACKAGE)
# =============================================================================
# One specification (setup cell): the labels are built from the orders of the setup cell
# (ORDER_MCP / SEASONAL_MCP / ORDER_AFRR / SEASONAL_AFRR), so they are not typed a second time here.
def _p3_order_label(order, seasonal):
    return "SARIMAX({},{},{})({},{},{}){}".format(*order, *seasonal)

SELECTED_ORDERS = {"MCP": _p3_order_label(ORDER_MCP, SEASONAL_MCP),
                   "a-FRR": _p3_order_label(ORDER_AFRR, SEASONAL_AFRR)}
ORDER_SELECTION_NOTE = (
    "The SARIMAX order selection (AIC-minimising grid search, 216 combinations per model on a stratified four-week sample) "
    "was run as a preliminary script that is not part of this package. It selected " + SELECTED_ORDERS["MCP"] + " for the MCP "
    "and " + SELECTED_ORDERS["a-FRR"] + " for the a-FRR price. These orders are kept and fixed in Phase 4 (MCP with an "
    "intercept; a-FRR, d = 1, without intercept or drift). The a-FRR alternative in levels, SARIMAX(1,0,2)(1,0,1)24 with an "
    "intercept, is reported as a robustness row in Supplementary Table C1, which is produced by the code in Phase 10 "
    "(Appendix C) together with the MA root moduli."
)

# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))

    if target_df is not None:
        global_start = time.time()

        perform_seasonality_diagnostics(target_df)

        print("\n" + "="*80)
        print("PHASE 3.2: ORDER SELECTION")
        print("="*80)
        print(">> " + ORDER_SELECTION_NOTE)
        for _market, _order in SELECTED_ORDERS.items():
            print(f"   {_market:<6}: {_order}  (fixed in Phase 4)")

        duration = (time.time() - global_start) / 60
        print("\n" + "="*80)
        print(f"PHASE 3 COMPLETED IN {duration:.2f} MINUTES.")
        print("="*80)
    else:
        print("!! Critical Error: DataFrame 'df' not found. Please execute Phase 1.")
        raise RuntimeError("DataFrame 'df' not found: execute Phase 1 first.")   # errors stop the cell

# run report and comparison with the independent pre-test
# (the pre-test reference file holds no Phase 3 quantities, so this block only records the outputs: report() alone)
_k45_files = [DIAG_PLOT_PATH, ACF24_PATH]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 3: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
_acf24 = pd.read_csv(ACF24_PATH)
report("Phase 3: seasonality diagnostics (ACF/PACF) and order selection note",
       lines=["Lag-24 autocorrelation on the representative subset (four one-week samples): "
              + ", ".join(f"{_r.Variable} {_r.Lag24_Autocorrelation:.4f}" for _r in _acf24.itertuples()) + ".",
              "Orders fixed in Phase 4 (preliminary grid search, not re-run here): "
              + "; ".join(f"{_m} {_o}" for _m, _o in SELECTED_ORDERS.items()) + ".",
              "No reference values for this phase in the independent pre-test: nothing to compare."],
       table=_acf24.round(4), files=[DIAG_PLOT_PATH, ACF24_PATH])


## Phase 4: Robust Estimation with Optimized Parameters

Re-estimates the SARIMAX models with the orders fixed in Phase 3 – SARIMAX(2,0,2)(1,0,1)24 with an intercept for the MCP and
SARIMAX(1,1,2)(1,0,1)24 without an intercept for the a-FRR price – for all eight SARIMAX models of Supplementary Table 3
(Models 1–6, 7.2 and 7.3). Saves full regression summaries, a coefficient table with standard errors, z statistics and p-values,
and diagnostic metrics (AIC, BIC, log-likelihood, convergence status, AR/MA roots, Ljung–Box statistics, gradient at the
solution and the result of every start).

**Estimation.** All models are
estimated by profile maximum likelihood (setup cell) from three AR/MA starting points: statsmodels' default, the Phase 2 AR/MA
estimates of the same model, and the best point of a fixed grid (if the best point is not invertible, its mirror
image is estimated as a further start, 'mirror'; setup cell). The Phase 2 values are only a different starting point: they
are placed into the Phase 4 parameters of the same name, and `ar.L2` / `ma.L2`, which have no Phase 2 counterpart, start at
zero; because the regression coefficients are solved by GLS a start is defined by its AR/MA values alone. The invertible
representation of the best point is reported (a non-invertible mirror image with a slightly higher log-likelihood stays in the
start records), the result of every start is saved, and models with the same dependent variable are cross-checked
at each other's solutions. Coefficients are in the original units.

**Convergence status.** `certified`: the profile Hessian at the estimate is negative definite and the predicted
remaining gain is below 1e-8 log-likelihood units. `plateau`: the a-FRR likelihood with d = 1 has a flat ridge (an MA root close
to the unit circle) on which this certificate may not be attainable in floating point; an estimate is accepted as a plateau if
a full extra search-and-polish round gains less than 1e-4 log-likelihood units, or if the Hessian is negative definite and the
predicted remaining gain is below 1e-4. Anything else is `not converged` and raises a warning. `Starts_within_0.01` counts the
starts that end within 0.01 log-likelihood units of the best one.

**Specification.** An intercept (regressor `const`) in the three MCP models; none in the five a-FRR models
(d = 1, no drift). The a-FRR model in levels, SARIMAX(1,0,2)(1,0,1)24 with an intercept, is estimated only as a robustness row
of Supplementary Table C1 (Phase 10).

**Outputs.** `Phase4_All_Coefficients.csv` holds `Std_Error`, `z` and `P_Value` based on OPG standard errors
and `Std_Error_*` / `P_Value_*` columns for `GLS` (conditional on the AR/MA parameters), `Hessian`, `Robust` (sandwich),
`Cluster_Day` (sandwich clustered by calendar day; the inference reported in Supplementary Table D1), `Cluster_Day_adj` (the
same with the factor G/(G−1)), `HAC24` and `HAC48`. `Phase4_Model_Diagnostics.csv` holds the diagnostics (`Converged` =
certified maximum, `Iterations` = Kalman-filter passes). `Phase4_LR_Tests.csv` holds the likelihood-ratio tests of the solar
coefficient in Models 1 and 4: the restricted models are estimated from the same starts, the likelihood ordering is checked in
both directions and repaired by re-estimation, and a remaining inconsistency stops the cell.
`Phase4_Baseline_vs_Optimized_Solar.csv` puts the solar coefficient of Models 1 and 4 from Phase 2 and Phase 4 side by side
(Supplementary S1.5). Every finished start is saved in `results/Phase_4/starts`, so an interrupted run resumes with the missing
starts. The models are estimated in parallel (`SARIMAX_N_JOBS`); on the full sample the cell took about 65 minutes in the reference run (8 logical CPUs, four models in parallel).

In [ ]:
# =============================================================================
# PHASE 4: ROBUST ESTIMATION WITH OPTIMIZED PARAMETERS (CACHED)
# =============================================================================
# (i) the estimation loop covers the eight SARIMAX models of Supplementary Table 3 (Models 1-6, 7.2 and 7.3);
# (ii) standard errors, z statistics and p-values are stored next to the coefficients, and
#      Table D1 (Appendix D) is generated from this file;
# (iii) per-model cache: models found in the cached output are loaded, models missing from it
#      are estimated and appended;
# (iv) profile maximum likelihood from several starts (setup cell); MCP models SARIMAX(2,0,2)(1,0,1)24 with an
#      intercept, a-FRR models SARIMAX(1,1,2)(1,0,1)24 without an intercept;
# (v) diagnostics (AR/MA roots, Ljung-Box, gradient, every start), likelihood-ratio tests of the solar
#      coefficient in Models 1 and 4, and the Phase 2 / Phase 4 comparison of Supplementary S1.5 are saved.
# =============================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2
import os
import time
import json
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings('always', category=ConvergenceWarning)   # a convergence problem is never hidden in the SARIMAX cells

OUTPUT_DIR_P4 = os.path.join(RESULTS_BASE, "Phase_4")
os.makedirs(OUTPUT_DIR_P4, exist_ok=True)

REPORT_FILE_P4 = os.path.join(OUTPUT_DIR_P4, "Phase4_Full_Estimation_Report.txt")
COEF_FILE_P4 = os.path.join(OUTPUT_DIR_P4, "Phase4_All_Coefficients.csv")
DIAG_FILE_P4 = os.path.join(OUTPUT_DIR_P4, "Phase4_Model_Diagnostics.csv")
LR_FILE_P4 = os.path.join(OUTPUT_DIR_P4, "Phase4_LR_Tests.csv")                          # likelihood-ratio tests
S15_FILE_P4 = os.path.join(OUTPUT_DIR_P4, "Phase4_Baseline_vs_Optimized_Solar.csv")      # Supplementary S1.5
P4_INFERENCE_COLS = ['Std_Error', 'z', 'P_Value']
# additional standard errors and their p-values (GLS, Hessian, sandwich, sandwich clustered by day with and
# without the small-sample factor, HAC with 24 and 48 lags); the columns above keep the OPG standard errors
P4_EXTRA_COLS = [f'{c}_{k}' for k in SARIMAX_EXTRA_SE for c in ('Std_Error', 'P_Value')]
P4_LR_TESTS = [('Model 4', 'Solar_Generation_Share'), ('Model 1', 'Solar_Generation_Share')]

def _p4_coef_frame(table: pd.DataFrame, model_id: str) -> pd.DataFrame:
    """Rows of Phase4_All_Coefficients.csv for one model, from the table returned by fit_sarimax."""
    coef_df = pd.DataFrame({'Coefficient': table['coef'], 'Std_Error': table['std err'], 'z': table['z'], 'P_Value': table['P>|z|']})
    coef_df['Model_ID'] = model_id
    for k in SARIMAX_EXTRA_SE:
        coef_df[f'Std_Error_{k}'] = table[f'std err ({k})']
        coef_df[f'P_Value_{k}'] = table[f'P>|z| ({k})']
    return coef_df

_P4_RUN = {'config': {}, 'info': {}, 'lr': {}}   # configuration and fit records of this run (read by the checkpoints)

def run_phase_4(df):
    print("\n" + "="*80 + "\nPHASE 4: ROBUST ESTIMATION PROCESS STARTED\n" + "="*80)

    # orders and intercepts of the specification of the setup cell: MCP SARIMAX(2,0,2)(1,0,1)24 with an
    # intercept (a regressor column of ones, scale factor 1); a-FRR SARIMAX(1,1,2)(1,0,1)24 without an intercept (d = 1).
    # The a-FRR model in levels, (1,0,2) with an intercept, is a robustness row of Supplementary Table C1 only (Appendix C).
    ORDER_aFRR, SEASONAL_aFRR = ORDER_AFRR, SEASONAL_AFRR

    common_controls = [
        'Consumption', 'Wind_Generation', 'Import_Coal_Generation',
        'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
        'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
        'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday'
    ]
    hourly_cols = [f'Solar_Share_x_Hour_{h}' for h in range(8, 18)]   # same columns and order as created below

    models_config = [
        {'id': 'Model 1', 'name': 'a-FRR Baseline', 'y': 'a-FRR_Price', 'params': 'aFRR', 'X': common_controls + ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday']},
        {'id': 'Model 2', 'name': 'a-FRR Hourly', 'y': 'a-FRR_Price', 'params': 'aFRR', 'X': common_controls + hourly_cols},
        # Supplementary Table 3, Model 3 (a-FRR peak stress): core controls + Interaction_Solar_Peak
        {'id': 'Model 3', 'name': 'a-FRR Peak Stress', 'y': 'a-FRR_Price', 'params': 'aFRR', 'X': common_controls + ['Interaction_Solar_Peak']},
        # Supplementary Table 3, Model 7.2 (a-FRR regime check): core controls + System_Direction_Down
        {'id': 'Model 7.2', 'name': 'a-FRR Regime Check', 'y': 'a-FRR_Price', 'params': 'aFRR', 'X': common_controls + ['System_Direction_Down']},
        {'id': 'Model 7.3', 'name': 'a-FRR Flexibility Trap', 'y': 'a-FRR_Price', 'params': 'aFRR', 'X': common_controls + ['Solar_Generation_Share', 'System_Direction_Down', 'System_Direction_Up', 'Interaction_Solar_Dir_Down', 'Interaction_Solar_Dir_Up']},
        {'id': 'Model 4', 'name': 'MCP Baseline', 'y': 'Mcp', 'params': 'MCP', 'X': common_controls + ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday']},
        {'id': 'Model 5', 'name': 'MCP Hourly', 'y': 'Mcp', 'params': 'MCP', 'X': common_controls + hourly_cols},
        # Supplementary Table 3, Model 6 (MCP peak depression): core controls + Interaction_Solar_Peak
        {'id': 'Model 6', 'name': 'MCP Peak Depression', 'y': 'Mcp', 'params': 'MCP', 'X': common_controls + ['Interaction_Solar_Peak']},
    ]
    config_by_id = {c['id']: c for c in models_config}
    _P4_RUN['config'] = config_by_id                 # read by the checkpoints
    rank = {c['id']: k for k, c in enumerate(models_config)}

    # per-model cache; results produced with another method or specification are re-estimated
    cached = None
    if os.path.exists(REPORT_FILE_P4) and os.path.exists(COEF_FILE_P4) and os.path.exists(DIAG_FILE_P4):
        _diag = pd.read_csv(DIAG_FILE_P4, float_precision='round_trip')
        if 'Method' in _diag.columns and (_diag['Method'] == SARIMAX_METHOD_TAG).all():
            # round_trip: re-reading must not alter the last digit of the stored coefficients
            cached = pd.read_csv(COEF_FILE_P4, index_col=0, float_precision='round_trip')
        else:
            print(">> Cached Phase 4 results were produced with another method or specification "
                  "-> all models are re-estimated.")
    done = [] if cached is None else list(dict.fromkeys(cached['Model_ID']))
    todo = [c for c in models_config if c['id'] not in done]
    lr_cached = None
    if cached is not None and os.path.exists(LR_FILE_P4):
        _lr = pd.read_csv(LR_FILE_P4, float_precision='round_trip')   # round_trip: rows kept from the cache are rewritten unchanged
        if 'Method' in _lr.columns and (_lr['Method'] == SARIMAX_METHOD_TAG).all():
            lr_cached = _lr
            # a cached row whose restricted estimate is a non-invertible representation is recomputed; its starts are read
            # back from results/Phase_4/starts, so nothing is re-estimated. A row whose start record cannot be read is
            # recomputed as well (its restricted model is then estimated again where starts are missing).
            def _restricted_invertible(r):
                p = os.path.join(OUTPUT_DIR_P4, "starts", re.sub(r'[^A-Za-z0-9_.-]+', '_', "P4_" + r['Model_ID'].replace(' ', '_')
                                 + f"__without_{r['Variable']}__{r['Restricted_best_start']}") + '.json')
                try:
                    with open(p, encoding='utf-8') as fh:
                        return bool(json.load(fh)['invertible'])
                except Exception as exc:
                    print(f"   [LR cache] {r['Model_ID']} without {r['Variable']}: start record {os.path.basename(p)} not readable "
                          f"({type(exc).__name__}) -> the likelihood-ratio test is recomputed")
                    return False
            lr_cached = lr_cached[[_restricted_invertible(r) for r in lr_cached.to_dict('records')]]
    # a likelihood-ratio test is (re)computed if its model is estimated in this run or its row is not in the cache
    lr_todo = [(m, v) for m, v in P4_LR_TESTS if m in {c['id'] for c in todo} or lr_cached is None
               or not ((lr_cached['Model_ID'] == m) & (lr_cached['Variable'] == v)).any()]

    if cached is not None:
        print(f">> [CACHE] Results found in '{OUTPUT_DIR_P4}' for: {', '.join(done)}.\n>> Loading cached report...\n" + "-" * 80)
        with open(REPORT_FILE_P4, "r", encoding="utf-8") as f:
            print(f.read())
        print("-" * 80)
        if not todo and not lr_todo:
            print(f">> Coefficients loaded from '{COEF_FILE_P4}'.")
            print(pd.read_csv(DIAG_FILE_P4).to_string(index=False))
            print(f"\n>> Likelihood-ratio tests loaded from '{LR_FILE_P4}'.")
            print(lr_cached.drop(columns=['Method']).to_string(index=False))
            _p4_write_s15()
            return
        if todo:
            print(f">> Models missing from the cache: {', '.join(c['id'] for c in todo)} -> estimating them now...")

    # Feature engineering (create the interaction terms if missing)
    if 'Interaction_Solar_Sunday' not in df.columns:
        df['Sunday'] = (df.index.dayofweek == 6).astype(int)
        df['Interaction_Solar_Sunday'] = df['Solar_Generation_Share'] * df['Sunday']
        df['Interaction_Solar_Holiday'] = df['Solar_Generation_Share'] * df.get('Public_Holiday', 0)
        for h in range(8, 18):
            df[f'Solar_Share_x_Hour_{h}'] = df['Solar_Generation_Share'] * (df['Hour'] == h).astype(int)
        df['Interaction_Solar_Peak'] = df['Solar_Generation_Share'] * ((df['Hour'] >= 10) & (df['Hour'] <= 15)).astype(int)
        df['Interaction_Solar_Dir_Down'] = df['Solar_Generation_Share'] * df['System_Direction_Down']
        df['Interaction_Solar_Dir_Up'] = df['Solar_Generation_Share'] * df.get('System_Direction_Up', 0)

    def design(config, drop=None):
        """Estimation sample of a model: (endog, exog, order, seasonal order); `drop` leaves one regressor out (LR tests)."""
        order, seasonal = (ORDER_MCP, SEASONAL_MCP) if config['params'] == 'MCP' else (ORDER_aFRR, SEASONAL_aFRR)
        data_mod = df[[config['y']] + config['X']].dropna()        # drop rows with missing values
        exog = data_mod[[c for c in config['X'] if c != drop]].copy()
        if SARIMAX_INTERCEPT[config['params']]:
            exog.insert(0, 'const', 1.0)
        return data_mod[config['y']], exog, order, seasonal

    def p4_starts(config):
        """AR/MA starting points: statsmodels' default, the Phase 2 (baseline-order) AR/MA estimates of the same
        model, and the best point of a fixed grid. The Phase 2 values are only a different starting point: they are placed
        into the Phase 4 parameters of the same name and terms without a Phase 2 counterpart (ar.L2, ma.L2) start at zero.
        Since the regression coefficients are solved by GLS, a start is defined by its AR/MA values alone (the Phase 2
        regression coefficients are not used)."""
        starts = {'default': 'default'}
        p2 = os.path.join(RESULTS_BASE, "Phase_2", f"{config['id'].replace(' ', '_')}_summary.csv")
        if os.path.exists(p2):
            w = pd.read_csv(p2, float_precision='round_trip')
            w = dict(zip(w[w.columns[0]].astype(str).str.strip(), pd.to_numeric(w['coef'], errors='coerce')))
            w = {k: float(v) for k, v in w.items() if k.startswith(('ar.', 'ma.')) and np.isfinite(v)}
            if w:
                starts['phase2'] = w
        if 'phase2' not in starts:
            print(f"   {config['id']}: no Phase 2 AR/MA estimates found -> the 'phase2' start is not used (run Phase 2 first).")
        starts['grid'] = 'grid'
        return starts

    start_sets = {c['id']: p4_starts(c) for c in models_config if c['id'] in {t['id'] for t in todo} | {m for m, _ in lr_todo}}
    jobs = [('fit', config, None) + design(config) + (start_sets[config['id']],) for config in todo]
    # restricted models of the likelihood-ratio tests (one regressor left out), estimated from the same starts
    jobs += [('lr', config_by_id[m], v) + design(config_by_id[m], drop=v) + (start_sets[m],) for m, v in lr_todo]
    print(f">> Estimating {len(todo)} SARIMAX model(s) and {len(lr_todo)} restricted model(s), up to {min(SARIMAX_N_JOBS, len(jobs))} in parallel "
          f"(profile maximum likelihood, {len(next(iter(start_sets.values())))} starts each; roughly 10-40 min per model on the full sample with four models in parallel)...")

    # tables of every model: cached ones as stored, new ones as they arrive
    coef_by_model = {} if cached is None else {m: cached[cached['Model_ID'] == m] for m in done}
    diag_by_model = {} if cached is None else {r['Model_ID']: r for r in pd.read_csv(DIAG_FILE_P4, float_precision='round_trip').to_dict('records')}
    fitted, restricted = {}, {}
    _P4_RUN.update(info=fitted, lr=restricted)       # fit records of this run (checkpoints)

    def store(config, table, info, text, secs, f, tag=''):
        """Report block, coefficient rows and diagnostics row of one model; the two CSV files are rewritten after every
        model (sorted in the configuration order), so an interrupted run resumes with the missing ones."""
        status = sarimax_status_text(info)                         # certified maximum, plateau or not converged
        arma = info['arma']
        f.write(f"\n{'#'*90}\nRESULTS FOR {config['id']}: {config['name']}{tag}\n{'#'*90}\n"
                f"{status} ({info['iterations']} filter passes) | log-likelihood {info['llf']:.4f} | "
                f"AIC {info['aic']:.3f} | BIC {info['bic']:.3f}\n"
                f"AR root moduli {[round(v, 5) for v in info['AR_roots']]} | MA root moduli {[round(v, 5) for v in info['MA_roots']]} | "
                f"Ljung-Box Q(24) {info['LjungBox_Q24']:.3f} (p {info['LjungBox_p24']:.4g}), Q(48) {info['LjungBox_Q48']:.3f} (p {info['LjungBox_p48']:.4g}) | "
                f"max |gradient| {info['max_abs_gradient']:.3g}\n\nCoefficients (original units; std err, z, P>|z| and the interval: OPG):\n"
                f"{table.to_string(float_format=lambda v: f'{v:.4f}')}\n\n{text}\n")
        f.flush()
        print("-" * 90 + f"\nRESULTS SUMMARY FOR {config['id']}{tag}  [{status}, {info['iterations']} filter passes, {secs / 60:.1f} min]\n" + "-" * 90)
        print(table[SARIMAX_SUMMARY_COLS].to_string(float_format=lambda v: f"{v:,.4f}"))
        print(f"   log-likelihood {info['llf']:.4f} | best start '{info['best_start']}' | AR root moduli {[round(v, 4) for v in info['AR_roots']]} | "
              f"MA root moduli {[round(v, 4) for v in info['MA_roots']]} | max |gradient| {info['max_abs_gradient']:.3g}")
        if info['status_class'] == 'not converged':
            warnings.warn(f"Phase 4, {config['id']}: the selected estimate did NOT converge ({info['status']})", ConvergenceWarning)
        if info['se_flags']:
            print(f"   !! standard-error checks: {info['se_flags']}")
        coef_by_model[config['id']] = _p4_coef_frame(table, config['id'])
        order, seasonal = (ORDER_MCP, SEASONAL_MCP) if config['params'] == 'MCP' else (ORDER_aFRR, SEASONAL_aFRR)
        diag_by_model[config['id']] = {
            'Model_ID': config['id'], 'AIC': info['aic'], 'BIC': info['bic'], 'LogLik': info['llf'],
            'Converged': info['converged'], 'Status': info['status_class'], 'Starts_within_0.01': info['starts_within_001'],
            'Iterations': info['iterations'], 'Best_start': info['best_start'],
            'Burn': info['burn'], 'Observations': info['nobs'] - info['burn'], 'N_days': info['n_days'], 'Parameters': info['k_params'],
            'Starts': json.dumps(info['starts']), 'Order': f"{order}{seasonal}",
            # diagnostics: AR/MA roots, seasonal coefficients, Ljung-Box (24 and 48 lags), gradient at the solution
            'AR_root_moduli': json.dumps([round(v, 5) for v in info['AR_roots']]),
            'MA_root_moduli': json.dumps([round(v, 5) for v in info['MA_roots']]),
            'Seasonal_AR': arma.get('ar.S.L24', np.nan), 'Seasonal_MA': arma.get('ma.S.L24', np.nan),
            'LjungBox_Q24': info['LjungBox_Q24'], 'LjungBox_p24': info['LjungBox_p24'],
            'LjungBox_Q48': info['LjungBox_Q48'], 'LjungBox_p48': info['LjungBox_p48'],
            'Max_abs_gradient': info['max_abs_gradient'], 'Remaining_gain': info['remaining_gain'],
            'SE_checks': info['se_flags'], 'Method': SARIMAX_METHOD_TAG}
        ids = sorted(coef_by_model, key=lambda m: rank.get(m, 99))
        pd.concat([coef_by_model[m] for m in ids])[['Coefficient'] + P4_INFERENCE_COLS + ['Model_ID'] + P4_EXTRA_COLS].to_csv(COEF_FILE_P4, **CSV_KW)
        pd.DataFrame([diag_by_model[m] for m in ids if m in diag_by_model]).to_csv(DIAG_FILE_P4, index=False, **CSV_KW)

    with open(REPORT_FILE_P4, "a" if cached is not None else "w", encoding="utf-8") as f:
        if cached is None:
            f.write("PHASE 4: FULL ESTIMATION RESULTS (OPTIMIZED PARAMETERS)\n" + "="*80 + "\n"
                    f"[Revision_R2] {SARIMAX_METHOD_TAG}; coefficient tables in original units.\n\n")
        for kind, config, var, table, info, text, secs in run_parallel(_p4_job, jobs):
            if kind == 'lr':
                restricted[(config['id'], var)] = info
                print(f">> restricted model for the likelihood-ratio test: {config['id']} without {var}: log-likelihood {info['llf']:.4f} "
                      f"({sarimax_status_text(info).lower()}, {secs / 60:.1f} min)")
                continue
            fitted[config['id']] = info
            store(config, table, info, text, secs, f)

        # cross-check between models with the same dependent variable (they share the order): if the AR/MA
        # estimate of another model gives a model a higher log-likelihood than its own solution, it is re-estimated from there
        redo = sarimax_cross_check({m: design(config_by_id[m]) + (i,) for m, i in fitted.items()})
        for kind, config, var, table, info, text, secs in run_parallel(_p4_job, [('fit', config_by_id[m], None) + design(config_by_id[m]) + (s,)
                                                                                 for m, s in redo.items()]):
            info['starts'] = {**fitted[config['id']]['starts'], **info['starts']}
            fitted[config['id']] = info
            store(config, table, info, text, secs, f, tag=' (re-estimated after the cross-check between models)')

        # ---- likelihood-ratio tests: LR = 2 (LogLik unrestricted - LogLik restricted), chi2 with 1 df ----
        def unrestricted(model_id):
            """Log-likelihood, AR/MA estimate and certificate of a Phase 4 model (from this run or from the cache)."""
            if model_id in fitted:
                return {'llf': fitted[model_id]['llf'], 'arma': fitted[model_id]['arma'], 'certified': fitted[model_id]['converged'],
                        'status': fitted[model_id]['status_class']}
            c = coef_by_model[model_id]['Coefficient']; d = diag_by_model[model_id]
            return {'llf': float(d['LogLik']), 'arma': {k: float(v) for k, v in c.items() if str(k).startswith(('ar.', 'ma.'))},
                    'certified': bool(d['Converged']), 'status': str(d['Status'])}

        lr_rows = [] if lr_cached is None else [r for r in lr_cached.to_dict('records') if (r['Model_ID'], r['Variable']) not in lr_todo]
        for model_id, var in lr_todo:
            config = config_by_id[model_id]
            y, X, order, seasonal = design(config); _, Xr, _, _ = design(config, drop=var)
            r, u = restricted[(model_id, var)], unrestricted(model_id)
            # (a) the restricted model, evaluated at the unrestricted AR/MA estimate, must not beat the restricted maximum;
            #     otherwise the restricted model is re-estimated from that point
            if sarimax_profile_loglike(y, Xr, order, seasonal, u['arma']) > r['llf'] + 1e-4:
                print(f"   [LR check] {model_id} without {var}: a higher log-likelihood exists near the unrestricted AR/MA estimate -> re-estimated from there")
                # through run_parallel (a single job: main process), so that the warnings of the job are counted
                _, _, _, _, r2, _, _ = next(run_parallel(_p4_job, [('lr', config, var, y, Xr, order, seasonal,
                                                                    {'unrestricted': u['arma'], 'previous best': r['arma']})]))
                r2['starts'] = {**r['starts'], **r2['starts']}; r = restricted[(model_id, var)] = r2
            # (b) the unrestricted model, evaluated at the restricted AR/MA estimate, must not beat the unrestricted maximum,
            #     and LR must not be negative; otherwise the unrestricted model is re-estimated from the restricted estimate
            check = sarimax_profile_loglike(y, X, order, seasonal, r['arma']) - u['llf']
            if not (np.isfinite(check) and check <= 1e-4 and 2.0 * (u['llf'] - r['llf']) >= -1e-6):
                print(f"   [LR check] {model_id}: likelihood ordering not consistent (unrestricted model at the restricted AR/MA estimate "
                      f"minus its maximum: {check:.6f}) -> the unrestricted model is re-estimated from the restricted estimate")
                _, _, _, table, info, text, secs = next(run_parallel(_p4_job, [('fit', config, None, y, X, order, seasonal,
                                                                                {'from restricted': r['arma'], 'previous best': u['arma']})]))
                if model_id in fitted:
                    info['starts'] = {**fitted[model_id]['starts'], **info['starts']}
                fitted[model_id] = info
                store(config, table, info, text, secs, f, tag=' (re-estimated after the likelihood-ratio check)')
                u = unrestricted(model_id)
                check = sarimax_profile_loglike(y, X, order, seasonal, r['arma']) - u['llf']
            LR = 2.0 * (u['llf'] - r['llf'])
            consistent = bool(np.isfinite(check) and check <= 1e-4 and LR >= -1e-6)
            c = coef_by_model[model_id].loc[var]
            lr_rows.append({
                'Model_ID': model_id, 'Variable': var, 'Restriction': f'{var} = 0', 'LogLik_unrestricted': u['llf'],
                'LogLik_restricted': r['llf'], 'df': 1, 'LR': LR, 'P_Value': float(chi2.sf(max(LR, 0.0), 1)),
                # for one restriction LR is asymptotically the Wald statistic with the Hessian standard error; under quasi-ML
                # it is overstated by (sandwich SE / Hessian SE)^2, hence the adjusted p-values
                'P_Value_adjusted_Robust': float(chi2.sf(max(LR, 0.0) * (c['Std_Error_Hessian'] / c['Std_Error_Robust']) ** 2, 1)),
                'P_Value_adjusted_Cluster_Day': float(chi2.sf(max(LR, 0.0) * (c['Std_Error_Hessian'] / c['Std_Error_Cluster_Day']) ** 2, 1)),
                'Coefficient': c['Coefficient'], 'Std_Error_Hessian': c['Std_Error_Hessian'], 'Std_Error_Robust': c['Std_Error_Robust'],
                'Std_Error_Cluster_Day': c['Std_Error_Cluster_Day'],
                'Unrestricted_certified_maximum': u['certified'], 'Restricted_certified_maximum': r['converged'],
                'Unrestricted_status': u['status'], 'Restricted_status': r['status_class'],
                'Restricted_best_start': r['best_start'],
                'Restricted_LogLik_by_start': json.dumps({k: round(v['llf'], 4) for k, v in r['starts'].items()}),
                'Unrestricted_at_restricted_ARMA_minus_unrestricted_LogLik': check, 'Consistent': consistent,
                'Method': SARIMAX_METHOD_TAG})
            if 'not converged' in (u['status'], r['status_class']):
                warnings.warn(f"Phase 4, LR test of {var} in {model_id}: the restricted or the unrestricted estimate did NOT converge", ConvergenceWarning)
        if lr_rows:
            lr_tab = pd.DataFrame(lr_rows)
            lr_tab = lr_tab.iloc[sorted(range(len(lr_tab)), key=lambda i: P4_LR_TESTS.index((lr_tab['Model_ID'].iat[i], lr_tab['Variable'].iat[i])))]
            lr_tab.to_csv(LR_FILE_P4, index=False, **CSV_KW)
            print("\n>> Likelihood-ratio tests (chi2, 1 degree of freedom; adjusted p-values: LR scaled by (Hessian SE / sandwich SE)^2):")
            print(lr_tab.drop(columns=['Method', 'Restricted_LogLik_by_start']).to_string(index=False))
            bad = lr_tab.loc[~lr_tab['Consistent'].astype(bool), 'Model_ID'].tolist()
            if bad:                 # checkpoint K15 (level 'fail') records it below; the final summary raises
                print("!! Likelihood-ratio consistency check failed even after re-estimation for: " + ", ".join(bad)
                      + ". The unrestricted estimate of these models is not a global maximum (checkpoint K15).")

    _p4_write_s15()
    alld = pd.read_csv(DIAG_FILE_P4)
    failed = [c['id'] for c in models_config if c['id'] not in set(alld['Model_ID'])]
    print(f"\n{'='*80}")
    print(alld.drop(columns=['Starts']).to_string(index=False))
    if failed:
        raise RuntimeError(f"PHASE 4 INCOMPLETE: {', '.join(failed)} could not be estimated (see the errors above); re-run this cell.")
    print(f"PHASE 4 COMPLETED. Outputs saved to '{OUTPUT_DIR_P4}'. Convergence: "
          + "; ".join(f"{s}: {', '.join(g)}" for s, g in alld.groupby('Status', sort=False)['Model_ID']))

def _p4_job(kind, config, var, endog, exog, order, seasonal, starts=None):
    """One Phase 4 estimation (runs in a parallel worker): kind 'fit' = a model of the configuration,
    'lr' = the restricted model of a likelihood-ratio test (regressor `var` left out; no standard errors needed)."""
    t0 = time.time()
    # every finished start is saved in results/Phase_4/starts (an interrupted run resumes with the missing starts)
    key = "P4_" + config['id'].replace(' ', '_') + ('' if kind == 'fit' else f"__without_{var}")
    table, info, text = fit_sarimax(endog, exog, order, seasonal, starts=starts, inference=(kind == 'fit'),
                                    job_dir=os.path.join(OUTPUT_DIR_P4, "starts"), job_key=key)
    return kind, config, var, table, info, text, time.time() - t0

def _p4_write_s15():
    """Supplementary S1.5: the solar coefficient of Models 1 and 4 under the baseline order (Phase 2) and the
    optimized order (Phase 4), side by side. Rebuilt from the two result folders on every run."""
    coef = pd.read_csv(COEF_FILE_P4, index_col=0, float_precision='round_trip')
    diag = pd.read_csv(DIAG_FILE_P4, float_precision='round_trip').set_index('Model_ID')
    conv_path = os.path.join(RESULTS_BASE, "Phase_2", "Phase2_Convergence.csv")
    conv = pd.read_csv(conv_path).set_index('Model_ID') if os.path.exists(conv_path) else None
    rows, var = [], 'Solar_Generation_Share'
    for model_id in ['Model 4', 'Model 1']:
        p2_path = os.path.join(RESULTS_BASE, "Phase_2", f"{model_id.replace(' ', '_')}_summary.csv")
        r4 = coef[(coef['Model_ID'] == model_id) & (coef.index == var)]
        if not os.path.exists(p2_path) or len(r4) != 1 or conv is None or model_id not in conv.index:
            continue
        p2 = pd.read_csv(p2_path, index_col=0, float_precision='round_trip'); r4 = r4.iloc[0]
        if var not in p2.index or conv.loc[model_id, 'Method'] != SARIMAX_METHOD_TAG:
            continue
        r2 = p2.loc[var]
        row = {'Model_ID': model_id, 'Variable': var, 'Phase2_Order': f"{ORDER_BASELINE}{SEASONAL_BASELINE}", 'Phase2_Intercept': False,
               'Phase2_LogLik': conv.loc[model_id, 'LogLik'], 'Phase2_Coefficient': r2['coef'], 'Phase2_Std_Error': r2['std err'],
               'Phase2_P_Value': r2['P>|z|']}
        row.update({f'Phase2_Std_Error_{k}': r2.get(f'std err ({k})', np.nan) for k in SARIMAX_EXTRA_SE})
        row.update({'Phase4_Order': diag.loc[model_id, 'Order'], 'Phase4_Intercept': bool((coef[coef['Model_ID'] == model_id].index == 'const').any()),
                    'Phase4_LogLik': diag.loc[model_id, 'LogLik'], 'Phase4_Coefficient': r4['Coefficient'], 'Phase4_Std_Error': r4['Std_Error'],
                    'Phase4_P_Value': r4['P_Value']})
        row.update({f'Phase4_Std_Error_{k}': r4[f'Std_Error_{k}'] for k in SARIMAX_EXTRA_SE})
        row['Change_pct'] = (r4['Coefficient'] - r2['coef']) / abs(r2['coef']) * 100.0
        rows.append(row)
    if rows:
        pd.DataFrame(rows).to_csv(S15_FILE_P4, index=False, **CSV_KW)
        print(f"\n>> Solar coefficient, baseline order (Phase 2) vs optimized order (Phase 4) [Supplementary S1.5] -> '{S15_FILE_P4}'")
        print(pd.DataFrame(rows).to_string(index=False))
    else:
        print("\n>> Phase 2 / Phase 4 comparison (Supplementary S1.5) not written: Phase 2 results of Models 1 and 4 with the current "
              "specification are needed (run Phase 2 first).")

# Run
if 'df' in globals() and df is not None:
    run_phase_4(df)

# run report and comparison with the independent pre-test (built from the files of results/Phase_4)
_coef = pd.read_csv(COEF_FILE_P4, index_col=0, float_precision='round_trip')
_diag = pd.read_csv(DIAG_FILE_P4, float_precision='round_trip')
_lr = pd.read_csv(LR_FILE_P4, float_precision='round_trip')
# checkpoints K10-K16 of the reported models and of the likelihood-ratio tests, read from the files above (and from the
# fit records of this run for the models estimated in it); K17 is written with the comparisons below
_dfi = generate_interaction_terms(df)[0]        # the regressors of every model (the terms of run_phase_4, Phase 2 helper)
_ck_rows = []
for _d in _diag.to_dict('records'):
    _m = _d['Model_ID']; _c = _coef[_coef['Model_ID'] == _m]; _i = _P4_RUN['info'].get(_m, {}); _cfg = _P4_RUN['config'].get(_m)
    _nd = {'N_days column': _d.get('N_days'),
           'day-cluster factor of the standard errors': sarimax_implied_clusters(_c.get('Std_Error_Cluster_Day'), _c.get('Std_Error_Cluster_Day_adj'))}
    _ck_rows.append({'model': _m, 'kind': 'SARIMAX', 'status': str(_d['Status']), 'within_001': _d['Starts_within_0.01'],
                     'ma_roots': json.loads(_d['MA_root_moduli']), 'seasonal_ma': float(_d['Seasonal_MA']),
                     'representation': _i.get('representation'), 'llf': float(_d['LogLik']),
                     'peak': (_i['peak_llf'], _i['peak_start']) if 'peak_llf' in _i else sarimax_peak(_d['Starts']),
                     'regressors': [str(_v) for _v in _c.index if not str(_v).startswith(('ar.', 'ma.')) and _v != 'sigma2'],
                     'intercept': None if _cfg is None else bool(SARIMAX_INTERCEPT[_cfg['params']]),
                     'n_days': {_k: _v for _k, _v in _nd.items() if _v is not None and np.isfinite(_v)},
                     'days_expected': None if _cfg is None else int(_dfi[[_cfg['y']] + _cfg['X']].dropna().index.normalize().nunique()),
                     'se_checks': '' if pd.isna(_d.get('SE_checks', np.nan)) else str(_d['SE_checks'])})
sarimax_checkpoints("Phase 4", _ck_rows)
# K15: restricted and unrestricted estimate of every likelihood-ratio test invertible, consistency flag True ('fail')
_inv, _k15 = {_r['model']: sarimax_invertible(_r) for _r in _ck_rows}, []
for _r in _lr.to_dict('records'):
    _ri = _P4_RUN['lr'].get((_r['Model_ID'], _r['Variable']))
    if _ri is not None:
        _rinv = _ri.get('representation') == 'invertible'
    else:
        _rec = sarimax_start_record(os.path.join(OUTPUT_DIR_P4, "starts"), "P4_" + _r['Model_ID'].replace(' ', '_') + f"__without_{_r['Variable']}",
                                    _r['Restricted_best_start'])
        _rinv = None if _rec is None or 'invertible' not in _rec else bool(_rec['invertible'])
    _k15.append((_r['Model_ID'], _r['Variable'], _inv.get(_r['Model_ID']), _rinv, bool(_r['Consistent']), _r['Restricted_best_start']))
_k15_txt = {True: 'invertible', False: 'NOT invertible', None: 'not available'}
_k15_all = [v for t in _k15 for v in t[2:5]]
checkpoint("K15 Phase 4: likelihood-ratio tests (restricted and unrestricted estimates invertible, consistency flag True)",
           None if not _k15 else (False if False in _k15_all else (None if None in _k15_all else True)),
           "; ".join(f"{m} ({v} = 0): unrestricted {_k15_txt[a]}, restricted {_k15_txt[b]} (start '{s}'), consistent {c}" for m, v, a, b, c, s in _k15)
           or "no likelihood-ratio test", level='fail')
_k45_files = [REPORT_FILE_P4, COEF_FILE_P4, DIAG_FILE_P4, LR_FILE_P4, S15_FILE_P4]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 4: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 4: optimized models (MCP SARIMAX(2,0,2)(1,0,1)24 with intercept; a-FRR SARIMAX(1,1,2)(1,0,1)24 without intercept)",
       ["Status: " + "; ".join(f"{s}: {', '.join(g)}" for s, g in _diag.groupby('Status', sort=False)['Model_ID']) + ".",
        "Likelihood-ratio tests of the solar coefficient: " + "; ".join(f"{r.Model_ID} LR = {r.LR:.2f} (p {'< 1e-300' if r.P_Value < 1e-300 else f'= {r.P_Value:.3g}'})" for r in _lr.itertuples()) + ".",
        sarimax_start_warnings(os.path.join(OUTPUT_DIR_P4, "starts"))],
       table=_diag[['Model_ID', 'Order', 'Status', 'LogLik', 'AIC', 'BIC', 'Best_start', 'Starts_within_0.01', 'Burn', 'N_days']],
       files=[REPORT_FILE_P4, COEF_FILE_P4, DIAG_FILE_P4, LR_FILE_P4, S15_FILE_P4])
_hyp = ('Solar_', 'Interaction_Solar', 'System_Direction')
_rows, _se_rows = [], []
for _m in _diag['Model_ID']:
    _k, _c, _d = _m.replace('Model ', 'M'), _coef[_coef['Model_ID'] == _m], _diag[_diag['Model_ID'] == _m].iloc[0]
    _rows += [(f"{_m}: log-likelihood", _d['LogLik'], f"{_k}.tani.LogLik"), (f"{_m}: AIC", _d['AIC'], f"{_k}.tani.AIC")]
    for _v in _c.index:
        if str(_v).startswith(_hyp):
            _rows += [(f"{_m}: {_v}, coefficient", _c.loc[_v, 'Coefficient'], f"{_k}.coef.{_v}"),
                      (f"{_m}: {_v}, day-clustered SE", _c.loc[_v, 'Std_Error_Cluster_Day'], f"{_k}.se.Cluster_Day.{_v}")]
        elif str(_v).startswith(('ar.', 'ma.')):
            _rows.append((f"{_m}: {_v}", _c.loc[_v, 'Coefficient'], f"{_k}.arma.{_v}"))
    if _m in ('Model 1', 'Model 4'):               # every kind of standard error of the solar coefficient
        _se_rows += [(f"{_m}: solar share, {_kind} standard error", _c.loc['Solar_Generation_Share', _col], f"{_k}.se.{_ref_kind}.Solar_Generation_Share")
                     for _kind, _col, _ref_kind in [('OPG', 'Std_Error', 'OPG')] + [(_e, f'Std_Error_{_e}', 'Cluster_Day_duz' if _e == 'Cluster_Day_adj' else _e)
                                                                                   for _e in SARIMAX_EXTRA_SE]]
for _r in _lr.itertuples():
    _k = _r.Model_ID.replace('Model ', 'M')
    _rows += [(f"LR test, {_r.Model_ID}: restricted log-likelihood", _r.LogLik_restricted, f"LR.{_k}.LogLik_restricted"),
              (f"LR test, {_r.Model_ID}: LR statistic", _r.LR, f"LR.{_k}.LR")]
# a seasonal MA coefficient that equals the reciprocal of the pre-test value is the mirror image of the same
# model (same autocovariances); the comparison below then shows DIFFERS for that model's ma.S.L24, log-likelihood and AIC
_mirror_notes = [f"{_lab.split(':')[0]}: the pre-test value of ma.S.L24 ({REFERENCE[_id][0]:.4f}) is the reciprocal of "
                 f"this run's {float(_val):.4f}. The pre-test reports the non-invertible mirror image of the same model "
                 "(higher log-likelihood through the initialisation), this run the invertible representation (setup cell)."
                 for _lab, _val, _id in _rows if str(_lab).endswith('ma.S.L24') and _id in REFERENCE
                 and abs(float(_val) - REFERENCE[_id][0]) > 1e-3 and abs(float(_val) * REFERENCE[_id][0] - 1.0) < 1e-3]
if _mirror_notes:
    report("Phase 4: rows that differ from the pre-test (mirror image of the seasonal MA term)", _mirror_notes)
# K17: DIFFERS rows of a model whose pre-test value of ma.S.L24 is the mirror image (see _mirror_notes) are named as such
_mirror_models = {_n.split(':')[0] for _n in _mirror_notes}
_explained = {_lab: 'mirror image of the seasonal MA term in the pre-test' for _lab, _, _ in _rows
              if str(_lab).split(':')[0] in _mirror_models and str(_lab).endswith(('log-likelihood', 'AIC', 'ma.S.L24'))}
compare_with_reference("Phase 4: optimized models", _rows, check="K17 Phase 4: comparison with the independent pre-test (optimized models)",
                       explained=_explained)
compare_with_reference("Phase 4: standard errors of the solar coefficient (Models 1 and 4)", _se_rows,
                       check="K17 Phase 4: comparison with the independent pre-test (standard errors of the solar coefficient)")

## Phase 5: Advanced Visualization (SARIMAX Results)


In [ ]:
# =============================================================================
# PHASE 5: ADVANCED VISUALIZATION (SARIMAX) - FULLY DYNAMIC EXPORT MODE
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# no blanket warning filter (see the setup cell)

# Output directory configuration
# RESULTS_BASE is set once by the setup cell; it is not detected again here
OUTPUT_DIR_P5 = os.path.join(RESULTS_BASE, "Phase_5")
COEF_FILE_P4 = os.path.join(RESULTS_BASE, "Phase_4", "Phase4_All_Coefficients.csv")
os.makedirs(OUTPUT_DIR_P5, exist_ok=True)

# Academic visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (12, 7),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def get_dynamic_coefficients(csv_path: str, model_id: str) -> dict:
    """
    Reads hourly Solar_Share_x_Hour coefficients from the Phase 4 CSV output 
    for a given model and returns them as a dictionary.
    
    Args:
        csv_path (str): Path to the Phase 4 coefficient file.
        model_id (str): Model ID to extract (e.g. 'Model 2', 'Model 5').
        
    Returns:
        dict: {hour (int): coefficient (float)} dictionary.
    """
    coeffs = {}
    try:
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"Coefficient file not found: {csv_path}")
            
        df_coef = pd.read_csv(csv_path, index_col=0)
        
        # Filter coefficients for the target model
        model_data = df_coef[df_coef['Model_ID'] == model_id]
        
        if model_data.empty:
            raise ValueError(f"Model '{model_id}' not found in coefficient file.")
            
        # Extract hourly coefficients from hour 8 to 17
        for h in range(8, 18):
            col_name = f'Solar_Share_x_Hour_{h}'
            if col_name in model_data.index:
                val = model_data.loc[col_name, 'Coefficient']
                # Take first value if multiple matches (duplicate index guard)
                if isinstance(val, pd.Series):
                    val = val.iloc[0]
                coeffs[h] = float(val)
                
    except Exception as e:
        print(f"!! COEFFICIENT READ ERROR ({model_id}): {e}")
        raise   # errors stop the cell (a missing Phase 4 coefficient file is named above)
        
    return coeffs

def run_phase_5_final(df: pd.DataFrame):
    """
    Visualizes market profiles and SARIMAX coefficients.
    Reads the coefficients dynamically from the Phase 4 outputs (no hardcoded values).
    """
    print(f">> Phase 5: Generating dynamic publication-ready figures to '{OUTPUT_DIR_P5}'...")

    # df['Date'] is not modified here; the 2025 subset below is selected from the time index

    # =========================================================================
    # FIGURE 1 & 2: DAILY MARKET PROFILE
    # =========================================================================
    h_sum = df.groupby('Hour').agg(
        M_Sol=('Solar_Generation', 'mean'),
        Q25=('Solar_Generation', lambda x: x.quantile(0.25)),
        Q75=('Solar_Generation', lambda x: x.quantile(0.75)),
        M_MCP=('Mcp', 'mean'),
        M_aFRR=('a-FRR_Price', 'mean')
    ).reset_index()

    def plot_profile(data, title, filename, panel=None):
        fig, ax1 = plt.subplots(figsize=(12, 6))

        ax1.set_xlabel('Hour of the Day', fontweight='bold')
        ax1.set_ylabel('Average Generation (MWh)', color='teal', fontweight='bold')
        ax1.plot(data['Hour'], data['M_Sol'], color='teal', linewidth=2.5, label='Solar Generation')
        # [Revision_R2_Reviewer4_C9] the shaded band is data (inter-quartile range), so it is kept,
        # drawn lighter and named in the legend.
        ax1.fill_between(data['Hour'], data['Q25'], data['Q75'], color='teal', alpha=0.12,
                         linewidth=0, label='Solar Generation (IQR)')
        ax1.tick_params(axis='y', labelcolor='teal')

        ax2 = ax1.twinx()
        ax2.set_ylabel('Average MCP (TL/MWh)', color='darkorange', fontweight='bold')
        ax2.plot(data['Hour'], data['M_MCP'], color='darkorange', linewidth=2.5, label='MCP')
        ax2.tick_params(axis='y', labelcolor='darkorange')

        # [Revision_Review_Referee_2.7] the a-FRR capacity price (TL/MW) is not commensurable with the
        # MCP (TL/MWh): it gets its own right-hand axis instead of sharing the MCP axis.
        ax3 = ax1.twinx()
        ax3.spines['right'].set_position(('axes', 1.09))
        ax3.spines['right'].set_visible(True)
        ax3.set_ylabel('Average a-FRR Capacity Price (TL/MW)', color='crimson', fontweight='bold')
        ax3.plot(data['Hour'], data['M_aFRR'], color='crimson', linestyle='--', linewidth=2.5, label='a-FRR Price')
        ax3.tick_params(axis='y', labelcolor='crimson')
        ax3.grid(False)

        lines, labels = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        lines3, labels3 = ax3.get_legend_handles_labels()
        ax3.legend(lines + lines2 + lines3, labels + labels2 + labels3, loc='upper center', bbox_to_anchor=(.5, 1.08), ncol=4, frameon=False)

        # [Revision_R2_M6] panel label matching the caption: (a) full period, (b) 2025
        if panel:
            ax1.text(-0.075, 1.04, panel, transform=ax1.transAxes, fontsize=16,
                     fontweight='bold', va='bottom', ha='left')

        plt.xticks(np.arange(0, 24, 1))
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR_P5, filename), dpi=FIG_DPI, bbox_inches='tight')
        plt.show()

    plot_profile(h_sum, "Full Dataset", 'Phase_5_Figure_1_Daily_Market_Profile.png', panel='(a)')

    if isinstance(df.index, pd.DatetimeIndex):   # the year comes from the time index, not from the 'Date' column
        print(">> Generating 2025 subset profile...")
        df_2025 = df[df.index.year == 2025].copy()
        if not df_2025.empty:
            h_sum_25 = df_2025.groupby('Hour').agg(
                M_Sol=('Solar_Generation', 'mean'),
                Q25=('Solar_Generation', lambda x: x.quantile(0.25)),
                Q75=('Solar_Generation', lambda x: x.quantile(0.75)),
                M_MCP=('Mcp', 'mean'),
                M_aFRR=('a-FRR_Price', 'mean')
            ).reset_index()
            plot_profile(h_sum_25, "2025", 'Phase_5_Figure_2_Daily_Market_Profile.png', panel='(b)')
    # the 2025 profile (manuscript Figure 1b) must exist and must not be an empty file
    _fig2 = os.path.join(OUTPUT_DIR_P5, 'Phase_5_Figure_2_Daily_Market_Profile.png')
    if os.path.exists(_fig2) and os.path.getsize(_fig2) > 0:
        print(f">> 2025 profile saved: '{_fig2}' ({os.path.getsize(_fig2):,} bytes).")
    else:
        print(f"!! WARNING: '{_fig2}' is missing or empty (no 2025 observations in the data?).")

    # =========================================================================
    # FIGURE 4 & 6: DYNAMIC SARIMAX HOURLY IMPACT COEFFICIENTS
    # =========================================================================
    print(f">> Extracting dynamic coefficients from {COEF_FILE_P4}...")
    
    # Dynamic coefficient extraction for Model 5 (MCP) and Model 2 (a-FRR)
    ptf_coeffs = get_dynamic_coefficients(COEF_FILE_P4, 'Model 5')
    sfk_coeffs = get_dynamic_coefficients(COEF_FILE_P4, 'Model 2')

    configs = []
    if ptf_coeffs:
        configs.append((ptf_coeffs, 'Phase_5_Figure_4_Hourly_Merit_Order_Effect_MCP.png', 'mako', 'SARIMAX Impact on MCP (TL/MWh)'))
    if sfk_coeffs:
        configs.append((sfk_coeffs, 'Phase_5_Figure_6_Hourly_Volatility_Effect_aFRR.png', 'rocket_r', 'SARIMAX Impact on a-FRR (TL/MW)'))  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW)

    if not configs:
        print("!! WARNING: Could not read dynamic coefficients. Bar plots skipped.")
    else:
        for coeffs, filename, pal, y_label in configs:
            plt.figure(figsize=(10, 5))
            ser = pd.Series(coeffs)
            # palette through an explicit hue (seaborn >= 0.13 deprecates palette without hue; same colours)
            sns.barplot(x=ser.index, y=ser.values, hue=ser.index.astype(str), hue_order=[str(h) for h in sorted(ser.index)],
                        palette=pal, legend=False)
            
            plt.axhline(0, color='black', linewidth=1.2)
            plt.xlabel('Hour of the Day', fontweight='bold')
            plt.ylabel(y_label, fontweight='bold')
            plt.tight_layout()

            plt.savefig(os.path.join(OUTPUT_DIR_P5, filename), dpi=FIG_DPI, bbox_inches='tight')
            plt.show()

    print(f"\n>> Phase 5 completed dynamically. Outputs saved to '{OUTPUT_DIR_P5}'.")

# --- Execution ---
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_phase_5_final(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found in memory. Please execute Phase 1 first.")
        raise RuntimeError("DataFrame 'df' not found: execute Phase 1 first.")   # errors stop the cell (no report from earlier figures)

# run report and comparison with the independent pre-test
# (figures only; the bar figures show the hourly Solar_Share_x_Hour coefficients of Model 5 and Model 2 stored by Phase 4)
_p5_files = [os.path.join(OUTPUT_DIR_P5, _f) for _f in (
    'Phase_5_Figure_1_Daily_Market_Profile.png', 'Phase_5_Figure_2_Daily_Market_Profile.png',
    'Phase_5_Figure_4_Hourly_Merit_Order_Effect_MCP.png', 'Phase_5_Figure_6_Hourly_Volatility_Effect_aFRR.png')]
_p5_bars = pd.DataFrame({_m: get_dynamic_coefficients(COEF_FILE_P4, _m) for _m in ('Model 5', 'Model 2')}).rename_axis('Hour')
_k45_files = _p5_files + [COEF_FILE_P4]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 5: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 5: daily market profiles and hourly SARIMAX coefficients (figures)",
       lines=[f"Bar figures: hourly Solar_Share_x_Hour coefficients (hours 8-17) read from '{COEF_FILE_P4}'.",
              f"Model 5 (MCP): {_p5_bars['Model 5'].min():,.0f} to {_p5_bars['Model 5'].max():,.0f} TL/MWh per unit of solar share.",
              f"Model 2 (a-FRR): {_p5_bars['Model 2'].min():,.0f} to {_p5_bars['Model 2'].max():,.0f} TL/MW per unit of solar share."],
       table=_p5_bars.reset_index(), files=_k45_files)
compare_with_reference("Phase 5: hourly coefficients drawn in the bar figures (Phase 4 file)",
                       [(f"{_m}: Solar_Share_x_Hour_{_h}", _p5_bars.loc[_h, _m], f"{_m.replace('Model ', 'M')}.coef.Solar_Share_x_Hour_{_h}")
                        for _m in ('Model 5', 'Model 2') for _h in _p5_bars.index],
                       check="K17 Phase 5: comparison with the independent pre-test")

## Phase 6: Non-Linear Analysis (XGBoost Cross-Checks)

Partial-dependence analysis (6.1), the flexibility-trap interaction simulation (6.2) and the out-of-sample validation of these models (6.4; the number 6.3 is not used, so that the heading matches the R1 output files `Phase_6_4_*`).

### Phase 6.1: Partial Dependence Analysis


In [ ]:
# =================================================================================================
# PHASE 6.1: NON-LINEAR IMPACT ANALYSIS (XGBOOST) - REVISED (DECILE LINES REMOVED)
# =================================================================================================
# [Revision_Review_Referee_2.7] Axis labels: PartialDependenceDisplay.from_estimator(ax=...) replaces the
# Axes it is given, so labels set on ax5/ax7/ax1 never reached the saved figures (sklearn defaults
# 'Partial dependence' / 'Solar_Generation_Share' were shown). Labels are now set on disp.axes_[0, 0]
# and carry the price units (MCP: TL/MWh; a-FRR capacity price: TL/MW). The PDP grid values are also
# exported (Phase_6_1_PDP_Values.csv) so the curves can be compared numerically across re-runs.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.inspection import PartialDependenceDisplay
import os
import warnings

# no blanket warning filter (see the setup cell)

OUTPUT_DIR_P6 = os.path.join(RESULTS_BASE, "Phase_6")
if not os.path.exists(OUTPUT_DIR_P6):
    os.makedirs(OUTPUT_DIR_P6)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def check_phase6_plots():
    """Checks if the required individual plot files already exist."""
    fig_5 = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_5_PDP_Merit_Order_and_Volatility.png')
    fig_7 = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_7_PDP_Merit_Order_and_Volatility.png')
    fig_8 = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_8_Surplus_Probability_and_Distribution.png')
    pdp_csv = os.path.join(OUTPUT_DIR_P6, 'Phase_6_1_PDP_Values.csv')   # [Revision] the numerical record is part of the cache
    return os.path.exists(fig_5) and os.path.exists(fig_7) and os.path.exists(fig_8) and os.path.exists(pdp_csv)

def run_phase_6_1_final(df):
    print(f"PHASE 6.1: ML Analysis (Directory: '{OUTPUT_DIR_P6}')...")
    print("="*80)

    if check_phase6_plots():
        print(f">> [CACHE] Existing analyses found in '{OUTPUT_DIR_P6}'.")
        print(">> Skipping XGBoost training, loading saved plots...\n")

        fig_5_path = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_5_PDP_Merit_Order_and_Volatility.png')
        fig_7_path = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_7_PDP_Merit_Order_and_Volatility.png')
        fig_8_path = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_8_Surplus_Probability_and_Distribution.png')

        plt.figure(figsize=(10, 6)); plt.imshow(plt.imread(fig_5_path)); plt.axis('off')
        plt.title("Phase 6.1 - Figure 5 (Preview)", fontsize=14); plt.show()

        plt.figure(figsize=(10, 6)); plt.imshow(plt.imread(fig_7_path)); plt.axis('off')
        plt.title("Phase 6.1 - Figure 7 (Preview)", fontsize=14); plt.show()

        plt.figure(figsize=(16, 6)); plt.imshow(plt.imread(fig_8_path)); plt.axis('off')
        plt.title("Phase 6.1 - Figure 8 (Preview)", fontsize=14); plt.show()

        print(f">> Cached data from '{OUTPUT_DIR_P6}' loaded successfully.")
        print(">> To regenerate, delete the PNG files and re-run this cell.")
        return

    print(">> No cached results found. Training XGBoost models...")

    df_ml = df.copy()
    if df_ml['Solar_Generation_Share'].max() <= 1.0:
        df_ml['Solar_Generation_Share'] *= 100

    features_core = ['Solar_Generation_Share', 'Consumption', 'Wind_Generation', 'Price_Cap', 'Hour_Sin', 'Hour_Cos']
    features_with_mcp = features_core + ['Mcp']

    m_a = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(df_ml[features_core], df_ml['Mcp'])
    m_b = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(df_ml[features_with_mcp], df_ml['a-FRR_Price'])
    m_c = xgb.XGBClassifier(n_estimators=200, random_state=42).fit(df_ml[features_core], df_ml['System_Direction_Down'])

    # FIGURE 5: MCP MERIT ORDER EFFECT
    print(">> Plotting Figure 4 (MCP; file Phase_6_Figure_5, R1 number)...")
    fig5, ax5 = plt.subplots(figsize=(10, 6))
    disp5 = PartialDependenceDisplay.from_estimator(
        m_a, df_ml[features_core], ['Solar_Generation_Share'],
        ax=ax5, line_kw={"color": "orange", "linewidth": 3}
    )
    # Remove decile markers for cleaner output
    for vlines in disp5.deciles_vlines_:
        for v in vlines:
            v.set_visible(False)
    _ax5 = disp5.axes_[0, 0]   # [Revision_Review_Referee_2.7] the Axes actually drawn on
    _ax5.set_xlabel('Solar Generation Share (%)')
    _ax5.set_ylabel('Partial dependence of MCP (TL/MWh)')
    plt.tight_layout()
    save_path_fig5 = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_5_PDP_Merit_Order_and_Volatility.png')
    plt.savefig(save_path_fig5, dpi=FIG_DPI, bbox_inches='tight')
    plt.title("Phase 6.1 - Figure 5: MCP Merit-Order Effect (Preview Only)", fontsize=14)
    plt.show()

    # FIGURE 7: a-FRR VOLATILITY EFFECT
    print(">> Plotting Figure 6 (a-FRR; file Phase_6_Figure_7, R1 number)...")
    fig7, ax7 = plt.subplots(figsize=(10, 6))
    disp7 = PartialDependenceDisplay.from_estimator(
        m_b, df_ml[features_with_mcp], ['Solar_Generation_Share'],
        ax=ax7, line_kw={"color": "crimson", "linewidth": 3}
    )
    for vlines in disp7.deciles_vlines_:
        for v in vlines:
            v.set_visible(False)
    _ax7 = disp7.axes_[0, 0]   # [Revision_Review_Referee_2.7]
    _ax7.set_xlabel('Solar Generation Share (%)')
    _ax7.set_ylabel('Partial dependence of a-FRR price (TL/MW)')
    plt.tight_layout()
    save_path_fig7 = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_7_PDP_Merit_Order_and_Volatility.png')
    plt.savefig(save_path_fig7, dpi=FIG_DPI, bbox_inches='tight')
    plt.title("Phase 6.1 - Figure 7: a-FRR Volatility Effect (Preview Only)", fontsize=14)
    plt.show()

    # FIGURE 8: SURPLUS ANALYSIS
    print(">> Plotting Figure 7 (Surplus Analysis; file Phase_6_Figure_8, R1 number)...")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    disp8 = PartialDependenceDisplay.from_estimator(
        m_c, df_ml[features_core], ['Solar_Generation_Share'],
        ax=ax1, line_kw={"color": "blue", "linewidth": 3}
    )
    for vlines in disp8.deciles_vlines_:
        for v in vlines:
            v.set_visible(False)
    _ax1 = disp8.axes_[0, 0]   # [Revision_Review_Referee_2.7]
    _ax1.set_xlabel('Solar Generation Share (%)')
    _ax1.set_ylabel('Predicted probability of energy surplus')

    # palette through an explicit hue (seaborn >= 0.13 deprecates palette without hue; same colours) and
    # labels set together with their fixed positions (set_ticklabels alone warns: no FixedLocator)
    sns.boxplot(x=df_ml['System_Direction_Down'], y=df_ml['Solar_Generation_Share'], hue=df_ml['System_Direction_Down'].astype(str),
                hue_order=['0', '1'], palette="coolwarm", legend=False, ax=ax2)
    ax2.set_xticks([0, 1], ['Balanced/Deficit', 'Surplus'])
    ax2.set_xlabel('System State')
    ax2.set_ylabel('Solar Generation Share (%)')
    plt.tight_layout()
    save_path_surplus = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_8_Surplus_Probability_and_Distribution.png')
    plt.savefig(save_path_surplus, dpi=FIG_DPI, bbox_inches='tight')
    plt.suptitle("Phase 6.1 - Figure 8: Surplus Analysis (Preview Only)", fontsize=16)
    plt.show()

    # [Revision_Review_Referee_2.7] export the PDP grid values (numerical record of Figures 4, 6, 7-left)
    def _pd_xy(disp):
        r = disp.pd_results[0]
        grid = r['grid_values'][0] if 'grid_values' in r else r['values'][0]
        return np.asarray(grid), np.asarray(r['average'][0])
    _rows = []
    for _name, _disp in [('MCP (Figure 4)', disp5), ('a-FRR price (Figure 6)', disp7), ('P(surplus) (Figure 7, left)', disp8)]:
        _x, _y = _pd_xy(_disp)
        _rows += [{'Figure': _name, 'Solar_Generation_Share_pct': float(a), 'Partial_dependence': float(b)} for a, b in zip(_x, _y)]
    _pdp_path = os.path.join(OUTPUT_DIR_P6, 'Phase_6_1_PDP_Values.csv')
    pd.DataFrame(_rows).to_csv(_pdp_path, index=False, **CSV_KW)
    print(f">> PDP grid values saved to '{_pdp_path}'.")

    print(f"\n>> All ML analyses and figures saved to '{OUTPUT_DIR_P6}'.")

if __name__ == "__main__":
    if 'df' in locals():
        run_phase_6_1_final(df)
    elif 'df' in globals():
        run_phase_6_1_final(globals()['df'])

# run report and comparison with the independent pre-test
_pdp_file = os.path.join(OUTPUT_DIR_P6, 'Phase_6_1_PDP_Values.csv')
_pdp = pd.read_csv(_pdp_file, float_precision='round_trip')
_pdp_ids = {'MCP (Figure 4)': 'MCP_Figure_4', 'a-FRR price (Figure 6)': 'aFRR_price_Figure_6',
            'P(surplus) (Figure 7, left)': 'P_surplus_Figure_7_left'}
# checkpoint K41: the three partial-dependence curves are in the file and all their values are finite
_k41_bad = [_f for _f in _pdp_ids if not ((_pdp['Figure'] == _f).any() and np.isfinite(
    _pdp.loc[_pdp['Figure'] == _f, ['Solar_Generation_Share_pct', 'Partial_dependence']].to_numpy(dtype=float)).all())]
checkpoint("K41 Phase 6.1: partial-dependence values", not _k41_bad,
           f"{len(_pdp_ids) - len(_k41_bad)} of {len(_pdp_ids)} curves present with finite values ({len(_pdp)} grid points in all)"
           + "".join(f"; missing or not finite: {_f}" for _f in _k41_bad))
_pdp_lines, _pdp_rows = [], []
for _fig, _key in _pdp_ids.items():
    _c = _pdp[_pdp['Figure'] == _fig].reset_index(drop=True)
    _pdp_lines.append(f"{_fig}: {_c['Partial_dependence'].iloc[0]:,.3f} at a solar share of {_c['Solar_Generation_Share_pct'].iloc[0]:.2f}%"
                      f" -> {_c['Partial_dependence'].iloc[-1]:,.3f} at {_c['Solar_Generation_Share_pct'].iloc[-1]:.2f}% ({len(_c)} grid points).")
    _pdp_rows += [(f"{_fig}, grid point {_i + 1} ({_c['Solar_Generation_Share_pct'].iloc[_i]:.2f}%)", _c['Partial_dependence'].iloc[_i],
                   f"PDP.{_key}.{_i + 1:03d}") for _i in range(9, len(_c), 10)]
_k45_files = [os.path.join(OUTPUT_DIR_P6, _f) for _f in (
    'Phase_6_Figure_5_PDP_Merit_Order_and_Volatility.png', 'Phase_6_Figure_7_PDP_Merit_Order_and_Volatility.png',
    'Phase_6_Figure_8_Surplus_Probability_and_Distribution.png')] + [_pdp_file]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 6.1: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 6.1: XGBoost partial dependence on the solar generation share",
       lines=_pdp_lines + [f"xgboost {xgb.__version__} in this kernel (a cached file keeps the values of the session that fitted the models). Tree ensembles depend on library version and platform:"
                           " agreement with the pre-test is expected in the reference environment only."],
       files=_k45_files)
compare_with_reference("Phase 6.1: partial dependence, every 10th grid point of each curve", _pdp_rows,
                       check="K17 Phase 6.1: comparison with the independent pre-test")

### Phase 6.2: Flexibility Trap Analysis

The crossover of the two curves – the solar share beyond which the surplus curve stays above the deficit/balanced curve up to the end of the common range (45%) – is computed from the simulated curves by `last_upward_crossover()` (linear interpolation between grid points; Appendix I uses the same function) and saved to `Phase_6_2_Flexibility_Trap_Crossover.csv` together with the figure; the two curves are saved to `Phase_6_2_Flexibility_Trap_Curves.csv`. The value is printed in the run report of the cell and is read from the file by Table J4.

In [ ]:
# =============================================================================
# PHASE 6.2: NON-LINEAR IMPACT ANALYSIS (XGBOOST) - FLEXIBILITY TRAP
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import os
import warnings

# no blanket warning filter (see the setup cell)

OUTPUT_DIR_P6 = os.path.join(RESULTS_BASE, "Phase_6")
if not os.path.exists(OUTPUT_DIR_P6):
    os.makedirs(OUTPUT_DIR_P6)

# Academic plot settings
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def check_phase6_2_plot():
    """Checks if Figure 9 (Flexibility Trap) and its numerical record already exist."""
    fig_9 = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_9_Flexibility_Trap.png')
    # the crossover and curve files are part of the cache: they belong to the fit that produced the figure
    return os.path.exists(fig_9) and os.path.exists(CROSSOVER_FILE_P6) and os.path.exists(CURVES_FILE_P6)

# crossover point of the two curves and the curves themselves, saved together with the figure
CROSSOVER_FILE_P6 = os.path.join(OUTPUT_DIR_P6, 'Phase_6_2_Flexibility_Trap_Crossover.csv')
CURVES_FILE_P6 = os.path.join(OUTPUT_DIR_P6, 'Phase_6_2_Flexibility_Trap_Curves.csv')

def last_upward_crossover(x, y_surplus, y_deficit):
    """Crossover of the surplus and the deficit/balanced curve on a common grid (Phase 6.2 and Appendix I).

    x: solar share at the grid points of the common range; y_surplus, y_deficit: the two simulated curves at these points.
    Returns (crossover, up, down). `up` / `down` list the solar shares at which the surplus curve rises above / falls
    below the deficit/balanced curve (linear interpolation between grid points). `crossover` is the last upward crossing,
    i.e. the share beyond which the surplus curve stays above up to the end of the common range; it is NaN if there is
    no upward crossing or if the surplus curve is not above at the end of the range."""
    diff = y_surplus - y_deficit
    cross = lambda i: float(x[i] + (x[i + 1] - x[i]) * (0.0 - diff[i]) / (diff[i + 1] - diff[i]))
    up = [cross(i) for i in range(len(diff) - 1) if diff[i] <= 0 < diff[i + 1]]
    down = [cross(i) for i in range(len(diff) - 1) if diff[i] > 0 >= diff[i + 1]]
    crossover = up[-1] if (up and diff[-1] > 0) else np.nan
    return crossover, up, down

def run_phase_6_2_final(df):
    print(f">> Phase 6.2: Flexibility Trap Analysis (Directory: '{OUTPUT_DIR_P6}')...")
    print("="*80)

    fig_9_path = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_9_Flexibility_Trap.png')

    # Cache check
    if check_phase6_2_plot():
        print(f">> [CACHE] Existing analysis found in '{OUTPUT_DIR_P6}'.")
        print(">> Skipping XGBoost training, loading saved plot...\n")

        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(fig_9_path))
        plt.axis('off')
        plt.title("Phase 6.2 - Figure 9: Flexibility Trap (Preview)", fontsize=14)
        plt.show()

        print(f">> Cached data from '{OUTPUT_DIR_P6}' loaded successfully.")
        print(">> To regenerate, delete the PNG file and re-run this cell.")
        # the crossover point belongs to the fit that produced the figure; it is written only together with it
        print(f">> Crossover point (from '{CROSSOVER_FILE_P6}'):")
        print(pd.read_csv(CROSSOVER_FILE_P6).T.to_string(header=False))
        return

    print(">> No cached results found. Training XGBoost models (Flexibility Trap)...")

    # Data preparation
    dm = df.copy()
    if dm['Solar_Generation_Share'].max() <= 1:
        dm['Solar_Generation_Share'] *= 100

    sur = dm[dm['System_Direction_Down'] == 1]
    dfc = dm[dm['System_Direction_Down'] == 0]

    ft = ['Solar_Generation_Share', 'Mcp', 'Consumption', 'Wind_Generation', 'Hour_Sin', 'Hour_Cos']

    # Model training
    print(">> Training conditional XGBoost models...")
    ms = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(sur[ft], sur['a-FRR_Price'])
    md = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(dfc[ft], dfc['a-FRR_Price'])

    # Ceteris paribus simulation
    sr = np.linspace(0, 60, 100)
    avg = dm[ft].median()
    sim = pd.DataFrame([avg]*100, columns=ft)
    sim['Solar_Generation_Share'] = sr

    mk = sr <= 45
    yd = md.predict(sim[mk][ft])
    ys = ms.predict(sim[ft])
    ysc = ys[:len(yd)]
    sc = sr[mk]

    # -------------------------------------------------------------------------
    # FIGURE 9: FLEXIBILITY TRAP
    # -------------------------------------------------------------------------
    print("  [Figure 8; file Phase_6_Figure_9, R1 number] Plotting Flexibility Trap...")
    fig, ax = plt.subplots(figsize=(10, 6))


    ax.plot(sc, yd, color='green', lw=3.0, label='Energy Deficit & Balanced')
    ax.plot(sr, ys, color='red', lw=3.0, label='Energy Surplus')
    
    # [Revision_R2_M6] legend terms aligned with Section 4.5: deficit/balanced curve above the surplus
    # curve = 'deficit premium'; the reverse = 'Flexibility Trap'.
    # Zone 1: deficit premium (was 'Elasticity Risk Gap')
    ax.fill_between(
        sc, ysc, yd, 
        where=(yd >= ysc), interpolate=True, 
        color='gray', alpha=.12, hatch='...', 
        label='Deficit premium (deficit/balanced > surplus)'
    )
    
    # Zone 2: Flexibility Trap
    ax.fill_between(
        sc, ysc, yd, 
        where=(ysc > yd), interpolate=True, 
        color='red', alpha=.08, hatch='xxx', 
        label='Flexibility Trap (surplus > deficit/balanced)'
    )

    ax.set_xlabel('Solar Generation Share (%)', fontweight='bold')
    ax.set_ylabel('Predicted a-FRR Price (TL/MW)', fontweight='bold')  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW)
    ax.legend(loc='upper left', fontsize=11, frameon=True)
    ax.axvline(x=45, color='green', ls=':', alpha=.5)

    plt.tight_layout()
    plt.savefig(fig_9_path, dpi=FIG_DPI, bbox_inches='tight')
    plt.show()

    # Zone dominance statistics
    trap_pct = (ysc > yd).sum() / len(yd) * 100
    gap_pct = (yd >= ysc).sum() / len(yd) * 100
    print(f"\n>> Zone dominance in the 0-45% solar range:")
    print(f"   - Deficit premium (Deficit > Surplus): {gap_pct:.1f}%")
    print(f"   - Flexibility Trap (Surplus > Deficit): {trap_pct:.1f}%")

    # crossover of the two curves, computed from the simulated curves: the solar share at which the
    # surplus curve rises above the deficit/balanced curve (linear interpolation between the grid points of the simulation).
    # 'Crossover' is the last such point, i.e. the share beyond which the surplus curve stays above up to the end of the
    # common range (45%); it is empty if the surplus curve is not above at 45%. All crossings are listed.
    crossover, up, down = last_upward_crossover(sc, ysc, yd)
    pd.DataFrame([{'Crossover_Solar_Share_pct': crossover,
                   'Upward_crossings_pct': "; ".join(f"{v:.2f}" for v in up),        # surplus curve rises above the deficit/balanced curve
                   'Downward_crossings_pct': "; ".join(f"{v:.2f}" for v in down),    # surplus curve falls below it
                   'Flexibility_Trap_share_of_range_pct': trap_pct, 'Deficit_premium_share_of_range_pct': gap_pct,
                   'Common_range_pct': '0-45', 'Grid_step_pct': float(sr[1] - sr[0]),
                   'Curves': 'XGBoost (n_estimators=200, random_state=42), other features at their medians',
                   'xgboost_version': xgb.__version__}]).to_csv(CROSSOVER_FILE_P6, index=False, **CSV_KW)
    # the two curves of the figure as numbers (the deficit/balanced curve exists on the common range, up to 45%, only)
    pd.DataFrame({'Solar_Generation_Share_pct': sr,
                  'Predicted_aFRR_Price_Deficit_Balanced': pd.Series(yd, dtype=float),
                  'Predicted_aFRR_Price_Surplus': pd.Series(ys, dtype=float)}).to_csv(CURVES_FILE_P6, index=False, **CSV_KW)
    print(f"\n>> Crossover point (surplus curve above the deficit/balanced curve from there to 45%): "
          + (f"{crossover:.1f}% solar share" if np.isfinite(crossover) else "none (the surplus curve is not above at 45%)"))
    print(f"   upward crossings: {[round(v, 2) for v in up]} | downward crossings: {[round(v, 2) for v in down]}")
    print(f"   Saved: '{CROSSOVER_FILE_P6}' and '{CURVES_FILE_P6}'")

    print(f"\n>> Phase 6.2 complete. Flexibility Trap figure saved to '{OUTPUT_DIR_P6}'.")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    if 'df' in globals() and df is not None:
        run_phase_6_2_final(df)
    else:
        print("ERROR: DataFrame 'df' not found. Please run Phase 1 first.")

# run report and comparison with the independent pre-test
_x62 = pd.read_csv(CROSSOVER_FILE_P6, float_precision='round_trip').iloc[0]
_c62 = pd.read_csv(CURVES_FILE_P6)
# checkpoint K42: the crossover with its definition; no crossover that holds to the end of the range -> WARNING
_k42_x = float(_x62['Crossover_Solar_Share_pct'])
_k42_list = lambda v: str(v) if pd.notna(v) and str(v).strip() else 'none'
checkpoint("K42 Phase 6.2: flexibility-trap crossover", bool(np.isfinite(_k42_x)),
           "crossover = the last upward crossing of the surplus curve over the deficit/balanced curve after which it stays above "
           f"up to the end of the common range ({_x62['Common_range_pct']}%; linear interpolation between the grid points): "
           + (f"{_k42_x:.2f}% solar share" if np.isfinite(_k42_x) else "none (no sustained crossover)")
           + f"; upward crossings: {_k42_list(_x62['Upward_crossings_pct'])}; downward crossings: {_k42_list(_x62['Downward_crossings_pct'])} (% solar share)",
           level='warn')
_k45_files = [os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_9_Flexibility_Trap.png'), CROSSOVER_FILE_P6, CURVES_FILE_P6]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 6.2: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 6.2: flexibility trap (XGBoost interaction simulation)",
       lines=["Crossover (the surplus curve stays above the deficit/balanced curve from there to 45%): "
              + (f"{_x62['Crossover_Solar_Share_pct']:.2f}% solar share." if np.isfinite(_x62['Crossover_Solar_Share_pct'])
                 else "none (no upward crossing after which the surplus curve stays above up to 45%)."),
              f"Upward crossings: {_x62['Upward_crossings_pct']} | downward crossings: {_x62['Downward_crossings_pct']} (% solar share).",
              f"Flexibility Trap zone: {_x62['Flexibility_Trap_share_of_range_pct']:.1f}% of the 0-45% range; curves saved at {len(_c62)} grid points.",
              f"xgboost {_x62['xgboost_version']} (version of the fit). Tree ensembles depend on library version and platform:"
              " agreement with the pre-test is expected in the reference environment only."],
       files=_k45_files)
compare_with_reference("Phase 6.2: crossover of the flexibility-trap curves",
                       [("Crossover, % solar share", _x62['Crossover_Solar_Share_pct'], "KESISIM.Phase_6_2_Figure_8")],
                       check="K17 Phase 6.2: comparison with the independent pre-test")

### Phase 6.4: Out-of-Sample Validation of the XGBoost Cross-Checks (Revision_Review_Referee_2.3)

XGBoost is used as an interpretive cross-check (partial-dependence plots), not as a forecasting model. To show that
the patterns are not in-sample artefacts, held-out performance is evaluated with 5-fold expanding-window
(walk-forward) time-series cross-validation (`sklearn.model_selection.TimeSeriesSplit`): always train on the past,
test on the future. The models evaluated are **exactly** the Phase 6.1 models – same features (`features_core`,
`features_with_mcp`) and the same estimator call (`n_estimators=200, random_state=42`, XGBoost library defaults
otherwise) – so the metrics refer to the models whose partial-dependence plots are interpreted (Figures 4, 6, 7).
Regression targets report MAE, RMSE and out-of-sample R²; the system-direction classifier reports accuracy and F1.
The in-sample fit of the full-sample model is reported alongside for comparison.

Produces: `Phase_6_4_Out_of_Sample_CV.csv` (fold averages) and `Phase_6_4_Out_of_Sample_CV_Folds.csv` (per fold) in `results/Phase_6/`.
(Numbering: Phase 6.3 is not used; this check keeps its R1 number 6.4 so that the heading matches its output files.)


In [ ]:
# =============================================================================
# PHASE 6.4: OUT-OF-SAMPLE VALIDATION (EXPANDING-WINDOW TIME-SERIES CV) - CACHED
# =============================================================================
# Revision_Review_Referee_2.3 – Out-of-sample validation of XGBoost
# Feeds Supplementary Appendix J.3.
# Evaluates EXACTLY the Phase 6.1 models (same features, same estimator
# call: n_estimators=200, random_state=42, XGBoost library defaults otherwise), so
# that the held-out metrics refer to the models whose partial-dependence plots are
# interpreted (Figures 4, 6, 7). Per-fold metrics and the in-sample fit of the
# full-sample model are reported alongside.
# =============================================================================
import pandas as pd, numpy as np, os, warnings
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (mean_absolute_error, mean_squared_error, accuracy_score, f1_score,
                             roc_auc_score, balanced_accuracy_score)
# no blanket warning filter (see the setup cell)

OUTPUT_DIR_P6 = os.path.join(RESULTS_BASE, "Phase_6")
os.makedirs(OUTPUT_DIR_P6, exist_ok=True)

# Identical to Phase 6.1 (features_core / features_with_mcp and the estimator call)
CV_FEATURES_CORE = ['Solar_Generation_Share', 'Consumption', 'Wind_Generation', 'Price_Cap', 'Hour_Sin', 'Hour_Cos']
CV_FEATURES_AFRR = CV_FEATURES_CORE + ['Mcp']
CV_XGB_PARAMS = dict(n_estimators=200, random_state=42)   # as in Phase 6.1; every other XGBoost setting at its library default

def _r2(y, p):
    return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

def run_out_of_sample_cv(df: pd.DataFrame, n_splits: int = 5):
    print("\n>>> PHASE 6.4: OUT-OF-SAMPLE VALIDATION (EXPANDING-WINDOW TIME-SERIES CV)")
    path = os.path.join(OUTPUT_DIR_P6, "Phase_6_4_Out_of_Sample_CV.csv")
    path_f = os.path.join(OUTPUT_DIR_P6, "Phase_6_4_Out_of_Sample_CV_Folds.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    d = df.dropna(subset=CV_FEATURES_AFRR + ['a-FRR_Price', 'System_Direction_Down']).copy()
    # NOTE: Phase 6.1 hands NaNs to XGBoost unchanged; the dataset has none (N = 35,064 in every
    # phase), so this row set is identical to the one the Phase 6.1 models were fitted on.
    if d['Solar_Generation_Share'].max() <= 1.0:        # Phase 6.1 works in percent
        d['Solar_Generation_Share'] *= 100
    tscv = TimeSeriesSplit(n_splits=n_splits)   # expanding window: train on the past, test on the future

    rows, folds = [], []
    try:
        for dep, label, feats in [('Mcp', 'MCP (price)', CV_FEATURES_CORE),
                                  ('a-FRR_Price', 'a-FRR (price)', CV_FEATURES_AFRR)]:
            X = d[feats].astype(float).values; y = d[dep].astype(float).values
            mae, rmse, r2 = [], [], []
            for k, (tr, te) in enumerate(tscv.split(X), 1):
                p = XGBRegressor(**CV_XGB_PARAMS).fit(X[tr], y[tr]).predict(X[te])
                mae.append(mean_absolute_error(y[te], p)); rmse.append(np.sqrt(mean_squared_error(y[te], p))); r2.append(_r2(y[te], p))
                folds.append({'Target': label, 'Fold': k, 'Train_N': len(tr), 'Test_N': len(te),
                              'Test_start': str(d.index[te[0]]), 'Test_end': str(d.index[te[-1]]),
                              'MAE': mae[-1], 'RMSE': rmse[-1], 'R2_oos': r2[-1], 'Accuracy': np.nan, 'F1': np.nan})
            r2_in = _r2(y, XGBRegressor(**CV_XGB_PARAMS).fit(X, y).predict(X))   # full-sample (Phase 6.1) model, in-sample
            rows.append({'Target': label, 'MAE': np.mean(mae), 'RMSE': np.mean(rmse), 'R2_oos': np.mean(r2),
                         'Accuracy': np.nan, 'F1': np.nan, 'R2_insample': r2_in, 'Accuracy_insample': np.nan})

        X = d[CV_FEATURES_CORE].astype(float).values; yc = d['System_Direction_Down'].astype(int).values
        acc, f1, auc, bacc, base = [], [], [], [], []
        for k, (tr, te) in enumerate(tscv.split(X), 1):
            clf = XGBClassifier(**CV_XGB_PARAMS).fit(X[tr], yc[tr])
            p = clf.predict(X[te]); pr = clf.predict_proba(X[te])[:, 1]
            acc.append(accuracy_score(yc[te], p)); f1.append(f1_score(yc[te], p))
            auc.append(roc_auc_score(yc[te], pr)); bacc.append(balanced_accuracy_score(yc[te], p))
            base.append(max(yc[te].mean(), 1 - yc[te].mean()))   # majority-class accuracy in the test fold
            # (share of the larger class in that held-out fold, i.e. an oracle 'always-majority' baseline;
            #  the surplus share differs between the folds - see Majority_baseline_acc in the folds file - so the baseline is fold-specific)
            folds.append({'Target': 'System direction (classifier)', 'Fold': k, 'Train_N': len(tr), 'Test_N': len(te),
                          'Test_start': str(d.index[te[0]]), 'Test_end': str(d.index[te[-1]]),
                          'MAE': np.nan, 'RMSE': np.nan, 'R2_oos': np.nan, 'Accuracy': acc[-1], 'F1': f1[-1],
                          'ROC_AUC': auc[-1], 'Balanced_Acc': bacc[-1], 'Majority_baseline_acc': base[-1]})
        clf_full = XGBClassifier(**CV_XGB_PARAMS).fit(X, yc)
        acc_in = accuracy_score(yc, clf_full.predict(X))
        rows.append({'Target': 'System direction (classifier)', 'MAE': np.nan, 'RMSE': np.nan, 'R2_oos': np.nan,
                     'Accuracy': np.mean(acc), 'F1': np.mean(f1), 'R2_insample': np.nan, 'Accuracy_insample': acc_in,
                     'ROC_AUC': np.mean(auc), 'Balanced_Acc': np.mean(bacc), 'Majority_baseline_acc': np.mean(base)})

        out = pd.DataFrame(rows)
        pd.DataFrame(folds).to_csv(path_f, index=False, **CSV_KW)
        out.to_csv(path, index=False, **CSV_KW)
        print(out.to_string(index=False, na_rep='-'))
        print("\n  Note: metrics are averages over the 5 held-out folds (sklearn TimeSeriesSplit, expanding window).")
        print("        Models, features and hyper-parameters are identical to Phase 6.1 (the PDP models).")
        print("        The low F1 reflects the minority (surplus) class; XGBoost is an interpretive tool (PDP), not a forecasting model.")
        print("        For the imbalanced system-direction target, accuracy is compared with the majority-class baseline;")
        print("        ROC-AUC and balanced accuracy are the informative held-out measures.")
        print("        '-' = measure not defined for that target (empty / NaN in the CSV).")
        print(f"  Saved: {path}\n  Saved: {path_f}")
    except Exception as e:
        print(f"  !! Error in out-of-sample validation: {e}")
        raise   # errors stop the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_out_of_sample_cv(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
import xgboost
_cv_path = os.path.join(OUTPUT_DIR_P6, "Phase_6_4_Out_of_Sample_CV.csv")
_cv = pd.read_csv(_cv_path, float_precision='round_trip')
_cvt = _cv.set_index('Target')
# checkpoint K41: number of walk-forward folds per target = the number configured (n_splits of
# run_out_of_sample_cv) and every fit measure defined for the target finite (fold averages, in-sample fit, every fold)
import inspect
_k41_n = inspect.signature(run_out_of_sample_cv).parameters['n_splits'].default
_k41_f = pd.read_csv(os.path.join(OUTPUT_DIR_P6, "Phase_6_4_Out_of_Sample_CV_Folds.csv"), float_precision='round_trip')
_k41_folds = _k41_f.groupby('Target', sort=False)['Fold'].nunique().to_dict()
_k41_cols = lambda t: (['Accuracy', 'F1', 'ROC_AUC', 'Balanced_Acc', 'Majority_baseline_acc'] if 'classifier' in t else ['MAE', 'RMSE', 'R2_oos'])
_k41_val = lambda r, c: float(r[c]) if c in r.index and pd.notna(r[c]) else np.nan
_k41_bad = ([f"{_r['Target']} (fold average): {_c}" for _, _r in _cv.iterrows()
             for _c in _k41_cols(_r['Target']) + (['Accuracy_insample'] if 'classifier' in _r['Target'] else ['R2_insample'])
             if not np.isfinite(_k41_val(_r, _c))]
            + [f"{_r['Target']} fold {_r['Fold']}: {_c}" for _, _r in _k41_f.iterrows() for _c in _k41_cols(_r['Target'])
               if not np.isfinite(_k41_val(_r, _c))])
_k41_fold_ok = set(_k41_folds) == set(_cv['Target']) and all(_v == _k41_n for _v in _k41_folds.values())
checkpoint("K41 Phase 6.4: walk-forward folds and fit metrics", _k41_fold_ok and not _k41_bad,
           "folds per target: " + ", ".join(f"{_t} {_v}" for _t, _v in _k41_folds.items()) + f" (configured: {_k41_n})"
           + ("" if set(_k41_folds) == set(_cv['Target']) else "; the targets of the folds file differ from those of the summary")
           + ("; every fit measure finite" if not _k41_bad else "".join(f"; not finite: {_b}" for _b in _k41_bad[:8])))
_k45_files = [_cv_path, os.path.join(OUTPUT_DIR_P6, "Phase_6_4_Out_of_Sample_CV_Folds.csv")]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 6.4: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 6.4: out-of-sample validation of the XGBoost models (5-fold expanding window)",
       lines=[f"Out-of-sample R2 (mean of the held-out folds): MCP {_cvt.loc['MCP (price)', 'R2_oos']:.3f}, a-FRR {_cvt.loc['a-FRR (price)', 'R2_oos']:.3f}"
              f" (in-sample: {_cvt.loc['MCP (price)', 'R2_insample']:.3f} and {_cvt.loc['a-FRR (price)', 'R2_insample']:.3f}).",
              f"System direction: accuracy {_cvt.loc['System direction (classifier)', 'Accuracy']:.3f} against a majority-class baseline of"
              f" {_cvt.loc['System direction (classifier)', 'Majority_baseline_acc']:.3f}; ROC-AUC {_cvt.loc['System direction (classifier)', 'ROC_AUC']:.3f}.",
              f"xgboost {xgboost.__version__} in this kernel (a cached file keeps the values of the session that fitted the models). Tree ensembles depend on library version and platform:"
              " agreement with the pre-test is expected in the reference environment only."],
       table=_cv, files=_k45_files)
compare_with_reference("Phase 6.4: walk-forward validation (fold averages)",
                       [(f"{_r['Target']}: {_c}", _r[_c], f"WF.{_r['Target']}.{_c}") for _, _r in _cv.iterrows()
                        for _c in _cv.columns if _c != 'Target' and pd.notna(_r[_c])],
                       check="K17 Phase 6.4: comparison with the independent pre-test")

## Phase 7: Robustness Checks (Quantile Regression)

### Phase 7.1: Quantile Regression at the 10th, 50th and 90th Percentiles


In [ ]:
# =============================================================================
# PHASE 7: ROBUSTNESS CHECKS (QUANTILE REGRESSION) - CACHED
# =============================================================================

import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# [Revision] QR convergence warnings are shown, not silenced (statsmodels sets 'always' for both categories at import)
from statsmodels.tools.sm_exceptions import IterationLimitWarning, ConvergenceWarning
warnings.simplefilter("always", IterationLimitWarning)
warnings.simplefilter("always", ConvergenceWarning)

OUTPUT_DIR = os.path.join(RESULTS_BASE, "Phase_7")
REPORT_FILE = os.path.join(OUTPUT_DIR, "Phase_7_Robustness_Quantile_Regression_Results.txt")
CSV_FILE = os.path.join(OUTPUT_DIR, "Phase_7_Quantile_Regression_Coefficients.csv")
PLOT_FILE = os.path.join(OUTPUT_DIR, "Phase_7_Figure_11_Quantile_Regression_Coefficient_Robustness.png")
P71_FITS = []   # per full-sample fit: (hypothesis, quantile, IRLS iterations, limit reached); checkpoint K30

def flag_iteration_limit(fit):
    """Calls fit() (a statsmodels QuantReg fit) and returns (result, limit_reached). Every warning of the fit
    is passed on unchanged (shown and recorded); the IterationLimitWarning is also noted as a flag for checkpoint K30."""
    w = []
    try:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            res = fit()
    finally:                             # also when the fit raises: what was caught before the error is passed on
        for x in w:
            warnings.warn_explicit(x.message, x.category, x.filename, x.lineno)
    return res, any(issubclass(x.category, IterationLimitWarning) for x in w)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def check_existing_results():
    """Checks if all required output files exist."""
    return os.path.exists(REPORT_FILE) and os.path.exists(CSV_FILE) and os.path.exists(PLOT_FILE)

def run_phase_7_exact_style(df):
    print("PHASE 7: Robustness Checks (Quantile Regression) Process Started...")
    print("="*80)

    # Cache check
    if check_existing_results():
        print(f">> [CACHE] Results found in '{OUTPUT_DIR}'. Loading...\n")

        with open(REPORT_FILE, "r", encoding="utf-8") as f:
            print(f.read())

        img = plt.imread(PLOT_FILE)
        plt.figure(figsize=(12, 6)); plt.imshow(img); plt.axis('off'); plt.show()

        print("="*80)
        print(f">> Results loaded from '{OUTPUT_DIR}' successfully.")
        return

    # Run quantile regression
    print(">> No cached results found. Running Quantile Regression (Q10, Q50, Q90)...")

    try:
        df_qr = df.copy()
        df_qr.rename(columns={'a-FRR_Price': 'aFRR_Price', 'Solar_Generation_Share': 'Solar_Share'}, inplace=True)

        controls = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation',
                    'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
                    'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
                    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
        control_str = " + ".join(controls)

        # [Revision_Review_Referee_2.5] standardised regressors for the convergence of the IRLS solver
        reg_cols = ['Solar_Share'] + controls
        sd = df_qr[reg_cols].std()
        df_qr_s = df_qr.copy()
        df_qr_s[reg_cols] = df_qr_s[reg_cols] / sd      # regressors only; y stays in original units

        qs = [0.10, 0.50, 0.90]
        mcp_coefs, afrr_coefs, qr_results_data = [], [], []

        with open(REPORT_FILE, "w", encoding="utf-8") as f:
            f.write("PHASE 7: QUANTILE REGRESSION ROBUSTNESS REPORT\n")
            f.write("="*80 + "\n\n")

            # H1: Merit-Order Effect (MCP)
            h1_title = ">>> HYPOTHESIS 1: MERIT-ORDER EFFECT ON MCP (ROBUSTNESS CHECK)\n"
            sep = "-" * 75 + "\n"
            header = f"{'Quantile':<10} | {'Coef (Solar)':<15} | {'P-Value':<10} | {'Status'}\n"

            f.write(h1_title + sep + header + sep)
            print("\n" + h1_title.strip())
            print(sep.strip())
            print(header.strip())
            print(sep.strip())

            formula_mcp = f"Mcp ~ Solar_Share + {control_str}"
            for q in qs:
                res, hit = flag_iteration_limit(lambda: smf.quantreg(formula_mcp, df_qr_s).fit(q=q, max_iter=5000))
                P71_FITS.append(('MCP', q, int(res.iterations), hit))     # checkpoint K30
                coef = res.params['Solar_Share'] / sd['Solar_Share']   # back to original scale
                pval = res.pvalues['Solar_Share']
                mcp_coefs.append(coef)
                status = "CONFIRMED (-)" if (coef < 0 and pval < 0.05) else "Rejected"
                line = f"Q{int(q*100):<8} | {coef:>10.4f}      | {pval:.4f}     | {status}\n"
                print(line.strip()); f.write(line)
                qr_results_data.append({'Hypothesis': 'MCP', 'Quantile': q, 'Coef': coef, 'Pval': pval})

            # H2: Volatility Effect (a-FRR)
            h2_title = "\n\n>>> HYPOTHESIS 2: VOLATILITY EFFECT ON a-FRR (ROBUSTNESS CHECK)\n"
            header_v = f"{'Quantile':<10} | {'Coef (Solar)':<15} | {'P-Value':<10} | {'Interpretation'}\n"

            f.write(h2_title + sep + header_v + sep)
            print(h2_title.strip())
            print(sep.strip())
            print(header_v.strip())
            print(sep.strip())

            formula_afrr = f"aFRR_Price ~ Solar_Share + {control_str}"
            for q in qs:
                res, hit = flag_iteration_limit(lambda: smf.quantreg(formula_afrr, df_qr_s).fit(q=q, max_iter=5000))
                P71_FITS.append(('aFRR', q, int(res.iterations), hit))    # checkpoint K30
                coef = res.params['Solar_Share'] / sd['Solar_Share']   # back to original scale
                pval = res.pvalues['Solar_Share']
                afrr_coefs.append(coef)
                if q == 0.90 and coef > 5000: interp = "CRITICAL STRESS (!)"
                elif coef > 0 and pval < 0.05: interp = "Positive (+)"
                else: interp = "Insignificant"
                line = f"Q{int(q*100):<8} | {coef:>10.4f}      | {pval:.4f}     | {interp}\n"
                print(line.strip()); f.write(line)
                qr_results_data.append({'Hypothesis': 'aFRR', 'Quantile': q, 'Coef': coef, 'Pval': pval})

        # Visualization
        plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman']})
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
        categories = ['Q10 (Low)', 'Q50 (Median)', 'Q90 (High)']

        ax1.bar(categories, mcp_coefs, color='darkorange', alpha=0.7)
        ax1.axhline(0, color='black', linewidth=0.8)
        ax1.set_ylabel('Solar Impact Coefficient (MCP)')

        ax2.bar(categories, afrr_coefs, color='crimson', alpha=0.7)
        ax2.axhline(0, color='black', linewidth=0.8)
        ax2.set_ylabel('Solar Impact Coefficient (a-FRR)')

        plt.tight_layout()
        plt.savefig(PLOT_FILE, dpi=FIG_DPI, bbox_inches='tight')
        plt.suptitle("Phase 7 - Quantile Regression Robustness Checks (Preview)", fontsize=16, y=1.02)
        plt.show()

        pd.DataFrame(qr_results_data).to_csv(CSV_FILE, index=False, **CSV_KW)   # LF line endings on every platform
        print(f"\n>> Results saved to '{OUTPUT_DIR}'.")

    except Exception as e:
        print(f"!! ERROR in Phase 7: {e}")
        raise   # an error stops the cell

# --- Execution ---
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_phase_7_exact_style(target_df)

# run report and comparison with the independent pre-test
_q71 = pd.read_csv(CSV_FILE)
_q71['Q'] = 'Q' + (_q71['Quantile'] * 100).round().astype(int).astype(str)
# checkpoints K30 (the six full-sample fits stopped before the 5,000-iteration limit) and K45 (output files)
if P71_FITS:
    _k30_hit = [f"{h} Q{int(round(q * 100))}" for h, q, it, hit in P71_FITS if hit or it >= 5000]
    checkpoint("K30 Phase 7.1: full-sample quantile regressions converged", not _k30_hit,
               f"{len(P71_FITS) - len(_k30_hit)} of {len(P71_FITS)} fits converged before the 5,000-iteration limit "
               f"(IRLS iterations {min(f[2] for f in P71_FITS):,} to {max(f[2] for f in P71_FITS):,})"
               + (f"; at the limit (the reported coefficient is not a converged estimate): {', '.join(_k30_hit)}" if _k30_hit else ""),
               level='fail')
else:
    checkpoint("K30 Phase 7.1: full-sample quantile regressions converged", None,
               "results loaded from the cache of an earlier run (iteration counts not recorded)", level='fail')
_k45_files = [REPORT_FILE, CSV_FILE, PLOT_FILE]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]
checkpoint("K45 Phase 7.1: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 7.1 - Quantile regression (Q10, Q50, Q90)",
       lines=["Solar coefficient per unit solar share (0-1): " +
              "; ".join(f"{h} {qq} {c:,.1f}" for h, qq, c in zip(_q71['Hypothesis'], _q71['Q'], _q71['Coef'])) + "."],
       table=_q71[['Hypothesis', 'Quantile', 'Coef', 'Pval']], files=[REPORT_FILE, CSV_FILE, PLOT_FILE])
compare_with_reference("Phase 7.1 - quantile-regression point estimates",
                       [(f"{h} {qq} solar coefficient (per unit share)", float(c), f"QR.{h}.{qq}.Coef")
                        for h, qq, c in zip(_q71['Hypothesis'], _q71['Q'], _q71['Coef'])] +
                       [(f"{h} {qq} effect per percentage point", float(c) / 100, f"QR.{h}.{qq}.yp")
                        for h, qq, c in zip(_q71['Hypothesis'], _q71['Q'], _q71['Coef'])],
                       check="K17 Phase 7.1: comparison with the independent pre-test")

### Phase 7.2: Solver Iteration Sensitivity (Revision_Review_Referee_2.5)

Estimates the Phase 7 specification with statsmodels `QuantReg` on the unscaled regressors at 1,000 / 2,000 /
5,000 / 10,000 iterations, and fits it with the formula interface on the unscaled regressors at 2,000 iterations
(column `Phase7_Code_Rerun`). The Phase 7.1 cell standardises the regressors and allows 5,000 iterations, so that column
is not the Phase 7.1 estimate. Fits that stop at the iteration limit are flagged
(`Converged_iter*` in the CSV; `*` in the printed table, also for `Phase7_Code_Rerun`, whose flags are written to the run
report). The iteration-limit warning of statsmodels is captured per fit and recorded in these flags
instead of being printed.

Produces: `Phase_7_2_QR_Iteration_Sensitivity.csv` in `results/Phase_7/`.

In [ ]:
# =============================================================================
# PHASE 7.2: QR SOLVER ITERATION SENSITIVITY - CACHED
# =============================================================================
# Revision_Review_Referee_2.5 – QR convergence diagnostics & dependence-robust SE
# Feeds Supplementary Appendix J.5.
# =============================================================================
import pandas as pd, numpy as np, os, warnings
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import IterationLimitWarning
# [Revision] QR convergence warnings are never silenced: the iteration-limit warning of every fit in Phases 7.2-7.4 is
# captured per fit and reported (capture_iteration_limit); statsmodels sets 'always' for both categories at import
from statsmodels.tools.sm_exceptions import IterationLimitWarning, ConvergenceWarning
warnings.simplefilter("always", IterationLimitWarning)
warnings.simplefilter("always", ConvergenceWarning)

OUTPUT_DIR_P7 = os.path.join(RESULTS_BASE, "Phase_7")
os.makedirs(OUTPUT_DIR_P7, exist_ok=True)

# Same control set as Phase 7
QR_CONTROLS = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation',
               'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
               'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
               'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
QUANTILES  = [0.10, 0.50, 0.90]
QR_TARGETS = [('Mcp', 'MCP'), ('a-FRR_Price', 'aFRR')]


def qr_design(df: pd.DataFrame):
    """Design matrix [const, solar, controls] and the std. deviation of each regressor."""
    Xr = df[['Solar_Generation_Share'] + QR_CONTROLS].astype(float)
    return sm.add_constant(Xr, has_constant='add').values, Xr.std().values

def check_loss(y, X, beta, q):
    """QR objective (sum of check-function losses); lower = better fit."""
    u = y - X @ beta
    return float(np.sum(u * (q - (u < 0))))

def capture_iteration_limit(fit):
    """Calls fit() (a statsmodels QuantReg fit) and returns (result, limit_reached). The IterationLimitWarning
    is captured per fit and returned as a flag, so it is reported in the tables and in the run report instead of being
    printed; any other warning raised during the fit is passed on unchanged."""
    w = []
    try:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            res = fit()
    finally:                             # also when the fit raises: what was caught before the error is passed on
        for x in w:
            if not issubclass(x.category, IterationLimitWarning):
                warnings.warn_explicit(x.message, x.category, x.filename, x.lineno)
    return res, any(issubclass(x.category, IterationLimitWarning) for x in w)

def fit_quantreg(y, X, q, max_iter):
    """statsmodels QuantReg fit that also reports whether the iteration limit was hit."""
    res, limit_reached = capture_iteration_limit(lambda: sm.QuantReg(y, X).fit(q=q, max_iter=max_iter))
    return res, not limit_reached

P72_RERUN_LIMIT = []   # per Phase7_Code_Rerun fit (unscaled, 2,000 iterations): True = stopped at the iteration
                       # limit (printed table and run report; not a CSV column)
P72_FITS = []          # per fit of this step: (hypothesis, quantile, iteration budget, column, converged
                       # flag from the captured warning, IRLS iterations); checkpoint K31

def run_iteration_sensitivity(df: pd.DataFrame, iter_grid=(1000, 2000, 5000, 10000)):
    print("\n>>> PHASE 7.2: QR SOLVER ITERATION SENSITIVITY (UNSCALED REGRESSORS)")
    path = os.path.join(OUTPUT_DIR_P7, "Phase_7_2_QR_Iteration_Sensitivity.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    X, _ = qr_design(df)
    # column Phase7_Code_Rerun: the Phase 7.1 formula (formula interface, hyphen-free names) fitted with
    # unscaled regressors and 2,000 iterations; Phase 7.1 itself standardises the regressors and allows 5,000 iterations
    df_qr = df.rename(columns={'a-FRR_Price': 'aFRR_Price', 'Solar_Generation_Share': 'Solar_Share'})
    control_str = " + ".join(QR_CONTROLS)

    rows = []
    try:
        print(f"  {'':10}" + "".join(f"{f'iter={mi}':>14}" for mi in iter_grid) + f"{'Phase7 rerun':>14}")
        for dep, label in QR_TARGETS:
            y = df[dep].astype(float).values
            dep_f = 'Mcp' if dep == 'Mcp' else 'aFRR_Price'
            for q in QUANTILES:
                row = {'Hypothesis': label, 'Quantile': q}
                for mi in iter_grid:
                    res, conv = fit_quantreg(y, X, q, mi)
                    P72_FITS.append((label, q, mi, f'Converged_iter{mi}', conv, int(res.iterations)))   # K31
                    row[f'Coef_iter{mi}'] = round(res.params[1], 1)
                    row[f'Converged_iter{mi}'] = conv
                res_rerun, limit_rerun = capture_iteration_limit(       # flagged like the other columns
                    lambda: smf.quantreg(f"{dep_f} ~ Solar_Share + {control_str}", df_qr).fit(q=q, max_iter=2000))
                row['Phase7_Code_Rerun'] = round(res_rerun.params['Solar_Share'], 1)
                P72_RERUN_LIMIT.append(limit_rerun)
                P72_FITS.append((label, q, 2000, 'Phase7_Code_Rerun', not limit_rerun, int(res_rerun.iterations)))
                rows.append(row)
                cells = "".join(f"{row[f'Coef_iter{mi}']:>13,.1f}{' ' if row[f'Converged_iter{mi}'] else '*'}"
                                for mi in iter_grid)
                print(f"  {label:<5} Q{int(q*100):<3}{cells}{row['Phase7_Code_Rerun']:>13,.1f}"
                      f"{'*' if P72_RERUN_LIMIT[-1] else ' '}")

        out = pd.DataFrame(rows)
        out.to_csv(path, index=False, **CSV_KW)   # LF line endings on every platform
        print("\n  * = iteration limit reached (solver did not converge).")
        print("  Note: a converged estimator would give the same coefficient in every column.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in iteration-sensitivity check: {e}")
        raise   # an error stops the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_iteration_sensitivity(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
# (no comparison in this step: the pre-test file holds the converged estimates only; see Phases 7.1, 7.3 and 7.4)
_p72 = os.path.join(OUTPUT_DIR_P7, "Phase_7_2_QR_Iteration_Sensitivity.csv")
_t72 = pd.read_csv(_p72)
_c72 = _t72.filter(like='Coef_iter')
# checkpoint K31: fits at the iteration limit (by design: unscaled regressors); the marks of the table (from the
# captured warnings) must agree with the iteration counts of the fits; checkpoint K45: output file
_k31_n = "; ".join(f"{int((_t72[c].astype(str) != 'True').sum())} of {len(_t72)} at {int(c.replace('Converged_iter', '')):,} iterations"
                   for c in _t72.columns if c.startswith('Converged_iter'))
if P72_FITS:
    _k31_grid = [f for f in P72_FITS if f[3] != 'Phase7_Code_Rerun']
    _k31_rerun = [f for f in P72_FITS if f[3] == 'Phase7_Code_Rerun']
    _k31_iter = [f"{h} Q{int(round(q * 100))} {col}" for h, q, b, col, conv, it in P72_FITS if bool(conv) != (it < b)]
    _k31_csv = ([str(v) for v in _t72[[c for c in _t72.columns if c.startswith('Converged_iter')]].to_numpy().ravel()]
                == [str(bool(f[4])) for f in _k31_grid])
    checkpoint("K31 Phase 7.2: iteration-limit marks", not _k31_iter and _k31_csv,
               f"fits at the iteration limit (by design, unscaled regressors): {_k31_n}; re-run of the first-submission call: "
               f"{sum(not f[4] for f in _k31_rerun)} of {len(_k31_rerun)} at 2,000 iterations; "
               + ("the marks agree with the iteration counts of the fits and with the CSV" if not _k31_iter and _k31_csv else
                  "marks that disagree with the iteration counts: " + (", ".join(_k31_iter) or "none")
                  + ("" if _k31_csv else "; the marks of the CSV differ from those of the fits")), level='warn')
else:
    checkpoint("K31 Phase 7.2: iteration-limit marks", None,
               f"table loaded from the cache of an earlier run: {_k31_n} (marks not re-checked)", level='warn')
_k45_bad = [p for p in [_p72] if not (os.path.isfile(p) and os.path.getsize(p) > 0)]
checkpoint("K45 Phase 7.2: output files", not _k45_bad,
           f"{1 - len(_k45_bad)} of 1 listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 7.2 - QR solver iteration sensitivity (unscaled regressors)",
       lines=["Fits stopped at the iteration limit: " +
              "; ".join(f"{int((_t72[c].astype(str) != 'True').sum())} of {len(_t72)} at {int(c.replace('Converged_iter', '')):,} iterations"
                        for c in _t72.columns if c.startswith('Converged_iter')) + ".",
              ("Phase 7 call of the first submission, re-run (formula interface, unscaled regressors, 2,000 iterations): "
               f"{sum(P72_RERUN_LIMIT)} of {len(P72_RERUN_LIMIT)} fits stopped at the iteration limit (marked * in the printed table)."
               if P72_RERUN_LIMIT else
               "Phase 7 call of the first submission, re-run: iteration-limit flags not recorded (table loaded from the cache)."),
              f"Largest spread of the solar coefficient across the iteration budgets: {(_c72.max(axis=1) - _c72.min(axis=1)).max():,.1f}.",
              "No reference values exist for this step (the pre-test file holds the converged estimates only)."],
       table=_t72, files=[_p72])

### Phase 7.3: Exact QR Solution (Linear Programming) (Revision_Review_Referee_2.5)

Computes the exact quantile-regression solution with the HiGHS linear-programming solver and compares it with
statsmodels IRLS on standardised regressors. The check-function loss identifies the minimising estimate.

Produces: `Phase_7_3_QR_Exact_Solution.csv` in `results/Phase_7/`.

In [ ]:
# =============================================================================
# PHASE 7.3: EXACT QR SOLUTION (LP / HiGHS) vs SCALED IRLS - CACHED
# =============================================================================
# Revision_Review_Referee_2.5 – QR convergence diagnostics & dependence-robust SE
# Feeds Supplementary Appendix J.5.
# =============================================================================
import pandas as pd, numpy as np, os, time
import statsmodels.api as sm
from sklearn.linear_model import QuantileRegressor
import warnings
from sklearn.exceptions import ConvergenceWarning as SkConvergenceWarning

P73_LP = []   # per estimate: (hypothesis, quantile, LP solved); checkpoint K30

def run_exact_solution(df: pd.DataFrame):
    print("\n>>> PHASE 7.3: EXACT QR SOLUTION (LINEAR PROGRAMMING) vs SCALED IRLS")
    path = os.path.join(OUTPUT_DIR_P7, "Phase_7_3_QR_Exact_Solution.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    X, sd = qr_design(df)
    Xs = X.copy(); Xs[:, 1:] = X[:, 1:] / sd     # standardised regressors -> well-conditioned IRLS
    scale = np.r_[1.0, sd]                        # converts scaled coefficients back

    rows = []
    try:
        for dep, label in QR_TARGETS:
            y = df[dep].astype(float).values
            for q in QUANTILES:
                t0 = time.time()
                w_lp = []
                try:
                    with warnings.catch_warnings(record=True) as w_lp:   # solver status of the LP -> K30
                        warnings.simplefilter("always")
                        lp = QuantileRegressor(quantile=q, alpha=0, solver='highs', fit_intercept=False).fit(X, y)
                finally:
                    for x in w_lp:                  # passed on unchanged: shown and recorded (also when the fit raises)
                        warnings.warn_explicit(x.message, x.category, x.filename, x.lineno)
                b_lp = lp.coef_
                P73_LP.append((label, q, not any(issubclass(x.category, SkConvergenceWarning) for x in w_lp)))
                t_lp = time.time() - t0
                res_s, conv_s = fit_quantreg(y, Xs, q, 5000)
                b_s = res_s.params / scale
                res_u, conv_u = fit_quantreg(y, X, q, 1000)   # statsmodels default
                rows.append({'Hypothesis': label, 'Quantile': q,
                             'Coef_Exact_LP': round(b_lp[1], 1),
                             'Coef_Scaled_IRLS': round(b_s[1], 1),
                             'Coef_Unscaled_IRLS_1000': round(res_u.params[1], 1),
                             'Loss_Exact_LP': round(check_loss(y, X, b_lp, q)),
                             'Loss_Scaled_IRLS': round(check_loss(y, X, b_s, q)),
                             'Loss_Unscaled_IRLS_1000': round(check_loss(y, X, res_u.params, q)),
                             'Scaled_IRLS_Converged': conv_s,
                             'Unscaled_IRLS_Converged': conv_u})
                r = rows[-1]
                print(f"  {label:<5} Q{int(q*100):<3} LP={r['Coef_Exact_LP']:>10,.1f}  "
                      f"scaled IRLS={r['Coef_Scaled_IRLS']:>10,.1f} (conv={conv_s})  "
                      f"unscaled IRLS={r['Coef_Unscaled_IRLS_1000']:>10,.1f} (conv={conv_u})  "
                      f"| loss LP={r['Loss_Exact_LP']:,} vs unscaled={r['Loss_Unscaled_IRLS_1000']:,}  [{t_lp:.0f}s]")

        out = pd.DataFrame(rows)
        out.to_csv(path, index=False, **CSV_KW)   # LF line endings on every platform
        print("\n  Note: LP and scaled IRLS coincide and attain the lowest loss -> they are the QR estimates.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in exact QR solution: {e}")
        raise   # an error stops the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_exact_solution(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
_p73 = os.path.join(OUTPUT_DIR_P7, "Phase_7_3_QR_Exact_Solution.csv")
_t73 = pd.read_csv(_p73)
_q73 = 'Q' + (_t73['Quantile'] * 100).round().astype(int).astype(str)
# checkpoint K30: the exact LP solutions (solver status) and the scaled IRLS fits of this step converged;
# checkpoint K45: output file
_k30_irls = [f"{h} Q{int(round(q * 100))}" for h, q, c in zip(_t73['Hypothesis'], _t73['Quantile'], _t73['Scaled_IRLS_Converged'])
             if str(c) != 'True']
_k30_lp = [f"{h} Q{int(round(q * 100))}" for h, q, solved in P73_LP if not solved]
checkpoint("K30 Phase 7.3: exact LP solutions and scaled IRLS fits converged",
           False if (_k30_irls or _k30_lp) else (True if P73_LP else None),
           f"scaled IRLS (5,000 iterations): {len(_t73) - len(_k30_irls)} of {len(_t73)} converged"
           + (f" (at the limit: {', '.join(_k30_irls)})" if _k30_irls else "") + "; exact LP (HiGHS): "
           + ((f"{len(P73_LP) - len(_k30_lp)} of {len(P73_LP)} solved" + (f" (not solved: {', '.join(_k30_lp)})" if _k30_lp else ""))
              if P73_LP else "solver status not recorded (table loaded from the cache of an earlier run)"), level='fail')
_k45_bad = [p for p in [_p73] if not (os.path.isfile(p) and os.path.getsize(p) > 0)]
checkpoint("K45 Phase 7.3: output files", not _k45_bad,
           f"{1 - len(_k45_bad)} of 1 listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 7.3 - Exact QR solution (linear programming) vs IRLS",
       lines=["Largest absolute difference of the solar coefficient from the exact LP solution: scaled IRLS "
              f"{(_t73['Coef_Scaled_IRLS'] - _t73['Coef_Exact_LP']).abs().max():,.1f}, "
              f"unscaled IRLS (1,000 iterations) {(_t73['Coef_Unscaled_IRLS_1000'] - _t73['Coef_Exact_LP']).abs().max():,.1f}.",
              f"Fits that converged: scaled IRLS {int((_t73['Scaled_IRLS_Converged'].astype(str) == 'True').sum())} of {len(_t73)}, "
              f"unscaled IRLS {int((_t73['Unscaled_IRLS_Converged'].astype(str) == 'True').sum())} of {len(_t73)}.",
              "Largest check loss relative to the exact LP solution: scaled IRLS "
              f"{(_t73['Loss_Scaled_IRLS'] / _t73['Loss_Exact_LP']).max():.6f}, "
              f"unscaled IRLS {(_t73['Loss_Unscaled_IRLS_1000'] / _t73['Loss_Exact_LP']).max():.6f}."],
       table=_t73, files=[_p73])
compare_with_reference("Phase 7.3 - exact (LP) quantile-regression point estimates",
                       [(f"{h} {qq} solar coefficient, exact LP (per unit share)", float(c), f"QR.{h}.{qq}.Coef")
                        for h, qq, c in zip(_t73['Hypothesis'], _q73, _t73['Coef_Exact_LP'])],
                       check="K17 Phase 7.3: comparison with the independent pre-test")

### Phase 7.4: Day-Clustered Block-Bootstrap Standard Errors (Revision_Review_Referee_2.5)

Re-estimates the quantile regressions with statsmodels IRLS on standardised regressors (at most 5,000 iterations; a
full-sample fit that reaches the limit is a failed checkpoint (K30); Phase 7.3 compares these fits with the exact LP
solution) and computes standard errors from a day-clustered block bootstrap (whole 24-hour blocks resampled; 200
replications, seed 42).

Produces: `Phase_7_4_QR_Clustered_Bootstrap_SE.csv` in `results/Phase_7/`.

The calendar day of each observation is taken from the time index. A bootstrap replicate
that raises an error is counted (`Boot_OK` below 200), no standard error is
computed for that estimate and checkpoint K32 fails. A replicate whose IRLS fit stops at the 5,000-iteration limit is
kept in the standard error; the iteration-limit warning is captured per replicate, and the number of such replicates
per estimate is printed and written to the run report (checkpoint K32). As a diagnostic (not used in Table J6) each of
them is re-solved with up to 50,000 iterations and the standard error is recomputed, printed and written to the run report
(not to the CSV). A failed checkpoint does not stop the cell: it is reported, and the summary at the end of the run
stops the run.

In [ ]:
# =============================================================================
# PHASE 7.4: DAY-CLUSTERED BLOCK-BOOTSTRAP QR STANDARD ERRORS - CACHED
# =============================================================================
# Revision_Review_Referee_2.5 – QR convergence diagnostics & dependence-robust SE
# Feeds Supplementary Appendix J.5.
# =============================================================================
import pandas as pd, numpy as np, os
import statsmodels.api as sm

N_BOOT_QR = 200
SEED_QR   = 42
MAX_ITER_RESOLVE_QR = 50000   # diagnostic re-solve of the replicates that stop at the 5,000-iteration limit
P74_ITER_LIMIT = []           # per estimate: replicates at the iteration limit and the diagnostic SE (run report;
                              # not CSV columns)
P74_FULL = []                 # per full-sample fit: (hypothesis, quantile, limit reached, IRLS iterations); K30
P74_FAILS = []                # per estimate with failed replicates: (hypothesis, quantile, number, first error); K32

def run_clustered_bootstrap(df: pd.DataFrame, n_boot: int = N_BOOT_QR, seed: int = SEED_QR):
    print("\n>>> PHASE 7.4: DAY-CLUSTERED BLOCK-BOOTSTRAP QR STANDARD ERRORS")
    path = os.path.join(OUTPUT_DIR_P7, "Phase_7_4_QR_Clustered_Bootstrap_SE.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    X, sd = qr_design(df)
    Xs = X.copy(); Xs[:, 1:] = X[:, 1:] / sd      # standardised regressors -> well-conditioned IRLS (see Phase 7.3)
    days = df.index.date                           # calendar day from the time index, not from the Date column
    uniq = np.unique(days)
    day_idx = {dd: np.where(days == dd)[0] for dd in uniq}
    rng = np.random.default_rng(seed)

    rows = []
    try:
        for dep, label in QR_TARGETS:
            y = df[dep].astype(float).values
            for q in QUANTILES:
                pt_res, pt_limit = capture_iteration_limit(lambda: sm.QuantReg(y, Xs).fit(q=q, max_iter=5000))
                P74_FULL.append((label, q, pt_limit, int(pt_res.iterations)))   # checkpoint K30, no stop
                pt = pt_res.params[1] / sd[0]
                bs = []; fails = []                 # failed replicates are recorded, not skipped silently
                at_limit = []                       # per replicate: True = IRLS stopped at the iteration limit
                bs_resolved = []; n_resolve_open = 0   # diagnostic: limit replicates re-solved
                for _ in range(n_boot):
                    sdays = rng.choice(uniq, size=len(uniq), replace=True)   # resample whole days
                    idx = np.concatenate([day_idx[dd] for dd in sdays])
                    try:
                        res_b, hit = capture_iteration_limit(lambda: sm.QuantReg(y[idx], Xs[idx]).fit(q=q, max_iter=5000))
                        bs.append(res_b.params[1] / sd[0]); at_limit.append(hit)
                        if hit:                     # diagnostic: the same replicate with a larger budget
                            res_r, hit_r = capture_iteration_limit(
                                lambda: sm.QuantReg(y[idx], Xs[idx]).fit(q=q, max_iter=MAX_ITER_RESOLVE_QR))
                            bs_resolved.append(res_r.params[1] / sd[0]); n_resolve_open += int(hit_r)
                        else:
                            bs_resolved.append(bs[-1])
                    except Exception as e_rep:
                        fails.append(repr(e_rep))
                if fails:                           # no standard error from an incomplete set of replicates;
                    P74_FAILS.append((label, q, len(fails), fails[0]))   # checkpoint K32 fails, the run continues
                    print(f"  !! {len(fails)} of {n_boot} bootstrap replicates failed for {label} Q{int(q*100)}; "
                          f"first error: {fails[0]}")
                se = float(np.std(bs)) if not fails else np.nan; t = pt / se if se else np.nan
                n_limit = int(np.sum(at_limit))
                se_resolved = float(np.std(bs_resolved)) if not fails else np.nan   # diagnostic only, not used in Table J6
                stars = '***' if abs(t) > 2.58 else ('**' if abs(t) > 1.96 else ('*' if abs(t) > 1.645 else ''))
                rows.append({'Hypothesis': label, 'Quantile': q, 'Coef': round(pt, 1),
                             'Clustered_SE': round(se, 1), 't_stat': round(t, 2), 'Sig': stars,
                             'Boot_OK': len(bs)})
                P74_ITER_LIMIT.append({'Hypothesis': label, 'Quantile': q, 'Boot_Iter_Limit': n_limit,
                                       'Boot_Resolve_At_Limit': n_resolve_open, 'Clustered_SE_Resolved': se_resolved, 'Clustered_SE': se})
                print(f"  {label:<5} Q{int(q*100):<3} beta={pt:>10,.0f}  clustered-SE={se:>7,.0f}  t={t:>6.1f} {stars:<3}"
                      f"  replicates at the iteration limit: {n_limit} of {len(bs)}; SE with them re-solved "
                      + (f"{se_resolved:,.1f} ({(se_resolved - se) / se:+.4%})" if np.isfinite(se) and se > 0
                         else "not computed (failed replicates)"))

        out = pd.DataFrame(rows)
        out.to_csv(path, index=False, **CSV_KW)   # LF line endings on every platform
        print("\n" + out.to_string(index=False))
        print("\n  Note: *** p<0.01, ** p<0.05, * p<0.10.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in clustered bootstrap: {e}")
        raise   # an error stops the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_clustered_bootstrap(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
_p74 = os.path.join(OUTPUT_DIR_P7, "Phase_7_4_QR_Clustered_Bootstrap_SE.csv")
_t74 = pd.read_csv(_p74)
_q74 = 'Q' + (_t74['Quantile'] * 100).round().astype(int).astype(str)
# checkpoints K30 (full-sample fits of this step converged), K32 (bootstrap replicates: errors, iteration limit)
# and K45 (output file). The error count is read from Boot_OK of the CSV, so it also covers a table loaded from the cache.
if P74_FULL:
    _k30_hit = [f"{h} Q{int(round(q * 100))}" for h, q, hit, _ in P74_FULL if hit]
    checkpoint("K30 Phase 7.4: full-sample quantile regressions converged", not _k30_hit,
               f"{len(P74_FULL) - len(_k30_hit)} of {len(P74_FULL)} fits converged before the 5,000-iteration limit "
               f"(IRLS iterations {min(n for *_, n in P74_FULL):,} to {max(n for *_, n in P74_FULL):,})"
               + (f"; at the limit (the reported coefficient is not a converged estimate): {', '.join(_k30_hit)}" if _k30_hit else ""),
               level='fail')
else:
    checkpoint("K30 Phase 7.4: full-sample quantile regressions converged", None,
               "table loaded from the cache of an earlier run (convergence of the full-sample fits not recorded)", level='fail')
_k32_lab = [f"{h} {qq}" for h, qq in zip(_t74['Hypothesis'], _q74)]
_k32_err = [f"{l}: {int(b)} of {N_BOOT_QR}" for l, b in zip(_k32_lab, _t74['Boot_OK']) if int(b) != N_BOOT_QR]
checkpoint("K32 Phase 7.4: bootstrap replicates without an error", not _k32_err,
           (f"{N_BOOT_QR} of {N_BOOT_QR} replicates without an error for each of the {len(_t74)} estimates" if not _k32_err else
            "replicates without an error (no standard error for these estimates): " + "; ".join(_k32_err)
            + "".join(f"; first error {h} Q{int(round(q * 100))}: {str(err)[:200]}" for h, q, _, err in P74_FAILS)), level='fail')
if len(P74_ITER_LIMIT) == len(_t74):
    _k32_n, _k32_tot = sum(r['Boot_Iter_Limit'] for r in P74_ITER_LIMIT), int(_t74['Boot_OK'].sum())
    _k32_dse = [abs(r['Clustered_SE_Resolved'] - r['Clustered_SE']) / r['Clustered_SE'] for r in P74_ITER_LIMIT
                if r['Clustered_SE'] and np.isfinite(r['Clustered_SE'])]
    checkpoint("K32 Phase 7.4: bootstrap replicates at the iteration limit", _k32_n == 0,
               f"{_k32_n:,} of {_k32_tot:,} replicates ({_k32_n / max(_k32_tot, 1):.2%}) stopped at the 5,000-iteration limit "
               "and are kept in the standard error: "
               + "; ".join(f"{l} {r['Boot_Iter_Limit']} of {int(b)} ({r['Boot_Iter_Limit'] / max(int(b), 1):.1%})"
                           for l, r, b in zip(_k32_lab, P74_ITER_LIMIT, _t74['Boot_OK']))
               + f"; diagnostic re-solve with up to {MAX_ITER_RESOLVE_QR:,} iterations: "
               f"{sum(r['Boot_Resolve_At_Limit'] for r in P74_ITER_LIMIT)} still at the limit, largest change of the clustered SE "
               + (f"{max(_k32_dse):.3%}" if _k32_dse else "not available") + " (Table J6 keeps the replicates as estimated)", level='warn')
else:
    checkpoint("K32 Phase 7.4: bootstrap replicates at the iteration limit", None,
               "table loaded from the cache of an earlier run (replicates at the iteration limit not recorded)", level='warn')
_k45_bad = [p for p in [_p74] if not (os.path.isfile(p) and os.path.getsize(p) > 0)]
checkpoint("K45 Phase 7.4: output files", not _k45_bad,
           f"{1 - len(_k45_bad)} of 1 listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 7.4 - Day-clustered block-bootstrap QR standard errors (Supplementary Table J6)",
       lines=[f"Successful bootstrap replications per estimate: {int(_t74['Boot_OK'].min())} to {int(_t74['Boot_OK'].max())} "
              f"of {N_BOOT_QR} (seed {SEED_QR}, whole days resampled).",
              *([("Bootstrap replicates whose IRLS fit stopped at the 5,000-iteration limit (kept in the standard error): " +
                  "; ".join(f"{r['Hypothesis']} Q{int(round(r['Quantile'] * 100))} {r['Boot_Iter_Limit']}" for r in P74_ITER_LIMIT) +
                  f" ({sum(r['Boot_Iter_Limit'] for r in P74_ITER_LIMIT)} of {int(_t74['Boot_OK'].sum()):,} replicates)."),
                 (f"Diagnostic (not used in Table J6): clustered SE with these replicates re-solved with up to "
                  f"{MAX_ITER_RESOLVE_QR:,} iterations: " +
                  "; ".join(f"{r['Hypothesis']} Q{int(round(r['Quantile'] * 100))} {r['Clustered_SE_Resolved']:,.1f} (reported {s:,.1f})"
                            for r, s in zip(P74_ITER_LIMIT, _t74['Clustered_SE'])) +
                  f"; re-solved replicates still at the limit: {sum(r['Boot_Resolve_At_Limit'] for r in P74_ITER_LIMIT)}.")]
                if len(P74_ITER_LIMIT) == len(_t74) else
                ["Bootstrap replicates at the iteration limit: not recorded (table loaded from the cache)."]),
              f"|t| of the solar coefficient: {_t74['t_stat'].abs().min():.2f} to {_t74['t_stat'].abs().max():.2f}; "
              f"|t| > 2.58 in {int((_t74['t_stat'].abs() > 2.58).sum())} of {len(_t74)} estimates."],
       table=_t74, files=[_p74])
compare_with_reference("Phase 7.4 - quantile regression with day-clustered bootstrap (Table J6)",
                       [(f"{h} {qq} {name}", value, f"QR.{h}.{qq}.{rid}")
                        for h, qq, c, se, t, b in zip(_t74['Hypothesis'], _q74, _t74['Coef'], _t74['Clustered_SE'],
                                                      _t74['t_stat'], _t74['Boot_OK'])
                        for name, value, rid in [('solar coefficient (per unit share)', float(c), 'Coef'),
                                                 ('effect per percentage point', float(c) / 100, 'yp'),
                                                 ('clustered bootstrap SE', float(se), 'Clustered_SE'),
                                                 ('t statistic', float(t), 't_stat'),
                                                 ('successful bootstrap replications', int(b), 'Boot_OK')]],
                       check="K17 Phase 7.4: comparison with the independent pre-test")

## Phase 8: Non-Linear GAM Robustness (Cubic Spline)

### Phase 8.1: GAM Flexibility-Stress Analysis

Fits separate GAM models (OLS with natural cubic splines) for the surplus and deficit regimes to validate the non-linear flexibility-stress pattern identified in Phase 6, providing confidence intervals around the regime-specific a-FRR price trajectories.

In [ ]:
# =============================================================================
# PHASE 8: ROBUSTNESS CHECKS - NON-LINEAR GAM ANALYSIS (CUBIC SPLINE)
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from patsy import cr
import warnings
import os

# no blanket warning filter (see the setup cell)

# cell-specific names (OUTPUT_DIR_P8 / REPORT_PATH_P8 / PLOT_PATH_P8)
OUTPUT_DIR_P8 = os.path.join(RESULTS_BASE, "Phase_8")
os.makedirs(OUTPUT_DIR_P8, exist_ok=True)

REPORT_PATH_P8 = os.path.join(OUTPUT_DIR_P8, "Phase_8_Flexibility_Stress_Report.txt")
PLOT_PATH_P8 = os.path.join(OUTPUT_DIR_P8, "Phase_8_Figure_10_GAM_Flexibility_Stress.png")

plt.rcParams.update({
    'figure.figsize': (12, 7),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],   # same typography as the other figures (was sans-serif)
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def execute_final_gam_analysis(df):
    print("PHASE 8: GAM Robustness Analysis (Cubic Spline)...")
    print("-" * 80)

    # Regime separation
    df_up_combined = df[df['System_Direction_Down'] == 0].copy()
    df_down = df[df['System_Direction_Down'] == 1].copy()

    print(f">> [Step 1/3] Data split: Deficit+Balanced ({len(df_up_combined)} obs), Surplus ({len(df_down)} obs)")
    print(">> [Step 2/3] Fitting OLS + Natural Cubic Splines models...")

    # Cache check
    if os.path.exists(REPORT_PATH_P8) and os.path.exists(PLOT_PATH_P8):
        print(f"\n>> [CACHE] Results found in '{OUTPUT_DIR_P8}'. Loading...")

        with open(REPORT_PATH_P8, "r", encoding="utf-8") as f:
            print(f.read())

        img = plt.imread(PLOT_PATH_P8)
        plt.figure(figsize=(12, 7))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Phase 8 - Figure 10: Non-Linear Flexibility-Stress Pattern (Preview - Loaded)", fontsize=14)
        plt.show()

        print(f">> Figure loaded: '{os.path.basename(PLOT_PATH_P8)}'")
        return

    # Fit GAM models
    formula = 'Q("a-FRR_Price") ~ cr(Solar_Generation_Share, df=6) + Consumption + Wind_Generation + Import_Coal_Generation'
    model_up = smf.ols(formula, data=df_up_combined).fit()
    model_down = smf.ols(formula, data=df_down).fit()

    solar_grid = np.linspace(df['Solar_Generation_Share'].min(), df['Solar_Generation_Share'].max(), 100)
    pred_df = pd.DataFrame({
        'Solar_Generation_Share': solar_grid,
        'Consumption': df['Consumption'].mean(),
        'Wind_Generation': df['Wind_Generation'].mean(),
        'Import_Coal_Generation': df['Import_Coal_Generation'].mean()
    })

    res_up = model_up.get_prediction(pred_df).summary_frame(alpha=0.05)
    res_down = model_down.get_prediction(pred_df).summary_frame(alpha=0.05)

    # Visualization
    print(">> [Step 3/3] Generating figure...")
    plt.figure(figsize=(12, 7))
    solar_pct = solar_grid * 100

    # [Revision_R2_Reviewer4_C9] confidence bands are data: kept, drawn lighter and named in the legend
    plt.plot(solar_pct, res_up['mean'], color='green', lw=3, label='Energy Deficit & Balanced')
    plt.fill_between(solar_pct, res_up['mean_ci_lower'], res_up['mean_ci_upper'], color='green', alpha=0.12,
                     linewidth=0, label='95% CI (Deficit & Balanced)')

    plt.plot(solar_pct, res_down['mean'], color='red', lw=3, label='Energy Surplus')
    plt.fill_between(solar_pct, res_down['mean_ci_lower'], res_down['mean_ci_upper'], color='red', alpha=0.12,
                     linewidth=0, label='95% CI (Surplus)')

    plt.xlabel("Solar Generation Share (%)")
    plt.ylabel("Simulated a-FRR Price (TL/MW)")  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW); plain text, not mathtext
    plt.axvline(x=20, color='grey', linestyle='--', alpha=0.6, label='~20% onset')
    plt.legend(loc='upper left', frameon=True, shadow=False)  # [Revision_Editor_E6] no shaded legend
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    # Save clean, then preview with title
    plt.savefig(PLOT_PATH_P8, dpi=FIG_DPI, bbox_inches='tight')
    plt.title("Phase 8 - Figure 10: Non-Linear Flexibility-Stress Pattern (Preview Only)", fontsize=14)
    plt.show()

    # Numerical analysis
    gap_start = res_up['mean'].iloc[0] - res_down['mean'].iloc[0]
    gap_end = res_up['mean'].iloc[-1] - res_down['mean'].iloc[-1]

    report_content = (
        "\n============================================================\n"
        "PHASE 8 NUMERICAL ANALYSIS RESULTS:\n"
        "------------------------------------------------------------\n"
        f"Starting Price Gap (0% Solar): {gap_start:.2f} TL\n"
        f"Ending Price Gap (Max Solar):  {gap_end:.2f} TL\n"
        f"Gap Expansion Factor:      {gap_end/gap_start:.2f}x\n"
        "============================================================"
    )

    with open(REPORT_PATH_P8, "w", encoding="utf-8") as f:
        f.write(report_content)

    print(report_content)
    print(f">> Figure saved: '{os.path.basename(PLOT_PATH_P8)}'")

# --- Execution ---
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        execute_final_gam_analysis(target_df)
    else:
        print("Error: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
import re
with open(REPORT_PATH_P8, "r", encoding="utf-8") as _f:
    _gam_txt = _f.read()
_gam = {k: float(re.search(re.escape(k) + r"[^:]*:\s*(-?\d+(?:\.\d+)?)", _gam_txt).group(1))
        for k in ['Starting Price Gap', 'Ending Price Gap', 'Gap Expansion Factor']}   # the three numbers of the saved text report
_k45_files = [REPORT_PATH_P8, PLOT_PATH_P8]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 8.1: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 8.1 - GAM flexibility-stress pattern (Supplementary Figure K.1)",
       lines=[f"Fitted a-FRR price, deficit+balanced minus surplus: {_gam['Starting Price Gap']:,.2f} TL/MW at the lowest solar share "
              f"and {_gam['Ending Price Gap']:,.2f} TL/MW at the highest (ratio {_gam['Gap Expansion Factor']:.2f}x).",
              "Read from the saved text report; the fitted curves and their 95% confidence bands are in the figure."],
       files=_k45_files)
compare_with_reference("Phase 8.1 - GAM report vs. pre-test (pre-test sheet Hizli_Analizler, quick analyses)",
                       [("Starting price gap (TL/MW)", _gam['Starting Price Gap'], "GAMRAPOR.Starting_Price_Gap_TL"),
                        ("Ending price gap (TL/MW)", _gam['Ending Price Gap'], "GAMRAPOR.Ending_Price_Gap_TL"),
                        ("Gap expansion factor (x)", _gam['Gap Expansion Factor'], "GAMRAPOR.Gap_Expansion_Factor_x")],
                       check="K17 Phase 8.1: comparison with the independent pre-test")

### Phase 8.2: Statistical Identification of the Transition Onset (Revision_Review_Referee_2.4)

Estimates a **continuous regression-kink model** with an unknown threshold (Hansen, 2017) for the a-FRR price,

$$y_t = a + b\,s_t + \delta\,(s_t-\gamma)_+ + X_t'c + u_t,$$

where $s_t$ is the solar share (%), $(s_t-\gamma)_+ = \max(s_t-\gamma, 0)$ and $X_t$ is the manuscript's control
set. The threshold $\gamma$ is located by least squares over candidate solar shares of 5–44% (step 1 pp); it is a
*kink* (the slope changes from $b$ to $b+\delta$, the fitted line stays continuous), not the discontinuous
sample-splitting model of Hansen (2000). Reported: the 90% confidence interval for $\gamma$ from a day-clustered
bootstrap (whole days resampled; 200 replications, seed 42); the kink coefficient $\delta$ with its day-clustered
bootstrap standard error; and a sup-F test of $H_0{:}\ \delta = 0$ (no kink) whose p-value comes from a
day-clustered wild bootstrap under the null (Rademacher weights per day; 200 replications). This identifies the
~20% onset of balancing-cost acceleration. The higher crossover is not estimated here (it is computed and saved by Phase 6.2): it is the surplus–deficit
crossover of the Phase 6.2 XGBoost interaction simulation.

Reference: Hansen, B. E. (2017). Regression kink with an unknown threshold. *Journal of Business & Economic
Statistics*, 35(2), 228–240. https://doi.org/10.1080/07350015.2015.1073595

Produces: `Phase_8_2_Hansen_Threshold.csv` in `results/Phase_8/`.


In [ ]:
# =============================================================================
# PHASE 8.2: REGRESSION-KINK THRESHOLD (HANSEN, 2017) + DAY-CLUSTERED BOOTSTRAP - CACHED
# =============================================================================
# Revision_Review_Referee_2.4 – Statistical identification of the ~20% threshold
# Feeds Supplementary Appendix J.4.
# Model: continuous regression kink  y = a + b*s + delta*(s - gamma)_+ + Xc*c
#        Hansen, B. E. (2017). Regression kink with an unknown threshold.
#        Journal of Business & Economic Statistics, 35(2), 228-240.
#        (NOT the discontinuous sample-splitting model of Hansen, 2000.)
# Outputs: gamma_hat (grid search); 90% CI from a day-clustered (whole-day) bootstrap;
#          delta (slope change at the kink) with day-clustered bootstrap SE and t;
#          sup-F test of H0: delta = 0 with a day-clustered wild-bootstrap p-value.
# NOTE: the upper crossover is NOT estimated here; it is the surplus/deficit crossover of the
#       Phase 6.2 XGBoost interaction simulation (confounder-controlled PDP).
# =============================================================================
import pandas as pd, numpy as np, os

THRESHOLD_CONTROLS = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation',
                      'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
                      'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
                      'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
N_BOOT_THR = 200
SEED_THR   = 42
KINK_BOOT_RECORD = {}   # finite bootstrap replicates of an estimation in this kernel (checkpoint K40); empty = cache

# the cell defines its own output folder (Phase_8 under RESULTS_BASE)
OUTPUT_DIR_P8 = os.path.join(RESULTS_BASE, "Phase_8")
os.makedirs(OUTPUT_DIR_P8, exist_ok=True)

def kink_design(s, Xc, g):
    """Design matrix of the continuous-kink model: [1, s, (s-g)_+, Xc]."""
    return np.column_stack([np.ones(len(s)), s, np.maximum(s - g, 0.0), Xc])

def kink_fit(y, s, Xc, g):
    """OLS of the kink model at threshold g -> (beta, SSR); beta = [a, b, delta, c...]."""
    X = kink_design(s, Xc, g)
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    return beta, float(((y - X @ beta) ** 2).sum())

def kink_ssr(y, s, Xc, g):
    """SSR of the continuous-kink model y = a + b*s + d*(s-g)_+ + Xc*c."""
    return kink_fit(y, s, Xc, g)[1]

def kink_profile(y, s, Xc, grid):
    """Least-squares grid search -> (gamma_hat, beta_hat at gamma_hat, min SSR)."""
    ssr = np.array([kink_ssr(y, s, Xc, g) for g in grid])
    i = int(ssr.argmin())
    beta, _ = kink_fit(y, s, Xc, grid[i])
    return float(grid[i]), beta, float(ssr[i])

def run_threshold_regression(df: pd.DataFrame, dep: str = 'a-FRR_Price',
                             n_boot: int = N_BOOT_THR, seed: int = SEED_THR):
    print("\n>>> PHASE 8.2: REGRESSION-KINK THRESHOLD (HANSEN, 2017) + DAY-CLUSTERED BOOTSTRAP")
    path = os.path.join(OUTPUT_DIR_P8, "Phase_8_2_Hansen_Threshold.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).T.to_string(header=False)); return

    grid = np.arange(5, 45, 1.0)   # candidate thresholds: 5%-44% solar share
    d = df.assign(solar_pct=df['Solar_Generation_Share'] * 100.0).dropna(subset=[dep, 'solar_pct'] + THRESHOLD_CONTROLS)
    y, s = d[dep].astype(float).values, d['solar_pct'].values
    Xc = d[THRESHOLD_CONTROLS].astype(float).values
    n, k = len(y), Xc.shape[1] + 3            # k = parameters of the kink model
    try:
        # --- point estimates -------------------------------------------------
        ghat, beta_hat, ssr1 = kink_profile(y, s, Xc, grid)
        b_pre, delta = float(beta_hat[1]), float(beta_hat[2])
        # H0 (no kink): the same regression without the (s - gamma)_+ term
        X0 = np.column_stack([np.ones(n), s, Xc])
        b0, _, _, _ = np.linalg.lstsq(X0, y, rcond=None)
        fit0 = X0 @ b0; e0 = y - fit0; ssr0 = float((e0 ** 2).sum())
        supF = ((ssr0 - ssr1) / 1.0) / (ssr1 / (n - k))   # sup over the grid = F at gamma_hat

        days = d.index.date   # calendar day from the time index, not from the 'Date' column (both give the same days)
        uniq, day_id = np.unique(days, return_inverse=True)
        day_idx = {dd: np.where(days == dd)[0] for dd in uniq}
        rng = np.random.default_rng(seed)

        # --- (i) day-clustered bootstrap of (gamma, delta): CI for gamma, SE for delta ---
        boot_g, boot_d = [], []
        for _ in range(n_boot):
            sdays = rng.choice(uniq, size=len(uniq), replace=True)   # resample whole days
            idx = np.concatenate([day_idx[dd] for dd in sdays])
            g_b, beta_b, _ = kink_profile(y[idx], s[idx], Xc[idx], grid)
            boot_g.append(g_b); boot_d.append(float(beta_b[2]))
        boot_g, boot_d = np.array(boot_g), np.array(boot_d)
        delta_se = float(boot_d.std(ddof=1)); delta_t = delta / delta_se if delta_se > 0 else np.nan

        # --- (ii) sup-F p-value: day-clustered wild bootstrap under H0 (Rademacher weight per day) ---
        supF_b = []
        for _ in range(n_boot):
            w = rng.choice([-1.0, 1.0], size=len(uniq))
            y_b = fit0 + w[day_id] * e0                          # null model + sign-flipped day blocks of residuals
            _, _, ssr1_b = kink_profile(y_b, s, Xc, grid)
            b0b, _, _, _ = np.linalg.lstsq(X0, y_b, rcond=None)
            ssr0_b = float(((y_b - X0 @ b0b) ** 2).sum())
            supF_b.append(((ssr0_b - ssr1_b) / 1.0) / (ssr1_b / (n - k)))
        supF_b = np.array(supF_b)
        p_supF = float((supF_b >= supF).mean())
        KINK_BOOT_RECORD.update(n_boot=n_boot, gamma_delta=int((np.isfinite(boot_g) & np.isfinite(boot_d)).sum()),
                                supF=int(np.isfinite(supF_b).sum()))   # successful replicates (checkpoint K40)

        out = pd.DataFrame([{'Dependent': dep, 'Threshold_Point_Estimate_pct': ghat,
                             'CI90_Lower_pct': np.percentile(boot_g, 5), 'CI90_Upper_pct': np.percentile(boot_g, 95),
                             'Bootstrap_Median_pct': np.median(boot_g), 'N_Boot': n_boot, 'Seed': seed,
                             'Grid': '5-44% (step 1)', 'N': n,
                             'slope_pre_kink_b': b_pre, 'delta': delta, 'delta_se': delta_se, 'delta_t': delta_t,
                             'slope_post_kink_b_plus_delta': b_pre + delta,
                             'supF_stat': supF, 'supF_pval_bootstrap': p_supF, 'N_Boot_supF': n_boot,
                             'supF_boot_method': 'day-clustered wild bootstrap (Rademacher) under H0: delta = 0',
                             'Model': 'continuous regression kink, Hansen (2017) JBES 35(2)'}])
        out.to_csv(path, index=False, **CSV_KW)
        print(out.T.to_string(header=False))
        print(f"\n  gamma_hat = {ghat:.0f}%  90% CI [{np.percentile(boot_g, 5):.0f}%, {np.percentile(boot_g, 95):.0f}%]"
              f"  |  delta = {delta:,.1f} TL/MW per pp (boot SE {delta_se:,.1f}, t = {delta_t:.1f})"
              f"  |  sup-F = {supF:.1f}, bootstrap p = {p_supF:.3f} (resolution 1/{n_boot})")
        print("  Note: onset (this phase) and crossover (Phase 6.2, Phase_6_2_Flexibility_Trap_Crossover.csv) are complementary.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in threshold regression: {e}")
        raise   # errors stop the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_threshold_regression(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
_kink_csv = os.path.join(OUTPUT_DIR_P8, "Phase_8_2_Hansen_Threshold.csv")
_kink = pd.read_csv(_kink_csv).iloc[0]
# checkpoint K40: successful bootstrap replicates = N_BOOT_THR, and the 90% interval contains the estimate
if KINK_BOOT_RECORD:
    _k40_ok = (KINK_BOOT_RECORD['n_boot'] == N_BOOT_THR and KINK_BOOT_RECORD['gamma_delta'] == N_BOOT_THR
               and KINK_BOOT_RECORD['supF'] == N_BOOT_THR)
    _k40_txt = (f"estimated in this kernel: {KINK_BOOT_RECORD['gamma_delta']} of {KINK_BOOT_RECORD['n_boot']} (gamma, delta) replicates and "
                f"{KINK_BOOT_RECORD['supF']} of {KINK_BOOT_RECORD['n_boot']} sup-F replicates finite (N_BOOT_THR = {N_BOOT_THR})")
else:
    _k40_fin = bool(np.isfinite([float(_kink[_c]) for _c in ('CI90_Lower_pct', 'CI90_Upper_pct', 'delta_se', 'supF_pval_bootstrap')]).all())
    _k40_ok = int(_kink['N_Boot']) == N_BOOT_THR and int(_kink['N_Boot_supF']) == N_BOOT_THR and _k40_fin
    _k40_txt = (f"table loaded from its cache: {int(_kink['N_Boot'])} (gamma, delta) and {int(_kink['N_Boot_supF'])} sup-F replicates "
                f"(N_BOOT_THR = {N_BOOT_THR}); the table is written only after every replicate has finished (a failing replicate "
                "stops the cell); " + ("its interval and bootstrap SE are finite, so no (gamma, delta) replicate is NaN" if _k40_fin else
                "its interval, bootstrap SE or p-value is NOT finite") + "; finite sup-F replicates are not recorded in the table")
checkpoint("K40 Phase 8.2: kink bootstrap replicates", _k40_ok, _k40_txt, level='warn')
_k40_g, _k40_lo, _k40_hi = (float(_kink[_c]) for _c in ('Threshold_Point_Estimate_pct', 'CI90_Lower_pct', 'CI90_Upper_pct'))
checkpoint("K40 Phase 8.2: interval contains the threshold estimate", bool(_k40_lo <= _k40_g <= _k40_hi),
           f"gamma = {_k40_g:.0f}% solar share, 90% day-clustered bootstrap interval [{_k40_lo:.0f}%, {_k40_hi:.0f}%]", level='warn')
_k45_files = [_kink_csv]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 8.2: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 8.2 - regression-kink threshold (Supplementary Table J4)",
       lines=[f"Threshold (gamma) = {_kink['Threshold_Point_Estimate_pct']:.0f}% solar share; 90% interval [{_kink['CI90_Lower_pct']:.0f}%, "
              f"{_kink['CI90_Upper_pct']:.0f}%] from the day-clustered bootstrap ({int(_kink['N_Boot'])} replications, seed {int(_kink['Seed'])}); "
              f"N = {int(_kink['N']):,}.",
              f"Slope below / above the threshold: {_kink['slope_pre_kink_b']:,.2f} / {_kink['slope_post_kink_b_plus_delta']:,.2f} TL/MW per "
              f"percentage point (delta = {_kink['delta']:,.2f}, bootstrap SE {_kink['delta_se']:,.2f}, t = {_kink['delta_t']:.2f}).",
              f"sup-F = {_kink['supF_stat']:,.1f}; day-clustered wild-bootstrap p = {_kink['supF_pval_bootstrap']:.3f} "
              f"(resolution 1/{int(_kink['N_Boot_supF'])})."],
       files=_k45_files)
# Scale: the kink model and the reference both use the solar share in percent (threshold in %, slopes per percentage point).
_kink_cols = ['Threshold_Point_Estimate_pct', 'CI90_Lower_pct', 'CI90_Upper_pct', 'Bootstrap_Median_pct', 'slope_pre_kink_b', 'delta',
              'slope_post_kink_b_plus_delta', 'delta_se', 'delta_t', 'supF_stat', 'supF_pval_bootstrap']
compare_with_reference("Phase 8.2 - regression kink vs. pre-test (sheet Kink_J4)", [(c, _kink[c], f"KINK.{c}") for c in _kink_cols],
                       check="K17 Phase 8.2: comparison with the independent pre-test")


## Phase 9: Methodological Framework Visualization

Generates a structured flowchart summarizing the entire research methodology, from data preparation through robust estimation and validation to final reporting.

In [ ]:
# =============================================================================
# PHASE 9: METHODOLOGICAL FRAMEWORK (WITH LOGISTIC REGRESSION)
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
import os

# no blanket warning filter (see the setup cell)

OUTPUT_DIR = os.path.join(RESULTS_BASE, "Phase_9")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def generate_maximized_flowchart():
    print(f"Generating Methodological Framework (Saving to '{OUTPUT_DIR}')...")
    print("="*80)

    # Color palette
    COLOR_ORANGE = '#F58231'
    COLOR_BLUE_DARK = '#3A607E'
    COLOR_GREY = '#708090'
    COLOR_BLUE_LIGHT = '#9FD0E2'
    COLOR_WHITE = '#FFFFFF'
    COLOR_LINE = '#2C3E50'

    fig, ax = plt.subplots(figsize=(16, 22))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 22)
    ax.axis('off')

    # --- Helper: Section Banner ---
    # [Revision_Editor_E6] no background colour: white panel, section colour kept on frame and title
    def draw_banner(x, y, w, h, color, title):
        t_color = '#2E7FA0' if color == COLOR_BLUE_LIGHT else color   # light blue is too pale for text
        rect = patches.Rectangle((x, y), w, h, linewidth=4, edgecolor=t_color, facecolor=COLOR_WHITE, zorder=1)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h - 0.6, title.upper(), ha='center', va='center',
                fontsize=24, fontweight='bold', color=t_color, zorder=3)

    # --- Helper: Inner Box ---
    def draw_inner_box(x, y, w, h, text, font_size=18):
        rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor=COLOR_LINE, facecolor=COLOR_WHITE, zorder=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center',
                fontsize=font_size, wrap=True, zorder=3, linespacing=1.3, fontweight='medium')

    # --- Helper: Transition Arrows ---
    def draw_thick_transition(x, y_start, y_end):
        ax.annotate('', xy=(x, y_end), xytext=(x, y_start),
                    arrowprops=dict(arrowstyle='simple, head_width=1.5, head_length=1.5',
                                   facecolor=COLOR_LINE, edgecolor=COLOR_LINE, zorder=5))

    def draw_arrow(x_start, y_start, x_end, y_end):
        ax.annotate('', xy=(x_end, y_end), xytext=(x_start, y_start),
                    arrowprops=dict(arrowstyle='->', color=COLOR_LINE, lw=2.5, mutation_scale=10, zorder=4))

    # =============================================================================
    # ROW 1: DATA PREPARATION (Orange)
    # =============================================================================
    r1_y = 17.5
    draw_banner(0, r1_y, 14, 4.0, COLOR_ORANGE, "1. Data Preparation & Initial Modeling")

    box_y1 = r1_y + 0.5
    b_w = 2.4

    draw_inner_box(0.3, box_y1, b_w, 2.5, "Input:\nMCP, a-FRR,\nSolar, Cons.")
    draw_inner_box(3.0, box_y1, b_w, 2.5, "Feature Eng:\nSeasonality &\nInteractions")
    draw_inner_box(5.7, box_y1, b_w, 2.5, "Stationarity\nChecks\n(ADF & KPSS)")  # [Revision_R2_M6] both tests, as in Section 3.2.2
    draw_inner_box(8.4, box_y1, b_w, 2.5, "Initial Models\n(8 SARIMAX)")
    draw_inner_box(11.1, box_y1, b_w, 2.5, "Logistic\nRegression")

    draw_arrow(0.3 + b_w, box_y1 + 1.25, 3.0, box_y1 + 1.25)
    draw_arrow(3.0 + b_w, box_y1 + 1.25, 5.7, box_y1 + 1.25)
    draw_arrow(5.7 + b_w, box_y1 + 1.25, 8.4, box_y1 + 1.25)
    draw_arrow(8.4 + b_w, box_y1 + 1.25, 11.1, box_y1 + 1.25)

    draw_thick_transition(7.0, 17.5, 16.5)

    # =============================================================================
    # ROW 2: OPTIMIZATION (Dark Blue)
    # =============================================================================
    r2_y = 12.5
    draw_banner(0, r2_y, 14, 4.0, COLOR_BLUE_DARK, "2. Parameter Optimization & Robust Estimation")

    box_y2 = r2_y + 0.5
    draw_inner_box(1.0, box_y2, 3.5, 2.5, "Grid Search:\n216 Combinations\nper Model")
    draw_inner_box(5.25, box_y2, 3.5, 2.5, "Optimal Params\n(p,d,q)(P,D,Q)s")
    draw_inner_box(9.5, box_y2, 3.5, 2.5, "Robust Re-Est. &\nDiagnostic Checks")

    draw_arrow(4.5, box_y2 + 1.25, 5.25, box_y2 + 1.25)
    draw_arrow(8.75, box_y2 + 1.25, 9.5, box_y2 + 1.25)
    draw_thick_transition(7.0, 12.5, 11.5)

    # =============================================================================
    # ROW 3: CHECKS & VALIDATION (Grey)
    # =============================================================================
    r3_y = 7.5
    draw_banner(0, r3_y, 14, 4.0, COLOR_GREY, "3. Visual, ML & Robustness Checks")

    box_y3 = r3_y + 0.5
    draw_inner_box(0.5, box_y3, 3.0, 2.5, "Visual:\nHourly Coefficient\nProfiles")  # [Revision_R2_M6] no IRF analysis in the paper; Figures 3 and 5
    draw_inner_box(3.85, box_y3, 3.0, 2.5, "ML (XGBoost):\nNon-Linear\nPDP Curves")
    draw_inner_box(7.15, box_y3, 3.0, 2.5, "GAM Analysis:\nTransition Regime")
    draw_inner_box(10.5, box_y3, 3.0, 2.5, "Quantile Reg:\nDistributional\nValidation")

    # =============================================================================
    # ROW 4: REPORTING & FUTURE (Light Blue)
    # =============================================================================
    r4_y = 1.0
    draw_banner(0, r4_y, 6.5, 5.5, COLOR_BLUE_LIGHT, "4. Final Reporting")
    draw_inner_box(0.75, r4_y + 0.5, 5.0, 3.5,
                   "Supplementary Tables\nHeatmaps & Matrices\nFull Model Specs\nMethodological Report",
                   font_size=20)

    draw_banner(7.5, r4_y, 6.5, 5.5, COLOR_BLUE_LIGHT, "Future Extensions")
    rect_f = patches.Rectangle((8.0, r4_y + 0.5), 5.5, 3.5, linewidth=2, edgecolor=COLOR_LINE, facecolor=COLOR_WHITE, zorder=2)
    ax.add_patch(rect_f)
    future_text = "Battery Storage Integration\nPolicy Scenario Analysis\nCapacity Expansion Impact"
    ax.text(10.75, r4_y + 2.25, future_text, ha='center', va='center',
            fontsize=20, fontweight='medium', linespacing=2.0, zorder=3)

    draw_thick_transition(3.25, 7.5, 6.5)
    draw_thick_transition(10.75, 7.5, 6.5)

    # Save
    output_path = os.path.join(OUTPUT_DIR, 'Figure_3_Methodological_Framework_Updated.png')
    plt.savefig(output_path, dpi=FIG_DPI_LARGE, bbox_inches='tight')  # [Revision_R2_Reviewer4_C9] large canvas -> ~9,500 px wide
    retag_png_dpi(output_path, FIG_DPI)  # same pixels, dpi tag 1000 like every other figure

    print(f"\n>> Methodological framework saved to: '{output_path}'")
    plt.show()

if __name__ == "__main__":
    generate_maximized_flowchart()

# run report (figure only) and checkpoint K45
_k45_files = [os.path.join(OUTPUT_DIR, 'Figure_3_Methodological_Framework_Updated.png')]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 9: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 9: methodological framework (figure)",
       lines=[f"Flow chart of the analysis (manuscript Figure 2), saved at {FIG_DPI_LARGE} dpi and tagged {FIG_DPI} dpi; no numbers."],
       files=_k45_files)

## Phase 10: Appendix Generation (A-H)

Generates the supplementary appendix tables and figures produced in this cell: **A** descriptive statistics, **B** ADF unit-root
tests (extended with KPSS in the Appendix B cell below), **C** model selection (Table C1: log-likelihood, AIC, BIC, AR/MA root moduli and
Ljung-Box statistics of five models, estimated here on a common likelihood sample), **D** full
SARIMAX estimation results (Table D1, read from the Phase 4 output), **E** XGBoost settings, **F** correlation matrices
(Figures F.1, F.2) and **H** manual down-regulation volume (Figure H.1). Supplementary **Appendix G** contains the mathematical
specifications only (no code output); variance-inflation factors are reported by Phase 11.2 (Appendix J.1). **Appendix I**
(BOTAŞ-tariff robustness) and **Appendix J** (reviewer robustness checks) are produced by their own cells below; **Appendix K**
(Figure K.1) is the Phase 8 GAM figure.

**Table C1 specification.** The five rows use the orders of the setup cell: for each market the
baseline order of Phase 2, SARIMAX(1,1,1)(1,0,1)24 without an intercept, and the order used in Phase 4 (status *Selected*):
SARIMAX(2,0,2)(1,0,1)24 with an intercept for the MCP and SARIMAX(1,1,2)(1,0,1)24 without an intercept for the a-FRR price. The
fifth row, the a-FRR price in levels, SARIMAX(1,0,2)(1,0,1)24 with an intercept, is a *robustness check (d = 0)*: this order is
estimated only here and is not used in any other phase. Every fit leaves the first 28 observations out of the log-likelihood,
so AIC and BIC refer to the same observations. The selected a-FRR model has 28 states, so its row has the log-likelihood
function of Phase 4 Model 1 and is estimated from the Phase 4 solution only; the other rows start from the stored Phase 2 /
Phase 4 estimates and from generic starting points (the starts of every row are printed). As in Phases 2 and 4, every
finished start is saved (`results/Phase_10/starts/`, with a `progress.log`), so an interrupted run resumes with the missing
starts. Run this cell after Phase 2 and Phase 4: without the Phase 4 results the cell stops.

**Table D1 inference.** P-values and significance stars come from the standard errors clustered by calendar day
(Phase 4 columns `Std_Error_Cluster_Day` / `P_Value_Cluster_Day`), the default inference of the paper.
`Appendix_D_All_Hypothesis_Variables.csv` lists every hypothesis variable of the eight models with the day-clustered standard
error, z and p-value and keeps the OPG values and the other standard errors of Phase 4 as additional columns.

An error in any appendix stops the cell. The cell ends with a run report (`results/RUN_REPORT.md`): the output files and the
comparison of Tables A1, C1 and D1 and of the ADF statistics with the independent pre-test.

*Note: Appendix J and some revision tables depend on Phase 11 outputs, so in the code flow they are produced after Phase 11.*

In [ ]:
# =============================================================================
# PHASE 10: APPENDIX GENERATION (APPENDIX A-H)
# Specification: Table C1 is estimated with ORDER_* / SEASONAL_* / SARIMAX_INTERCEPT of the setup cell
# (the a-FRR order (1,0,2) with an intercept only as a robustness row); Table D1 reports the standard errors clustered by
# calendar day; an error stops the cell; the cell ends with the run report and the comparison with the pre-test.
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
import warnings
import os
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# no blanket warning filter (see the setup cell)
warnings.filterwarnings('always', category=ConvergenceWarning)   # Appendix C estimates SARIMAX models

OUTPUT_DIR_P10 = os.path.join(RESULTS_BASE, "Phase_10")
os.makedirs(OUTPUT_DIR_P10, exist_ok=True)

# Academic plot settings
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 8),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold'
})

def get_or_create_data() -> pd.DataFrame:
    """Returns the Phase 1 DataFrame 'df'.
    The appendices are produced from the dataset only."""
    target_df = globals().get('df')
    if target_df is None:
        raise RuntimeError("DataFrame 'df' not found. Please run Phase 1 first.")
    return target_df

def generate_appendix_a(df: pd.DataFrame):
    print("\n>>> APPENDIX A: DESCRIPTIVE STATISTICS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_A_Descriptive_Statistics.csv")
    
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path, index_col=0).to_string())
        return

    cols_to_desc = ['Mcp', 'a-FRR_Price', 'Solar_Generation_Share', 'Consumption', 'System_Direction_Down']
    if 'Natural_Gas_Generation' in df.columns:
        cols_to_desc.insert(2, 'Natural_Gas_Generation')

    valid_cols = [c for c in cols_to_desc if c in df.columns]
    desc_stats = df[valid_cols].describe().T[['mean', 'std', 'min', 'max', '50%']]
    desc_stats.columns = ['Mean', 'Std. Dev.', 'Min', 'Max', 'Median']

    desc_stats.to_csv(path, **CSV_KW)   # LF line endings on every platform
    print(desc_stats.to_string())
    print(f"  Saved: {path}")

def generate_appendix_b(df: pd.DataFrame):
    print("\n>>> APPENDIX B: ADF UNIT ROOT TESTS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_B_ADF_Unit_Root_Tests.csv")
    
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    def run_adf(series: pd.Series, name: str) -> dict:
        result = adfuller(series.dropna())
        return {
            "Variable": name,
            "ADF Statistic": round(result[0], 4),
            "P-Value": round(result[1], 4),
            "Result": "I(0) - Stationary" if result[1] < 0.05 else "I(1) - Non-Stationary"
        }

    try:
        adf_results = []
        if 'Mcp' in df.columns: adf_results.append(run_adf(df['Mcp'], "MCP (Day-Ahead Price)"))
        if 'a-FRR_Price' in df.columns: 
            adf_results.append(run_adf(df['a-FRR_Price'], "a-FRR Price (Level)"))
            adf_results.append(run_adf(df['a-FRR_Price'].diff(), "a-FRR Price (1st Difference)"))
        if 'Solar_Generation_Share' in df.columns: adf_results.append(run_adf(df['Solar_Generation_Share'], "Solar Generation Share"))
        
        df_adf = pd.DataFrame(adf_results)
        df_adf.to_csv(path, index=False, **CSV_KW)   # LF line endings on every platform
        print(df_adf.to_string(index=False))
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in ADF tests: {e}")
        raise   # errors stop the cell

C1_BURN = 28   # observations left out of every Table C1 log-likelihood: the largest number of states among its models

def _c1_job(row, endog, exog, order, seasonal, starts):
    """One fit of Supplementary Table C1 (runs in a parallel worker). As in Phases 2 and 4, every finished
    start is saved (results/Phase_10/starts, with a progress.log), so an interrupted run resumes with the missing starts."""
    import inspect
    checkpoint = {}
    if 'job_dir' in inspect.signature(fit_sarimax).parameters:
        checkpoint = {'job_dir': os.path.join(OUTPUT_DIR_P10, "starts"),
                      'job_key': "C1_row%d_%s_%d%d%d" % ((row + 1, endog.name) + tuple(order))}
    _, info, _ = fit_sarimax(endog, exog, order, seasonal, starts=starts, burn=C1_BURN, inference=False, **checkpoint)
    return row, info

def generate_appendix_c(df: pd.DataFrame):
    """Supplementary Table C1 is produced by the code.
    Full-sample profile-ML fits with the regressors of Model 4 (MCP) and Model 1 (a-FRR)
    under the specification of the setup cell: for each market the baseline order of Phase 2 (no intercept) and the
    order used in Phase 4 ('Selected': MCP with an intercept, a-FRR without); for the a-FRR price also the order (1,0,2) in
    levels with an intercept, as a robustness check (d = 0; estimated only here). statsmodels leaves as many initial
    observations out of the log-likelihood as the model has states (27 or 28 here); so that the AIC and BIC of different
    orders refer to the same observations, every fit of this table leaves out the first C1_BURN = 28. The selected MCP row
    therefore differs from the Phase 4 log-likelihood of Model 4 by one observation; the selected a-FRR model has 28 states,
    so its row has the log-likelihood function of Phase 4 Model 1 and is estimated from the Phase 4 solution only."""
    print("\n>>> APPENDIX C: MODEL SELECTION (Table C1: log-likelihood, AIC and BIC on a common sample)")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_C_Model_Comparison.csv")

    if os.path.exists(path):
        cached = pd.read_csv(path)
        if 'Method' in cached.columns and (cached['Method'] == SARIMAX_METHOD_TAG).all():
            print(f"  [CACHE] Loaded from {path}")
            print(cached.drop(columns=['Method']).to_string(index=False))
            return
        print("  the cached table carries no method tag or a different one -> recomputed.")

    def stored_arma(phase, model_id, required=False):
        """AR/MA estimates of a Phase 2 / Phase 4 model in results/ (Method = SARIMAX_METHOD_TAG only), else None.
        required=True (selected rows): a result file that cannot be read is an error, not a missing estimate."""
        try:
            if phase == 2:
                conv = pd.read_csv(os.path.join(RESULTS_BASE, "Phase_2", "Phase2_Convergence.csv")).set_index('Model_ID')
                t = pd.read_csv(os.path.join(RESULTS_BASE, "Phase_2", f"{model_id.replace(' ', '_')}_summary.csv"), index_col=0)['coef']
                ok = conv.loc[model_id, 'Method'] == SARIMAX_METHOD_TAG
            else:
                diag = pd.read_csv(os.path.join(RESULTS_BASE, "Phase_4", "Phase4_Model_Diagnostics.csv")).set_index('Model_ID')
                c = pd.read_csv(os.path.join(RESULTS_BASE, "Phase_4", "Phase4_All_Coefficients.csv"), index_col=0)
                t = c.loc[c['Model_ID'] == model_id, 'Coefficient']
                ok = diag.loc[model_id, 'Method'] == SARIMAX_METHOD_TAG
            return ({k: float(v) for k, v in t.items() if str(k).startswith(('ar.', 'ma.'))} or None) if ok else None
        except Exception as exc:
            if required:
                raise   # errors stop the cell
            print(f"   !! stored Phase {phase} estimate of {model_id} could not be read ({exc!r}): generic starting points are used")
            return None

    try:
        controls = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation', 'Hydroelectric_Generation', 'Domestic_Coal_Generation',
                    'Price_Cap', 'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
        x_vars = controls + ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday']   # Models 1 and 4
        data, _ = generate_interaction_terms(df)
        # Specification of the setup cell: the selected rows use the orders and intercepts of
        # Phase 4 (ORDER_MCP with an intercept, ORDER_AFRR without), the baseline rows the Phase 2 order (ORDER_BASELINE, no
        # intercept). The a-FRR order (1,0,2) in levels with an intercept is not used in any phase: it is estimated only
        # here, as a robustness check of the differenced a-FRR specification.
        # (market, model, dependent variable, order, seasonal order, intercept, status, Phase 2 / Phase 4 model with a stored
        #  estimate, its order)
        rows_cfg = [
            ("MCP (Day-Ahead)", "Baseline", 'Mcp', ORDER_BASELINE, SEASONAL_BASELINE, False, "Rejected", (2, 'Model 4'), ORDER_BASELINE),
            ("MCP (Day-Ahead)", "Optimized", 'Mcp', ORDER_MCP, SEASONAL_MCP, SARIMAX_INTERCEPT['MCP'], "Selected", (4, 'Model 4'), ORDER_MCP),
            ("a-FRR (Balancing)", "Baseline", 'a-FRR_Price', ORDER_BASELINE, SEASONAL_BASELINE, False, "Rejected", (2, 'Model 1'), ORDER_BASELINE),
            ("a-FRR (Balancing)", "Optimized", 'a-FRR_Price', ORDER_AFRR, SEASONAL_AFRR, SARIMAX_INTERCEPT['aFRR'], "Selected", (4, 'Model 1'), ORDER_AFRR),
            ("a-FRR (Balancing)", "Alternative in levels", 'a-FRR_Price', (1, 0, 2), SEASONAL_AFRR, True, "Robustness check (d = 0)",
             (2, 'Model 1'), ORDER_BASELINE),
        ]

        jobs = []
        for k, (market, model, y, order, seasonal, intercept, status, source, source_order) in enumerate(rows_cfg):
            d = data[[y] + x_vars].dropna()
            exog = d[x_vars].copy()
            if intercept:
                exog.insert(0, 'const', 1.0)
            near = stored_arma(*source, required=(source[0] == 4))
            if near is None and source[0] == 4:
                # a selected row is the Phase 4 model itself: without its stored estimate it is not estimated
                raise RuntimeError(f"Appendix C: the Phase 4 estimate of {source[1]} (current method and specification) was not found in "
                                   f"'{os.path.join(RESULTS_BASE, 'Phase_4')}'. Run Phase 4 first.")
            if near is not None and source_order == order:
                # the same model was estimated in Phase 2 / Phase 4 (with its own burn-in): its solution is next to the maximum
                starts = {'phase estimate': near, 'default': 'default'}
                if order == ORDER_AFRR:
                    # the selected a-FRR model has C1_BURN states, so this row has the log-likelihood function of
                    # Phase 4 Model 1: it is estimated from the Phase 4 solution only (the multi-start search of Phase 4, which
                    # records every start, is not repeated here)
                    del starts['default']
            else:
                starts = {'default': 'default', 'arma0': 'arma0', **({'phase2': near} if near is not None else {}), 'grid': 'grid'}
            print(f"   {market}, order {order}{' with intercept' if intercept else ''} [{status}]: starts {list(starts)}")
            jobs.append((k, d[y], exog, order, seasonal, starts))
        print(f"  Estimating {len(jobs)} models, up to {min(SARIMAX_N_JOBS, len(jobs))} in parallel "
              f"(profile maximum likelihood, first {C1_BURN} observations left out of every log-likelihood)...")
        out = {}
        for k, info in run_parallel(_c1_job, jobs):
            market, model, y, order, seasonal, intercept, status = rows_cfg[k][:7]
            # three-valued convergence status of the setup cell: certified / plateau / not converged
            cls = info.get('status_class', 'certified' if info['converged'] else 'not converged')
            out[k] = {"Market": market, "Model": model, "Order": "(%d,%d,%d)" % tuple(order), "Intercept": intercept,
                      "LogLik": info['llf'], "Parameters": info['k_params'], "AIC": info['aic'], "BIC": info['bic'],
                      "AR_root_moduli": str([round(float(v), 4) for v in info['AR_roots']]),
                      "MA_root_moduli": str([round(float(v), 4) for v in info['MA_roots']]),
                      "LjungBox_Q24": info['LjungBox_Q24'], "LjungBox_p24": info['LjungBox_p24'],
                      "LjungBox_Q48": info['LjungBox_Q48'], "LjungBox_p48": info['LjungBox_p48'],
                      "Solar_Coefficient": info['key_coef'], "Status": status, "Certified_maximum": info['converged'], "Convergence": cls,
                      "Starts_within_0.01": info.get('starts_within_001', np.nan),
                      "Best_start": info['best_start'], "Observations": info['nobs'] - info['burn'], "Burn": info['burn'],
                      "Method": SARIMAX_METHOD_TAG}
            if cls == 'not converged':      # a plateau (flat ridge) is an accepted status, as in Phases 2 and 4
                warnings.warn(f"Appendix C, {market} {model} {order}: the estimate did NOT converge ({info['status']})", ConvergenceWarning)
        df_compare = pd.DataFrame([out[k] for k in sorted(out)])
        df_compare.to_csv(path, index=False, **CSV_KW)
        print(df_compare.drop(columns=['Method']).to_string(index=False))
        seasonal_txt = ", ".join(sorted({"(%d,%d,%d)%d" % tuple(r[4]) for r in rows_cfg}))   # from SEASONAL_* (setup cell)
        print(f"  All rows: seasonal order {seasonal_txt}, {int(df_compare['Observations'].iloc[0]):,} observations in the log-likelihood "
              f"(first {C1_BURN} left out), so AIC and BIC are comparable across orders.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in Appendix C: {e}")
        raise   # errors stop the cell

def generate_appendix_d():
    """Table D1 is generated from the Phase 4 output (Phase4_All_Coefficients.csv),
    for all eight SARIMAX models of Supplementary Table 3. Rebuilt on every run (it only
    formats the Phase 4 file). Writes:
      Appendix_D_Full_SARIMAX_Results.csv       - Table D1 layout (Model, Variable, Coefficient, P-Value, Significance)
      Appendix_D_All_Hypothesis_Variables.csv   - every Table 3 hypothesis variable + Consumption of the eight models,
                                                  with standard error, z and p-value
    Inference: standard errors clustered by calendar day (sandwich; Phase 4 columns Std_Error_Cluster_Day and
    P_Value_Cluster_Day), the default inference of the paper. P-Value and Significance of Table D1 and Std_Error, z,
    P_Value and Significance of the second file are the day-clustered ones; the second file keeps the OPG values
    (Std_Error_OPG, z_OPG, P_Value_OPG) and the other standard errors of Phase 4 as additional columns.
    Significance: *** p < 0.001, ** p < 0.01, * p < 0.05 (as in the note printed with Table D1); NS otherwise."""
    print("\n>>> APPENDIX D: FULL SARIMAX ESTIMATION RESULTS (from the Phase 4 output)")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_D_Full_SARIMAX_Results.csv")
    path_all = os.path.join(OUTPUT_DIR_P10, "Appendix_D_All_Hypothesis_Variables.csv")
    coef_path = os.path.join(RESULTS_BASE, "Phase_4", "Phase4_All_Coefficients.csv")
    # a missing or incomplete Phase 4 output stops the cell
    if not os.path.exists(coef_path):
        raise RuntimeError(f"Phase 4 output not found ({coef_path}). Run Phase 4 first.")
    coef = pd.read_csv(coef_path, index_col=0, float_precision='round_trip')
    # Table D1 uses the standard errors clustered by calendar day (default inference of the paper)
    if not {'Std_Error_Cluster_Day', 'P_Value_Cluster_Day'} <= set(coef.columns):
        raise RuntimeError("Phase 4 output has no day-clustered standard errors and p-values yet. Run the Phase 4 cell of this notebook first.")
    need = ['Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5', 'Model 6', 'Model 7.2', 'Model 7.3']
    lacking = [m for m in need if m not in set(coef['Model_ID'])]
    if lacking:
        raise RuntimeError(f"Phase 4 output lacks {', '.join(lacking)}; Appendix D not rebuilt. Re-run Phase 4.")

    def get(model, var):
        r = coef[(coef['Model_ID'] == model) & (coef.index == var)]
        if len(r) != 1:
            raise KeyError(f"{model} / {var} not found in {coef_path}")
        # a missing day-clustered standard error must not end up as 'not significant' in the table
        if r[['Coefficient', 'Std_Error_Cluster_Day', 'P_Value_Cluster_Day']].isna().to_numpy().any():
            raise ValueError(f"{model} / {var}: coefficient or day-clustered standard error / p-value missing in {coef_path}")
        return r.iloc[0]

    def fmt_coef(b):
        return f"{b:,.3f}" if abs(b) < 1 else f"{b:,.2f}"

    def fmt_p(p):
        return "< 0.001" if p < 0.001 else f"= {p:.3f}"

    def stars(p):
        return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "NS (Insignificant)"

    hours = [f'Solar_Share_x_Hour_{h}' for h in range(8, 18)]
    def extreme_hour(model, pick):
        vals = {h: get(model, h)['Coefficient'] for h in hours}
        return pick(vals, key=vals.get)
    h_drop = extreme_hour('Model 5', min)   # largest MCP depression across 08:00-17:00
    h_risk = extreme_hour('Model 2', max)   # largest a-FRR increase across 08:00-17:00
    lab_h = lambda v: f"Hour_{int(v.rsplit('_', 1)[1]):02d}:00 * Solar"

    # (model id, model label, variable, displayed variable name, note appended to the significance)
    d1_rows = [
        ('Model 4', 'Model 4 (MCP Baseline)', 'Solar_Generation_Share', 'Solar_Generation_Share', ''),
        ('Model 4', 'Model 4 (MCP Baseline)', 'Consumption', 'Consumption', ''),
        ('Model 1', 'Model 1 (a-FRR Baseline)', 'Solar_Generation_Share', 'Solar_Generation_Share', ''),
        ('Model 1', 'Model 1 (a-FRR Baseline)', 'Consumption', 'Consumption', ''),
        ('Model 5', 'Model 5 (MCP Hourly)', h_drop, lab_h(h_drop), ' (Max Drop)'),
        ('Model 2', 'Model 2 (a-FRR Hourly)', h_risk, lab_h(h_risk), ' (Max Risk)'),
        ('Model 7.3', 'Model 7.3 (Regime)', 'Solar_Generation_Share', 'Solar (Base/Balanced)', ''),
        ('Model 7.3', 'Model 7.3 (Regime)', 'Interaction_Solar_Dir_Up', 'Solar * Deficit (Up)', ' (Risk Premium)'),
        ('Model 7.3', 'Model 7.3 (Regime)', 'Interaction_Solar_Dir_Down', 'Solar * Surplus (Down)', ''),
        # models defined in Supplementary Table 3 and estimated in Phase 4
        ('Model 3', 'Model 3 (a-FRR Peak Stress)', 'Interaction_Solar_Peak', 'Solar * Peak (10:00-15:00)', ''),
        ('Model 6', 'Model 6 (MCP Peak Depression)', 'Interaction_Solar_Peak', 'Solar * Peak (10:00-15:00)', ''),
        ('Model 7.2', 'Model 7.2 (Regime Check)', 'System_Direction_Down', 'System_Direction_Down (Surplus)', ''),
    ]
    rows = []
    for mid, mlab, var, vlab, note in d1_rows:
        r = get(mid, var)
        sig = stars(r['P_Value_Cluster_Day'])           # day-clustered p-value
        rows.append({"Model": mlab, "Variable": vlab, "Coefficient": fmt_coef(r['Coefficient']),
                     "P-Value": fmt_p(r['P_Value_Cluster_Day']), "Significance": sig + (note if sig.startswith('*') else '')})
    df_d_full = pd.DataFrame(rows)
    df_d_full.to_csv(path, index=False, **CSV_KW)
    print(df_d_full.to_string(index=False))
    print("  Note: *** p < 0.001, ** p < 0.01, * p < 0.05; NS = not significant at 5%. P-values from standard errors clustered by calendar day.")
    print(f"  Saved: {path}")

    # complete list: every Table 3 hypothesis variable (+ Consumption) of the eight SARIMAX models
    hyp = {
        'Model 1': ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday'],
        'Model 2': hours,
        'Model 3': ['Interaction_Solar_Peak'],
        'Model 4': ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday'],
        'Model 5': hours,
        'Model 6': ['Interaction_Solar_Peak'],
        'Model 7.2': ['System_Direction_Down'],
        'Model 7.3': ['Solar_Generation_Share', 'System_Direction_Down', 'System_Direction_Up',
                      'Interaction_Solar_Dir_Down', 'Interaction_Solar_Dir_Up'],
    }
    full = []
    for mid, vars_ in hyp.items():
        for var in vars_ + ['Consumption']:
            r = get(mid, var)
            # Std_Error, z, P_Value and Significance: clustered by calendar day; the OPG values and the
            # other standard errors of Phase 4 follow as additional columns
            row = {'Model': mid, 'Variable': var, 'Coefficient': r['Coefficient'], 'Std_Error': r['Std_Error_Cluster_Day'],
                   'z': r['Coefficient'] / r['Std_Error_Cluster_Day'], 'P_Value': r['P_Value_Cluster_Day'],
                   'Significance': stars(r['P_Value_Cluster_Day']), 'SE_Type': 'Cluster_Day',
                   'Std_Error_OPG': r['Std_Error'], 'z_OPG': r['z'], 'P_Value_OPG': r['P_Value']}
            row.update({f'{c}_{kind}': r[f'{c}_{kind}'] for kind in SARIMAX_EXTRA_SE if kind != 'Cluster_Day' for c in ('Std_Error', 'P_Value')})
            full.append(row)
    pd.DataFrame(full).to_csv(path_all, index=False, **CSV_KW)
    print(f"  Saved: {path_all} ({len(full)} rows)")

def generate_appendix_e():
    print("\n>>> APPENDIX E: XGBOOST HYPERPARAMETERS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_E_XGBoost_Params.csv")
    
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    # [Revision] Appendix E lists the settings used in Phases 6.1 / 6.2 / 6.4
    # (xgb.XGBRegressor / XGBClassifier(n_estimators=200, random_state=42); every other
    # setting at the XGBoost library default). No hyper-parameter tuning was performed:
    # XGBoost is an interpretive cross-check (partial-dependence plots), not a forecaster.
    xgb_params = {
        "Parameter": ["n_estimators", "learning_rate", "max_depth", "subsample", "colsample_bytree",
                      "objective (regressors)", "objective (classifier)", "random_state", "tuning"],
        "Value": [200, "0.3 (library default)", "6 (library default)", "1.0 (library default)",
                  "1.0 (library default)", "reg:squarederror", "binary:logistic", 42, "none (fixed a priori)"],
        "Rationale": ["Number of boosting rounds, fixed a priori", "Library default", "Library default",
                      "Library default (no row subsampling)", "Library default (no column subsampling)",
                      "Squared-error loss for MCP and a-FRR price", "Log-loss for system direction",
                      "Reproducibility (same seed in Phases 6.1, 6.2, 6.4)",
                      "Interpretive cross-check (PDP), not a tuned forecaster; out-of-sample fit in Phase 6.4"]
    }
    df_xgb = pd.DataFrame(xgb_params)
    df_xgb.to_csv(path, index=False, **CSV_KW)
    print(df_xgb.to_string(index=False))
    print(f"  Saved: {path}")

def generate_appendix_f(df: pd.DataFrame):
    print("\n>>> APPENDIX F: CORRELATION HEATMAPS")
    mcp_vars = ['Mcp', 'Consumption', 'Natural_Gas_Generation', 'Solar_Generation_Share', 'Wind_Generation', 'Import_Coal_Generation']
    afrr_vars = ['a-FRR_Price', 'Consumption', 'Natural_Gas_Generation', 'Solar_Generation_Share', 'Hydroelectric_Generation', 'System_Direction_Down']

    mcp_vars = [c for c in mcp_vars if c in df.columns]
    afrr_vars = [c for c in afrr_vars if c in df.columns]

    configs = [
        (mcp_vars, "Appendix_F1_MCP_Correlation", "MCP Market Correlations"),
        (afrr_vars, "Appendix_F2_aFRR_Correlation", "a-FRR Market Correlations")
    ]

    for vars_list, fname, title in configs:
        if len(vars_list) > 1:
            save_path = os.path.join(OUTPUT_DIR_P10, f"{fname}.png")
            
            if os.path.exists(save_path):
                print(f"  [CACHE] Loading {fname}...")
                plt.figure(figsize=(8, 6))
                plt.imshow(plt.imread(save_path))
                plt.axis('off')
                plt.show()
                continue

            try:
                plt.figure(figsize=(10, 8))
                corr = df[vars_list].corr()
                mask = np.triu(np.ones_like(corr, dtype=bool))
                # [Revision_R2_Reviewer4_C9] readable variable names, no in-figure title (the caption carries it)
                # and no background grid through the matrix; values are unchanged.
                _lab = {'Mcp': 'MCP', 'a-FRR_Price': 'a-FRR Price', 'Consumption': 'Consumption',
                        'Natural_Gas_Generation': 'Nat. Gas Gen.', 'Solar_Generation_Share': 'Solar Share',
                        'Wind_Generation': 'Wind Gen.', 'Import_Coal_Generation': 'Imp. Coal Gen.',
                        'Hydroelectric_Generation': 'Hydro Gen.', 'System_Direction_Down': 'Surplus (Binary)'}
                _ax_hm = sns.heatmap(corr.rename(index=_lab, columns=_lab), mask=mask, annot=True, fmt=".2f",
                                     cmap='coolwarm', vmin=-1, vmax=1)
                _ax_hm.grid(False)
                _ax_hm.tick_params(axis='y', labelrotation=0)
                _ax_hm.tick_params(axis='x', labelrotation=90)
                plt.tight_layout()
                plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight')
                plt.show()
                print(f"  Saved: {save_path}")
            except Exception as e:
                print(f"  !! Error generating heatmap {fname}: {e}")
                raise   # errors stop the cell

# Supplementary Appendix G holds the mathematical specifications (text only); variance-inflation factors are
# reported by Phase 11.2 (Appendix J.1).

def generate_appendix_h(df: pd.DataFrame):
    print("\n>>> APPENDIX H: MANUAL DOWN-REGULATION VOLUME ANALYSIS")
    save_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_H_Down_Regulation_Volume.png')

    if os.path.exists(save_path):
        print(f"  [CACHE] Loading from {save_path}...")
        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(save_path))
        plt.axis('off')
        plt.show()
        return

    dm = df.copy()
    if 'BPM_Down_Regulation_Volume' not in dm.columns:   # the column must be in the dataset
        raise RuntimeError("'BPM_Down_Regulation_Volume' column missing in the dataset; Appendix H cannot be produced.")   # no silent skip

    if dm['Solar_Generation_Share'].max() <= 1:
        dm['Solar_Generation_Share'] *= 100

    dm['Solar_Regime'] = dm['Solar_Generation_Share'].apply(lambda x: '≥ 34%' if x >= 34 else '< 34%')   # regime split at 34%

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(
        x='Solar_Regime', y='BPM_Down_Regulation_Volume', data=dm,
        order=['< 34%', '≥ 34%'], hue='Solar_Regime', hue_order=['< 34%', '≥ 34%'], palette='coolwarm', legend=False, ax=ax   # explicit hue (same colours)
    )
    
    ax.set_xlabel('Solar Penetration Regime', fontweight='bold')
    ax.set_ylabel('Manual Down-Regulation Volume (MWh)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {save_path}")

def run_phase_10(df):
    print(f">> PHASE 10: Appendix Generation (Directory: '{OUTPUT_DIR_P10}')...")
    print("="*80)
    
    try:
        # appendices that do not depend on SARIMAX results (A, B, E, F, H) first: an error in Appendix C or D
        # stops the cell and must not keep them from being written
        generate_appendix_a(df)
        generate_appendix_b(df)
        generate_appendix_e()
        generate_appendix_f(df)
        # Appendix G: mathematical specifications (text only). VIFs: Phase 11.2 -> Appendix J.1.
        generate_appendix_h(df)
        generate_appendix_c(df)
        generate_appendix_d()

        print("\n" + "="*80)
        print(f"PHASE 10 COMPLETE. Appendices A-F and H saved to '{OUTPUT_DIR_P10}'.")
        print("="*80)

    except Exception as e:
        print(f"\n!! CRITICAL PIPELINE ERROR: {e}")
        raise   # errors stop the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    df_target = get_or_create_data()
    run_phase_10(df_target)

# run report and comparison with the independent pre-test
# (read back from the files of this cell, so the record is the same when the cell loads its cache)
import json
_p10 = {k: os.path.join(OUTPUT_DIR_P10, v) for k, v in {
    'A': "Appendix_A_Descriptive_Statistics.csv", 'B': "Appendix_B_ADF_Unit_Root_Tests.csv", 'C': "Appendix_C_Model_Comparison.csv",
    'D': "Appendix_D_Full_SARIMAX_Results.csv", 'D_all': "Appendix_D_All_Hypothesis_Variables.csv", 'E': "Appendix_E_XGBoost_Params.csv",
    'F1': "Appendix_F1_MCP_Correlation.png", 'F2': "Appendix_F2_aFRR_Correlation.png", 'H': "Appendix_H_Down_Regulation_Volume.png"}.items()}
_a1 = pd.read_csv(_p10['A'], index_col=0).loc['Solar_Generation_Share']
_b1 = pd.read_csv(_p10['B'])
_c1 = pd.read_csv(_p10['C'])                                                   # five rows, in the order (i)-(v) of the reference table
_d1 = pd.read_csv(_p10['D_all'])
_d1 = _d1[_d1['Variable'] != 'Consumption']                                    # hypothesis variables only
_p4_solar = _d1[_d1['Variable'] == 'Solar_Generation_Share'].set_index('Model')['Coefficient']
# checkpoints of Supplementary Table C1 (K10-K14) and of the Phase 4 standard errors behind Table D1 (K16).
# They read the table and the start records that fit_sarimax wrote for it (results/Phase_10/starts), so they also hold
# when the table is loaded from its cache. Rows (i)-(v) as in the comparison with the pre-test below.
import glob, re
_c1_lab = [f"({_i}) {_r['Market']} {_r['Order']}" for _i, _r in zip(['i', 'ii', 'iii', 'iv', 'v'], _c1.to_dict('records'))]
_c1_rec = []      # per row: {start: record} of the row, and the record of the reported start (None if missing or not the reported point)
for _k, _r in enumerate(_c1.to_dict('records')):
    _recs = {}
    for _f in sorted(glob.glob(os.path.join(OUTPUT_DIR_P10, "starts", f"C1_row{_k + 1}_*_{re.sub(r'[^0-9]', '', _r['Order'])}__*.json"))):
        with open(_f, encoding='utf-8') as _fh:
            _j = json.load(_fh)
        _recs[_j.get('start')] = _j
    _best = _recs.get(_r['Best_start'])
    _c1_rec.append((_recs, _best if _best is not None and abs(float(_best['llf']) - float(_r['LogLik'])) < 1e-6 else None))
_c1_mkt = ['MCP' if str(_r['Market']).startswith('MCP') else 'aFRR' for _r in _c1.to_dict('records')]
_p4c = pd.read_csv(os.path.join(RESULTS_BASE, "Phase_4", "Phase4_All_Coefficients.csv"), index_col=0)
_c1_regs = {_m: [str(_v) for _v in _p4c.index[_p4c['Model_ID'] == _p4m] if not str(_v).startswith(('ar.', 'ma.', 'sigma2')) and _v != 'const']
            for _m, _p4m in (('MCP', 'Model 4'), ('aFRR', 'Model 1'))}   # regressors of Table C1 = those of Models 4 and 1
# K10: convergence status of every row (not converged -> FAILED; plateau -> WARNING)
_k10 = dict(zip(_c1_lab, _c1['Convergence'].astype(str)))
_k10_nc = [_l for _l, _s in _k10.items() if _s not in ('certified', 'plateau')]
_k10_txt = "; ".join(f"{_l}: {_s}" for _l, _s in _k10.items())
if _k10_nc:
    checkpoint("K10 Table C1: convergence", False, f"NOT CONVERGED: {', '.join(_k10_nc)} | {_k10_txt}", level='fail')
else:
    checkpoint("K10 Table C1: convergence", 'plateau' not in _k10.values(),
               _k10_txt + (" (plateau = flat ridge, an accepted status)" if 'plateau' in _k10.values() else ""), level='warn')
# K11: the reported representation is invertible (flag of the start record of the reported start; it covers the seasonal MA)
_k11 = []
for _l, _r, (_recs, _best) in zip(_c1_lab, _c1.to_dict('records'), _c1_rec):
    _ma = json.loads(_r['MA_root_moduli'])
    if _best is not None:
        _k11.append((_l, bool(_best['invertible']), "invertible" if _best['invertible'] else
                     f"NOT invertible (start '{_r['Best_start']}', MA root moduli {_ma})"))
    elif any(_v <= 1.0 for _v in _ma):
        _k11.append((_l, False, f"NOT invertible (MA root moduli {_ma})"))
    else:
        _k11.append((_l, None, f"start record of '{_r['Best_start']}' not found or not the reported point; MA root moduli {_ma} "
                               "above 1, the seasonal MA term is not in the table"))
checkpoint("K11 Table C1: reported representation invertible",
           False if any(_o is False for _, _o, _ in _k11) else (None if any(_o is None for _, _o, _ in _k11) else True),
           "; ".join(f"{_l}: {_t}" for _l, _o, _t in _k11))
# K12: at least two starts end within 0.01 log-likelihood units of the reported point (rows estimated from one start: not applicable)
_k12 = [(_l, int(_r['Starts_within_0.01'])) for _l, _r, (_recs, _) in zip(_c1_lab, _c1.to_dict('records'), _c1_rec) if len(_recs) != 1]
_k12_one = [_l for _l, (_recs, _) in zip(_c1_lab, _c1_rec) if len(_recs) == 1]
checkpoint("K12 Table C1: agreement of the starts", all(_n >= 2 for _, _n in _k12) if _k12 else None,
           "; ".join(f"{_l}: {_n} start(s) within 0.01" for _l, _n in _k12)
           + "".join(f"; {_l}: one start only (by design: estimated from the Phase 4 solution, whose starts are compared in Phase 4)" for _l in _k12_one),
           level='warn')
# K13: 'const' exactly where the design puts it (selected rows: SARIMAX_INTERCEPT; baseline rows: Phase 2 order without
# intercept; level row: with intercept), and the fitted model has the regressors of Model 4 / Model 1 plus that intercept
_k13 = []
for _l, _r, _m, (_, _best) in zip(_c1_lab, _c1.to_dict('records'), _c1_mkt, _c1_rec):
    _want = SARIMAX_INTERCEPT[_m] if _r['Status'] == 'Selected' else (_r['Status'] != 'Rejected')
    _has = str(_r['Intercept']).strip().lower() == 'true'
    _n_exog = int(_r['Parameters']) - len(_best['arma']) - 1 if _best is not None else None   # minus AR/MA terms and sigma2
    _n_want = len(_c1_regs[_m]) + int(_has)
    _k13.append((_l, _has == _want and _n_exog in (None, _n_want),
                 f"const {'yes' if _has else 'no'} (design: {'yes' if _want else 'no'}), "
                 + (f"{_n_exog} exogenous terms (expected {_n_want})" if _n_exog is not None else "number of terms not verified (no start record)")))
checkpoint("K13 Table C1: intercept and regressors", all(_o for _, _o, _ in _k13), "; ".join(f"{_l}: {_t}" for _l, _o, _t in _k13))
# K14: Table C1 has no standard errors and hence no day clusters; every row must use the whole estimation sample
_c1_data = generate_interaction_terms(df)[0] if 'generate_interaction_terms' in globals() else df
_k14 = []
for _l, _r, _m in zip(_c1_lab, _c1.to_dict('records'), _c1_mkt):
    _cols = ['Mcp' if _m == 'MCP' else 'a-FRR_Price'] + _c1_regs[_m]
    _smp = _c1_data[[_c for _c in _cols if _c in _c1_data.columns]].dropna() if all(_c in _c1_data.columns for _c in _cols) else None
    _k14.append((_l, int(_r['Observations']) + int(_r['Burn']), None if _smp is None else len(_smp),
                 None if _smp is None else int(pd.DatetimeIndex(_smp.index).normalize().nunique())))
checkpoint("K14 Table C1: estimation sample", None if any(_n is None for _, _, _n, _ in _k14) else all(_a == _n for _, _a, _n, _ in _k14),
           "Table C1 is estimated without standard errors (inference=False), so it has no day clusters; checked instead that every row "
           "uses the whole sample: " + "; ".join(f"{_l}: {_a:,} observations incl. burn-in" + ("" if _n is None else
           f" (sample {_n:,} hours, {_d:,} calendar days)") for _l, _a, _n, _d in _k14))
# K16: no standard-error flag for the Phase 4 models whose day-clustered standard errors Table D1 reports
_p4d = pd.read_csv(os.path.join(RESULTS_BASE, "Phase_4", "Phase4_Model_Diagnostics.csv")).set_index('Model_ID')
_k16_m = list(dict.fromkeys(_d1['Model']))
_k16_flag = {_m: str(_p4d.loc[_m, 'SE_checks']) for _m in _k16_m if 'SE_checks' in _p4d.columns and _m in _p4d.index
             and pd.notna(_p4d.loc[_m, 'SE_checks']) and str(_p4d.loc[_m, 'SE_checks']).strip()}
_k16_miss = [_m for _m in _k16_m if _m not in _p4d.index]
checkpoint("K16 Table D1: standard-error flags of the Phase 4 models",
           None if 'SE_checks' not in _p4d.columns else not (_k16_flag or _k16_miss),
           f"Table C1 has no standard errors; Table D1 reports the day-clustered standard errors of Phase 4 for {len(_k16_m)} models: "
           + ("no SE_checks column in Phase4_Model_Diagnostics.csv" if 'SE_checks' not in _p4d.columns else
              ("no standard-error flag" if not (_k16_flag or _k16_miss) else
               "; ".join([f"{_m}: {_v}" for _m, _v in _k16_flag.items()] + [f"{_m}: not in the Phase 4 diagnostics" for _m in _k16_miss]))),
           level='warn')
_k45_files = list(_p10.values())
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 10: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 10: Appendices A-F and H (Tables A1, B1 (ADF), C1, D1, E1; Figures F.1, F.2, H.1)",
       lines=[f"Table A1: solar generation share, mean {_a1['Mean']:.4f}, maximum {_a1['Max']:.4f} (share, 0-1).",
              f"Table C1 (common likelihood sample, first {int(_c1['Burn'].iloc[0])} observations left out), AIC: "
              + "; ".join(f"{_r['Market']} {_r['Order']} {_r['AIC']:,.1f} [{_r['Status']}; {_r['Convergence']}]" for _r in _c1.to_dict('records')) + ".",
              f"Table C1, AIC of the baseline minus the selected order: MCP {_c1['AIC'].iloc[0] - _c1['AIC'].iloc[1]:,.1f}, "
              f"a-FRR {_c1['AIC'].iloc[2] - _c1['AIC'].iloc[3]:,.1f}.",
              f"Solar coefficient, selected rows of Table C1 against Phase 4: MCP {_c1['Solar_Coefficient'].iloc[1]:,.4f} (Model 4: "
              f"{_p4_solar['Model 4']:,.4f}; one observation fewer in the Table C1 likelihood), a-FRR {_c1['Solar_Coefficient'].iloc[3]:,.4f} "
              f"(Model 1: {_p4_solar['Model 1']:,.4f}; same likelihood function, estimated from the Phase 4 solution).",
              f"Table D1 (standard errors clustered by calendar day): {int((_d1['P_Value'] < 0.05).sum())} of the {len(_d1)} "
              "hypothesis variables of the eight models are significant at 5%.",
              sarimax_start_warnings(os.path.join(OUTPUT_DIR_P10, "starts"))],
       table=_c1[['Market', 'Model', 'Order', 'Intercept', 'LogLik', 'Parameters', 'AIC', 'BIC', 'MA_root_moduli',
                  'Solar_Coefficient', 'Status', 'Convergence', 'Best_start', 'Starts_within_0.01']],
       files=_k45_files)
compare_with_reference("Table A1 (solar generation share) and the ADF statistics of Table B1",
                       [(f"A1 solar share, {_c}", _a1[_c], f"A1.Solar_Generation_Share.{_i}") for _c, _i in
                        [('Mean', 'mean'), ('Std. Dev.', 'std'), ('Min', 'min'), ('Max', 'max'), ('Median', 'median')]]
                       + [(f"ADF {_r['Variable']}, {_c}", _r[_c], f"B1.{_r['Variable']}.ADF {_i}") for _r in _b1.to_dict('records')
                          for _c, _i in [('ADF Statistic', 'Statistic'), ('P-Value', 'P-Value')]],
                       check="K17 Table A1 / B1 (ADF): comparison with the independent pre-test")
_c1_rows = []
for _id, _r in zip(['i', 'ii', 'iii', 'iv', 'v'], _c1.to_dict('records')):
    _c1_rows += [(f"C1 ({_id}) {_r['Market']} {_r['Order']}: {_c}", _r[_c], f"C1.{_id}.{_c}")
                 for _c in ['LogLik', 'Parameters', 'AIC', 'BIC', 'LjungBox_Q24', 'LjungBox_Q48', 'Solar_Coefficient', 'Burn']]
    _c1_rows += [(f"C1 ({_id}) {_r['Market']} {_r['Order']}: MA root modulus #{_j}", _v, f"C1.{_id}.MA_root_moduli.{_j}")
                 for _j, _v in enumerate(json.loads(_r['MA_root_moduli']), 1)]
_c1_rows += [("C1 AIC, MCP baseline minus selected", _c1['AIC'].iloc[0] - _c1['AIC'].iloc[1], "C1.turetilmis.dAIC_MCP"),
             ("C1 AIC, a-FRR baseline minus selected", _c1['AIC'].iloc[2] - _c1['AIC'].iloc[3], "C1.turetilmis.dAIC_aFRR")]
compare_with_reference("Supplementary Table C1 (model selection on a common likelihood sample)", _c1_rows,
                       check="K17 Table C1: comparison with the independent pre-test")
compare_with_reference("Supplementary Table D1: hypothesis variables of the eight models (coefficient; standard error clustered by calendar day)",
                       [(f"{_r['Model']} {_r['Variable']}: {_lab}", _r[_col], f"M{_r['Model'].split()[1]}.{_fam}.{_r['Variable']}")
                        for _r in _d1.to_dict('records')
                        for _lab, _col, _fam in [('coefficient', 'Coefficient', 'coef'), ('SE (day-clustered)', 'Std_Error', 'se.Cluster_Day')]],
                       check="K17 Table D1: comparison with the independent pre-test")

### Recalculation switch

With `FORCE_FRESH = True` the cache files of four analyses – the ADF & KPSS stationarity tests
(Appendix B), the inflation-robustness FE-OLS (Phase 11) and the XGBoost and GAM performance metrics – are deleted, and the
cells below recompute them **from scratch, from the data on this machine**. With `False` (the default) the cache is kept. The
switch does not touch the caches of Phases 2–8; `FORCE_FIGURES` is in the setup cell, because the figure phases run before
this cell.

A run starts from an empty results folder stamped for the data file and the
method (`RUN_MANIFEST.json`, setup cell), and no code edits result files after they have been written.

In [ ]:
# =============================================================================
# RECALCULATION SWITCH (CLEARS THE CACHE)
# =============================================================================
# FORCE_FRESH = True  -> the cache files of four analysis cells are deleted; the results are
#                        recomputed FROM SCRATCH from the data on this machine.
# FORCE_FRESH = False -> the cache is kept and the stored results are read.
import os

FORCE_FRESH = False

_fresh_targets = [
    os.path.join(RESULTS_BASE, "Phase_10", "Appendix_B_Stationarity_Tests.csv"),
    os.path.join(RESULTS_BASE, "Phase_11_Inflation", "Inflation_Robustness_Results.csv"),
    os.path.join(RESULTS_BASE, "Supplementary_Metrics", "XGBoost_Performance_Metrics.csv"),
    os.path.join(RESULTS_BASE, "Supplementary_Metrics", "GAM_Performance_Metrics.csv"),
]
if FORCE_FRESH:
    print(">> FORCE_FRESH = True: clearing four analysis caches; the results will be recomputed from scratch.")
    for p in _fresh_targets:
        if os.path.exists(p):
            os.remove(p)
            print("   Deleted:", p)
        else:
            print("   Already absent:", p)
else:
    print(">> FORCE_FRESH = False — cache kept, the stored results will be used.")

# A run starts from an empty, stamped results folder (setup cell); result files are never edited afterwards.
# Note: the FORCE_FIGURES switch lives in the setup cell (top of the notebook) –
# the figure phases run before this cell, so their caches have to be cleared there.

### Phase 10 (Extension): Appendix B – ADF & KPSS Stationarity Tests

Extends Appendix B with joint ADF & KPSS stationarity testing (Supplementary Table B1).
Saves `Appendix_B_Stationarity_Tests.csv` to `results/Phase_10/`.

In [ ]:
# =============================================================================
# PHASE 10 (EXTENSION): APPENDIX B - ADF & KPSS STATIONARITY TESTS (CACHED)
# =============================================================================
import pandas as pd
import os
import warnings
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning

# no blanket warning filter (see the setup cell)
KPSS_BOUND_NOTES = []   # KPSS p-values at the bounds of the statsmodels table (filled by run_tests)

OUTPUT_DIR_P10 = os.path.join(RESULTS_BASE, "Phase_10")
os.makedirs(OUTPUT_DIR_P10, exist_ok=True)

def generate_appendix_b_stationarity(df: pd.DataFrame):
    print("\n>>> APPENDIX B: ADF & KPSS STATIONARITY TESTS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_B_Stationarity_Tests.csv")

    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    def run_tests(series: pd.Series, name: str) -> dict:
        s = series.dropna()
        adf = adfuller(s)                                              # H0: unit root (non-stationary)
        # statsmodels interpolates the KPSS p-value in a table that covers 0.01-0.10; outside it returns the bound
        # and issues an InterpolationWarning. That warning is captured and reported with the table; others are passed on.
        _kw = []
        try:
            with warnings.catch_warnings(record=True) as _kw:
                warnings.simplefilter('always', InterpolationWarning)
                kpss_stat, kpss_p, _, kpss_crit = kpss(s, regression='c', nlags='auto')  # H0: stationary
        except Exception:
            for _x in _kw:                  # warnings caught before an error are passed on, then the error is raised
                warnings.warn_explicit(_x.message, _x.category, _x.filename, _x.lineno)
            raise
        for _x in _kw:
            if issubclass(_x.category, InterpolationWarning):
                KPSS_BOUND_NOTES.append(f"{name}: KPSS statistic {kpss_stat:.4f} lies outside the p-value table, "
                                        f"p {'<' if kpss_p <= 0.01 else '>'} {kpss_p:.2f}")
            else:
                warnings.warn_explicit(_x.message, _x.category, _x.filename, _x.lineno)
        adf_stationary  = adf[1] < 0.05                               # reject unit root
        kpss_stationary = kpss_stat < kpss_crit['5%']                 # fail to reject stationarity
        if adf_stationary and kpss_stationary:
            verdict = "I(0) - Stationary (both)"
        elif (not adf_stationary) and (not kpss_stationary):
            verdict = "I(1) - Non-Stationary (both)"
        else:
            verdict = "Long memory / persistence (tests diverge)"
        return {
            "Variable": name,
            "ADF Statistic": round(adf[0], 4), "ADF P-Value": round(adf[1], 4),
            "KPSS Statistic": round(kpss_stat, 4), "KPSS P-Value": round(kpss_p, 4),
            "KPSS 5% Crit.": round(kpss_crit['5%'], 4),
            "Conclusion": verdict
        }

    try:
        results = []
        if 'Mcp' in df.columns: results.append(run_tests(df['Mcp'], "MCP (Day-Ahead Price)"))
        if 'a-FRR_Price' in df.columns:
            results.append(run_tests(df['a-FRR_Price'], "a-FRR Price (Level)"))
            results.append(run_tests(df['a-FRR_Price'].diff(), "a-FRR Price (1st Difference)"))
        if 'Solar_Generation_Share' in df.columns: results.append(run_tests(df['Solar_Generation_Share'], "Solar Generation Share"))

        df_out = pd.DataFrame(results)
        df_out.to_csv(path, index=False, **CSV_KW)   # LF line endings on every platform
        print(df_out.to_string(index=False))
        for _n in KPSS_BOUND_NOTES:
            print(f"  Note: {_n}")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in stationarity tests: {e}")
        raise   # errors stop the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        generate_appendix_b_stationarity(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
_b1_csv = os.path.join(OUTPUT_DIR_P10, "Appendix_B_Stationarity_Tests.csv")
_b1 = pd.read_csv(_b1_csv)
# checkpoint K43: KPSS p-values at a bound of the statsmodels look-up table (0.01-0.10) are bounds, not values
_k43 = [f"{_r['Variable']}: p {'<' if _r['KPSS P-Value'] <= 0.01 else '>'} {_r['KPSS P-Value']:.2f} (statistic {_r['KPSS Statistic']:.4f})"
        for _, _r in _b1.iterrows() if _r['KPSS P-Value'] <= 0.01 or _r['KPSS P-Value'] >= 0.10]
_k43_agree = not KPSS_BOUND_NOTES or len(KPSS_BOUND_NOTES) == len(_k43)   # captured warnings (computed in this kernel) vs. table
checkpoint("K43 Appendix B: KPSS p-values at the table bounds", not _k43 and _k43_agree,
           (f"{len(_k43)} of {len(_b1)} KPSS p-values are bounds of the table, reported as bounds: " + "; ".join(_k43)) if _k43 else
           f"all {len(_b1)} KPSS p-values inside the table (0.01-0.10)"
           + ("" if _k43_agree else f"; {len(KPSS_BOUND_NOTES)} InterpolationWarning(s) captured, {len(_k43)} bound(s) in the table"),
           level='warn')
_k45_files = [_b1_csv]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Appendix B: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Appendix B - ADF and KPSS stationarity tests (Supplementary Table B1)",
       lines=[f"{r['Variable']}: {r['Conclusion']}" for _, r in _b1.iterrows()]
             + ["KPSS p-values are interpolated in the statsmodels table, which covers 0.01-0.10: 0.01 means p < 0.01 and 0.10 "
                "means p > 0.10. ADF p-values are rounded to four decimals (0.0 = below 0.00005)."],
       table=_b1.drop(columns='Conclusion'), files=_k45_files)
compare_with_reference("Table B1 vs. pre-test (pre-test sheet Hizli_Analizler, quick analyses)",
                       [(f"{r['Variable']}: {c}", r[c], f"B1.{r['Variable']}.{c}") for _, r in _b1.iterrows()
                        for c in ['ADF Statistic', 'ADF P-Value', 'KPSS Statistic', 'KPSS P-Value', 'KPSS 5% Crit.']],
                       check="K17 Appendix B: comparison with the independent pre-test")

## Phase 10 (Appendix I): Robustness Check with Natural Gas Cost Control

Re-estimates the Flexibility Trap (XGBoost) and GAM flexibility-stress models with the
Botaş natural-gas tariff as an additional control variable (column `Botas_Tariff` of the
data loaded and verified in Phase 1; the data file is not read again).

Produces **Appendix I** under `results/Phase_10/`: **Figure I.1** (Flexibility Trap +
Botaş Tariff) and **Figure I.2** (GAM Flexibility-Stress + Botaş Tariff).

The crossover of the two curves of Figure I.1 is computed with the same function as in
Phase 6.2 (`last_upward_crossover()`, defined in the Phase 6.2 cell, which must have been run in this kernel)
and saved to `Appendix_I_Flexibility_Trap_Crossover.csv`.

In [ ]:
# =============================================================================
# PHASE 10 – APPENDIX I: ROBUSTNESS CHECK WITH NATURAL GAS COST CONTROL (BOTAŞ)
# =============================================================================
# Produces: Appendix I – Figure I.1 (Flexibility Trap + Botaş), Figure I.2 (GAM Flexibility-Stress + Botaş)
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import statsmodels.formula.api as smf
from patsy import cr
import os
import warnings

# no blanket warning filter (see the setup cell)

OUTPUT_DIR_P10 = os.path.join(RESULTS_BASE, "Phase_10")
os.makedirs(OUTPUT_DIR_P10, exist_ok=True)

# Settings closest to the original academic style
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def run_appendix_i_botas(df_b):
    print(f"\n>> PHASE 10 — APPENDIX I: Botaş Robustness Check (Directory: '{OUTPUT_DIR_P10}')...")
    print("="*80)

    if 'Botas_Tariff' not in df_b.columns:
        print("!! WARNING: 'Botas_Tariff' column not found in dataset. Skipping Appendix I.")
        return

    fig11_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Figure_I1_Flexibility_Trap_Botas.png')
    fig12_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Figure_I2_GAM_Flexibility_Stress_Botas.png')
    report_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Botas_Report.txt')
    # crossover of the two curves of Figure I.1, saved together with the figure
    crossover_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Flexibility_Trap_Crossover.csv')

    # =========================================================================
    # 1. XGBOOST FLEXIBILITY TRAP (+ BOTAS_TARIFF)
    # =========================================================================
    if os.path.exists(fig11_path) and os.path.exists(crossover_path):   # the crossover file is part of the cache
        print(f"  [CACHE] Loading {os.path.basename(fig11_path)}...")
        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(fig11_path))
        plt.axis('off')
        plt.show()
    else:
        if 'last_upward_crossover' not in globals():   # the crossover function of Phase 6.2 is used
            raise RuntimeError("last_upward_crossover() is not defined: run the Phase 6.2 cell first (same kernel).")
        print("  [Figure I.1] Training XGBoost with Botaş Tariff...")
        dm = df_b.copy()
        if dm['Solar_Generation_Share'].max() <= 1:
            dm['Solar_Generation_Share'] *= 100

        sur = dm[dm['System_Direction_Down'] == 1]
        dfc = dm[dm['System_Direction_Down'] == 0]

        ft = ['Solar_Generation_Share','Mcp','Consumption','Wind_Generation','Hour_Sin','Hour_Cos','Botas_Tariff']

        ms = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(sur[ft], sur['a-FRR_Price'])
        md = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(dfc[ft], dfc['a-FRR_Price'])

        sr = np.linspace(0, 60, 100)
        avg = dm[ft].median()
        sim = pd.DataFrame([avg]*100, columns=ft)
        sim['Solar_Generation_Share'] = sr

        mk = sr <= 45
        yd = md.predict(sim[mk][ft])
        ys = ms.predict(sim[ft])
        ysc = ys[:len(yd)]
        sc = sr[mk]

        fig, ax = plt.subplots(figsize=(10, 6))
        
        ax.plot(sc, yd, color='#1f77b4', lw=3.0, label='Energy Deficit & Balanced')
        ax.plot(sr, ys, color='red', lw=3.0, label='Energy Surplus')
        
        # [Revision_R2_M6] legend terms aligned with Section 4.5 (same as Figure 8)
        ax.fill_between(sc, ysc, yd, where=(yd>=ysc), interpolate=True, color='gray', alpha=.12, hatch='...', label='Deficit premium (deficit/balanced > surplus)')
        ax.fill_between(sc, ysc, yd, where=(ysc>yd), interpolate=True, color='red', alpha=.08, hatch='xxx', label='Flexibility Trap (surplus > deficit/balanced)')
        
        ax.set_xlabel('Solar Generation Share (%)')
        ax.set_ylabel('Predicted a-FRR Price (TL/MW)')  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW)
        
        ax.legend(loc='upper left', fontsize=11, frameon=True)
        ax.grid(True, ls='--', alpha=.4)
        ax.axvline(x=45, color='gray', ls=':', alpha=.5)
        
        plt.tight_layout()
        plt.savefig(fig11_path, dpi=FIG_DPI, bbox_inches='tight')
        plt.show()
        print(f"  Saved: {fig11_path}")

        # crossover of the two curves, computed with the same function as Phase 6.2 and saved with the figure
        crossover, up, down = last_upward_crossover(sc, ysc, yd)
        pd.DataFrame([{'Crossover_Solar_Share_pct': crossover,
                       'Upward_crossings_pct': "; ".join(f"{v:.2f}" for v in up),        # surplus curve rises above the deficit/balanced curve
                       'Downward_crossings_pct': "; ".join(f"{v:.2f}" for v in down),    # surplus curve falls below it
                       'Flexibility_Trap_share_of_range_pct': (ysc > yd).sum() / len(yd) * 100,
                       'Deficit_premium_share_of_range_pct': (yd >= ysc).sum() / len(yd) * 100,
                       'Common_range_pct': '0-45', 'Grid_step_pct': float(sr[1] - sr[0]),
                       'Curves': 'XGBoost (n_estimators=200, random_state=42) with Botas_Tariff, other features at their medians',
                       'xgboost_version': xgb.__version__}]).to_csv(crossover_path, index=False, **CSV_KW)
        print("  Crossover point (surplus curve above the deficit/balanced curve from there to 45%): "
              + (f"{crossover:.1f}% solar share" if np.isfinite(crossover) else "none (no upward crossing that holds up to 45%)"))
        print(f"  Saved: {crossover_path}")

    # =========================================================================
    # 2. GAM FLEXIBILITY-STRESS (+ BOTAS_TARIFF)
    # =========================================================================
    if os.path.exists(fig12_path) and os.path.exists(report_path):
        print(f"  [CACHE] Loading {os.path.basename(fig12_path)} and Report...")
        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(fig12_path))
        plt.axis('off')
        plt.show()
        
        with open(report_path, "r", encoding="utf-8") as f:
            print(f.read())
    else:
        print("\n  [Figure I.2] Training GAM Flexibility-Stress with Botaş Tariff...")
        
        du = df_b[df_b['System_Direction_Down'] == 0]
        dd = df_b[df_b['System_Direction_Down'] == 1]

        formula = 'Q("a-FRR_Price") ~ cr(Solar_Generation_Share,df=6)+Consumption+Wind_Generation+Import_Coal_Generation+Botas_Tariff'

        mu = smf.ols(formula, data=du).fit()
        md = smf.ols(formula, data=dd).fit()

        sg = np.linspace(df_b['Solar_Generation_Share'].min(), df_b['Solar_Generation_Share'].max(), 100)
        pred_df = pd.DataFrame({
            'Solar_Generation_Share': sg,
            'Consumption': df_b['Consumption'].mean(),
            'Wind_Generation': df_b['Wind_Generation'].mean(),
            'Import_Coal_Generation': df_b['Import_Coal_Generation'].mean(),
            'Botas_Tariff': df_b['Botas_Tariff'].mean()
        })

        ru = mu.get_prediction(pred_df).summary_frame(alpha=.05)
        rd = md.get_prediction(pred_df).summary_frame(alpha=.05)
        
        # Multiply by 100 for the x-axis (plot) only
        sp = sg * 100

        fig, ax = plt.subplots(figsize=(10, 6))
        
        
        ax.plot(sp, ru['mean'], color='#1f77b4', lw=3.0, label='Energy Deficit & Balanced')
        ax.fill_between(sp, ru['mean_ci_lower'], ru['mean_ci_upper'], color='#1f77b4', alpha=.12, linewidth=0, label='95% CI (Deficit & Balanced)')
        ax.plot(sp, rd['mean'], color='red', lw=3.0, label='Energy Surplus')
        ax.fill_between(sp, rd['mean_ci_lower'], rd['mean_ci_upper'], color='red', alpha=.12, linewidth=0, label='95% CI (Surplus)')
        
        ax.set_xlabel('Solar Generation Share (%)')
        ax.set_ylabel('Simulated a-FRR Price (TL/MW)')  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW)
        
        ax.axvline(x=20, color='grey', ls='--', alpha=.5, label='~20% onset')
        ax.legend(loc='upper left', fontsize=11, frameon=True)
        ax.grid(True, alpha=.3)
        
        plt.tight_layout()
        plt.savefig(fig12_path, dpi=FIG_DPI, bbox_inches='tight')
        plt.show()
        print(f"  Saved: {fig12_path}")

        # Save and Print R² Report
        report_content = f"=== Appendix I: GAM Flexibility-Stress (Botaş Controlled) ===\n" \
                         f"Deficit/Balanced R²: {mu.rsquared:.4f}\n" \
                         f"Surplus R²: {md.rsquared:.4f}\n"
                         
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        print(f"\n{report_content}")
        print(f"  Report Saved: {report_path}")

    print("\n>> Appendix I (Botaş) complete.")
    print("="*80)

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = globals().get('df', locals().get('df'))
    
    if target_df is not None and 'Botas_Tariff' in target_df.columns:
        run_appendix_i_botas(target_df)
    else:
        # only the verified in-memory data of Phase 1 is used; the data file is not read again
        raise RuntimeError("DataFrame 'df' with the column 'Botas_Tariff' not found in memory. Please run Phase 1 first.")

# run report and comparison with the independent pre-test
import re
_i_cross_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Flexibility_Trap_Crossover.csv')
_i_report_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Botas_Report.txt')
_i_cross = pd.read_csv(_i_cross_path, float_precision='round_trip').iloc[0]
with open(_i_report_path, "r", encoding="utf-8") as _f:
    _i_text = _f.read()
_i_r2 = {_k: float(re.search(re.escape(_k) + r" R²:\s*([-0-9.]+)", _i_text).group(1)) for _k in ('Deficit/Balanced', 'Surplus')}
# Figure I.1 crossover in words: when the surplus curve does not stay above the deficit/balanced curve up to 45%
# the crossings saved with the figure are listed instead of 'nan' (the comparison below keeps the value of the file)
_i_pts = lambda v: [float(t) for t in str(v).split(';') if t.strip() not in ('', 'nan')]
_i_up, _i_down = _i_pts(_i_cross['Upward_crossings_pct']), _i_pts(_i_cross['Downward_crossings_pct'])
_i_x, _i_trap = float(_i_cross['Crossover_Solar_Share_pct']), float(_i_cross['Flexibility_Trap_share_of_range_pct'])
_i_events = sorted([(v, 'up') for v in _i_up] + [(v, 'down') for v in _i_down])
_i_end_above = (_i_events[-1][1] == 'up') if _i_events else _i_trap > 50.0
_i_fmt = lambda xs: (", ".join(f"{v:.2f}" for v in xs) + "% solar share") if xs else "no point"
_i_explained = False   # True when the DIFFERS row of the crossover is explained in the report (checkpoint K17)
if np.isfinite(_i_x):
    _i_cross_lines = [f"Figure I.1 crossover (the surplus curve stays above the deficit/balanced curve from there to 45%): {_i_x:.2f}% solar share."]
else:
    _i_cross_lines = [f"Figure I.1 crossover (the surplus curve stays above the deficit/balanced curve from there to 45%): none. "
                      f"The surplus curve rises above the deficit/balanced curve at {_i_fmt(_i_up)} and falls below it at "
                      f"{_i_fmt(_i_down)}; at the end of the 0-45% range it is {'above' if _i_end_above else 'below'} it "
                      f"(above over {_i_trap:.1f}% of the range)."]
    _i_ref = REFERENCE.get("KESISIM.Appendix_I_Figure_I1")
    _i_after = [v for v in _i_down if _i_up and v > _i_up[-1]]
    if _i_ref is not None and _i_up and _i_after and abs(_i_ref[0] - _i_up[-1]) < 0.01:
        _i_cross_lines.append(f"The pre-test value {_i_ref[0]:.2f}% coincides with the last upward crossing; the surplus curve falls below the "
                              f"deficit/balanced curve again at {min(_i_after):.2f}%, so the comparison row below shows DIFFERS.")
        _i_explained = True
# checkpoint K42: no crossover that holds to the end of the range -> WARNING with the crossings (a documented
# property of Appendix I, not an error)
checkpoint("K42 Appendix I: flexibility-trap crossover", bool(np.isfinite(_i_x)),
           "crossover = the last upward crossing of the surplus curve over the deficit/balanced curve after which it stays above "
           f"up to the end of the common range ({_i_cross['Common_range_pct']}%): "
           + (f"{_i_x:.2f}% solar share" if np.isfinite(_i_x) else "none (no sustained crossover)")
           + f"; upward crossings: {_i_fmt(_i_up)}; downward crossings: {_i_fmt(_i_down)}", level='warn')
_k45_files = [os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Figure_I1_Flexibility_Trap_Botas.png'), _i_cross_path,
              os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Figure_I2_GAM_Flexibility_Stress_Botas.png'), _i_report_path]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Appendix I: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Appendix I: flexibility trap and GAM flexibility-stress with the Botas tariff as a control",
       lines=_i_cross_lines + [
              f"Figure I.2 GAM R2: deficit/balanced {_i_r2['Deficit/Balanced']:.4f}, surplus {_i_r2['Surplus']:.4f}.",
              f"xgboost {_i_cross['xgboost_version']} (version of the fit). Tree ensembles depend on library version and platform:"
              " agreement with the pre-test is expected in the reference environment only."],
       files=_k45_files)
compare_with_reference("Appendix I: crossover (Figure I.1) and GAM fit (Figure I.2)",
                       [("Figure I.1 crossover, % solar share", _i_cross['Crossover_Solar_Share_pct'], "KESISIM.Appendix_I_Figure_I1"),
                        ("GAM R2, deficit/balanced", _i_r2['Deficit/Balanced'], "GAMRAPOR.AppendixI_Deficit_Balanced_R2"),
                        ("GAM R2, surplus", _i_r2['Surplus'], "GAMRAPOR.AppendixI_Surplus_R2")],
                       check="K17 Appendix I: comparison with the independent pre-test",
                       explained={'Figure I.1 crossover, % solar share': 'pre-test value being the last upward crossing, after which the surplus curve falls below again (section above)'} if _i_explained else None)

## Phase 11: Inflation-Robustness Checks (Ratio & Cap-Normalised FE-OLS)

### Phase 11.1: Ratio and Cap-Normalised FE-OLS

Verifies that the MCP / a-FRR divergence is not an artefact of inflation or currency
drift (Supplementary Appendix J.6, summarised in manuscript Section 4.6). Estimates fixed-effects OLS models
(Hour + Month + Year FE, standard errors clustered by day) for:
the a-FRR/MCP normalisation ratio and cap-normalised MCP & a-FRR prices.

*Standard errors are clustered by calendar day, taken from the time index (`day_key`, Phase 1). The number of day clusters of each model is written to the output file (column `Clusters (days)`).*

Produces: `Inflation_Robustness_Results.csv` in `results/Phase_11_Inflation/`.

*The two cells after this check compute the in-sample model-fit metrics of the XGBoost and GAM models (Supplementary Material S1; `results/Supplementary_Metrics/`). They are not part of the inflation check and keep their R1 position.*

In [ ]:
# =============================================================================
# PHASE 11: INFLATION-ROBUSTNESS CHECKS (RATIO & CAP-NORMALISED FE-OLS) - CACHED
# =============================================================================
import pandas as pd, numpy as np, os, warnings
import statsmodels.formula.api as smf
# no blanket warning filter (see the setup cell)

OUTPUT_DIR_P11 = os.path.join(RESULTS_BASE, "Phase_11_Inflation")
os.makedirs(OUTPUT_DIR_P11, exist_ok=True)

def generate_inflation_robustness(df: pd.DataFrame):
    print("\n>>> PHASE 11: INFLATION-ROBUSTNESS (RATIO & CAP-NORMALISED FE-OLS)")
    path = os.path.join(OUTPUT_DIR_P11, "Inflation_Robustness_Results.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    d = df.copy()
    # statsmodels formulas cannot contain hyphens -> rename
    d = d.rename(columns={'a-FRR_Price': 'aFRR',
                          'a-FRR_Price_Divide_Price_Cap': 'aFRR_over_Cap',
                          'Mcp_Divide_Price_Cap': 'Mcp_over_Cap'})
    d['Year'] = d.index.year   # year from the time index (the 'Date' column is not used)

    # inflation-neutral ratio (drop MCP<1 to avoid blow-up; winsorise 1/99)
    d_ratio = d[d['Mcp'] >= 1].copy()
    d_ratio['Ratio'] = d_ratio['aFRR'] / d_ratio['Mcp']
    lo, hi = d_ratio['Ratio'].quantile([.01, .99]); d_ratio['Ratio'] = d_ratio['Ratio'].clip(lo, hi)

    CTRL = ("Consumption + Wind_Generation_Share + Hydroelectric_Generation_Share "
            "+ Botas_Tariff + Weekend + Public_Holiday + C(Hour) + C(Month) + C(Year)")

    def day_id(data):
        # cluster by calendar day, derived from the time index (day_key, Phase 1)
        return day_key(data.index)

    def fe_ols(dep, data, label):
        m = smf.ols(f"{dep} ~ Solar_Generation_Share + {CTRL}", data=data)\
               .fit(cov_type='cluster', cov_kwds={'groups': day_id(data)})
        b = m.params['Solar_Generation_Share']
        return {"Dependent Variable": label,
                "Solar Coef (per unit share)": round(b, 4),
                "Effect per +1pp": round(b/100, 4),
                "Std. Error": round(m.bse['Solar_Generation_Share'], 4),
                "P-Value": round(m.pvalues['Solar_Generation_Share'], 4),
                "R-squared": round(m.rsquared, 3),
                "N": int(m.nobs),
                "Clusters (days)": int(np.unique(day_id(data)).size)}   # number of day clusters behind the std. error
    try:
        rows = [
            fe_ols("Mcp",           d,       "MCP (level)"),        # merit-order, reference
            fe_ols("aFRR",          d,       "a-FRR (level)"),      # balancing, reference
            fe_ols("Ratio",         d_ratio, "a-FRR / MCP ratio"),  # inflation-neutral
            fe_ols("Mcp_over_Cap",  d,       "MCP / Price Cap"),    # inflation-neutral
            fe_ols("aFRR_over_Cap", d,       "a-FRR / Price Cap"),  # inflation-neutral
        ]
        out = pd.DataFrame(rows)
        out.to_csv(path, index=False, **CSV_KW)   # LF line endings on every platform
        print(out.to_string(index=False))
        print("\n  Note: Hour+Month+Year fixed effects; std. errors clustered by calendar day (day_key of the time index).")
        print(f"  Day clusters per model: {out['Clusters (days)'].tolist()} (calendar days in the full sample: {np.unique(day_id(d)).size:,}).")
        print(f"  Ratio model drops {len(d)-len(d_ratio)} hours with MCP<1 and winsorises at 1/99 pct.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in inflation-robustness checks: {e}")
        raise   # errors stop the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        generate_inflation_robustness(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
_j6_csv = os.path.join(OUTPUT_DIR_P11, "Inflation_Robustness_Results.csv")
_j6 = pd.read_csv(_j6_csv)
# checkpoint K44: one day cluster per calendar day of each model's sample (and N = hours of that sample);
# the ratio model uses the hours with MCP >= 1, the other models the whole sample
_k44 = []
if 'Clusters (days)' in _j6.columns:
    for _, _r in _j6.iterrows():
        _smp = df[df['Mcp'] >= 1] if _r['Dependent Variable'] == 'a-FRR / MCP ratio' else df
        _k44.append((_r['Dependent Variable'], int(_r['Clusters (days)']), int(np.unique(day_key(_smp.index)).size), int(_r['N']), len(_smp)))
checkpoint("K44 Appendix J.6: day clusters", all(_c == _d and _n == _h for _, _c, _d, _n, _h in _k44) if _k44 else None,
           "; ".join(f"{_v}: {_c:,} clusters for {_d:,} calendar days, N = {_n:,} of {_h:,} hours" for _v, _c, _d, _n, _h in _k44)
           if _k44 else "the table has no 'Clusters (days)' column (produced before release R2)")
_k45_files = [_j6_csv]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Appendix J.6: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 11.1 - inflation robustness, ratio and cap-normalised FE-OLS (Supplementary Appendix J.6)",
       lines=["Hour, month and year fixed effects; standard errors clustered by calendar day (day_key of the time index).",
              "Day clusters: " + ", ".join(f"{r['Dependent Variable']} = {int(r['Clusters (days)']):,}" for _, r in _j6.iterrows()) + "."],
       table=_j6, files=_k45_files)
compare_with_reference("Appendix J.6 vs. pre-test (pre-test sheet Hizli_Analizler, quick analyses)",
                       [(f"{r['Dependent Variable']}: {c}", r[c], f"J6.{r['Dependent Variable']}.{c}") for _, r in _j6.iterrows()
                        for c in ['Solar Coef (per unit share)', 'Effect per +1pp', 'Std. Error', 'P-Value', 'R-squared', 'N']],
                       check="K17 Appendix J.6: comparison with the independent pre-test")

In [ ]:
# =============================================================================
# SUPPLEMENTARY: XGBOOST PERFORMANCE METRICS (R² AND ACCURACY) - CACHED
# =============================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score, accuracy_score
import os
import warnings

# no blanket warning filter (see the setup cell)

OUTPUT_DIR_SUPP = os.path.join(RESULTS_BASE, "Supplementary_Metrics")
os.makedirs(OUTPUT_DIR_SUPP, exist_ok=True)

def run_xgboost_performance_metrics():
    print(">> Computing XGBoost performance metrics...")
    path = os.path.join(OUTPUT_DIR_SUPP, "XGBoost_Performance_Metrics.csv")

    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    # 1. Data: the verified in-memory data of Phase 1 is used; the data file is not read again
    df_ml = df.copy()

    # 2. Data preparation
    if df_ml['Solar_Generation_Share'].max() <= 1.0:
        df_ml['Solar_Generation_Share'] *= 100

    features_core = ['Solar_Generation_Share', 'Consumption', 'Wind_Generation', 'Price_Cap', 'Hour_Sin', 'Hour_Cos']
    features_with_mcp = features_core + ['Mcp']
    # Exactly the Phase 6.2 (Flexibility Trap) feature set – the regime models are evaluated with this set
    features_trap = ['Solar_Generation_Share', 'Mcp', 'Consumption', 'Wind_Generation', 'Hour_Sin', 'Hour_Cos']

    # 3. General models (Phase 6.1)
    print(">> Training general models...")

    m_a = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(df_ml[features_core], df_ml['Mcp'])
    m_b = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(df_ml[features_with_mcp], df_ml['a-FRR_Price'])
    m_c = xgb.XGBClassifier(n_estimators=200, random_state=42).fit(df_ml[features_core], df_ml['System_Direction_Down'])

    # 4. Regime-conditional models (Phase 6.2 - Flexibility Trap)
    print(">> Training regime-conditional models...")

    surplus_data = df_ml[df_ml['System_Direction_Down'] == 1]
    deficit_bal_data = df_ml[df_ml['System_Direction_Down'] == 0]

    model_surplus = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(surplus_data[features_trap], surplus_data['a-FRR_Price'])
    model_deficit = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(deficit_bal_data[features_trap], deficit_bal_data['a-FRR_Price'])

    # 5. Metrics
    y_pred_mcp = m_a.predict(df_ml[features_core])
    r2_mcp = r2_score(df_ml['Mcp'], y_pred_mcp)

    y_pred_afrr = m_b.predict(df_ml[features_with_mcp])
    r2_afrr = r2_score(df_ml['a-FRR_Price'], y_pred_afrr)

    y_pred_dir = m_c.predict(df_ml[features_core])
    acc_dir = accuracy_score(df_ml['System_Direction_Down'], y_pred_dir)

    y_pred_def = model_deficit.predict(deficit_bal_data[features_trap])
    r2_def = r2_score(deficit_bal_data['a-FRR_Price'], y_pred_def)

    y_pred_sur = model_surplus.predict(surplus_data[features_trap])
    r2_sur = r2_score(surplus_data['a-FRR_Price'], y_pred_sur)

    # 6. Save + report
    out = pd.DataFrame([
        {"Metric": "MCP Model R2 Score",            "Value": round(r2_mcp, 3)},
        {"Metric": "a-FRR (Volatility) R2 Score",   "Value": round(r2_afrr, 3)},
        {"Metric": "System Direction Accuracy (%)", "Value": round(acc_dir * 100, 1)},
        {"Metric": "Deficit Regime Model R2",       "Value": round(r2_def, 3)},
        {"Metric": "Surplus Regime Model R2",       "Value": round(r2_sur, 3)},
    ])
    out.to_csv(path, index=False, **CSV_KW)   # LF line endings on every platform

    print("\n" + "=" * 50)
    print("XGBOOST PERFORMANCE METRICS SUMMARY")
    print("=" * 50)
    print(f"1. MCP Model R2 Score:             {r2_mcp:.3f}")
    print(f"2. a-FRR (Volatility) R2 Score:    {r2_afrr:.3f}")
    print(f"3. System Direction Accuracy:      {acc_dir*100:.1f}%")
    print(f"4. Deficit Regime Model R2:        {r2_def:.3f}")
    print(f"5. Surplus Regime Model R2:        {r2_sur:.3f}")
    print("=" * 50)
    print(f"  Saved: {path}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    run_xgboost_performance_metrics()

# run report and comparison with the independent pre-test
_xgb_path = os.path.join(OUTPUT_DIR_SUPP, "XGBoost_Performance_Metrics.csv")
_xgb_m = pd.read_csv(_xgb_path, float_precision='round_trip')
# checkpoint K41: the in-sample fit metrics of the five XGBoost models are finite
_k41_bad = [str(_r['Metric']) for _, _r in _xgb_m.iterrows() if not np.isfinite(float(_r['Value']))]
checkpoint("K41 Supplementary metrics: XGBoost in-sample fit", bool(len(_xgb_m)) and not _k41_bad,
           f"{len(_xgb_m) - len(_k41_bad)} of {len(_xgb_m)} metrics finite" + "".join(f"; not finite: {_b}" for _b in _k41_bad))
_k45_files = [_xgb_path]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Supplementary XGBoost metrics: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Supplementary: in-sample fit of the XGBoost models (Phase 6.1 and Phase 6.2)",
       lines=["In-sample R2 (accuracy for the system-direction classifier) of the five XGBoost models, rounded as stored in the file.",
              f"xgboost {xgb.__version__} in this kernel (a cached file keeps the values of the session that fitted the models). Tree ensembles depend on library version and platform:"
              " agreement with the pre-test is expected in the reference environment only."],
       table=_xgb_m, files=_k45_files)
compare_with_reference("XGBoost performance metrics (in-sample)",
                       [(_r['Metric'], _r['Value'], f"XGB.{_r['Metric']}") for _, _r in _xgb_m.iterrows()],
                       check="K17 Supplementary XGBoost metrics: comparison with the independent pre-test")

In [ ]:
# =============================================================================
# SUPPLEMENTARY: GAM PERFORMANCE METRICS (R²) - CACHED
# =============================================================================

import pandas as pd
import os
import statsmodels.formula.api as smf
from patsy import cr

OUTPUT_DIR_SUPP = os.path.join(RESULTS_BASE, "Supplementary_Metrics")
os.makedirs(OUTPUT_DIR_SUPP, exist_ok=True)

def check_gam_performance(df):
    print("\n>> Computing GAM model R-squared values...")
    print("-" * 50)
    path = os.path.join(OUTPUT_DIR_SUPP, "GAM_Performance_Metrics.csv")

    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        print("-" * 50)
        return

    # Regime separation
    df_up = df[df['System_Direction_Down'] == 0].copy()
    df_down = df[df['System_Direction_Down'] == 1].copy()

    # Same formula as Phase 8
    formula = 'Q("a-FRR_Price") ~ cr(Solar_Generation_Share, df=6) + Consumption + Wind_Generation + Import_Coal_Generation'

    m_up = smf.ols(formula, data=df_up).fit()
    m_down = smf.ols(formula, data=df_down).fit()

    out = pd.DataFrame([
        {"Model": "Energy DEFICIT (Up-Regulation)",   "R-squared": round(m_up.rsquared, 4),   "N": int(m_up.nobs)},
        {"Model": "Energy SURPLUS (Down-Regulation)", "R-squared": round(m_down.rsquared, 4), "N": int(m_down.nobs)},
    ])
    out.to_csv(path, index=False, **CSV_KW)   # LF line endings on every platform

    print(f"1. Energy DEFICIT Model (Up-Regulation) R-squared: {m_up.rsquared:.4f}")
    print(f"2. Energy SURPLUS Model (Down-Regulation) R-squared: {m_down.rsquared:.4f}")
    print(f"  Saved: {path}")
    print("-" * 50)

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        check_gam_performance(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
_gamfit_csv = os.path.join(OUTPUT_DIR_SUPP, "GAM_Performance_Metrics.csv")
_gamfit = pd.read_csv(_gamfit_csv)
_k45_files = [_gamfit_csv]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Supplementary GAM metrics: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Supplementary metrics - GAM fit by regime (R-squared)",
       lines=[f"{r['Model']}: R-squared = {r['R-squared']:.4f}, N = {int(r['N']):,}" for _, r in _gamfit.iterrows()],
       files=_k45_files)
compare_with_reference("GAM fit metrics vs. pre-test (pre-test sheet Hizli_Analizler, quick analyses)",
                       [(f"{r['Model']}: {c}", r[c], f"GAM.{r['Model']}.{c}") for _, r in _gamfit.iterrows() for c in ['R-squared', 'N']],
                       check="K17 Supplementary GAM metrics: comparison with the independent pre-test")

### Phase 11.2: Timing, Decision-Time Information & Confounding (Revision_Review_Referee_2.1)

Re-estimates the solar coefficients with information available at each market's decision time: the solar share
lagged one day (D−1) for the day-ahead MCP and two days (D−2) for the a-FRR capacity price, and the system
operator's real-time (TEİAŞ) production series (`RT_Solar_Generation_Share`, `RT_Consumption`). Also tests whether
the solar effect is distinct from consumption (correlations, VIF, horse-race) and the net-load ramp mechanism.
Interpretive OLS with Newey–West (HAC, 24 lags) standard errors.

Produces: `Phase_11_2_Timing_Robustness.csv` and `Phase_11_2_Confounding_Diagnostics.csv` in `results/Phase_11_Robustness/`.

In [ ]:
# =============================================================================
# PHASE 11.2: TIMING, DECISION-TIME INFORMATION & CONFOUNDING (OLS-HAC) - CACHED
# =============================================================================
# Revision_Review_Referee_2.1 – Timing, decision-time info & confounding
# Feeds Supplementary Appendix J.1.
# =============================================================================
import pandas as pd, numpy as np, os, warnings
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
# no blanket warning filter (see the setup cell)

OUTPUT_DIR_P11R = os.path.join(RESULTS_BASE, "Phase_11_Robustness")
os.makedirs(OUTPUT_DIR_P11R, exist_ok=True)

# Core control set of the manuscript (natural gas excluded)
REV_CONTROLS = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation',
                'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
                'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
                'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

# TEİAŞ real-time (ex-ante) series: RT solar share = solar MW / total generation MW, RT consumption = load MW
RT_SOLAR_SHARE_COL = 'RT_Solar_Generation_Share'
RT_CONSUMPTION_COL = 'RT_Consumption'

def ols_hac(data: pd.DataFrame, dep: str, xvars: list):
    """OLS of dep on xvars (+ constant) with Newey-West (HAC) standard errors, 24-hour lag window."""
    d = data.dropna(subset=[dep] + xvars)
    X = sm.add_constant(d[xvars].astype(float))
    return sm.OLS(d[dep].astype(float), X).fit(cov_type='HAC', cov_kwds={'maxlags': 24})

def run_timing_confounding(df: pd.DataFrame):
    print("\n>>> PHASE 11.2: TIMING / DECISION-TIME INFORMATION / CONFOUNDING (OLS-HAC)")
    path_t = os.path.join(OUTPUT_DIR_P11R, "Phase_11_2_Timing_Robustness.csv")
    path_c = os.path.join(OUTPUT_DIR_P11R, "Phase_11_2_Confounding_Diagnostics.csv")
    if os.path.exists(path_t) and os.path.exists(path_c):
        print(f"  [CACHE] Loaded from {path_t}")
        print(pd.read_csv(path_t).to_string(index=False))
        print(f"\n  [CACHE] Loaded from {path_c}")
        print(pd.read_csv(path_c).to_string(index=False)); return

    d = df.copy()   # hourly series, sorted by Date_Hour (Phase 1)
    d['solar_lag24'] = d['Solar_Generation_Share'].shift(24)   # day-ahead market: D-1
    d['solar_lag48'] = d['Solar_Generation_Share'].shift(48)   # a-FRR capacity: D-2
    d['Net_Load'] = d['Consumption'] - d['Solar_Generation']
    d['Net_Load_Ramp'] = d['Net_Load'].diff()
    cc = [c for c in REV_CONTROLS if c != 'Consumption']

    rows = []
    def add(block, label, dep, key, xvars):
        m = ols_hac(d, dep, xvars)
        rows.append({'Block': block, 'Specification': label, 'Dependent': dep, 'Regressor': key,
                     'Beta': m.params[key], 't_stat': m.tvalues[key], 'p_value': m.pvalues[key], 'N': int(m.nobs)})
    try:
        # [A] decision-time lags of the settlement (ex-post) solar share
        add('A: decision-time lag', 'MCP - realised, same hour', 'Mcp', 'Solar_Generation_Share', ['Solar_Generation_Share'] + REV_CONTROLS)
        add('A: decision-time lag', 'MCP - solar lagged one day (D-1)', 'Mcp', 'solar_lag24', ['solar_lag24'] + REV_CONTROLS)
        add('A: decision-time lag', 'a-FRR - realised, same hour', 'a-FRR_Price', 'Solar_Generation_Share', ['Solar_Generation_Share'] + REV_CONTROLS)
        add('A: decision-time lag', 'a-FRR - solar lagged two days (D-2)', 'a-FRR_Price', 'solar_lag48', ['solar_lag48'] + REV_CONTROLS)

        # [A'] real-time (ex-ante) TEİAŞ series; consumption also at decision time
        if RT_SOLAR_SHARE_COL in d.columns and RT_CONSUMPTION_COL in d.columns:
            d['rt_solar_lag24'] = d[RT_SOLAR_SHARE_COL].shift(24)
            d['rt_solar_lag48'] = d[RT_SOLAR_SHARE_COL].shift(48)
            d['rt_cons_lag24'] = d[RT_CONSUMPTION_COL].shift(24)
            d['rt_cons_lag48'] = d[RT_CONSUMPTION_COL].shift(48)
            add("A': real-time (ex-ante)", 'MCP - real-time solar (D-1) + real-time consumption', 'Mcp', 'rt_solar_lag24', ['rt_solar_lag24', RT_CONSUMPTION_COL] + cc)
            add("A': real-time (ex-ante)", 'MCP - fully ex-ante (real-time consumption, D-1)', 'Mcp', 'rt_solar_lag24', ['rt_solar_lag24', 'rt_cons_lag24'] + cc)
            add("A': real-time (ex-ante)", 'a-FRR - real-time solar (D-2) + real-time consumption', 'a-FRR_Price', 'rt_solar_lag48', ['rt_solar_lag48', RT_CONSUMPTION_COL] + cc)
            add("A': real-time (ex-ante)", 'a-FRR - fully ex-ante (real-time consumption, D-2)', 'a-FRR_Price', 'rt_solar_lag48', ['rt_solar_lag48', 'rt_cons_lag48'] + cc)
            for key in [RT_SOLAR_SHARE_COL, RT_CONSUMPTION_COL]:
                add("A': real-time horse-race", 'a-FRR ~ RT solar + RT consumption', 'a-FRR_Price', key, [RT_SOLAR_SHARE_COL, RT_CONSUMPTION_COL] + cc)
        else:
            print("  !! Real-time (TEİAŞ) columns not found -> block A' skipped.")

        # [B] horse-race: solar share and consumption in the same model
        for key in ['Solar_Generation_Share', 'Consumption']:
            add('B: horse-race', 'a-FRR ~ solar + consumption', 'a-FRR_Price', key, ['Solar_Generation_Share', 'Consumption'] + cc)

        # [C] mechanism: net load and its ramp
        add('C: net-load mechanism', 'a-FRR ~ net load', 'a-FRR_Price', 'Net_Load', ['Net_Load'] + cc)
        add('C: net-load mechanism', 'a-FRR ~ net-load ramp + solar', 'a-FRR_Price', 'Net_Load_Ramp', ['Net_Load_Ramp', 'Solar_Generation_Share'] + cc)

        timing = pd.DataFrame(rows)

        # Confounding diagnostics: correlations and VIF
        diag = []
        if RT_SOLAR_SHARE_COL in d.columns:
            diag.append({'Diagnostic': 'Correlation RT (ex-ante) vs settlement (ex-post) solar share', 'Item': 'all hours',
                         'Value': d[RT_SOLAR_SHARE_COL].corr(d['Solar_Generation_Share'])})
        diag.append({'Diagnostic': 'Correlation solar share vs consumption', 'Item': 'all hours',
                     'Value': d['Solar_Generation_Share'].corr(d['Consumption'])})
        for h in [10, 12, 14, 16, 17, 18, 19, 20]:
            sub = d[d['Hour'] == h]
            diag.append({'Diagnostic': 'Correlation solar share vs consumption', 'Item': f'hour {h}',
                         'Value': sub['Solar_Generation_Share'].corr(sub['Consumption'])})
        vifvars = ['Solar_Generation_Share', 'Consumption', 'Wind_Generation', 'Import_Coal_Generation',
                   'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap']
        Xv = sm.add_constant(d[vifvars].dropna().astype(float))
        for i, v in enumerate(vifvars):
            diag.append({'Diagnostic': 'VIF', 'Item': v, 'Value': variance_inflation_factor(Xv.values, i + 1)})
        conf = pd.DataFrame(diag)

        timing.to_csv(path_t, index=False, **CSV_KW)
        conf.to_csv(path_c, index=False, **CSV_KW)
        print(timing.to_string(index=False))
        print("\n" + conf.to_string(index=False))
        print("\n  Note: OLS with Newey-West (HAC, 24 lags) standard errors; interpretive check, SARIMAX models unchanged.")
        print(f"  Saved: {path_t}\n  Saved: {path_c}")
    except Exception as e:
        print(f"  !! Error in timing/confounding checks: {e}")
        raise   # errors stop the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_timing_confounding(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
_j1_csv = os.path.join(OUTPUT_DIR_P11R, "Phase_11_2_Timing_Robustness.csv")
_j1c_csv = os.path.join(OUTPUT_DIR_P11R, "Phase_11_2_Confounding_Diagnostics.csv")
_j1, _j1c = pd.read_csv(_j1_csv), pd.read_csv(_j1c_csv)
_rt = _j1c.loc[_j1c['Diagnostic'].str.startswith('Correlation RT'), 'Value']
_sc = _j1c.loc[(_j1c['Diagnostic'] == 'Correlation solar share vs consumption') & (_j1c['Item'] == 'all hours'), 'Value']
_vif = _j1c[_j1c['Diagnostic'] == 'VIF']
_k45_files = [_j1_csv, _j1c_csv]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 11.2: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 11.2 - timing, decision-time information and confounding (Supplementary Table J1)",
       lines=[(f"Correlation between the real-time (ex-ante) and the settlement (ex-post) solar share, all hours: r = {_rt.iloc[0]:.4f}."
               if len(_rt) else "Real-time solar share not in the data: its correlation with the settlement share was not computed."),
              f"Correlation between solar share and consumption, all hours: r = {_sc.iloc[0]:.4f}; largest VIF = {_vif['Value'].max():.2f} "
              f"({_vif.loc[_vif['Value'].idxmax(), 'Item']})."],
       table=_j1[['Specification', 'Regressor', 'Beta', 't_stat', 'N']], files=_k45_files)
# Rows 6-19 of the reference sheet J1_J2 are the 14 rows of the timing file, in the same order.
compare_with_reference("Table J1 vs. pre-test (sheet J1_J2)",
                       [(f"{r['Specification']} | {r['Regressor']}: {c}", r[c], f"J1.r{6 + i:02d}.{c}")
                        for i, r in _j1.iterrows() for c in ['Beta', 't_stat', 'p_value'] if len(_j1) == 14],
                       check="K17 Phase 11.2: comparison with the independent pre-test")   # the row numbers hold for the complete table only (blocks A, A', B, C)

### Phase 11.3: Sub-Period Parameter Stability (Revision_Review_Referee_2.2)

Re-estimates the solar coefficients separately for the earlier crisis/high-price period (2022–2023) and the later
high-solar period (2024–2025). Interpretive OLS with Newey–West (HAC, 24 lags) standard errors.

Produces: `Phase_11_3_SubPeriod.csv` in `results/Phase_11_Robustness/`.

In [ ]:
# =============================================================================
# PHASE 11.3: SUB-PERIOD PARAMETER STABILITY (OLS-HAC) - CACHED
# =============================================================================
# Revision_Review_Referee_2.2 – Sub-period (regime) stability
# Feeds Supplementary Appendix J.2.
# =============================================================================
import pandas as pd, os

def run_subperiod_stability(df: pd.DataFrame):
    print("\n>>> PHASE 11.3: SUB-PERIOD PARAMETER STABILITY (OLS-HAC)")
    path = os.path.join(OUTPUT_DIR_P11R, "Phase_11_3_SubPeriod.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    years = df.index.year   # years from the time index, not from the 'Date' column
    periods = [('Full sample', df),
               ('2022-2023', df[years <= 2023]),
               ('2024-2025', df[years >= 2024])]
    rows = []
    try:
        for dep, label in [('Mcp', 'MCP (merit-order)'), ('a-FRR_Price', 'a-FRR (balancing)')]:
            for pname, pdf in periods:
                m = ols_hac(pdf, dep, ['Solar_Generation_Share'] + REV_CONTROLS)
                rows.append({'Dependent': label, 'Period': pname,
                             'Beta': m.params['Solar_Generation_Share'],
                             't_stat': m.tvalues['Solar_Generation_Share'], 'N': int(m.nobs)})
        out = pd.DataFrame(rows)
        out.to_csv(path, index=False, **CSV_KW)   # LF line endings on every platform
        print(out.to_string(index=False))
        print(f"\n  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in sub-period checks: {e}")
        raise   # errors stop the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_subperiod_stability(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

# run report and comparison with the independent pre-test
_j2_csv = os.path.join(OUTPUT_DIR_P11R, "Phase_11_3_SubPeriod.csv")
_j2 = pd.read_csv(_j2_csv)
_k45_files = [_j2_csv]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Phase 11.3: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 11.3 - sub-period parameter stability (Supplementary Table J2)",
       lines=[f"{dep}, solar coefficient per unit share: " + "; ".join(f"{r['Period']} {r['Beta']:,.0f} (t = {r['t_stat']:.1f})" for _, r in g.iterrows())
              for dep, g in _j2.groupby('Dependent', sort=False)],
       table=_j2, files=_k45_files)
# Rows 23-28 of the reference sheet J1_J2 are the six rows of the output file, in the same order.
compare_with_reference("Table J2 vs. pre-test (sheet J1_J2)",
                       [(f"{r['Dependent']}, {r['Period']}: {c}", r[c], f"J2.r{23 + i}.{c}") for i, r in _j2.iterrows() for c in ['Beta', 't_stat']],
                       check="K17 Phase 11.3: comparison with the independent pre-test")

## Phase 10 (Appendix J): Additional Robustness Checks (Reviewer Response)

Assembles Supplementary **Appendix J** (Tables J1–J6) from the revision outputs of Phases 11.2, 11.3, 6.4, 8.2, 6.2 and 7.4 and,
for Table J5, from the dataset. Placed after Phase 11 because it reads its results; it only formats cached CSVs and counts
observations, so it is rebuilt on every run. Outputs are written next to Appendices A–I in `results/Phase_10/`.

Produces: `Appendix_J1_Timing_Robustness.csv`, `Appendix_J2_SubPeriod.csv`, `Appendix_J3_OutOfSample.csv`,
`Appendix_J4_Threshold.csv`, `Appendix_J5_Threshold_Support.csv` (+ `Appendix_J5_Threshold_Support_Notes.csv`),
`Appendix_J5_Clustered_QR.csv` (Supplementary Table J6; the R1 file name is kept).

Supplementary Table J5 (empirical support for the transition thresholds: hours above 20% and above 34% solar
share, by year) is computed from the dataset
(the calendar year is taken from the time index). The quantile-regression table – Supplementary Table J6 in both R1 and R2 – keeps its R1 file name,
`Appendix_J5_Clustered_QR.csv`.
Table J1 uses the Supplementary wording ("fully lagged") and Table J6 reports coefficients with one decimal, as printed in the
Supplementary. The last row of Table J4 (upper crossover of the surplus and deficit/balanced curves) is read from the Phase 6.2
output `results/Phase_6/Phase_6_2_Flexibility_Trap_Crossover.csv`.

A missing source file or an error stops the cell. The cell ends with a run report (`results/RUN_REPORT.md`) and the comparison
of Tables J1, J2 and J5 with the independent pre-test.


In [ ]:
# =============================================================================
# PHASE 10 (APPENDIX J): SUPPLEMENTARY APPENDIX J AGGREGATION (J1-J6)
# =============================================================================
# Revision – Supplementary Appendix J aggregation
# Sources: Phase 11.2 (J1), Phase 11.3 (J2), Phase 6.4 (J3), Phase 8.2 and Phase 6.2 (J4), dataset (J5), Phase 7.4 (J6)
# Table J5 (threshold support) is produced from the dataset (Appendix_J5_Threshold_Support.csv);
# the QR table (Supplementary Table J6) keeps its R1 file name, Appendix_J5_Clustered_QR.csv.
# The upper crossover of Table J4 is read from the Phase 6.2 output;
# a missing source or an error stops the cell; the cell ends with the run report and the comparison with
# the independent pre-test.
# =============================================================================
import pandas as pd, numpy as np, os

OUTPUT_DIR_P10 = os.path.join(RESULTS_BASE, "Phase_10")
os.makedirs(OUTPUT_DIR_P10, exist_ok=True)

J_SOURCES = {
    'J1': os.path.join(RESULTS_BASE, "Phase_11_Robustness", "Phase_11_2_Timing_Robustness.csv"),
    'J2': os.path.join(RESULTS_BASE, "Phase_11_Robustness", "Phase_11_3_SubPeriod.csv"),
    'J3': os.path.join(RESULTS_BASE, "Phase_6", "Phase_6_4_Out_of_Sample_CV.csv"),
    'J4': os.path.join(RESULTS_BASE, "Phase_8", "Phase_8_2_Hansen_Threshold.csv"),
    # crossover of the surplus and deficit/balanced curves (last row of Table J4), computed and saved by Phase 6.2
    'J4_crossover': os.path.join(RESULTS_BASE, "Phase_6", "Phase_6_2_Flexibility_Trap_Crossover.csv"),
    'J6': os.path.join(RESULTS_BASE, "Phase_7", "Phase_7_4_QR_Clustered_Bootstrap_SE.csv"),
}

def _bt(beta, t, stars='', decimals=0):
    """Formats 'beta (t)' as in the Supplementary tables (J2: whole numbers; J6: one decimal)."""
    return f"{beta:,.{decimals}f} ({t:.1f}){stars}".replace('-', '−')

def threshold_support(df: pd.DataFrame, thresholds=(0.20, 0.34), sparse=0.45):   # upper descriptive threshold 34%
    """Table J5: hours above each solar-share threshold, by year, and the notes cited in
    Appendix J.4 / Section 4.5 (maximum share, hours above 45%, share of the above-34% hours in the last year)."""
    share = df['Solar_Generation_Share'].astype(float)
    if share.max() > 1.0:                       # guard: proportion (0-1), as in the dataset
        share = share / 100.0
    years = df.index.year                       # calendar year from the time index, never from the raw Date column
    years = pd.Series(np.asarray(years), index=share.index)
    year_list = sorted(years.unique())
    rows = []
    for thr in thresholds:
        above = share > thr
        by_year = above.groupby(years).sum()
        row = {'Solar share': f"> {thr * 100:.0f}%"}
        row.update({str(y): int(by_year.get(y, 0)) for y in year_list})
        row.update({'Total': int(above.sum()), '% of sample': f"{above.mean() * 100:.1f}%"})
        rows.append(row)
    above_hi = share > max(thresholds)
    last = year_list[-1]
    notes = pd.DataFrame([
        {'Quantity': 'Sample hours', 'Value': f"{len(share):,}"},
        {'Quantity': 'Maximum observed solar share', 'Value': f"{share.max() * 100:.1f}%"},
        {'Quantity': f"Hours above {sparse * 100:.0f}% solar share", 'Value': f"{int((share > sparse).sum())} ({(share > sparse).mean() * 100:.1f}%)"},
        {'Quantity': f"Share of the above-{max(thresholds) * 100:.0f}% hours in {last}",
         'Value': f"{above_hi[years == last].sum() / max(above_hi.sum(), 1) * 100:.0f}%"},
    ])
    return pd.DataFrame(rows), notes

def generate_appendix_j(df: pd.DataFrame):
    print("\n>>> PHASE 10 (APPENDIX J): ADDITIONAL ROBUSTNESS CHECKS (J1-J6)")
    missing = [k for k, p in J_SOURCES.items() if not os.path.exists(p)]
    if missing:   # a missing source stops the cell
        raise RuntimeError(f"Missing sources for {missing}: " + ", ".join(J_SOURCES[k] for k in missing)
                           + ". Run Phases 6.2, 6.4, 7.4, 8.2, 11.2 and 11.3 first.")
    if df is None:
        raise RuntimeError("DataFrame 'df' not found (needed for Table J5). Please run Phase 1.")
    try:
        # J1 - timing / decision-time information (Phase 11.2, blocks A and A')
        t = pd.read_csv(J_SOURCES['J1'])
        order = ['MCP - realised, same hour', 'MCP - solar lagged one day (D-1)',
                 'MCP - real-time solar (D-1) + real-time consumption', 'MCP - fully ex-ante (real-time consumption, D-1)',
                 'a-FRR - realised, same hour', 'a-FRR - solar lagged two days (D-2)',
                 'a-FRR - real-time solar (D-2) + real-time consumption', 'a-FRR - fully ex-ante (real-time consumption, D-2)']
        t = t[t['Block'].str.startswith(('A:', "A':")) & t['Specification'].isin(order)]
        j1 = t.set_index('Specification').loc[order, ['Beta', 't_stat', 'p_value']].round(4).reset_index()
        j1.columns = ['Specification', 'Beta (solar)', 't-stat', 'p-value']
        # Supplementary wording ("fully lagged"); the Phase 11.2 output keeps its own labels
        j1['Specification'] = j1['Specification'].str.replace('fully ex-ante', 'fully lagged', regex=False)

        # J2 - sub-period stability (Phase 11.3)
        s = pd.read_csv(J_SOURCES['J2'])
        j2 = (s.assign(Cell=[_bt(b, tt) for b, tt in zip(s['Beta'], s['t_stat'])])
               .pivot(index='Dependent', columns='Period', values='Cell')[['Full sample', '2022-2023', '2024-2025']]
               .reset_index().rename(columns={'Dependent': 'Coefficient (t-stat)'}))

        # J3 - out-of-sample validation (Phase 6.4)
        o = pd.read_csv(J_SOURCES['J3'])
        fit = []
        for _, r in o.iterrows():
            if pd.notna(r['R2_oos']):
                fit.append(f"R² = {r['R2_oos']:.3f}")
            else:
                extra = (f"; ROC-AUC = {r['ROC_AUC']:.3f}; balanced acc = {r['Balanced_Acc']:.3f}; majority-class acc = {r['Majority_baseline_acc']:.3f}"
                         if 'ROC_AUC' in o.columns and pd.notna(r.get('ROC_AUC', np.nan)) else '')
                fit.append(f"Acc = {r['Accuracy']:.3f}; F1 = {r['F1']:.3f}" + extra)
        fit_in = [f"R² = {r:.3f}" if pd.notna(r) else f"Acc = {a:.3f}"
                  for r, a in zip(o.get('R2_insample', [np.nan] * len(o)), o.get('Accuracy_insample', [np.nan] * len(o)))]
        j3 = pd.DataFrame({'Target': o['Target'], 'MAE': o['MAE'].round(0), 'RMSE': o['RMSE'].round(0),
                           'Held-out fit': fit, 'In-sample fit (full-sample model)': fit_in})

        # J4 - threshold identification (Phase 8.2)
        h = pd.read_csv(J_SOURCES['J4']).iloc[0]
        # upper crossover: read from the Phase 6.2 output (empty there = the surplus curve is not
        # above the deficit/balanced curve at the end of the common range of the two curves)
        xo = pd.read_csv(J_SOURCES['J4_crossover'])['Crossover_Solar_Share_pct'].iloc[0]
        xo_txt = f"{xo:.1f}% solar share" if pd.notna(xo) else "no crossover within the common range of the two curves"
        p_txt = (f"p < {1 / h['N_Boot_supF']:.3f}" if h['supF_pval_bootstrap'] == 0 else f"p = {h['supF_pval_bootstrap']:.3f}")
        j4 = pd.DataFrame({'Quantity': ['Model',
                                        'Onset threshold γ (point estimate)', '90% bootstrap confidence interval for γ',
                                        'Slope change at the kink δ (TL/MW per percentage point)',
                                        'Test of no kink (H0: δ = 0), sup-F',
                                        'Upper crossover (surplus overtakes deficit)'],
                           'Estimate': ['Continuous regression kink (Hansen, 2017), least-squares grid 5–44%',
                                        f"{h['Threshold_Point_Estimate_pct']:.0f}% solar share",
                                        f"[{h['CI90_Lower_pct']:.0f}%, {h['CI90_Upper_pct']:.0f}%] (median {h['Bootstrap_Median_pct']:.0f}%; day-clustered bootstrap, {int(h['N_Boot'])} reps)",
                                        (f"{h['delta']:,.1f}".replace('-', '−') + f" (day-clustered bootstrap SE {h['delta_se']:,.1f}; t = {h['delta_t']:.1f}); slope "
                                         + f"{h['slope_pre_kink_b']:,.1f}".replace('-', '−') + " below vs " + f"{h['slope_post_kink_b_plus_delta']:,.1f}".replace('-', '−') + " above γ"),
                                        f"F = {h['supF_stat']:.1f}; day-clustered wild-bootstrap {p_txt} ({int(h['N_Boot_supF'])} reps)",
                                        xo_txt + ' (XGBoost interaction simulation, Phase 6.2; descriptive, no confidence interval)']})

        # J5 - empirical support for the thresholds (dataset)
        j5, j5_notes = threshold_support(df)
        j5_notes.to_csv(os.path.join(OUTPUT_DIR_P10, "Appendix_J5_Threshold_Support_Notes.csv"),
                        index=False, encoding='utf-8-sig', **CSV_KW)

        # J6 - dependence-robust quantile regression (Phase 7.4); keeps its R1 file name Appendix_J5_Clustered_QR.csv
        q = pd.read_csv(J_SOURCES['J6'])
        q['Cell'] = [_bt(b, tt, s if isinstance(s, str) else '', decimals=1)
                     for b, tt, s in zip(q['Coef'], q['t_stat'], q['Sig'])]
        q['Col'] = 'Q' + (q['Quantile'] * 100).round().astype(int).astype(str)
        j6 = (q.pivot(index='Hypothesis', columns='Col', values='Cell')[['Q10', 'Q50', 'Q90']]
               .reindex(['MCP', 'aFRR']).rename(index={'MCP': 'MCP β (t)', 'aFRR': 'a-FRR β (t)'})
               .reset_index().rename(columns={'Hypothesis': 'Dependent variable'}))

        for name, tab, title in [('Appendix_J1_Timing_Robustness', j1, 'J1. Timing, decision-time information'),
                                 ('Appendix_J2_SubPeriod', j2, 'J2. Sub-period parameter stability'),
                                 ('Appendix_J3_OutOfSample', j3, 'J3. Out-of-sample validation'),
                                 ('Appendix_J4_Threshold', j4, 'J4. Threshold identification'),
                                 ('Appendix_J5_Threshold_Support', j5, 'J5. Hours above the transition thresholds, by year'),
                                 ('Appendix_J5_Clustered_QR', j6, 'J6. Dependence-robust quantile regression (*** p<0.01)')]:
            path = os.path.join(OUTPUT_DIR_P10, name + ".csv")
            tab.to_csv(path, index=False, encoding='utf-8-sig', **CSV_KW)
            print(f"\n  Table {title}\n" + tab.to_string(index=False, na_rep='-'))   # '-' = not defined (J3: MAE/RMSE of the classifier)
            if name == 'Appendix_J5_Threshold_Support':
                print("  Notes (Appendix J.4 / Section 4.5):\n" + j5_notes.to_string(index=False))
            print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in Appendix J aggregation: {e}")
        raise   # errors stop the cell

# === MAIN EXECUTION ===
if __name__ == "__main__":
    generate_appendix_j(globals().get('df'))

# run report and comparison with the independent pre-test
# (read back from files: Table J5 from the output of this cell; Tables J1 and J2 hold rounded / formatted values, so
#  their unrounded values are taken from the two source files they are built from)
_j_out = [os.path.join(OUTPUT_DIR_P10, _n + ".csv") for _n in
          ["Appendix_J1_Timing_Robustness", "Appendix_J2_SubPeriod", "Appendix_J3_OutOfSample", "Appendix_J4_Threshold",
           "Appendix_J5_Threshold_Support", "Appendix_J5_Threshold_Support_Notes", "Appendix_J5_Clustered_QR"]]
_j4 = pd.read_csv(_j_out[3], encoding='utf-8-sig')['Estimate']
_j5 = pd.read_csv(_j_out[4], encoding='utf-8-sig')
_j5n = pd.read_csv(_j_out[5], encoding='utf-8-sig').set_index('Quantity')['Value']    # rows: sample hours, maximum share, hours above the sparse-data share, ...
_k45_files = _j_out + list(J_SOURCES.values())
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Appendix J: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
report("Phase 10 (Appendix J): Supplementary Tables J1-J6",
       lines=[f"Table J4: onset threshold {_j4.iloc[1]}; upper crossover {_j4.iloc[-1]}.",
              "Table J5, hours above each solar share: "
              + "; ".join(f"{_r['Solar share']} {int(_r['Total']):,} ({_r['% of sample']})" for _r in _j5.to_dict('records'))
              + f"; {_j5n.index[2]}: {_j5n.iloc[2]}; maximum observed share {_j5n.iloc[1]}."],
       table=_j5, files=_k45_files)
_n_hours = int(str(_j5n.iloc[0]).replace(',', ''))
_j_rows = []
for _r in _j5.to_dict('records'):
    _tag = 'ESIK.duzeltilmis.gt' + _r['Solar share'].strip('> %')
    _j_rows += [(f"J5 hours {_r['Solar share']}, {_c}", int(_r[_c]), f"{_tag}.{_c}") for _c in _j5.columns if _c.isdigit()]
    _j_rows += [(f"J5 hours {_r['Solar share']}, total", int(_r['Total']), _tag + '.Saat'),
                (f"J5 share of the sample {_r['Solar share']}", int(_r['Total']) / _n_hours, _tag + '.Pay')]
_j_rows.append((f"J5 notes: {_j5n.index[2]}", int(str(_j5n.iloc[2]).split()[0]),
                'ESIK.duzeltilmis.gt' + ''.join(_ch for _ch in _j5n.index[2] if _ch.isdigit()) + '.Saat'))
_s1 = pd.read_csv(J_SOURCES['J1'])
_s1 = _s1[_s1['Block'].str.startswith(('A:', "A':"))].set_index('Specification')          # the blocks of Table J1
_j_rows += [(f"J1 {_s}: {_c}", _s1.loc[_s, _c], f"J1.{_i}.{_c}") for _s, _i in [
    ('MCP - realised, same hour', 'r06'), ('MCP - solar lagged one day (D-1)', 'r07'),
    ('a-FRR - realised, same hour', 'r08'), ('a-FRR - solar lagged two days (D-2)', 'r09'),
    ('MCP - real-time solar (D-1) + real-time consumption', 'r10'), ('MCP - fully ex-ante (real-time consumption, D-1)', 'r11'),
    ('a-FRR - real-time solar (D-2) + real-time consumption', 'r12'), ('a-FRR - fully ex-ante (real-time consumption, D-2)', 'r13')]
            for _c in ('Beta', 't_stat')]
_s2 = pd.read_csv(J_SOURCES['J2']).set_index(['Dependent', 'Period'])
_j2_keys = [(_dep, _per) for _dep in ('MCP (merit-order)', 'a-FRR (balancing)') for _per in ('Full sample', '2022-2023', '2024-2025')]
_j_rows += [(f"J2 {_dep}, {_per}: {_c}", _s2.loc[(_dep, _per), _c], f"J2.r{_k}.{_c}")
            for _k, (_dep, _per) in enumerate(_j2_keys, 23) for _c in ('Beta', 't_stat')]             # reference rows r23-r28
compare_with_reference("Supplementary Tables J1, J2 (unrounded source values) and J5 (hours above the thresholds)", _j_rows,
                       check="K17 Appendix J: comparison with the independent pre-test")

## Submission Figures: export under the manuscript numbering

The cell below copies the manuscript figures into `results/Submission_Figures/` under their manuscript numbers
(`Figure_1a.png` … `Figure_9.png`; the manuscript has eight figures: `Figure_9.png`, the base-case GAM figure, is
Supplementary Figure K.1 and is exported here only under its R1 name, not as a manuscript figure) and reports, for each,
the pixel size, the dpi tag and the effective resolution in a 19 cm double-column print. **The files `Figure_1a.png` … `Figure_8.png` of this folder**
are the ones uploaded to Editorial Manager; `Figure_9.png` is not uploaded (it is embedded in the Supplementary Material as Figure K.1). A missing, ambiguous or empty source figure stops the cell
(after all problems have been listed), and the exported files are entered in the run report (`results/RUN_REPORT.md`).


In [ ]:
# =============================================================================
# SUBMISSION FIGURES: MANUSCRIPT-NUMBERED EXPORT (Figure_1a ... Figure_9)
# Figure_9 (base-case GAM) is Supplementary Figure K.1; it is exported under its R1 name
# =============================================================================
import glob, shutil
from PIL import Image

SUBMISSION_DIR = os.path.join(RESULTS_BASE, "Submission_Figures")
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# (target name, source pattern) -- manuscript number <- pipeline file
_submission_map = [
    ("Figure_1a", os.path.join("Phase_5", "Phase_5_Figure_1_*.png")),
    ("Figure_1b", os.path.join("Phase_5", "Phase_5_Figure_2_*.png")),
    ("Figure_2",  os.path.join("Phase_9", "Figure_3_*.png")),
    ("Figure_3",  os.path.join("Phase_5", "Phase_5_Figure_4_*.png")),
    ("Figure_4",  os.path.join("Phase_6", "Phase_6_Figure_5_*.png")),
    ("Figure_5",  os.path.join("Phase_5", "Phase_5_Figure_6_*.png")),
    ("Figure_6",  os.path.join("Phase_6", "Phase_6_Figure_7_*.png")),
    ("Figure_7",  os.path.join("Phase_6", "Phase_6_Figure_8_*.png")),
    ("Figure_8",  os.path.join("Phase_6", "Phase_6_Figure_9_*.png")),
    ("Figure_9",  os.path.join("Phase_8", "Phase_8_Figure_10_GAM_Flexibility_Stress.png")),   # Supplementary Figure K.1
]

print(f">>> SUBMISSION FIGURES -> {SUBMISSION_DIR}\n")
_problems = []
_k45_dpi = {}   # dpi tag of every exported figure (checkpoint K45)
for _target, _pattern in _submission_map:
    _hits = sorted(glob.glob(os.path.join(RESULTS_BASE, _pattern)))
    if len(_hits) != 1:
        _problems.append(f"{_target}: pattern '{_pattern}' matched {len(_hits)} files {_hits}")
        continue
    if os.path.getsize(_hits[0]) == 0:                    # an empty source file is reported, not copied
        _problems.append(f"{_target}: source file is empty -> {_hits[0]}")
        continue
    _dst = os.path.join(SUBMISSION_DIR, _target + ".png")
    shutil.copy2(_hits[0], _dst)
    with warnings.catch_warnings():
        # PIL warns for images above about 89.5 megapixels (a guard against decompression bombs in untrusted
        # files). These are the notebook's own figures (the flowchart and Figure 7 exceed the limit): no information here.
        warnings.simplefilter('ignore', Image.DecompressionBombWarning)
        with Image.open(_dst) as _im:
            _w, _h = _im.size
            _tag = _im.info.get("dpi", ("?", "?"))[0]
            _k45_dpi[_target] = _tag
    _eff = _w / (19.0 / 2.54)   # effective dpi in a 19 cm double-column print
    _tag = f"{_tag:.0f}" if isinstance(_tag, (int, float)) else _tag   # PNG stores pixels per metre: 39,370 = 999.998 dpi
    print(f"  {_target:<10} {_w}x{_h}px | dpi tag: {_tag} | ~{_eff:.0f} dpi at 19 cm print width  <- {os.path.basename(_hits[0])}")

if _problems:
    print("\n!! PROBLEM:")
    for _p in _problems: print("   -", _p)
    # a missing, ambiguous or empty source figure stops the cell
    raise RuntimeError(f"Submission figures: {len(_problems)} of {len(_submission_map)} source figures are missing, ambiguous or empty "
                       "(listed above). Run the phases that produce them (Phases 5, 6.1, 6.2, 8 and 9) and re-run this cell.")
else:
    print(f"\n>> {len(_submission_map)}/{len(_submission_map)} figures ready. Upload Figure_1a ... Figure_8 from "
          "'results/Submission_Figures/' to EM; Figure_9.png (base-case GAM) is Supplementary Figure K.1 and is not "
          "uploaded as a manuscript figure.")

# run report and comparison with the independent pre-test
# (the exported files with their sizes and hashes; figure copies have no reference values)
_k45_files = [os.path.join(SUBMISSION_DIR, _target + ".png") for _target, _pattern in _submission_map]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Submission figures: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
# checkpoint K45: every exported figure carries the dpi tag FIG_DPI (PNG stores pixels per metre: 39,370 = 999.998 dpi)
_k45_off = [f"{_t}: {_v}" for _t, _v in _k45_dpi.items() if not (isinstance(_v, (int, float)) and abs(_v - FIG_DPI) < 0.5)]
checkpoint("K45 Submission figures: dpi tag", len(_k45_dpi) == len(_submission_map) and not _k45_off,
           f"{len(_k45_dpi) - len(_k45_off)} of {len(_submission_map)} exported figures tagged {FIG_DPI} dpi"
           + "".join(f"; other tag: {_x}" for _x in _k45_off), level='warn')
report("Submission figures: manuscript-numbered export (Figure_1a ... Figure_9)",
       lines=[f"{len(_submission_map)} figures copied to '{SUBMISSION_DIR}' under their manuscript numbers (Figure_9.png, the "
              "base-case GAM, is Supplementary Figure K.1 since R2 and keeps its R1 name; it is not a manuscript figure).",
              "Copies of figures produced by Phases 5, 6.1, 6.2, 8 and 9: nothing to compare with the independent pre-test."],
       files=_k45_files)

## Supplementary Figures: export of the Supplementary Material figures

Copies the six figures embedded in the Supplementary Material (F.1, F.2, H.1, I.1, I.2, K.1) to
`results/Supplementary_Figures/` under their own labels and prints a resolution report.
**Note:** these are NOT uploaded to EM as separate 'Figure' items – they are embedded in the Supplementary docx;
this folder keeps the docx images up to date and the repository complete. A missing source figure stops
the cell (after all missing files have been listed), and the exported files are entered in the run report
(`results/RUN_REPORT.md`). The cell then closes the run: it writes the summary of all checkpoints and recorded warnings
(`results/RUN_CHECKPOINTS.csv` and the last section of `RUN_REPORT.md`) and raises an error after writing it if a checkpoint
FAILED.


In [ ]:
# =============================================================================
# SUPPLEMENTARY FIGURES: LABELLED EXPORT (F1, F2, H1, I1, I2, K1)
# =============================================================================
import shutil
from PIL import Image

SUPP_FIG_DIR = os.path.join(RESULTS_BASE, "Supplementary_Figures")
os.makedirs(SUPP_FIG_DIR, exist_ok=True)

# (target name, source file) -- supplementary label <- pipeline output (full name, no wildcard)
_supp_map = [
    ("Supplementary_Figure_F1", os.path.join("Phase_10", "Appendix_F1_MCP_Correlation.png")),
    ("Supplementary_Figure_F2", os.path.join("Phase_10", "Appendix_F2_aFRR_Correlation.png")),
    ("Supplementary_Figure_H1", os.path.join("Phase_10", "Appendix_H_Down_Regulation_Volume.png")),
    ("Supplementary_Figure_I1", os.path.join("Phase_10", "Appendix_I_Figure_I1_Flexibility_Trap_Botas.png")),
    ("Supplementary_Figure_I2", os.path.join("Phase_10", "Appendix_I_Figure_I2_GAM_Flexibility_Stress_Botas.png")),
    # base-case GAM, also exported as Figure_9 (its R1 name); distinct from I.2 (BOTAS-tariff variant)
    ("Supplementary_Figure_K1", os.path.join("Phase_8", "Phase_8_Figure_10_GAM_Flexibility_Stress.png")),
]

print(f">>> SUPPLEMENTARY FIGURES -> {SUPP_FIG_DIR}\n")
_missing = []
_k45_dpi = {}   # dpi tag of every exported figure (checkpoint K45)
for _target, _rel in _supp_map:
    _srcp = os.path.join(RESULTS_BASE, _rel)
    if not os.path.exists(_srcp):
        _missing.append(f"{_target}: source missing -> {_srcp}")
        continue
    _dst = os.path.join(SUPP_FIG_DIR, _target + ".png")
    shutil.copy2(_srcp, _dst)
    with Image.open(_dst) as _im:
        _w, _h = _im.size
        _tag = _im.info.get("dpi", ("?", "?"))[0]
        _k45_dpi[_target] = _tag
    _tag = f"{_tag:.0f}" if isinstance(_tag, (int, float)) else _tag   # PNG stores pixels per metre
    print(f"  {_target:<26} {_w}x{_h}px | dpi tag: {_tag}  <- {os.path.basename(_srcp)}")

if _missing:
    print("\n!! PROBLEM:")
    for _m in _missing: print("   -", _m)
    # a missing source figure stops the cell
    raise RuntimeError(f"Supplementary figures: {len(_missing)} of {len(_supp_map)} source figures are missing (listed above). Run the "
                       "cells that produce them (Phase 8.1, Phase 10 Appendices F and H, Appendix I) and re-run this cell.")
else:
    print(f"\n>> {len(_supp_map)}/{len(_supp_map)} supplementary figures copied.")
    print(">> They are embedded in the Supplementary docx; they are not uploaded to EM as separate Figure items.")

# run report and comparison with the independent pre-test
# (the exported files with their sizes and hashes; figure copies have no reference values)
_k45_files = [os.path.join(SUPP_FIG_DIR, _target + ".png") for _target, _rel in _supp_map]
_k45_bad = [p for p in _k45_files if not (os.path.isfile(p) and os.path.getsize(p) > 0)]   # checkpoint K45
checkpoint("K45 Supplementary figures: output files", not _k45_bad,
           f"{len(_k45_files) - len(_k45_bad)} of {len(_k45_files)} listed files exist and are not empty"
           + "".join(f"; MISSING or empty: {os.path.relpath(p, RESULTS_BASE)}" for p in _k45_bad))
# checkpoint K45: every exported figure carries the dpi tag FIG_DPI (PNG stores pixels per metre: 39,370 = 999.998 dpi)
_k45_off = [f"{_t}: {_v}" for _t, _v in _k45_dpi.items() if not (isinstance(_v, (int, float)) and abs(_v - FIG_DPI) < 0.5)]
checkpoint("K45 Supplementary figures: dpi tag", len(_k45_dpi) == len(_supp_map) and not _k45_off,
           f"{len(_k45_dpi) - len(_k45_off)} of {len(_supp_map)} exported figures tagged {FIG_DPI} dpi"
           + "".join(f"; other tag: {_x}" for _x in _k45_off), level='warn')
report("Supplementary figures: labelled export (F1, F2, H1, I1, I2, K1)",
       lines=[f"{len(_supp_map)} figures copied to '{SUPP_FIG_DIR}' under their Supplementary labels.",
              "Copies of figures produced by Phase 8.1, Phase 10 (Appendices F and H) and Appendix I: nothing to compare with the independent pre-test."],
       files=[os.path.join(SUPP_FIG_DIR, _target + ".png") for _target, _rel in _supp_map])

# end of the run: total of the comparisons with the independent pre-test (K90) and of the warnings recorded in this run
# (K91), then the summary of all checkpoints (results/RUN_CHECKPOINTS.csv and the last section of RUN_REPORT.md);
# checkpoint_summary() raises after writing them if a checkpoint FAILED
_cmp_full = [_c for _c in RUN_COMPARISONS if _c['full_sample']]
_cmp_n = {_s: sum(_c[_s] for _c in _cmp_full) for _s in ('match', 'DIFFERS', 'reported', 'no reference')}
_cmp_skip = sorted({_c['sample'] for _c in RUN_COMPARISONS if not _c['full_sample']})
_name = "K90 run: comparison with the independent pre-test, total of all comparison tables"
if not REFERENCE:
    checkpoint(_name, None, f"reference file not found ({REFERENCE_FILE})", level='warn')
elif not _cmp_full:
    checkpoint(_name, None, f"no comparison on the full sample ({'; '.join(_cmp_skip) or 'no comparison made'})", level='warn')
else:
    checkpoint(_name, _cmp_n['DIFFERS'] == 0,
               f"{len(_cmp_full)} tables: {_cmp_n['match']:,} match, {_cmp_n['DIFFERS']:,} DIFFERS, {_cmp_n['reported']:,} reported without "
               f"tolerance, {_cmp_n['no reference']:,} without reference"
               + ("; DIFFERS in: " + "; ".join(f"'{_c['title']}' ({_c['DIFFERS']})" for _c in _cmp_full if _c['DIFFERS']) if _cmp_n['DIFFERS'] else "")
               + (f"; {len(RUN_COMPARISONS) - len(_cmp_full)} table(s) not counted: {'; '.join(_cmp_skip)}" if _cmp_skip else ""), level='warn')
_w_total = sum(_w['count'] for _w in RUN_WARNINGS)
checkpoint("K91 run: warnings recorded", _w_total == 0,
           f"{_w_total:,} warning(s) in {len(RUN_WARNINGS)} distinct entries"
           + (": " + "; ".join(f"{_w['category']} x{_w['count']} ({_w['step'] or 'this step'}; {_w['where']}; {_w['location']})"
                               for _w in RUN_WARNINGS[:12]) + (" ..." if len(RUN_WARNINGS) > 12 else "") if _w_total else ""), level='warn')
checkpoint_summary()


### Phase 6.2 (check): Flexibility-Trap Crossover without the Day-Ahead Price

Repeats the Phase 6.2 interaction simulation with the day-ahead price (`Mcp`) removed from the features of
the two XGBoost models. It supports the statement in Section 4.5 of the manuscript that the crossover near 34% is not
reproduced when the MCP is removed from the simulation's features. The cell first repeats the base case (with the MCP)
and compares it with `Phase_6_2_Flexibility_Trap_Crossover.csv` and `Phase_6_2_Flexibility_Trap_Curves.csv`; those files,
the figure and the run report are not changed. Definitions are those of Phase 6.2: grid of 100 solar shares from 0 to
60%, other features at their medians, common range 0–45%, crossings by linear interpolation between grid points; the
crossover is the last upward crossing after which the surplus curve stays above the deficit/balanced curve up to the end
of the common range (empty if there is none).

This cell and the Phase 8.2 check below come after the cell that closes the run with the summary of the checkpoints: they
make no entry in `results/RUN_REPORT.md` or `results/RUN_CHECKPOINTS.csv`, and a run that stops at a FAILED checkpoint stops
before them. Each uses only the setup cell and Phase 1, so either can also be run on its own after those two cells.

Produces: `Phase_6_2_Crossover_Without_MCP.csv` in `results/Phase_6/`.

In [ ]:
# =============================================================================
# PHASE 6.2 (CHECK): FLEXIBILITY-TRAP CROSSOVER WITHOUT THE DAY-AHEAD PRICE
# =============================================================================
# Section 4.5 of the manuscript: the crossover near 34% is not reproduced when the MCP is removed from the
# features of the simulation. The cell uses only the setup cell and Phase 1 (RESULTS_BASE, CSV_KW, df). It does not change
# the outputs of Phase 6.2 and makes no entry in the run report.

import os
import numpy as np
import pandas as pd
import xgboost as xgb


def run_phase_6_2_check_without_mcp(df):
    """Phase 6.2 simulation with and without the MCP among the features -> Phase_6_2_Crossover_Without_MCP.csv."""
    out_dir = os.path.join(RESULTS_BASE, "Phase_6")
    os.makedirs(out_dir, exist_ok=True)
    out_file = os.path.join(out_dir, 'Phase_6_2_Crossover_Without_MCP.csv')
    crossover_file = os.path.join(out_dir, 'Phase_6_2_Flexibility_Trap_Crossover.csv')     # written by Phase 6.2
    curves_file = os.path.join(out_dir, 'Phase_6_2_Flexibility_Trap_Curves.csv')           # written by Phase 6.2
    features = ['Solar_Generation_Share', 'Mcp', 'Consumption', 'Wind_Generation', 'Hour_Sin', 'Hour_Cos']   # as in Phase 6.2
    features_without_mcp = [c for c in features if c != 'Mcp']
    print(f">> Phase 6.2 (check): crossover without the MCP (Directory: '{out_dir}')...")
    print("="*80)

    def crossings(x, y_surplus, y_deficit):
        """Same definition as last_upward_crossover() of Phase 6.2: (crossover, upward crossings, downward crossings)."""
        diff = y_surplus - y_deficit
        cross = lambda i: float(x[i] + (x[i + 1] - x[i]) * (0.0 - diff[i]) / (diff[i + 1] - diff[i]))
        up = [cross(i) for i in range(len(diff) - 1) if diff[i] <= 0 < diff[i + 1]]
        down = [cross(i) for i in range(len(diff) - 1) if diff[i] > 0 >= diff[i + 1]]
        crossover = up[-1] if (up and diff[-1] > 0) else np.nan
        return crossover, up, down

    def simulate(ft):
        """The Phase 6.2 simulation for the feature list ft (same statements, same order)."""
        dm = df.copy()
        if dm['Solar_Generation_Share'].max() <= 1:
            dm['Solar_Generation_Share'] *= 100
        sur = dm[dm['System_Direction_Down'] == 1]
        dfc = dm[dm['System_Direction_Down'] == 0]
        ms = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(sur[ft], sur['a-FRR_Price'])
        md = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(dfc[ft], dfc['a-FRR_Price'])
        sr = np.linspace(0, 60, 100)
        avg = dm[ft].median()
        sim = pd.DataFrame([avg]*100, columns=ft)
        sim['Solar_Generation_Share'] = sr
        mk = sr <= 45
        yd = md.predict(sim[mk][ft])
        ys = ms.predict(sim[ft])
        ysc = ys[:len(yd)]
        sc = sr[mk]
        crossover, up, down = crossings(sc, ysc, yd)
        return {'sr': sr, 'sc': sc, 'ys': ys, 'ysc': ysc, 'yd': yd, 'crossover': crossover, 'up': up, 'down': down}

    def row(label, ft, r, note):
        return {'Specification': label,
                'Features': "; ".join(ft),
                'Crossover_Solar_Share_pct': r['crossover'],
                'Upward_crossings_pct': "; ".join(f"{v:.2f}" for v in r['up']),          # surplus curve rises above the deficit/balanced curve
                'Downward_crossings_pct': "; ".join(f"{v:.2f}" for v in r['down']),      # surplus curve falls below it
                'Flexibility_Trap_share_of_range_pct': float((r['ysc'] > r['yd']).sum() / len(r['yd']) * 100),
                'Deficit_premium_share_of_range_pct': float((r['yd'] >= r['ysc']).sum() / len(r['yd']) * 100),
                'Last_grid_point_of_common_range_pct': float(r['sc'][-1]),
                'Surplus_curve_at_last_grid_point': float(r['ysc'][-1]),
                'Deficit_Balanced_curve_at_last_grid_point': float(r['yd'][-1]),
                'Common_range_pct': '0-45',
                'Grid_step_pct': float(r['sr'][1] - r['sr'][0]),
                'Curves': 'XGBoost (n_estimators=200, random_state=42), other features at their medians',
                'xgboost_version': xgb.__version__,
                'Note': note}

    # 1. base case: the Phase 6.2 simulation, compared with the files written by Phase 6.2
    base = simulate(features)
    print(">> 1. With the MCP (base case of Phase 6.2)")
    print("   crossover " + (f"{base['crossover']:.2f}%" if np.isfinite(base['crossover']) else "none")
          + f" | upward {[round(v, 2) for v in base['up']]} | downward {[round(v, 2) for v in base['down']]}")
    if os.path.exists(crossover_file) and os.path.exists(curves_file):
        _x = pd.read_csv(crossover_file, float_precision='round_trip',                   # the two lists are read as text
                         dtype={'Upward_crossings_pct': str, 'Downward_crossings_pct': str}).iloc[0]
        _c = pd.read_csv(curves_file, float_precision='round_trip')
        x_file = float(_x['Crossover_Solar_Share_pct'])
        if np.isfinite(base['crossover']) and np.isfinite(x_file):
            d_cross = abs(base['crossover'] - x_file)
        elif np.isnan(base['crossover']) and np.isnan(x_file):
            d_cross = 0.0                                                                # no sustained crossover in either
        else:
            d_cross = np.nan
        d_sur = float(np.max(np.abs(base['ys'] - _c['Predicted_aFRR_Price_Surplus'].to_numpy())))
        d_def = float(np.max(np.abs(base['yd'] - _c['Predicted_aFRR_Price_Deficit_Balanced'].to_numpy()[:len(base['yd'])])))
        as_list = lambda v: "" if pd.isna(v) else str(v)          # an empty list is read as NaN
        same_lists = ("; ".join(f"{v:.2f}" for v in base['up']) == as_list(_x['Upward_crossings_pct'])
                      and "; ".join(f"{v:.2f}" for v in base['down']) == as_list(_x['Downward_crossings_pct']))
        reproduced = bool(np.isfinite(d_cross) and d_cross <= 1e-3 and same_lists)
        note_base = (f"base case: crossover of Phase 6.2 {x_file:.6f}; difference {d_cross:.1e} percentage points; crossings "
                     f"{'equal to' if same_lists else 'DIFFERENT from'} the lists of Phase 6.2; largest curve difference "
                     f"{max(d_sur, d_def):.1e} TL/MW -> {'reproduced' if reproduced else 'NOT reproduced'}")
        if not reproduced:
            print("   !! The base case does not reproduce the files of Phase 6.2 (other library version or platform?).")
    else:
        note_base = "base case: files of Phase 6.2 not found; not compared"
    print(f"   {note_base}")

    # 2. the same simulation without the MCP
    alt = simulate(features_without_mcp)
    print(">> 2. Without the MCP")
    print("   crossover " + (f"{alt['crossover']:.2f}%" if np.isfinite(alt['crossover'])
                             else "none (the surplus curve is not above at the end of the common range)")
          + f" | upward {[round(v, 2) for v in alt['up']]} | downward {[round(v, 2) for v in alt['down']]}")
    for label, r in (("with the MCP", base), ("without the MCP", alt)):
        print(f"   at {r['sc'][-1]:.2f}% ({label}): surplus {r['ysc'][-1]:,.1f} | deficit/balanced {r['yd'][-1]:,.1f} TL/MW")

    # 3. result table
    pd.DataFrame([row("Phase 6.2 (with the MCP)", features, base, note_base),
                  row("Without the MCP", features_without_mcp, alt,
                      "same simulation with the MCP removed from the features")]).to_csv(out_file, index=False, **CSV_KW)
    print(f"\n>> Phase 6.2 (check) complete. Saved: '{out_file}'")


# === MAIN EXECUTION ===
if __name__ == "__main__":
    if 'df' in globals() and df is not None:
        run_phase_6_2_check_without_mcp(df)
    else:
        print("ERROR: DataFrame 'df' not found. Please run Phase 1 first.")

### Phase 8.2 (check): Kink Location with Year Effects and in Recent Samples

Repeats the point estimate of the Phase 8.2 regression-kink model (a) with year dummies for 2023, 2024 and
2025 added to the controls, (b) on the hours of 2024–2025 only and (c) on the hours of 2025 only. It supports the
statement in Section 4.6 of the manuscript and in Supplementary Appendix J.4 that the kink's location changes little when
year effects are added but rises when the sample is limited to the most recent years. The cell first repeats the point
estimate of Phase 8.2 and compares it with `Phase_8_2_Hansen_Threshold.csv`; that file and the run report are not changed.
The model, the controls and the grid (5–44% solar share, step 1 pp) are those of Phase 8.2. These are point estimates
only: no bootstrap interval and no sup-F test is computed for the three variants. The cell uses only the setup cell and
Phase 1.

Produces: `Phase_8_2_Kink_Location_Checks.csv` in `results/Phase_8/`.

In [ ]:
# =============================================================================
# PHASE 8.2 (CHECK): KINK LOCATION WITH YEAR EFFECTS AND IN RECENT SAMPLES
# =============================================================================
# Section 4.6 of the manuscript and Supplementary Appendix J.4: the kink's location changes little when year
# effects are added but rises when the sample is limited to the most recent years. The cell uses only the setup cell and
# Phase 1 (RESULTS_BASE, CSV_KW, df). It does not change the output of Phase 8.2 and makes no entry in the run report.

import os
import numpy as np
import pandas as pd


def run_phase_8_2_check_kink_location(df, dep='a-FRR_Price'):
    """Point estimate of the Phase 8.2 kink model with year effects and in recent samples
    -> Phase_8_2_Kink_Location_Checks.csv."""
    out_dir = os.path.join(RESULTS_BASE, "Phase_8")
    os.makedirs(out_dir, exist_ok=True)
    out_file = os.path.join(out_dir, 'Phase_8_2_Kink_Location_Checks.csv')
    threshold_file = os.path.join(out_dir, 'Phase_8_2_Hansen_Threshold.csv')                # written by Phase 8.2
    controls = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation',                 # as in Phase 8.2
                'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
                'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
                'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    grid = np.arange(5, 45, 1.0)      # candidate thresholds: 5%-44% solar share, as in Phase 8.2
    print(f">> Phase 8.2 (check): kink location with year effects and in recent samples (Directory: '{out_dir}')...")
    print("="*80)

    def design(s, Xc, g):
        """Design matrix of the continuous-kink model: [1, s, (s-g)_+, Xc] (as kink_design() of Phase 8.2)."""
        return np.column_stack([np.ones(len(s)), s, np.maximum(s - g, 0.0), Xc])

    def fit(y, s, Xc, g):
        """OLS of the kink model at threshold g -> (beta, SSR); beta = [a, b, delta, c...] (as kink_fit() of Phase 8.2)."""
        X = design(s, Xc, g)
        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        return beta, float(((y - X @ beta) ** 2).sum())

    def point_estimate(data, ctrl):
        """Least-squares grid search on the rows of data with the control columns ctrl (as kink_profile() of Phase 8.2)."""
        d = data.assign(solar_pct=data['Solar_Generation_Share'] * 100.0).dropna(subset=[dep, 'solar_pct'] + ctrl)
        y, s = d[dep].astype(float).values, d['solar_pct'].values
        Xc = d[ctrl].astype(float).values
        ssr = np.array([fit(y, s, Xc, g)[1] for g in grid])
        i = int(ssr.argmin())
        beta, _ = fit(y, s, Xc, grid[i])
        ghat = float(grid[i])
        return {'gamma': ghat, 'slope_below': float(beta[1]), 'delta': float(beta[2]), 'slope_above': float(beta[1] + beta[2]),
                'n': len(y), 'at_grid_edge': bool(ghat in (grid[0], grid[-1]))}

    def row(label, sample, controls_text, r, note):
        return {'Specification': label, 'Sample': sample, 'Controls': controls_text, 'N': r['n'],
                'Threshold_Point_Estimate_pct': r['gamma'],
                'slope_below_threshold': r['slope_below'], 'slope_above_threshold': r['slope_above'], 'delta': r['delta'],
                'Threshold_at_edge_of_grid': r['at_grid_edge'], 'Grid': '5-44% (step 1)',
                'Model': 'continuous regression kink, least-squares grid search (point estimate; no bootstrap)', 'Note': note}

    # 1. the kink model of Phase 8.2, compared with the file written by Phase 8.2
    base = point_estimate(df, controls)
    if os.path.exists(threshold_file):
        _h = pd.read_csv(threshold_file, float_precision='round_trip').iloc[0]
        rel = lambda a, b: abs(a - b) / abs(b)
        d_slopes = max(rel(base['slope_below'], float(_h['slope_pre_kink_b'])), rel(base['delta'], float(_h['delta'])),
                       rel(base['slope_above'], float(_h['slope_post_kink_b_plus_delta'])))
        same_gamma = base['gamma'] == float(_h['Threshold_Point_Estimate_pct'])
        same_n = base['n'] == int(_h['N'])
        reproduced = bool(same_gamma and same_n and d_slopes <= 1e-9)
        note_base = (f"base case: threshold of Phase 8.2 {float(_h['Threshold_Point_Estimate_pct']):.0f}%, N {int(_h['N']):,}; threshold "
                     f"{'equal' if same_gamma else 'DIFFERENT'}, N {'equal' if same_n else 'DIFFERENT'}, largest relative difference "
                     f"of the slopes {d_slopes:.1e} -> {'reproduced' if reproduced else 'NOT reproduced'}")
        if not reproduced:
            print("   !! The base case does not reproduce the file of Phase 8.2.")
    else:
        note_base = "base case: file of Phase 8.2 not found; not compared"

    # 2. the same point estimate with year effects and in recent samples
    year = df.index.year
    dy = df.assign(Year_2023=(year == 2023).astype(float), Year_2024=(year == 2024).astype(float),
                   Year_2025=(year == 2025).astype(float))
    variants = [
        ("Phase 8.2 (base case)", "2022-2025", "controls of Phase 8.2", base, note_base),
        ("With year effects", "2022-2025", "controls of Phase 8.2 and dummies for 2023, 2024 and 2025",
         point_estimate(dy, controls + ['Year_2023', 'Year_2024', 'Year_2025']), "year dummies added to the controls"),
        ("Recent sample", "2024-2025", "controls of Phase 8.2",
         point_estimate(df[year >= 2024], controls), "hours of 2024-2025 only"),
        ("Most recent year", "2025", "controls of Phase 8.2",
         point_estimate(df[year == 2025], controls), "hours of 2025 only"),
    ]
    for label, sample, _, r, _ in variants:
        print(f">> {label} ({sample}, N = {r['n']:,}): threshold {r['gamma']:.0f}% | slope below {r['slope_below']:.1f} | "
              f"slope above {r['slope_above']:.1f} TL/MW per percentage point"
              + (" | threshold at the edge of the grid" if r['at_grid_edge'] else ""))
    print(f"   {note_base}")

    # 3. result table
    pd.DataFrame([row(*v) for v in variants]).to_csv(out_file, index=False, **CSV_KW)
    print(f"\n>> Phase 8.2 (check) complete. Saved: '{out_file}'")


# === MAIN EXECUTION ===
if __name__ == "__main__":
    if 'df' in globals() and df is not None:
        run_phase_8_2_check_kink_location(df)
    else:
        print("ERROR: DataFrame 'df' not found. Please run Phase 1 first.")